# From Pixels to Provenance — full pipeline notebook

Automated galaxy-morphology classification in HST archival imaging (NUS-ISS Practice Module, Sangeeta Bahrani, A0339780R).

This notebook contains **the complete code of the 30 October 2026 release**: 28 stages (Phase 0 to Phase 9), `config.yaml`, the runner and the tools. It runs the same way on Google Colab and in a local Jupyter session, on **any** set of HST `_drc` / `_drz` files: one pointing or several pointings at different RA/Dec.

**What this release contains** (details in `DEVELOPMENT_LOG.md`)

| Area | Change |
|---|---|
| Data | 8 pointings (GOODS-S, GOODS-S-2..4, GOODS-N-1..4), F435W / F606W / F775W, files chosen jointly for three-filter coverage (`tools/get_fits.py`) |
| Stage 0 | split plan set per pointing (`partition.pointing_splits`): whole pointings or stripes sized by their share of GZH galaxies; real data footprints; GZH count table per pointing and split |
| Stage 1a | input check and pointing table (`name`, `splits`); no built-in download (use `tools/get_fits.py`) |
| Stage 1b | QC autoencoder v2: sky-cell split into fit / validation / calibration tiles, early stopping, bootstrap threshold; log shows `no gain k/patience` |
| Stage 1c, 1d | shorter, cleaned code; PSF borrowing between pointings with too few stars |
| Stage 2a | CAE (field-noise input + presence masks), PCA by Horn's parallel analysis |
| Stage 2b | full-covariance GMM, k by minimum BIC with elbow / ICL / held-out / seed / bootstrap checks; HDBSCAN density outliers; plain-language evaluation lines |
| Stage 4a | every galaxy cutout -> R, G, B (F775W, F606W, F435W), asinh, resized to 224 px -> frozen ImageNet **ResNet-18** (512 features, written with `residual_block`) and **EfficientNet-B0** (1,280 features) |
| Stage 4b | **ENSEMBLE (feature fusion)**: 512 + 1,280 CNN features + 14 physical features = 1,806 per galaxy, block weights w1 / w2 / w3 chosen by 5-fold CV -> **XGBoost** (5 classes) + merger XGBoost; summary, Loss / Accuracy history |
| Stage 4c / 4d | calibration of the fusion model; ablation (each feature source alone, 128 vs 224 px, previous three-model ensemble) |
| Stages 6a-7c | **sealed** while developing (`evaluation.test_sealed: true`): the test and external sky is evaluated once, at the end |

**How to use it**

1. **Section 1 – Setup**: choose the project folder (on Colab: Google Drive, so outputs survive the session) and install the packages.
2. **Section 2 – Write the code**: run every cell once. Each `%%writefile` cell writes one file of the project (`src/p2p/...`, `tools/...`, `config.yaml`, `run_pipeline.py`). To change a setting, edit the `config.yaml` cell and run it again.
3. **Section 3 – Data**: put your FITS files in `data/raw/` and the GZH / Gaia / CANDELS tables in `data/catalogs/` (or download them with the tool cells).
4. **Section 4 – Run**: either run everything in one cell, or stage by stage. After each stage, `show_stage("…")` prints what the stage did, its key numbers and its figures.
5. **Section 5 – Results**: inspect `catalog.csv`.

A GPU (Colab: *Runtime → Change runtime type → GPU*; Mac: Apple MPS) speeds up Stages 1b, 2a and 4a; the rest runs on CPU. Stage 4a downloads the ImageNet weights of ResNet-18 and EfficientNet-B0 once (~66 MB, internet needed the first time).

## 1. Setup

In [ ]:
import os, sys, pathlib

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/pixels2provenance"     # change if you like
else:
    PROJECT_DIR = os.path.abspath("pixels2provenance_nb")        # next to this notebook

for d in ["src/p2p/stages", "tools", "data/raw", "data/catalogs", "outputs"]:
    os.makedirs(os.path.join(PROJECT_DIR, d), exist_ok=True)
os.chdir(PROJECT_DIR)
print("Project folder:", os.getcwd())

The `%%writefile` cells below write relative paths, so the working directory must be the project folder. Re-run the cell above whenever you restart the kernel.

In [ ]:
%%writefile requirements.txt
# Python 3.10 - 3.13
numpy>=1.24
scipy>=1.10
pandas>=2.0
matplotlib>=3.7
pyyaml>=6.0
joblib>=1.3
astropy>=6.0
photutils>=2.0          # 2.x and 3.x both supported (src/p2p/compat.py)
statmorph>=0.5          # if the build fails: pip install --no-build-isolation statmorph
scikit-learn>=1.3
hdbscan>=0.8.33
xgboost>=2.0
torch>=2.1              # CPU build is enough; CUDA / Apple MPS are used automatically if present
torchvision>=0.16
astroquery>=0.4.7       # only for downloading from MAST / querying Gaia online
# --- optional: the field runner UI ---
streamlit>=1.40         # app.py (streamlit run app.py)
ipywidgets>=8.0         # the field-runner panel in the notebook (Colab has it already)

In [ ]:
# Install the packages (Colab already has numpy, scipy, pandas, matplotlib, scikit-learn, torch, torchvision)
%pip install -q -r requirements.txt
# statmorph sometimes needs this on fresh environments:
# %pip install -q --no-build-isolation statmorph

In [ ]:
import importlib
for m in ["numpy", "scipy", "pandas", "astropy", "photutils", "statmorph", "sklearn", "hdbscan", "xgboost", "torch", "torchvision"]:
    try:
        mod = importlib.import_module(m)
        print(f"{m:12s} {getattr(mod, '__version__', '?')}")
    except Exception as e:
        print(f"{m:12s} MISSING  ({e})")
import torch
print("CUDA available:", torch.cuda.is_available())

## 2. Write the project code

Run every cell in this section once. The files are identical to the release on your Mac (`src/`, `tools/`, `config.yaml`, `run_pipeline.py`).

### 2.1 Configuration — `config.yaml`
All parameters of every stage. Edit here and re-run this cell to change a setting.

In [ ]:
%%writefile config.yaml
# =====================================================================
# From Pixels to Provenance - pipeline configuration
# Every stage reads its settings from this file. Paths are relative to
# the project folder (the folder that holds this file).
# =====================================================================

project:
  name: "From Pixels to Provenance"
  seed: 42
  n_jobs: -1                 # CPU workers for statmorph / clustering (-1 = all cores)
  device: auto               # auto | cpu | cuda | mps   (deep-learning stages)

paths:
  raw: data/raw              # put your *_drc.fits / *_drz.fits files here
  catalogs: data/catalogs    # GZH / Gaia / CANDELS tables (CSV, CSV.GZ or FITS)
  outputs: outputs           # every stage writes into outputs/<stage folder>/

# ---------------------------------------------------------------- Stage 0
partition:
  mode: auto                 # auto = derive regions from the FITS footprints | manual
  collar_arcmin: 0.1         # buffer ring around every region (proposal: 2' for large mosaics;
                             # a single ACS pointing is only ~3.4' wide: 0.1' = 6" keeps stripes usable)
  fractions: {train: 0.70, dev: 0.15, test: 0.15}
  min_pointings_for_whole_assignment: 3   # fewer pointings -> split each pointing into stripes
  external_patterns: ["j8e611", "j8e688", "j8dna3", "j8dnb8"]
                             # file-name / TARGNAME substrings of the WHOLLY external pointings (never trained on;
                             # Stage 1a role). Must list exactly the pointings with split: external below (a pointing
                             # with an external STRIPE, like GOODS-S, is not listed here).
  pointing_splits:           # Sangeeta's plan (28 Oct, 16:44). Empty list [] = automatic 70/15/15 assignment.
    # match = file-name substrings of the pointing's three files; split = whole pointing, or stripes (in the order
    # given, along the pointing's longer axis; a collar separates neighbouring stripes) holding the given share of
    # the pointing's GZH galaxies (stripe_share below)
    - {name: GOODS-S,   match: [jb5x],           stripes: {train: 0.25, dev: 0.50, external: 0.25}}
    - {name: GOODS-S-2, match: [j8mt],           stripes: {test: 0.80, dev: 0.20}}
    - {name: GOODS-S-3, match: [j8e606, j8e621], split: train}
    - {name: GOODS-S-4, match: [j8e611, j8e688], split: external}   # Phase 7 (Kartaltepe human labels)
    - {name: GOODS-N-1, match: [j8dna1, j8dnh8], split: train}
    - {name: GOODS-N-2, match: [j8dna3, j8dnb8], split: external}   # held-out field (GZH labels, no Kartaltepe)
    - {name: GOODS-N-3, match: [j8dna0, j8dnb5], stripes: {train: 0.50, dev: 0.50}}
    - {name: GOODS-N-4, match: [j8dna4, j8dnb9], split: test}
  stripe_share: gzh_galaxies  # gzh_galaxies = stripe sizes by share of the pointing's GZH galaxies (positions only)
                             # | data_area = by share of the area with data (the 70/15/15 single-pointing rule)
  phase7_region: auto        # auto = external pointings if any, else the held-out TEST split
                             # (one field: Phase 7 is evaluated on galaxies never trained on) | external | test
  manual_regions: []         # used when mode: manual, e.g.
  #  - {name: COSMOS_A, ra: [150.05, 150.15], dec: [2.15, 2.25], split: train}

# ---------------------------------------------------------------- Stage 1a
acquisition:
  file_patterns: ["*_drc.fits", "*_drz.fits", "*_drc.fits.gz", "*_drz.fits.gz"]
  group_radius_arcsec: 60    # files whose centres are closer than this form one pointing ...
  group_min_overlap: 0.5     # ... and so do files whose footprints overlap by >= 50% of the smaller one
                             # (bands of one field taken by different HST programmes)
  bands: auto                # auto | list such as [F435W, F606W, F814W] (blue -> red)
  max_bands: 3
  detection_band: auto       # auto = reddest band in each pointing

# ---------------------------------------------------------------- Stage 1b
frame_qc:
  tile: 128
  train_stride: 128
  score_stride: 64           # 50% overlap so artefacts on a tile edge are not split
  min_tile_coverage: 0.9
  max_train_tiles: 4000
  qc_version: 2              # 2 = early stopping on validation tiles + GroupNorm CAE; 1 = original (fixed steps)
  latent_dim: 128
  channels: [16, 32, 64, 64] # conv channels of the 4 encoder blocks (v2)
  max_epochs: 120            # v2: upper limit (150 until 2026-10-30; the 28 Oct run was within ~2 % of its best by ~120);
                             # earlier stop only if ...
  min_epochs: 20
  patience: 10               # ... when the validation-tile loss has not improved ...
  min_delta: 0.005           # ... by at least 0.5 % (relative) for this many epochs; the epoch with the
                             # lowest validation loss is kept
  min_gain_vs_flat: 0.02     # never stop early while the model is < 2 % better than a flat tile (plateau)
  lr: 0.001
  lr_patience: 4             # halve the learning rate after this many epochs without improvement (< patience)
  augment: true              # random 90-degree rotations and flips while training
  train_splits: [train, dev] # Stage 0 sky the QC model may fit / stop / calibrate on (test + external: scored only)
  split_cell_arcsec: 30      # fit / validation / calibration tiles are split by whole sky cells of this size,
                             # so the same sky in another band never sits on both sides (a tile is 6.4")
  val_fraction: 0.1          # share of the tiles that decides when training stops (v2)
  cal_fraction: 0.2          # share of the tiles that sets the threshold (never trained on)
  epochs: 8                  # v1 only: minimum epochs ...
  min_steps: 1000            # ... v1 only: and at least this many optimisation steps
  batch_size: 64
  threshold_percentile: 99.0
  exclude_flagged: false     # true = drop sources in flagged tiles from training. Default false:
                             # with no labelled artefacts the CAE also flags rare bright galaxies.

# ---------------------------------------------------------------- Stage 1c
detection:
  background_box: 64
  background_filter: 3
  kernel_fwhm_px: 3.0
  nsigma: 2.0
  npixels: 10
  deblend_nlevels: 32
  deblend_contrast: 0.001
  correlated_noise_factor: 1.0   # drizzled noise is correlated; >1 inflates the rms (see README)
  # ---- band registration (bands from different HST programmes can disagree by ~1") ----
  register_bands: true           # measure each band's offset against the detection band and remove it
  register_search_arcsec: 3.0    # largest offset searched for
  max_band_offset_arcsec: 0.05   # Stage 1c FAILS if a band is still offset by more than this
  register_min_significance: 5.0 # the offset peak must hold >= 5x the pairs expected by chance
  band_offsets_arcsec: {}        # known offsets instead of measured ones, band minus detection band,
                                 # [dRA cos(dec), dDec] in arcsec, e.g. {F475W: [0.154, 0.866]}
  # ---- cosmic rays (single-exposure products cannot reject them) ----
  cr_rejection: two_band         # two_band = a source must be confirmed by a second band | none
  confirm_snr: 3.0               # S/N in another covering band that confirms a source as real ...
  confirm_max_shift_px: 1.5      # ... with that band's light centred within 1.5 px of the detection,
  confirm_max_shift_frac: 0.5    # or within 0.5 x the equivalent radius for large galaxies
  cr_pixel_nsigma: 5.0           # a cutout pixel is a cosmic-ray hit if it is > 5 sigma above ...
  cr_pixel_ratio: 4.0            # ... AND > 4 x what every other band predicts from the source colour
  cr_mask_grow_px: 0             # extra growth of the masked area (each hit is already grown by 1 px)
  mag_min_snr: 2.0               # mag_<band> (and colours) only where the band detects the source at S/N >= 2
  max_sources_per_pointing: 8000 # confirmed (real) sources: brightest kept if more are detected
  max_cr_candidates_per_pointing: 1000   # cosmic-ray candidates kept (as Noise), capped separately
  cutout_size: 128
  cutout_pixscale_arcsec: auto   # auto = detection band's pixel scale
  min_valid_fraction: 0.90
  core_radius_px: 16

# ---------------------------------------------------------------- Stage 1d
psf_match:
  stamp: 25
  min_stars: 5
  max_stars: 40
  star_min_snr: 50
  star_max_ellipticity: 0.3  # drizzled ACS stars measure up to ~0.23 in a single exposure
  star_isolation_px: 5       # no other segment this close to a PSF star's centre (further out: masked)
  star_min_snr_per_band: 30  # a star shapes a band's PSF only if it has S/N >= 30 in that band
  require_stars: true        # stop when no PSF star passes the cuts (false = continue with the Gaussian fallback)
  fallback_fwhm_arcsec: {ACS: 0.10, WFC3_UVIS: 0.08, WFC3_IR: 0.15, default: 0.10}  # approximate
  kernel: gaussian           # gaussian (robust default) | photutils (Fourier ratio, needs clean PSFs)
  window: 0.35               # photutils TopHatWindow beta (kernel: photutils only)

# ---------------------------------------------------------------- Stage 1e / 5b
morphology:
  min_snr: 3.0               # fainter sources get NaN morphology + flag 4
  sersic_maxiter: 500
  sersic_n_bounds: [0.05, 8.0]   # a fit ending at or beyond these limits gets flag_sersic = 1
  sersic_rhalf_max_px: 64        # ... and so does r_half beyond half the 128-px cutout
  pass2_mode: psf_fit        # psf_fit = statmorph Sersic fit convolved with the ePSF
                             # deconvolve = Richardson-Lucy deconvolution, then statmorph
  rl_iterations: 10

# ---------------------------------------------------------------- Stage 2a-2e
representation:
  cae_version: 2             # 2 = improved CAE (below); 1 = the original CAE (per-cutout stretch, fixed steps)
  latent_dim: 32             # CAE bottleneck size (cae_version 1 used 64)
  channels: [16, 32, 64, 64] # conv channels of the 4 encoder blocks (128 -> 8 px)
  band_mask_channels: true   # add a 0/1 "data present" channel per band (a band outside its footprint is not a dark source)
  asinh_softening_sigma: 3.0 # input scale: asinh((x - local bg) / (a * band sigma)); linear below ~a sigma ...
  asinh_max_sigma: 1000      # ... and 1.0 at this many sigma (same scale for every cutout of a pointing)
  target_weight: 10          # loss weight of the target footprint (grown 2 px) relative to background pixels
  ignore_cr_pixels: true     # cosmic-ray pixels found in Stage 1c (segmentation code 3) do not count in the loss
  augment: true              # random 90-degree rotations and flips while training
  max_epochs: 100            # upper limit; training normally stops earlier ...
  min_epochs: 20
  patience: 10               # ... when the validation (dev region) loss has not improved ...
  min_delta: 0.005           # ... by at least 0.5 % (relative) for this many epochs; the best epoch's weights are kept
  lr: 0.001
  lr_patience: 4             # halve the learning rate after this many epochs without validation improvement
  batch_size: 64
  epochs: 15                 # cae_version 1 only: minimum epochs ...
  min_steps: 1500            # ... cae_version 1 only: and at least this many optimisation steps
  n_pca: auto                # auto = Horn's parallel analysis (components above shuffled-data eigenvalues); or a number
  n_pca_max: 12              # cap for n_pca: auto
  pca_parallel_iterations: 50
  reuse_trained_cae: false   # true = load outputs/stage2a_representation/rep_cae.pt instead of retraining (PCA/fusion only)
  pca_min_variance: 0.02     # ...and drop any of them explaining < 2% of the latent variance (see pca_scree.png)
  spatial_weight: 0.5        # weight of the image block vs the physical block
  tabular_features: [fwhm_over_psf, band_consistency, gini, m20, concentration, asymmetry, smoothness,
                     sersic_n, r_half, ellipticity, mag, snr, colour_1, colour_2]
clustering:
  k_min: 2
  k_max: 20                        # BIC sweep range; the chosen k must not sit on the boundary (see gmm_k_selection.png)
  covariance_type: full            # full: the features are correlated; diag never reaches a BIC minimum here
  reg_covar: 0.001                 # variance floor added to each component (scaled feature units)
  n_init: 3
  robustness_seeds: [7, 1]         # repeat the BIC sweep near the chosen k with these seeds
  bootstrap_refits: 3              # stability (ARI) of the chosen GMM on bootstrap resamples
  compare_diag: true               # also draw the diagonal-covariance BIC curve in the figure
  hdbscan_min_cluster_size: auto   # auto = max(10, n_train // 50)
  hdbscan_min_samples: 5
  hdbscan_max_noise: 0.5           # ignore HDBSCAN fits that call more than half of the sources noise
  hdbscan_outlier_percentile: 95   # GLOSH above this train percentile -> density_outlier (Stage 2e)
baseline:
  point_source_tolerance: 0.2     # |FWHM / stellar-locus FWHM - 1| <= this -> stellar locus
  star_locus_min_snr: 150          # size separates stars from compact galaxies only above this S/N
                                   # (GOODS-S single exposures: ~ mag 22.3; check stage2d size_magnitude.png)
  class_star_threshold: 0.5
  noise_snr_max: 3.0
  sub_psf_factor: 0.7              # FWHM below 0.7 x the stellar locus -> sharper than the PSF
  single_band_max: 0.05            # only used if sources.csv has no cr_candidate column (older runs)
semantic_audit:
  gaia_radius_arcsec: 0.5
  gzh_radius_arcsec: 0.5
  gaia_mag_limit: 21.0             # compared with the detection-band AB magnitude (approximate)
  montage_n: 25
  min_evidence_members: 5          # a signal names a cluster only if >= this many train members carry it
gating:
  physical_gate: true              # object-level checks a cluster name must not contradict (Stage 2e docstring)
  transitional_p: 0.80
  outlier_max_p: 0.50
  recon_percentile: 99.0

# ---------------------------------------------------------------- Catalogues
catalogs:
  gzh:
    file: auto                     # auto = first file in data/catalogs matching *gz_hubble* or *gzh*
    match_radius_arcsec: 0.5
    estimate_offset: true          # remove the median astrometric offset before matching
    columns:                       # exact Galaxy Zoo: Hubble (Willett et al. 2017) names
      ra: RA
      dec: DEC
      smooth: t01_smooth_or_features_a01_smooth_debiased_fraction
      features: t01_smooth_or_features_a02_features_or_disk_debiased_fraction
      artifact: t01_smooth_or_features_a03_star_or_artifact_weighted_fraction
      edgeon: t02_edgeon_a01_yes_weighted_fraction
      spiral: t04_spiral_a01_spiral_weighted_fraction
      odd: t06_odd_a01_yes_weighted_fraction
      odd_irregular: t08_odd_feature_a04_irregular_weighted_fraction
      odd_disturbed: t08_odd_feature_a03_disturbed_weighted_fraction
      odd_merger: t08_odd_feature_a06_merger_weighted_fraction
      clumpy: t12_clumpy_a01_yes_weighted_fraction
    thresholds: {smooth: 0.6, features: 0.5, edgeon: 0.5, spiral: 0.5, irregular: 0.5,
                 odd: 0.4, merger: 0.5}
    boundary_agreement: 0.5
  gaia:
    file: auto                     # auto = *gaia* in data/catalogs (e.g. gaia_dr3.csv); or query online (below)
    query_online: false            # true = astroquery Gaia cone search per pointing
    columns: {source_id: source_id, ra: ra, dec: dec, parallax: parallax, pmra: pmra,
              pmdec: pmdec, g_mag: phot_g_mean_mag, teff: teff_gspphot}
    epoch: 2016.0
  candels:
    file: auto                     # auto = *kartaltepe* / *candels* / *k15* in data/catalogs (e.g. candels_k15.csv)
    columns:                       # Kartaltepe et al. (2015) FRACTIONAL table = VizieR J/ApJS/221/11/table3.
                                   # Lists = alternative names (VizieR web / TAP / astroquery spell them differently).
                                   # The five class fractions are REQUIRED. Check: python tools/check_catalog_columns.py
      ra: [RAJ2000, _RAJ2000, RA, ra]
      dec: [DEJ2000, _DEJ2000, DEC, dec]
      spheroid: ["f(Sph)", f_Sph_, fSph, f_spheroid]
      disk: ["f(Dk)", f_Dk_, fDk, f_disk]
      irregular: ["f(Ir)", f_Ir_, fIr, f_irr]
      point_source: ["f(PS)", f_PS_, fPS, f_ps]
      unclassifiable: ["f(UnCl)", f_UnCl_, fUnCl, f_unc]
      merger: ["f(M)", f_M_, fM, f_merger]
      n_classifiers: [NCl, Ncl, n_classifiers]   # used to keep the deepest row when a galaxy appears twice

# ---------------------------------------------------------------- Stage 4
supervised:
  mode: train                # train = learn from this field's GZH labels | apply = use a frozen model bundle
                             # (fields without Galaxy Zoo: Hubble labels; tools/run_field.py chooses automatically)
  model_bundle: null         # apply mode: folder written by tools/export_model.py, e.g. models/goods_2026-10-30
  # ---- Stage 4a: frozen ImageNet feature extractors (no CNN is trained) ----
  pretrained: true           # ImageNet weights for ResNet-18 and EfficientNet-B0 (downloaded once, ~66 MB)
  input_size: 224            # cutouts (128 px) are resized to this before the CNNs (ImageNet training size)
  feature_batch_size: 64
  # ---- Stage 4b: ENSEMBLE (feature fusion) + XGBoost ----
  cv_folds: 5
  cv_group_arcsec: 30        # galaxies in the same 30" sky cell stay in the same fold
  selection_metric: log_loss # number of trees / block weights chosen by minimum out-of-fold log-loss (or macro_f1)
  xgb_max_trees: 400         # trees per class chosen from 10, 25, 50, ... this
  fusion_block_weights:      # ENSEMBLE weights [w1 ResNet-18, w2 EfficientNet-B0, w3 physical]: the share of the
                             # features each tree may draw from each block; the best set is chosen by 5-fold CV
    - [0.333, 0.333, 0.334]  # equal
    - [0.284, 0.708, 0.008]  # proportional to the block sizes (512 : 1280 : 14) = no re-balancing
    - [0.40, 0.40, 0.20]     # image-heavy
    - [0.25, 0.25, 0.50]     # physics-heavy
    - [0.50, 0.25, 0.25]     # ResNet-heavy
    - [0.25, 0.50, 0.25]     # EfficientNet-heavy
  xgboost:                   # the "compile" step of XGBoost (p2p/xgb_worker.py has the same defaults)
    max_depth: 4             # shallow trees: ~1,100 galaxies, 1,806 features
    min_child_weight: 2
    learning_rate: 0.05
    subsample: 0.8           # random 80 % of the galaxies per tree     (tree equivalent of dropout)
    colsample_bytree: 0.3    # random 30 % of the features per tree, drawn by the ensemble block weights
    reg_lambda: 1.0          # L2 penalty on the leaf values             (L2 weight regularisation)
    reg_alpha: 0.0           # L1 penalty
    tree_method: hist
    n_jobs: 4
  # ---- Stage 4d: ablation ----
  ablation_input_size: 128   # the fusion is also cross-validated with features from 128 px (no resize)
  compare_previous_ensemble: Output_beofreF1   # outputs folder of the earlier three-model ensemble run (its
                             # out-of-fold predictions are scored on the same galaxies); null = no such row

# ---------------------------------------------------------------- Stages 6-7
evaluation:
  test_sealed: true          # true while developing: Stages 6a-7c do NOT score the test pointing or the Phase-7
                             # (external) pointings, so nothing can be tuned on them; all decisions use the
                             # cross-validation of train + dev. Final run: set false, then
                             # python run_pipeline.py --from 6a   (once; Stage 6a's ledger records every evaluation)

# ---------------------------------------------------------------- Stage 8
catalogue:
  designation_prefix: "P2P J"
  dedup_radius_arcsec: 0.2

# ---------------------------------------------------------------- Stage 9
atlas:
  enabled: true
  min_snr: 0                 # only image galaxies with snr >= this (0 = every galaxy)
  max_galaxies: null         # null = all; a number keeps the most confident ones
  clean_cosmic_rays: true    # display-only removal of sharp single-band hits (data are not changed)
  cr_nsigma: 5.0
  cr_ratio: 3.0              # a hit is > cr_ratio x brighter (in S/N) in its band than in every other band
  cr_passes: 2
  cr_suspect_fraction: 0.5   # a 'galaxy' that loses more than this share of its central light is flagged cr_suspect
  palette_colours: 256       # PNG palette size (compression); 0 = full 24-bit colour
  upscale: 1                 # 2 = save 256x256 instead of the native 128x128 cutout
  soft_sigma: 2.0            # asinh softening, in units of the sky noise of the cutout border
  top_percentile: 99.7       # brightest level shown, from the central part of the cutout ...
  min_top_sigma: 15          # ... but at least this many sky sigmas (faint galaxies are not over-stretched)
  keep_png_folder: true      # false = keep only galaxy_images.zip
  gallery_n: 36              # galaxies per class gallery (most confident first)
  gallery_columns: 6
  overview_per_class: 6

# ---------------------------------------------------------------- tools/run_field.py, app.py
field_runner:
  min_gzh_labels: 150        # fewer GZH galaxies than this in the field(s) -> apply mode (frozen model bundle)
  footprint_radius_deg: 0.028   # GZH / CANDELS counted within this radius of each target (ACS/WFC ~ 0.028 deg)
  gaia_radius_deg: 0.06      # Gaia DR3 cone per target

### 2.2 Runner — `run_pipeline.py`

In [ ]:
%%writefile run_pipeline.py
#!/usr/bin/env python
"""Run the From Pixels to Provenance pipeline.

Examples (from the project folder):
    python run_pipeline.py                  # every stage, in order
    python run_pipeline.py --list           # show the stages
    python run_pipeline.py --only 1c        # one stage (its inputs must exist)
    python run_pipeline.py --from 2a        # from 2a to the end
    python run_pipeline.py --from 1a --to 1e
"""
from __future__ import annotations

import argparse
import importlib
import json
import sys
import time
import traceback
import warnings
from pathlib import Path

ROOT = Path(__file__).resolve().parent
sys.path.insert(0, str(ROOT / "src"))

from p2p.context import STAGES, Ctx, MissingInput, StageSkipped, save_json  # noqa: E402

# pandas >= 2 prints "DataFrame is highly fragmented" when many catalogue columns are added one by one
# (Stage 8 builds 91). It is a speed hint only - the results are identical - so it is not shown.
try:
    from pandas.errors import PerformanceWarning
    warnings.simplefilter("ignore", PerformanceWarning)
except ImportError:
    pass


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--only", help="run a single stage, e.g. 1c")
    ap.add_argument("--from", dest="start", help="first stage to run")
    ap.add_argument("--to", dest="stop", help="last stage to run")
    ap.add_argument("--list", action="store_true", help="list the stages and exit")
    ap.add_argument("--config", default="config.yaml")
    ap.add_argument("--keep-going", action="store_true", help="continue after a failed stage")
    ap.add_argument("--summary", action="store_true",
                    help="print a model.summary() of every CNN (built from the config, untrained) and exit")
    args = ap.parse_args()
    if args.summary:
        _model_summaries(Ctx(ROOT, args.config))
        return

    keys = [k for k, *_ in STAGES]
    if args.list:
        for k, mod, folder, title in STAGES:
            print(f"  {k:>3}  {title:45s} -> outputs/{folder}/   (src/p2p/stages/{mod}.py)")
        return
    if args.only:
        todo = [args.only]
    else:
        i0 = keys.index(args.start) if args.start else 0
        i1 = keys.index(args.stop) if args.stop else len(keys) - 1
        todo = keys[i0:i1 + 1]
    unknown = [k for k in todo if k not in keys]
    if unknown:
        sys.exit(f"Unknown stage(s): {unknown}. Use --list.")

    warnings.filterwarnings("ignore", category=RuntimeWarning)
    ctx = Ctx(ROOT, args.config)
    runlog_path = ctx.out / "run_log.json"
    runlog = json.loads(runlog_path.read_text()) if runlog_path.exists() else {}
    info = {k: (m, t) for k, m, _, t in STAGES}

    for k in todo:
        mod_name, title = info[k]
        ctx.log.info("=" * 78)
        ctx.log.info(f"Stage {k} - {title}")
        ctx.log.info("=" * 78)
        t0 = time.time()
        status = "ok"
        try:
            mod = importlib.import_module(f"p2p.stages.{mod_name}")
            _limit_foreign_openmp()
            summary = mod.run(ctx) or {}
        except StageSkipped as e:
            status, summary = "skipped", {"reason": str(e)}
            ctx.log.info(f"[Stage {k}] SKIPPED: {e}")
            _clear_stale(ctx, k, str(e))
        except MissingInput as e:
            status, summary = "missing-input", {"reason": str(e)}
            ctx.log.info(f"[Stage {k}] CANNOT RUN: {e}")
            if not args.keep_going:
                _save(runlog, k, status, t0, summary, runlog_path)
                sys.exit(1)
        except Exception as e:
            status, summary = "failed", {"error": repr(e)}
            ctx.log.info(traceback.format_exc())
            if not args.keep_going:
                _save(runlog, k, status, t0, summary, runlog_path)
                sys.exit(1)
        _save(runlog, k, status, t0, summary, runlog_path)
        ctx.log.info(f"[Stage {k}] {status} in {time.time() - t0:.1f} s")
    ctx.log.info(f"Done. Outputs in {ctx.rel(ctx.out)} (final catalogue: stage8_catalogue/catalog.csv)")


def _model_summaries(ctx):
    """model.summary() of every network of the pipeline: the two autoencoders trained in Stages 1b / 2a and the two
    frozen ImageNet feature extractors of Stage 4a (built without ImageNet weights here: only the shapes matter)."""
    import torch
    from p2p.nets import ConvAutoencoder, EfficientNetB0, RepAutoencoder, ResNet18, model_summary
    qc, rep, sup = ctx.cfg["frame_qc"], ctx.cfg["representation"], ctx.cfg["supervised"]
    size = int(ctx.cfg.get("detection", {}).get("cutout_size", 128))
    bands = int(ctx.cfg["acquisition"].get("max_bands", 3))
    masks = bool(rep.get("band_mask_channels", True))
    tile, s4 = int(qc.get("tile", 128)), int(sup.get("input_size", 224))
    if int(qc.get("qc_version", 2)) == 1:
        m1 = ConvAutoencoder(in_ch=1, latent_dim=int(qc["latent_dim"]), size=tile)
    else:
        m1 = RepAutoencoder(bands=1, latent_dim=int(qc["latent_dim"]), size=tile,
                            channels=tuple(qc.get("channels", [16, 32, 64, 64])), mask_channels=False)
    m2 = RepAutoencoder(bands=bands, latent_dim=int(rep["latent_dim"]), size=size,
                        channels=tuple(rep.get("channels", [16, 32, 64, 64])), mask_channels=masks)
    x4 = torch.zeros(1, 3, s4, s4)
    models = [
        ("Stage 1b - frame-QC autoencoder", m1, (torch.zeros(1, 1, tile, tile),)),
        ("Stage 2a - representation autoencoder", m2, (torch.zeros(1, bands * (2 if masks else 1), size, size),)),
        (f"Stage 4a - ResNet-18 feature extractor (ImageNet, frozen; input {s4} px)", ResNet18(pretrained=False), (x4,)),
        (f"Stage 4a - EfficientNet-B0 feature extractor (ImageNet, frozen; input {s4} px)",
         EfficientNetB0(pretrained=False), (x4,)),
    ]
    txt = "\n\n".join(model_summary(m, *x, name=n) for n, m, x in models)
    txt += ("\n\nStage 4b - XGBoost (feature fusion): not a network; its summary (trees, leaves, parameters, "
            "importance by ENSEMBLE block) is written by Stage 4b to outputs/stage4b_xgboost_fusion/model_summary.txt")
    print(txt)
    path = ctx.out / "model_summaries.txt"
    path.write_text(txt + "\n")
    print(f"\nWritten to {ctx.rel(path)}")


def _limit_foreign_openmp():
    """On macOS, PyTorch ships its own libomp and scikit-learn uses the environment's (e.g. Anaconda's).
    Once Stage 2a has run torch, a multi-threaded OpenMP region in the second runtime (KMeans inside
    GaussianMixture, HDBSCAN, ...) crashes the process with a segmentation fault. Every OpenMP runtime
    that is NOT torch's is pinned to one thread; torch keeps its threads. Called after each stage
    import, because a runtime can only be limited once it is loaded."""
    try:
        from threadpoolctl import ThreadpoolController
    except ImportError:
        return
    for lib in ThreadpoolController().lib_controllers:
        if lib.user_api == "openmp" and "torch" not in Path(lib.filepath).parts:
            lib.set_num_threads(1)


KEEP_ON_SKIP = {"test_split_ledger.json"}


def _clear_stale(ctx, k, reason):
    """A skipped stage must not leave files from an EARLIER run behind: later stages would read them."""
    from p2p.context import STAGE_DIR
    d = ctx.out / STAGE_DIR[k]
    if d.exists():
        import shutil
        for f in d.iterdir():
            if f.is_dir():
                shutil.rmtree(f)
            elif f.name not in KEEP_ON_SKIP:
                f.unlink()
        (d / "SKIPPED.txt").write_text(reason + "\n")


def _save(runlog, k, status, t0, summary, path):
    runlog[k] = {"status": status, "seconds": round(time.time() - t0, 1),
                 "finished": time.strftime("%Y-%m-%d %H:%M:%S"), "summary": summary}
    save_json(runlog, path)


if __name__ == "__main__":
    main()

### 2.3 Shared package — `src/p2p/`
- `__init__.py`
- `catalogs.py`
- `compat.py`
- `context.py`
- `detection.py`
- `features.py`
- `fitsio.py`
- `inference.py`
- `morphology.py`
- `multiband.py`
- `nets.py`
- `utils.py`
- `xgb_worker.py`

In [ ]:
%%writefile src/p2p/__init__.py
"""From Pixels to Provenance - automated galaxy morphology from HST archival imaging."""
__version__ = "1.0.0"

In [ ]:
%%writefile src/p2p/archive.py
"""Find and download the best HST images for a sky position (MAST), and the Gaia DR3 cone around it.

Which file is "best" for this pipeline, per filter:
  1. it must CONTAIN the position (the observation footprint s_region, not just a nearby centre);
  2. a COMBINED product (several exposures drizzled together, cosmic rays rejected) beats a single exposure:
     HAP visit-level filter products (hst_<prop>_<visit>_<inst>_<det>_<filter>_<6 chars>) and classic association
     products (<9 chars> ending in digits) are combined; HAP exposure-level products (8-character suffix) are
     single exposures. Single exposures are the main error source of this pipeline (cosmic rays).
  3. then the longest total exposure time, then the position closest to the image centre.
Multi-visit mosaics (skycells) are left out by default: they are several GB.
"""
from __future__ import annotations

import re
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np

# one band per colour, bluest first; the first available filter of each group is taken in 'auto'
FILTER_GROUPS = {
    "ACS/WFC": [["F475W", "F435W", "F555W"], ["F606W", "F625W"], ["F814W", "F775W", "F850LP"]],
    "WFC3/UVIS": [["F475W", "F438W", "F390W"], ["F606W", "F555W"], ["F814W", "F775W"]],
    "WFC3/IR": [["F105W", "F110W"], ["F125W", "F140W"], ["F160W"]],
}


@dataclass
class Choice:
    filter: str
    obs_id: str
    filename: str
    combined: bool
    exptime_s: float
    offset_arcmin: float
    size_mb: float
    target_name: str
    proposal_id: str


def product_kind(obs_id: str) -> str:
    """'hap-visit' (combined), 'hap-exposure' (single), 'classic-assoc' (combined), 'classic-exp' (single), 'mosaic'."""
    o = obs_id.lower()
    if "skycell" in o:
        return "mosaic"
    if o.startswith("hst_"):
        last = o.split("_")[-1]
        return "hap-visit" if len(last) == 6 else "hap-exposure"
    if re.fullmatch(r"[a-z0-9]{6}\d{3}", o):
        return "classic-assoc"
    return "classic-exp"


def is_combined(obs_id: str) -> bool:
    return product_kind(obs_id) in ("hap-visit", "classic-assoc", "mosaic")


def _polygons(s_region: str):
    """All polygons of an s_region string ('POLYGON [ICRS] ra dec ra dec ...', possibly several)."""
    out = []
    for part in re.split(r"(?i)polygon", str(s_region))[1:]:
        nums = [float(x) for x in re.findall(r"[-+]?\d+\.?\d*(?:[eE][-+]?\d+)?", part)]
        if len(nums) >= 6:
            out.append(np.array(nums[: len(nums) // 2 * 2]).reshape(-1, 2))
    return out


def contains(s_region: str, ra: float, dec: float) -> bool:
    from matplotlib.path import Path as MPath
    for poly in _polygons(s_region):
        p = poly.copy()
        p[:, 0] = (p[:, 0] - ra + 180) % 360 - 180          # unwrap RA around the target
        p[:, 0] *= np.cos(np.radians(dec))
        p[:, 1] -= dec
        if MPath(p).contains_point((0.0, 0.0)):
            return True
    return False


def rank_observations(obs, ra: float, dec: float, allow_single=True, allow_mosaic=False):
    """obs: astropy Table / pandas DataFrame from Observations.query_criteria. Returns a DataFrame sorted best first,
    with columns kind, combined, contains, offset_arcmin."""
    import pandas as pd
    df = obs.to_pandas() if hasattr(obs, "to_pandas") else pd.DataFrame(obs)
    if df.empty:
        return df
    df["kind"] = df.obs_id.map(product_kind)
    df["combined"] = df.obs_id.map(is_combined)
    df["contains"] = [contains(s, ra, dec) for s in df.s_region]
    cosd = np.cos(np.radians(dec))
    df["offset_arcmin"] = 60 * np.hypot((df.s_ra - ra) * cosd, df.s_dec - dec)
    df = df[df.contains]
    if not allow_mosaic:
        df = df[df.kind != "mosaic"]
    if not allow_single:
        df = df[df.combined]
    return df.sort_values(["combined", "t_exptime", "offset_arcmin"], ascending=[False, False, True])


def overlap_fraction(region_a: str, region_b: str, ra: float, dec: float, n: int = 30) -> float:
    """Share of footprint A that footprint B also covers (grid sampling; fine for choosing files)."""
    from matplotlib.path import Path as MPath
    cosd = np.cos(np.radians(dec))
    def local(poly):
        p = poly.copy()
        p[:, 0] = ((p[:, 0] - ra + 180) % 360 - 180) * cosd
        return p
    A = [local(p) for p in _polygons(region_a)]
    B = [MPath(local(p)) for p in _polygons(region_b)]
    if not A or not B:
        return 0.0
    pts = np.vstack([a for a in A])
    xs = np.linspace(pts[:, 0].min(), pts[:, 0].max(), n)
    ys = np.linspace(pts[:, 1].min(), pts[:, 1].max(), n)
    grid = np.array([(x, y) for x in xs for y in ys])
    in_a = np.any([MPath(a).contains_points(grid) for a in A], axis=0)
    if not in_a.any():
        return 0.0
    in_b = np.any([b.contains_points(grid) for b in B], axis=0)
    return float((in_a & in_b).sum() / in_a.sum())


def _filters_of(v) -> set:
    return {x.strip().upper() for x in str(v).replace(",", ";").split(";") if x.strip()}


def search(ra: float, dec: float, filters="auto", instrument="ACS/WFC", radius_deg=0.03,
           allow_single=True, allow_mosaic=False, log=print) -> list[Choice]:
    """One Choice per filter (or per colour group when filters == 'auto'). The reddest group (the detection band)
    is chosen first; the other bands then prefer files that overlap it by >= 50 %, so Stage 1a groups all bands
    into ONE pointing."""
    from astroquery.mast import Observations
    groups = FILTER_GROUPS.get(instrument, FILTER_GROUPS["ACS/WFC"]) if filters == "auto" else [[f] for f in filters]
    flat = sorted({f.upper() for g in groups for f in g})
    obs = Observations.query_criteria(coordinates=f"{ra} {dec}", radius=f"{radius_deg} deg", obs_collection="HST",
                                      dataproduct_type="image", calib_level=3, instrument_name=instrument,
                                      filters=[f"*{f}*" for f in flat])   # ACS lists e.g. "F814W;CLEAR2L"
    log(f"[archive] ra={ra} dec={dec}: {len(obs)} {instrument} observations in {flat}")
    ranked = rank_observations(obs, ra, dec, allow_single, allow_mosaic)
    picked: dict[int, tuple] = {}
    ref_region = None
    for gi in sorted(range(len(groups)), key=lambda i: -_wavelength_nm(groups[i][0])):
        for f in groups[gi]:
            cand = ranked[[f.upper() in _filters_of(v) for v in ranked.filters]] if len(ranked) else ranked
            if cand.empty:
                continue
            if ref_region is not None:
                cand = cand.assign(ov=[overlap_fraction(ref_region, r, ra, dec) >= 0.5 for r in cand.s_region])
                cand = cand.sort_values(["ov", "combined", "t_exptime", "offset_arcmin"],
                                        ascending=[False, False, False, True])
            hit = _first_product(Observations, cand, f)
            if hit:
                picked[gi] = hit
                if ref_region is None:
                    ref_region = hit[1]
                break
    choices = [picked[i][0] for i in sorted(picked)]
    for c in choices:
        log(f"[archive]   {c.filter:7s} {c.filename:55s} {'combined' if c.combined else 'SINGLE exposure':16s} "
            f"{c.exptime_s:7.0f} s  {c.offset_arcmin:4.1f}' from centre  {c.size_mb:6.0f} MB")
    missing = [g[0] for i, g in enumerate(groups) if i not in picked]
    if missing:
        log(f"[archive]   no covering {instrument} image for: {missing}")
    if any(not c.combined for c in choices):
        log("[archive]   WARNING: single-exposure files carry every cosmic ray - expect many spurious detections")
    return choices


def _wavelength_nm(band: str) -> float:
    m = re.search(r"(\d+)", band)
    v = float(m.group(1)) if m else 0.0
    return v * 10 if v < 200 else v


def _first_product(Observations, cand, f):
    for _, row in cand.iterrows():
        prods = Observations.filter_products(Observations.get_product_list(str(row.obsid)),
                                             productType="SCIENCE", extension="fits").to_pandas()
        for sub in ("DRC", "DRZ"):
            hit = prods[(prods.productSubGroupDescription == sub) &
                        prods.productFilename.str.lower().str.startswith(str(row.obs_id).lower())]
            if len(hit):
                r = hit.iloc[0]
                return (Choice(f.upper(), str(row.obs_id), str(r.productFilename), bool(row.combined),
                               float(row.t_exptime), round(float(row.offset_arcmin), 2),
                               round(float(r["size"]) / 1e6, 1), str(row.target_name), str(row.proposal_id)),
                        str(row.s_region))
    return None


def download(choices: list[Choice], out_dir, log=print) -> list[Path]:
    from astroquery.mast import Observations
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for c in choices:
        dest = out_dir / c.filename
        if dest.exists():
            log(f"[archive] already there: {dest.name}")
            paths.append(dest)
            continue
        uri = f"mast:HST/product/{c.filename}"
        log(f"[archive] downloading {c.filename} ({c.size_mb:.0f} MB) ...")
        status, msg, _ = Observations.download_file(uri, local_path=str(dest))
        if status != "COMPLETE":
            raise RuntimeError(f"download of {c.filename} failed: {status} {msg}")
        paths.append(dest)
    return paths


def gaia_cone(ra: float, dec: float, radius_deg: float, out_dir, log=print) -> Path:
    from astroquery.gaia import Gaia
    out = Path(out_dir) / f"gaia_dr3_{ra:.3f}_{dec:+.3f}.csv"
    if out.exists():
        log(f"[gaia] already there: {out.name}")
        return out
    q = ("SELECT source_id, ra, dec, parallax, pmra, pmdec, phot_g_mean_mag, teff_gspphot FROM gaiadr3.gaia_source "
         f"WHERE 1 = CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', {ra}, {dec}, {radius_deg}))")
    t = Gaia.launch_job(q).get_results()
    out.parent.mkdir(parents=True, exist_ok=True)
    t.to_pandas().to_csv(out, index=False)
    log(f"[gaia] {len(t)} sources within {radius_deg} deg -> {out.name}")
    return out


def choices_table(choices: list[Choice]):
    import pandas as pd
    return pd.DataFrame([asdict(c) for c in choices])

In [ ]:
%%writefile src/p2p/catalogs.py
"""Loading the external catalogues (GZH, Gaia DR3, CANDELS) into a standard form.

Each loader returns a DataFrame with *standard* column names (ra, dec, smooth,
...) taken from the mapping in config.yaml, cut to the sky area of the
pointings so large tables (GZH has ~114k rows) stay small in memory."""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from .context import Ctx, StageSkipped

PATTERNS = {"gzh": ["*gz_hubble*", "*gzh*", "*galaxy_zoo*"],
            "gaia": ["*gaia*"],
            "candels": ["*kartaltepe*", "*candels*", "*k15*"]}


def find_catalog(ctx: Ctx, kind: str) -> Path | None:
    spec = ctx.cfg["catalogs"][kind].get("file", "auto")
    if spec and spec != "auto":
        p = ctx.root / spec if not Path(spec).is_absolute() else Path(spec)
        return p if p.exists() else None
    if not ctx.catalogs.exists():
        return None
    for pat in PATTERNS[kind]:
        hits = sorted(ctx.catalogs.glob(pat)) + sorted(ctx.catalogs.glob(f"*/{pat}"))     # also one folder down
        hits = [h for h in hits if h.is_file() and h.suffix.lower() in (".csv", ".gz", ".fits", ".fit", ".vot", ".xml", ".tsv")]
        if kind == "candels":                     # VizieR J/ApJS/221/11: table3 holds the class fractions
            hits = sorted(hits, key=lambda h: "table3" not in h.name.lower())
        if hits:
            return hits[0]
    return None


def find_catalogs(ctx: Ctx, kind: str) -> list[Path]:
    """Every matching file (Gaia: one cone per pointing when several fields are processed together)."""
    spec = ctx.cfg["catalogs"][kind].get("file", "auto")
    if spec and spec != "auto":
        p = find_catalog(ctx, kind)
        return [p] if p else []
    if not ctx.catalogs.exists():
        return []
    for pat in PATTERNS[kind]:
        hits = sorted(ctx.catalogs.glob(pat)) + sorted(ctx.catalogs.glob(f"*/{pat}"))
        hits = [h for h in hits if h.is_file() and h.suffix.lower() in (".csv", ".gz", ".fits", ".fit", ".vot", ".xml", ".tsv")]
        if hits:
            return hits
    return []


def _candidates(col) -> list[str]:
    """A mapping value may be one column name or a list of alternatives (first one found wins)."""
    return [str(c) for c in col] if isinstance(col, (list, tuple)) else [str(col)]


def _text_format(path: Path):
    """Separator and comment character of a text table. VizieR / Gaia-archive downloads may use ';' or
    tabs and '#' comment lines; GZH is a plain comma-separated file."""
    import gzip
    opener = gzip.open if path.name.lower().endswith(".gz") else open
    with opener(path, "rt", errors="replace") as f:
        first = ""
        for line in f:
            if line.strip() and not line.startswith("#"):
                first = line
                break
    sep = max([",", ";", "\t", "|"], key=first.count) if first else ","
    return (sep if first.count(sep) else ","), "#"


def read_table(path: Path, usecols=None) -> pd.DataFrame:
    name = path.name.lower()
    if name.endswith((".fits", ".fit", ".fits.gz")):
        from astropy.table import Table
        t = Table.read(path)
        names = [c for c in t.colnames if len(t[c].shape) <= 1]
        df = t[names].to_pandas()
        return df[[c for c in df.columns if c in usecols]] if usecols else df
    if name.endswith((".vot", ".xml", ".vot.gz")):
        from astropy.table import Table
        return Table.read(path, format="votable").to_pandas()
    sep, comment = _text_format(path)
    header = pd.read_csv(path, nrows=0, sep=sep, comment=comment).columns
    cols = [c for c in header if usecols is None or c in usecols]
    return pd.read_csv(path, usecols=cols, sep=sep, comment=comment, low_memory=False)


def read_mapped(path: Path, mapping: dict) -> pd.DataFrame:
    """Read only the mapped columns (case-insensitive) - keeps big tables light."""
    want = {c.lower() for v in mapping.values() for c in _candidates(v)}
    name = path.name.lower()
    if name.endswith((".csv", ".csv.gz", ".tsv", ".tsv.gz", ".txt", ".dat")):
        sep, comment = _text_format(path)
        header = pd.read_csv(path, nrows=0, sep=sep, comment=comment).columns
        cols = [c for c in header if c.strip().lower() in want]
        df = pd.read_csv(path, usecols=cols, sep=sep, comment=comment, low_memory=False)
        df.columns = [c.strip() for c in df.columns]
        return df
    df = read_table(path)
    return df[[c for c in df.columns if c.lower() in want]]


def _to_degrees(values, is_ra):
    """Numbers stay numbers; sexagesimal strings ('03:32:24.6' / '-27:48:31') are converted to degrees."""
    num = pd.to_numeric(values, errors="coerce")
    text = values.astype(str).str.strip()
    sexa = num.isna() & text.str.contains(r"^[+-]?\d+[: ]\d+[: ]\d", regex=True)
    if sexa.any():
        from astropy.coordinates import Angle
        import astropy.units as u
        unit = u.hourangle if is_ra else u.deg
        num = num.copy()
        num[sexa] = [Angle(t, unit=unit).degree for t in text[sexa]]
    return num


def standardise(df: pd.DataFrame, mapping: dict, kind: str, required=("ra", "dec")) -> pd.DataFrame:
    """Rename catalogue columns to standard names. Matching is case-insensitive and a mapping value
    may list alternatives. A missing REQUIRED column stops the run with the closest names; a missing
    optional one is filled with NaN. Rows without a usable position are dropped."""
    import difflib
    lower = {c.lower(): c for c in df.columns}
    out = pd.DataFrame(index=df.index)
    missing = []
    for std, col in mapping.items():
        src = next((lower[c.lower()] for c in _candidates(col) if c.lower() in lower), None)
        if src is None:
            missing.append((std, col))
            out[std] = np.nan
        elif std in ("ra", "dec"):
            out[std] = _to_degrees(df[src], is_ra=(std == "ra"))
        elif std in ("source_id", "id"):
            out[std] = df[src]
        else:
            out[std] = pd.to_numeric(df[src], errors="coerce")
    hard = [m for m in missing if m[0] in required]
    if hard:
        hints = {std: difflib.get_close_matches(str(_candidates(col)[0]), list(df.columns), n=3, cutoff=0.3)
                 for std, col in hard}
        raise ValueError(f"{kind}: required column(s) not found: {hard}. Closest names in the file: {hints}.\n"
                         f"Fix 'catalogs.{kind}.columns' in config.yaml (python tools/check_catalog_columns.py "
                         "lists every column).")
    if missing:
        print(f"[catalogs] {kind}: optional column(s) missing, filled with NaN: {missing}")
    bad = out["ra"].isna() | out["dec"].isna()
    if bad.mean() > 0.5:
        raise ValueError(f"{kind}: {bad.mean():.0%} of the rows have no readable RA/Dec - check the file format.")
    return out[~bad]


def sky_box(metas, margin_deg=0.02):
    pts = np.vstack([np.asarray(m["footprint"]) for m in metas])
    return (pts[:, 0].min() - margin_deg, pts[:, 0].max() + margin_deg,
            pts[:, 1].min() - margin_deg, pts[:, 1].max() + margin_deg)


def cut_to_box(df, box):
    r0, r1, d0, d1 = box
    return df[(df.ra >= r0) & (df.ra <= r1) & (df.dec >= d0) & (df.dec <= d1)].reset_index(drop=True)


# ---------------------------------------------------------------- GZH
def load_gzh(ctx: Ctx, metas, required=True) -> pd.DataFrame | None:
    path = find_catalog(ctx, "gzh")
    if path is None:
        if required:
            raise StageSkipped("No Galaxy Zoo: Hubble table in data/catalogs (e.g. gz_hubble_main.csv.gz). "
                               "See README section 'Catalogues'.")
        return None
    mapping = ctx.cfg["catalogs"]["gzh"]["columns"]
    df = standardise(read_mapped(path, mapping), mapping, "gzh")
    df["gzh_row"] = df.index.to_numpy()          # row number in the original GZH file
    return cut_to_box(df, sky_box(metas))


# ---------------------------------------------------------------- Gaia
def load_gaia(ctx: Ctx, metas) -> pd.DataFrame | None:
    cfg = ctx.cfg["catalogs"]["gaia"]
    paths = find_catalogs(ctx, "gaia")
    if paths:
        df = pd.concat([standardise(read_mapped(p, cfg["columns"]), cfg["columns"], "gaia") for p in paths],
                       ignore_index=True)
        if "source_id" in df:
            df = df.drop_duplicates("source_id").reset_index(drop=True)
        df = cut_to_box(df, sky_box(metas))
        if df.empty and cfg.get("query_online"):
            return query_gaia_online(ctx, metas)
        return df
    if cfg.get("query_online"):
        return query_gaia_online(ctx, metas)
    return None


def query_gaia_online(ctx: Ctx, metas) -> pd.DataFrame | None:
    try:
        from astroquery.gaia import Gaia
    except Exception:
        print("[catalogs] astroquery not installed - cannot query Gaia online.")
        return None
    r0, r1, d0, d1 = sky_box(metas)
    q = ("SELECT source_id, ra, dec, parallax, pmra, pmdec, phot_g_mean_mag, teff_gspphot "
         f"FROM gaiadr3.gaia_source WHERE ra BETWEEN {r0} AND {r1} AND dec BETWEEN {d0} AND {d1}")
    try:
        t = Gaia.launch_job_async(q).get_results().to_pandas()
    except Exception as e:
        print(f"[catalogs] Gaia query failed: {e}")
        return None
    out = ctx.catalogs / "gaia_dr3_query.csv"
    ctx.catalogs.mkdir(parents=True, exist_ok=True)
    t.to_csv(out, index=False)
    return standardise(t, ctx.cfg["catalogs"]["gaia"]["columns"], "gaia")


def propagate_gaia(gaia: pd.DataFrame, to_epoch: float, from_epoch: float = 2016.0) -> pd.DataFrame:
    """Move Gaia positions to the HST observation epoch with proper motions."""
    g = gaia.copy()
    if not np.isfinite(to_epoch):
        return g
    dt = to_epoch - from_epoch
    pmra = g["pmra"].fillna(0).to_numpy(float)          # mas/yr, includes cos(dec)
    pmdec = g["pmdec"].fillna(0).to_numpy(float)
    cosd = np.cos(np.deg2rad(g["dec"].to_numpy(float)))
    g["ra"] = g["ra"] + pmra * dt / 3.6e6 / np.maximum(cosd, 1e-6)
    g["dec"] = g["dec"] + pmdec * dt / 3.6e6
    return g


# ---------------------------------------------------------------- CANDELS
K15_CLASS_COLUMNS = ("spheroid", "disk", "irregular", "point_source", "unclassifiable")


def load_candels(ctx: Ctx, metas) -> pd.DataFrame | None:
    """Kartaltepe et al. (2015) FRACTIONAL table (VizieR J/ApJS/221/11/table3). The five class fractions
    are required: if they were silently NaN every galaxy would become 'spheroid'. The same galaxy can
    appear at two depths; the row with the most classifiers is kept."""
    path = find_catalog(ctx, "candels")
    if path is None:
        return None
    mapping = ctx.cfg["catalogs"]["candels"]["columns"]
    df = standardise(read_mapped(path, mapping), mapping, "candels", required=("ra", "dec") + K15_CLASS_COLUMNS)
    if "n_classifiers" in df and df.n_classifiers.notna().any():
        key = (df.ra * 3600 / 0.2).round().astype("int64").astype(str) + "_" + \
              (df.dec * 3600 / 0.2).round().astype("int64").astype(str)
        before = len(df)
        df = df.assign(_k=key).sort_values("n_classifiers", ascending=False).drop_duplicates("_k").drop(columns="_k")
        if before > len(df):
            print(f"[catalogs] candels: {before - len(df)} duplicate rows (same galaxy, other depth) removed")
    return cut_to_box(df.reset_index(drop=True), sky_box(metas))

In [ ]:
%%writefile src/p2p/compat.py
"""photutils renamed several arguments and attributes between 1.x/2.x and 3.x
(npixels -> n_pixels, nlevels -> n_levels, xcentroid -> x_centroid, ...).
These wrappers accept either version so the project runs on both."""
from __future__ import annotations

import inspect

import numpy as np

from photutils.segmentation import SourceCatalog, deblend_sources, detect_sources

_DEBLEND = inspect.signature(deblend_sources).parameters
_ALIASES = {"x_centroid": "xcentroid", "y_centroid": "ycentroid", "n_labels": "nlabels",
            "segment_area": "area", "semimajor_axis": "semimajor_sigma",
            "semiminor_axis": "semiminor_sigma", "kron_flux_err": "kron_fluxerr",
            "segment_flux_err": "segment_fluxerr"}


def detect(data, threshold, npixels, mask=None):
    return detect_sources(data, threshold, npixels, mask=mask)


def deblend(data, segm, npixels, nlevels=32, contrast=0.001):
    kw = {"contrast": contrast}
    kw["n_levels" if "n_levels" in _DEBLEND else "nlevels"] = nlevels
    if "progress_bar" in _DEBLEND:
        kw["progress_bar"] = False
    return deblend_sources(data, segm, npixels, **kw)


def attr(obj, name):
    if hasattr(obj, name):
        return getattr(obj, name)
    return getattr(obj, _ALIASES[name])


def values(obj, name):
    """Catalogue property as a plain float array (strips astropy units)."""
    v = attr(obj, name)
    v = getattr(v, "value", v)
    return np.asarray(v, dtype=float)


def n_labels(segm) -> int:
    if segm is None:
        return 0
    return int(attr(segm, "n_labels"))


__all__ = ["detect", "deblend", "attr", "values", "n_labels", "SourceCatalog"]

In [ ]:
%%writefile src/p2p/context.py
"""Shared run context: configuration, folders, logging and the stage registry.

Every stage module exposes ``run(ctx)``. A stage reads what earlier stages
wrote (through ``ctx.require`` / ``load_objects``) and writes its own files
into ``outputs/<stage folder>/`` so each step can be inspected on its own.
"""
from __future__ import annotations

import json
import logging
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

# (key, module name, output folder, title) in EXECUTION order.
# Phase 5 runs before Phase 4 so the supervised models train on the
# PSF-refined Pass-2 features they are deployed on.
STAGES = [
    ("0",  "stage0_partition",        "stage0_partition",        "Sky-region partitioning"),
    ("1a", "stage1a_acquisition",     "stage1a_acquisition",     "Archive acquisition / local scan"),
    ("1b", "stage1b_frame_qc",        "stage1b_frame_qc",        "Frame quality control (CAE)"),
    ("1c", "stage1c_detection",       "stage1c_detection",       "Detection, deblending, aligned cutouts"),
    ("1d", "stage1d_psf_match",       "stage1d_psf_match",       "Cross-filter PSF homogenisation"),
    ("1e", "stage1e_pass1_morphology", "stage1e_pass1_morphology", "Pass-1 photometry and morphology"),
    ("2a", "stage2a_representation",  "stage2a_representation",  "Representation learning + fusion"),
    ("2b", "stage2b_clustering",      "stage2b_clustering",      "Unsupervised clustering"),
    ("2c", "stage2c_baseline",        "stage2c_baseline",        "Baseline segregation + Gaia match"),
    ("2d", "stage2d_semantic_audit",  "stage2d_semantic_audit",  "Semantic audit"),
    ("2e", "stage2e_continuum_gating", "stage2e_continuum_gating", "Continuum handling + outlier gating"),
    ("3a", "stage3a_gzh_labels",      "stage3a_gzh_labels",      "GZH label anchoring"),
    ("3b", "stage3b_star_routing",    "stage3b_star_routing",    "Star calibration routing"),
    ("3c", "stage3c_noise_model",     "stage3c_noise_model",     "Noise false-positive model"),
    ("5a", "stage5a_empirical_psf",   "stage5a_empirical_psf",   "Empirical PSF construction"),
    ("5b", "stage5b_pass2_refinement", "stage5b_pass2_refinement", "Pass-2 morphology refinement"),
    ("4a", "stage4a_cnn_features",    "stage4a_cnn_features",    "CNN features (ResNet-18 + EfficientNet-B0)"),
    ("4b", "stage4b_xgboost_fusion",  "stage4b_xgboost_fusion",  "Ensemble (feature fusion) + XGBoost"),
    ("4c", "stage4c_calibration",     "stage4c_calibration",     "Probability calibration"),
    ("4d", "stage4d_ablation",        "stage4d_ablation",        "Controlled ablation"),
    ("6a", "stage6a_test_eval",       "stage6a_test_eval",       "In-domain test (run once)"),
    ("6b", "stage6b_uncertainty",     "stage6b_uncertainty",     "Uncertainty audit"),
    ("6c", "stage6c_xai",             "stage6c_xai",             "Explainable AI (SHAP)"),
    ("7a", "stage7a_taxonomy",        "stage7a_taxonomy",        "Taxonomy alignment + ceiling"),
    ("7b", "stage7b_crosscat",        "stage7b_crosscat",        "Cross-catalogue evaluation"),
    ("7c", "stage7c_generalisation",  "stage7c_generalisation",  "Generalisation disentanglement"),
    ("8",  "stage8_catalogue",        "stage8_catalogue",        "Unified catalogue (catalog.csv)"),
    ("9",  "stage9_galaxy_atlas",     "stage9_galaxy_atlas",     "Galaxy image atlas (PNG + zip)"),
]
STAGE_DIR = {k: d for k, _, d, _ in STAGES}


class MissingInput(RuntimeError):
    """Raised when a stage needs a file that an earlier stage has not written yet."""


class StageSkipped(RuntimeError):
    """Raised by a stage that cannot run with the data provided (e.g. no CANDELS table)."""


class Ctx:
    def __init__(self, root: str | Path, config_file: str = "config.yaml"):
        self.root = Path(root).resolve()
        with open(self.root / config_file) as f:
            self.cfg = yaml.safe_load(f)
        p = self.cfg["paths"]
        self.raw = self.root / p["raw"]
        self.catalogs = self.root / p["catalogs"]
        self.out = self.root / p["outputs"]
        self.out.mkdir(parents=True, exist_ok=True)
        self.log = _make_logger(self.out / "pipeline.log")
        self.seed = int(self.cfg["project"].get("seed", 42))
        self.set_seed()

    # ---------------- folders and files ----------------
    def stage_dir(self, key: str) -> Path:
        d = self.out / STAGE_DIR[key]
        d.mkdir(parents=True, exist_ok=True)
        return d

    def path(self, key: str, name: str) -> Path:
        return self.out / STAGE_DIR[key] / name

    def require(self, key: str, name: str) -> Path:
        p = self.path(key, name)
        if not p.exists():
            raise MissingInput(f"{self.rel(p)} not found - run stage {key} first "
                               f"(python run_pipeline.py --only {key}).")
        return p

    # ---------------- small helpers ----------------
    def rel(self, p) -> str:
        """Path relative to the project folder when possible (outputs may live elsewhere)."""
        try:
            return str(Path(p).resolve().relative_to(self.root))
        except ValueError:
            return str(p)

    def set_seed(self):
        random.seed(self.seed)
        np.random.seed(self.seed)
        try:
            import torch
            torch.manual_seed(self.seed)
        except Exception:
            pass

    @property
    def n_jobs(self) -> int:
        n = int(self.cfg["project"].get("n_jobs", -1))
        return (os.cpu_count() or 1) if n < 0 else max(1, n)

    def device(self):
        import torch
        want = str(self.cfg["project"].get("device", "auto")).lower()
        if want == "auto":
            if torch.cuda.is_available():
                return torch.device("cuda")
            if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
                return torch.device("mps")
            return torch.device("cpu")
        return torch.device(want)


# ---------------- JSON / table helpers ----------------
def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=_json_default)
    return path


def load_json(path: Path):
    with open(path) as f:
        return json.load(f)


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if np.isnan(o) else float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    return str(o)


# Which file holds which per-object table. load_objects() merges them on object_id.
OBJECT_TABLES = {
    "sources":     ("1c", "sources.csv"),
    "pass1":       ("1e", "features_pass1.csv"),
    "clusters":    ("2b", "clusters.csv"),
    "baseline":    ("2c", "baseline.csv"),
    "gaia":        ("2c", "gaia_match.csv"),
    "evidence":    ("2d", "evidence.csv"),
    "segregation": ("2e", "segregation.csv"),
    "labels":      ("3a", "labels.csv"),
    "routing":     ("3b", "star_routing.csv"),
    "noise":       ("3c", "noise_scores.csv"),
    "pass2":       ("5b", "features_pass2.csv"),
}


def load_objects(ctx: Ctx, parts) -> pd.DataFrame:
    """Merge per-object tables on object_id (left join on the source list).
    A later part OVERWRITES columns of the same name for the rows it holds -
    this is how Pass-2 features replace Pass-1 features for the galaxy track."""
    df = None
    for part in parts:
        key, name = OBJECT_TABLES[part]
        t = pd.read_csv(ctx.require(key, name))
        if df is None:
            df = t
            continue
        overlap = [c for c in t.columns if c in df.columns and c != "object_id"]
        df = df.merge(t, on="object_id", how="left", suffixes=("", "__new"), indicator="_has")
        rows = df["_has"] == "both"
        for c in overlap:
            df.loc[rows, c] = df.loc[rows, c + "__new"]
        # drop the helper columns in one go and de-fragment (avoids pandas' PerformanceWarning)
        df = df.drop(columns=[c + "__new" for c in overlap] + ["_has"]).copy()
    return df


def _make_logger(path: Path) -> logging.Logger:
    log = logging.getLogger("p2p")
    if log.handlers:
        return log
    log.setLevel(logging.INFO)
    fmt = logging.Formatter("%(asctime)s  %(message)s", "%H:%M:%S")
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    fh = logging.FileHandler(path)
    fh.setFormatter(fmt)
    log.addHandler(sh)
    log.addHandler(fh)
    return log


class Timer:
    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *a):
        self.seconds = time.time() - self.t0

In [ ]:
%%writefile src/p2p/detection.py
"""Background modelling and source detection shared by Stage 1c and Stage 3c."""
from __future__ import annotations

import numpy as np
from photutils.background import Background2D, MedianBackground
from scipy import ndimage

from .compat import deblend, detect
from .utils import gaussian_psf


def subtract_background(sci, valid, box=64, filt=3):
    """2-D background (median estimator) fitted on covered pixels only, so chip
    gaps and frame edges (WHT = 0) do not drag the estimate. Returns the
    background-subtracted image (0 where invalid) and the background rms map."""
    data = np.where(valid, sci, 0.0).astype(np.float32)
    bkg = Background2D(data, box, filter_size=(filt, filt), coverage_mask=~valid,
                       bkg_estimator=MedianBackground(), fill_value=0.0)
    sub = np.where(valid, data - bkg.background, 0.0).astype(np.float32)
    rms = np.where(valid, bkg.background_rms, np.nan).astype(np.float32)
    rms = np.where(np.isfinite(rms) & (rms > 0), rms, np.nanmedian(rms))
    return sub, rms


def smooth(sub, fwhm_px=3.0):
    k = gaussian_psf(fwhm_px, size=int(2 * round(1.5 * fwhm_px) + 1))
    return ndimage.convolve(sub, k, mode="constant", cval=0.0)


def detect_and_deblend(sub, rms, valid, cfg, deblend_sources=True):
    """Threshold = nsigma x rms x correlated-noise factor on the smoothed image,
    then multi-threshold deblending (SExtractor-style)."""
    conv = smooth(sub, cfg["kernel_fwhm_px"])
    thr = cfg["nsigma"] * rms * cfg.get("correlated_noise_factor", 1.0)
    segm = detect(conv, thr, cfg["npixels"], mask=~valid)
    if segm is not None and deblend_sources:
        segm = deblend(conv, segm, cfg["npixels"], cfg["deblend_nlevels"], cfg["deblend_contrast"])
    return segm, conv

In [ ]:
%%writefile src/p2p/features.py
"""Tabular pre-processing, the ImageNet input of the frozen CNNs, and the galaxy / label tables."""
from __future__ import annotations

import joblib
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

from .utils import signed_log1p


class TabularPrep:
    """Median imputation -> signed log -> RobustScaler. Columns that are entirely
    NaN in the training data (e.g. colours when only one band exists) are dropped."""

    def __init__(self, columns):
        self.columns = list(columns)
        self.kept = None
        self.imputer = SimpleImputer(strategy="median")
        self.scaler = RobustScaler()

    def fit(self, df: pd.DataFrame):
        cols = [c for c in self.columns if c in df.columns]
        X = df[cols].apply(pd.to_numeric, errors="coerce").to_numpy(float)
        keep = ~np.all(np.isnan(X), axis=0)
        self.kept = [c for c, k in zip(cols, keep) if k]
        X = signed_log1p(self.imputer.fit_transform(df[self.kept].apply(pd.to_numeric, errors="coerce")))
        self.scaler.fit(X)
        return self

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        X = df.reindex(columns=self.kept).apply(pd.to_numeric, errors="coerce")
        return self.scaler.transform(signed_log1p(self.imputer.transform(X))).astype(np.float32)

    def save(self, path):
        joblib.dump(self, path)

    @staticmethod
    def load(path) -> "TabularPrep":
        return joblib.load(path)


def load_cutouts(ctx):
    """PSF-matched cutouts written by Stage 1d (memory-mapped, row = cutout_index)."""
    return np.load(ctx.require("1d", "cutouts.npy"), mmap_mode="r")


IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)[:, None, None]
IMAGENET_STD = np.array([0.229, 0.224, 0.225], np.float32)[:, None, None]


def imagenet_input(cubes, size=224, q_lo=1.0, q_hi=99.5, soft=0.1):
    """PSF-matched cutouts (N, 3, 128, 128), bands blue -> red, in e-/s -> ImageNet-ready tensor (N, 3, size, size).
      1. one asinh stretch per cutout, the SAME for all three bands (percentiles of all its pixels), so the colour
         of the galaxy survives (a separate stretch per band would make every galaxy grey);
      2. channels R, G, B = reddest, middle, bluest band (F775W, F606W, F435W); a band without data stays 0;
      3. ImageNet mean / std per channel (the statistics the frozen BatchNorm layers expect);
      4. bilinear resize to size x size (224 = the image size the networks were trained on)."""
    import torch
    import torch.nn.functional as F
    x = np.nan_to_num(np.asarray(cubes, np.float32))[:, ::-1]                 # blue->red  ->  R, G, B
    out = np.empty_like(x)
    for n in range(len(x)):
        lo, hi = np.percentile(x[n], [q_lo, q_hi])
        v = np.clip((x[n] - lo) / max(hi - lo, 1e-8), 0, 1)
        out[n] = np.arcsinh(v / soft) / np.arcsinh(1.0 / soft)
    t = torch.from_numpy((out - IMAGENET_MEAN) / IMAGENET_STD)
    if size != t.shape[-1]:
        t = F.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)
    return t


def physical_matrix(df, columns):
    """The 14 physical (Pass-2) features as they are: trees need no scaling, and NaN (not measurable) is
    handled natively by XGBoost."""
    return df.reindex(columns=list(columns)).apply(pd.to_numeric, errors="coerce").to_numpy(np.float32)


def galaxy_table(ctx):
    """Every galaxy-track object with Pass-2 features (overwriting Pass 1) and labels."""
    from .context import load_objects
    from .utils import as_bool
    df = load_objects(ctx, ["sources", "pass1", "segregation", "labels", "pass2"])
    df = df[(df.cluster_semantic == "Galaxy") & (df.anomaly_flag == 0)].copy()
    df["is_pseudo_label"] = as_bool(df.is_pseudo_label.fillna(True))
    df["edge_flag"] = as_bool(df.edge_flag)
    return df


def supervised_splits(ctx):
    """GZH-labelled galaxies per split. Truncated (edge_flag) sources are excluded,
    and QC-flagged ones too when frame_qc.exclude_flagged is true."""
    df = galaxy_table(ctx)
    lab = df[~df.is_pseudo_label & df.split_tag.isin(["train", "dev", "test"]) & ~df.edge_flag]
    if ctx.cfg["frame_qc"].get("exclude_flagged"):
        lab = lab[lab.frame_quality_flag == 1]
    lab = lab.assign(gz_label_id=lab.gz_label_id.astype(int))
    from .modes import is_apply
    if is_apply(ctx):
        # the frozen model never saw this field: EVERY labelled galaxy (any split, buffer too) is test data
        from .context import StageSkipped
        allg = df[~df.is_pseudo_label & ~df.edge_flag]
        if ctx.cfg["frame_qc"].get("exclude_flagged"):
            allg = allg[allg.frame_quality_flag == 1]
        if allg.empty:
            raise StageSkipped("apply mode: no GZH-labelled galaxies in this field, so no in-field evaluation "
                               "(every galaxy is still classified in Stage 8)")
        allg = allg.assign(gz_label_id=allg.gz_label_id.astype(int)).reset_index(drop=True)
        return {"train": allg.iloc[0:0], "dev": allg.iloc[0:0], "test": allg}
    splits = {s: lab[lab.split_tag == s].reset_index(drop=True) for s in ("train", "dev", "test")}
    empty = [s for s, v in splits.items() if len(v) == 0]
    if empty:
        m = df[~df.is_pseudo_label]
        per = m.split_tag.fillna("collar").value_counts().to_dict()
        per_ok = m[~m.edge_flag].split_tag.fillna("collar").value_counts().to_dict()
        raise RuntimeError(
            f"No labelled galaxies in split(s) {empty}.\n"
            f"  GZH-matched galaxies per split: {per}\n"
            f"  ... after removing edge-truncated ones: {per_ok}\n"
            "  With one pointing, Stage 0 cuts it into stripes: look at outputs/stage0_partition/split_regions.png, "
            "lower partition.collar_arcmin, or add a second pointing.")
    return splits

In [ ]:
%%writefile src/p2p/field.py
"""From sky positions to a finished run: search MAST, download, choose train/apply mode, run the pipeline.

Used by tools/run_field.py (command line), app.py (Streamlit UI) and the notebook panel.

    from p2p.field import Target, plan, prepare, run_pipeline_stream
    targets = [Target(53.12, -27.80, "GOODS-S")]
    p = plan(root, targets)                  # search only: what would be downloaded, GZH / CANDELS coverage
    cfg_file = prepare(root, p, fresh=True)  # archive the old run, download, write config_run.yaml
    for line in run_pipeline_stream(root, cfg_file):
        print(line, end="")
"""
from __future__ import annotations

import json
import shutil
import subprocess
import sys
import time
import urllib.parse
import urllib.request
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np
import yaml

from . import archive

GZH_URL = "https://zooniverse-data.s3.amazonaws.com/galaxy-zoo-hubble/gz_hubble_main.csv.gz"
CANDELS_URL = ("https://tapvizier.cds.unistra.fr/TAPVizieR/tap/sync?REQUEST=doQuery&LANG=ADQL&FORMAT=csv&QUERY="
               + urllib.parse.quote('SELECT * FROM "J/ApJS/221/11/table3"'))


@dataclass
class Target:
    ra: float
    dec: float
    name: str = ""


def parse_coord(ra, dec) -> tuple[float, float]:
    """Degrees ('53.12', '-27.80') or sexagesimal ('03:32:28.8', '-27:48:00' / '3h32m28.8s', '-27d48m')."""
    ra, dec = str(ra).strip(), str(dec).strip()
    if any(ch in ra for ch in ": hHmM") or any(ch in dec for ch in ": dDmM'\""):
        from astropy.coordinates import SkyCoord
        import astropy.units as u
        c = SkyCoord(ra, dec, unit=(u.hourangle, u.deg))
        return float(c.ra.deg), float(c.dec.deg)
    return float(ra), float(dec)


def parse_targets(text: str) -> list:
    """One target per line: 'RA, Dec[, name]' (commas or tabs)."""
    out = []
    for line in str(text).splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = [p.strip() for p in line.replace("\t", ",").split(",")]
        ra, dec = parse_coord(parts[0], parts[1])
        out.append(Target(ra, dec, parts[2] if len(parts) > 2 else ""))
    return out


@dataclass
class Plan:
    targets: list
    instrument: str = "ACS/WFC"
    filters: object = "auto"
    choices: dict = field(default_factory=dict)      # target index -> list[archive.Choice as dict]
    gzh_counts: dict = field(default_factory=dict)   # target index -> GZH galaxies within the footprint radius
    candels_counts: dict = field(default_factory=dict)
    mode: str = "train"
    reason: str = ""

    def to_json(self):
        d = asdict(self)
        d["targets"] = [asdict(t) if not isinstance(t, dict) else t for t in self.targets]
        return d


def load_cfg(root: Path, config="config.yaml") -> dict:
    return yaml.safe_load((Path(root) / config).read_text())


# ------------------------------------------------------------------ reference catalogues
def ensure_catalogues(root: Path, log=print):
    cats = Path(root) / load_cfg(root)["paths"]["catalogs"]
    cats.mkdir(parents=True, exist_ok=True)
    if not any(cats.glob("*gz_hubble*")) and not any(cats.glob("*/*gz_hubble*")):
        log("[catalogues] downloading Galaxy Zoo: Hubble (about 25 MB) ...")
        _fetch(GZH_URL, cats / "gz_hubble_main.csv.gz")
    if not any(cats.glob("*candels*")) and not any(cats.glob("*/*candels*")) and not any(cats.glob("*k15*")) \
            and not any(cats.glob("*/*k15*")):
        log("[catalogues] downloading CANDELS Kartaltepe et al. 2015 (VizieR J/ApJS/221/11/table3) ...")
        _fetch(CANDELS_URL, cats / "candels_k15.csv")


def _fetch(url, out: Path):
    tmp = out.with_suffix(out.suffix + ".part")
    with urllib.request.urlopen(url) as r, open(tmp, "wb") as f:
        shutil.copyfileobj(r, f, length=1 << 20)
    tmp.rename(out)


def count_near(root: Path, kind: str, targets, radius_deg: float) -> dict:
    """Catalogue rows within radius_deg of each target (GZH: labelled galaxies; CANDELS: expert labels)."""
    sys.path.insert(0, str(Path(root) / "src"))
    from .catalogs import PATTERNS, read_mapped, standardise
    cfg = load_cfg(root)
    cats = Path(root) / cfg["paths"]["catalogs"]
    path = next((h for pat in PATTERNS[kind] for h in sorted(cats.glob(pat)) + sorted(cats.glob(f"*/{pat}"))
                 if h.is_file()), None)
    if path is None:
        return {i: None for i in range(len(targets))}
    mapping = {k: v for k, v in cfg["catalogs"][kind]["columns"].items() if k in ("ra", "dec")}
    df = standardise(read_mapped(path, mapping), mapping, kind)
    out = {}
    for i, t in enumerate(targets):
        d = np.hypot((df.ra.to_numpy() - t.ra) * np.cos(np.radians(t.dec)), df.dec.to_numpy() - t.dec)
        out[i] = int((d < radius_deg).sum())
    return out


# ------------------------------------------------------------------ plan / prepare / run
def plan(root, targets, filters="auto", instrument="ACS/WFC", search_radius_deg=0.03, allow_single=True,
         mode="auto", min_gzh_labels=None, log=print) -> Plan:
    root = Path(root)
    cfg = load_cfg(root).get("field_runner", {})
    min_gzh = int(min_gzh_labels if min_gzh_labels is not None else cfg.get("min_gzh_labels", 150))
    p = Plan(targets=list(targets), instrument=instrument, filters=filters)
    for i, t in enumerate(p.targets):
        ch = archive.search(t.ra, t.dec, filters, instrument, search_radius_deg, allow_single, log=log)
        p.choices[i] = [asdict(c) for c in ch]
    try:
        ensure_catalogues(root, log)
    except Exception as e:                     # no internet for catalogues: decide from what is there
        log(f"[catalogues] could not download ({e!r}); using the files already in data/catalogs")
    fp = float(cfg.get("footprint_radius_deg", 0.028))
    p.gzh_counts = count_near(root, "gzh", p.targets, fp)
    p.candels_counts = count_near(root, "candels", p.targets, fp)
    n_gzh = sum(v or 0 for v in p.gzh_counts.values())
    if mode in ("train", "apply"):
        p.mode, p.reason = mode, "chosen by the user"
    elif n_gzh >= min_gzh:
        p.mode, p.reason = "train", f"{n_gzh} Galaxy Zoo: Hubble galaxies in the field(s) (>= {min_gzh})"
    else:
        p.mode, p.reason = "apply", (f"only {n_gzh} Galaxy Zoo: Hubble galaxies in the field(s) (< {min_gzh}): "
                                     "the frozen model bundle classifies the galaxies")
    log(f"[plan] mode = {p.mode}: {p.reason}")
    for i, t in enumerate(p.targets):
        log(f"[plan] {t.name or f'target {i + 1}'} ({t.ra}, {t.dec}): {len(p.choices[i])} band(s), "
            f"GZH {p.gzh_counts.get(i)}, CANDELS {p.candels_counts.get(i)}")
    return p


def newest_bundle(root: Path) -> Path | None:
    cards = sorted(Path(root).glob("models/*/model_card.json"), key=lambda p: p.stat().st_mtime)
    return cards[-1].parent if cards else None


def archive_previous_run(root: Path, log=print) -> Path | None:
    """Move data/raw/* and outputs/ to runs/<timestamp>/ (nothing is deleted)."""
    root = Path(root)
    cfg = load_cfg(root)
    raw, outs = root / cfg["paths"]["raw"], root / cfg["paths"]["outputs"]
    files = [f for f in raw.glob("*") if f.is_file() and f.name.lower() != "readme.md"] if raw.exists() else []
    has_out = outs.exists() and any(outs.iterdir())
    if not files and not has_out:
        return None
    dest = root / "runs" / time.strftime("%Y-%m-%d_%H%M%S")
    (dest / "raw").mkdir(parents=True, exist_ok=True)
    for f in files:
        shutil.move(str(f), dest / "raw" / f.name)
    if has_out:
        shutil.move(str(outs), dest / "outputs")
        outs.mkdir(parents=True, exist_ok=True)
    log(f"[archive] previous run moved to {dest.relative_to(root)} ({len(files)} image file(s){', outputs' if has_out else ''})")
    return dest


def prepare(root, p: Plan, fresh=True, model_bundle=None, download_images=True, log=print) -> str:
    """Download what the plan chose and write config_run.yaml. Returns the config file name to run with."""
    root = Path(root)
    cfg = load_cfg(root)
    if fresh:
        archive_previous_run(root, log)
    raw = root / cfg["paths"]["raw"]
    cats = root / cfg["paths"]["catalogs"]
    if download_images:
        for i, t in enumerate(p.targets):
            ch = [archive.Choice(**c) for c in p.choices.get(i, [])]
            if not ch:
                log(f"[prepare] no images for target {i + 1} ({t.ra}, {t.dec}) - skipped")
                continue
            archive.download(ch, raw, log)
            try:
                archive.gaia_cone(t.ra, t.dec, float(cfg.get("field_runner", {}).get("gaia_radius_deg", 0.06)), cats, log)
            except Exception as e:
                log(f"[gaia] cone failed ({e!r}); Stage 2c will ask Gaia online instead")
                cfg["catalogs"]["gaia"]["query_online"] = True
    cfg.setdefault("supervised", {})["mode"] = p.mode
    if p.mode == "apply":
        b = Path(model_bundle) if model_bundle else (Path(cfg["supervised"]["model_bundle"])
                                                      if cfg["supervised"].get("model_bundle") else newest_bundle(root))
        if b is None:
            raise RuntimeError("apply mode needs a model bundle (models/<name>/). Export one from a training run: "
                               "python tools/export_model.py --name <name>")
        cfg["supervised"]["model_bundle"] = str(b if b.is_absolute() else b)
        log(f"[prepare] apply mode with model bundle {cfg['supervised']['model_bundle']}")
    (root / "config_run.yaml").write_text(
        "# Written by p2p.field.prepare() for this run - edit config.yaml, not this file.\n"
        + yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
    (root / cfg["paths"]["outputs"]).mkdir(parents=True, exist_ok=True)
    (root / cfg["paths"]["outputs"] / "field_request.json").write_text(json.dumps(p.to_json(), indent=1))
    return "config_run.yaml"


def run_pipeline_stream(root, config_file="config_run.yaml", start=None, stop=None, keep_going=False):
    """Run run_pipeline.py in a child process and yield its output line by line."""
    cmd = [sys.executable, "-u", "run_pipeline.py", "--config", config_file]
    if start:
        cmd += ["--from", str(start)]
    if stop:
        cmd += ["--to", str(stop)]
    if keep_going:
        cmd.append("--keep-going")
    proc = subprocess.Popen(cmd, cwd=str(root), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        yield line
    proc.wait()
    yield f"\n[run] finished with exit code {proc.returncode}\n"

In [ ]:
%%writefile src/p2p/fitsio.py
"""Reading HST drizzled products (_drc / _drz) and grouping them into pointings."""
from __future__ import annotations

import fnmatch
import warnings
from pathlib import Path

import numpy as np
from astropy.io import fits
from astropy.time import Time
from astropy.wcs import WCS, FITSFixedWarning
from astropy.wcs.utils import proj_plane_pixel_scales

warnings.simplefilter("ignore", FITSFixedWarning)


def find_products(raw_dir: Path, patterns) -> list[Path]:
    files = []
    for p in sorted(raw_dir.rglob("*")):
        if p.is_file() and any(fnmatch.fnmatch(p.name.lower(), pat.lower()) for pat in patterns):
            files.append(p)
    # prefer _drc over _drz for the same rootname
    by_root = {}
    for f in files:
        root = f.name.lower().replace("_drc", "").replace("_drz", "")
        if root not in by_root or "_drc" in f.name.lower():
            by_root[root] = f
    return sorted(by_root.values())


def filter_name(prim, sci) -> str:
    for hdr in (prim, sci):
        for k in ("FILTER", "FILTER1", "FILTER2"):
            v = str(hdr.get(k, "")).strip().upper()
            if v and not v.startswith("CLEAR") and v not in ("N/A", "NONE"):
                return v
    return "UNKNOWN"


def instrument_key(prim) -> str:
    inst = str(prim.get("INSTRUME", "")).upper()
    det = str(prim.get("DETECTOR", "")).upper()
    if inst == "WFC3":
        return f"WFC3_{det}" if det else "WFC3"
    return inst or "UNKNOWN"


def read_meta(path: Path) -> dict:
    """Header-only read: everything later stages need to know about a file."""
    with fits.open(path, memmap=True) as hdul:
        names = [h.name for h in hdul]
        prim = hdul[0].header
        sci_ext = "SCI" if "SCI" in names else (1 if len(hdul) > 1 else 0)
        sci = hdul[sci_ext].header
        w = WCS(sci).celestial
        nx, ny = int(sci.get("NAXIS1", 0)), int(sci.get("NAXIS2", 0))
    ok_wcs = w.has_celestial and str(sci.get("CTYPE1", "")).startswith("RA")
    scale = float(proj_plane_pixel_scales(w)[0] * 3600) if ok_wcs else np.nan
    fp = w.calc_footprint(axes=(nx, ny)) if ok_wcs else np.zeros((4, 2))
    centre = w.pixel_to_world_values((nx - 1) / 2, (ny - 1) / 2) if ok_wcs else (np.nan, np.nan)
    photflam = sci.get("PHOTFLAM", prim.get("PHOTFLAM"))
    photplam = sci.get("PHOTPLAM", prim.get("PHOTPLAM"))
    zp = float(-2.5 * np.log10(photflam) - 5 * np.log10(photplam) - 2.408) if photflam and photplam else np.nan
    date = prim.get("DATE-OBS") or sci.get("DATE-OBS")
    try:
        epoch = float(Time(date).decimalyear) if date else np.nan
    except Exception:
        epoch = np.nan
    return {
        "file": str(path), "name": path.name,
        "product_type": "drc" if "_drc" in path.name.lower() else ("drz" if "_drz" in path.name.lower() else "other"),
        "has_sci": "SCI" in names, "has_wht": "WHT" in names, "has_ctx": "CTX" in names,
        "wcs_ok": bool(ok_wcs), "naxis1": nx, "naxis2": ny,
        "instrument": instrument_key(prim), "telescope": str(prim.get("TELESCOP", "")),
        "filter": filter_name(prim, sci), "targname": str(prim.get("TARGNAME", "")),
        "proposid": prim.get("PROPOSID"), "exptime": float(prim.get("EXPTIME", sci.get("EXPTIME", np.nan))),
        "ndrizim": prim.get("NDRIZIM", sci.get("NDRIZIM")), "date_obs": str(date), "epoch": epoch,
        "bunit": str(sci.get("BUNIT", "")), "photflam": photflam, "photplam": photplam, "zp_ab": zp,
        "pixscale_arcsec": scale, "ra_centre": float(centre[0]), "dec_centre": float(centre[1]),
        "footprint": np.asarray(fp).tolist(),
    }


def load_image(path: str | Path):
    """SCI (float32), WHT (float32 or None) and the celestial WCS."""
    with fits.open(path, memmap=False) as hdul:
        names = [h.name for h in hdul]
        sci_ext = "SCI" if "SCI" in names else 1
        sci = np.asarray(hdul[sci_ext].data, dtype=np.float32)
        wht = np.asarray(hdul["WHT"].data, dtype=np.float32) if "WHT" in names else None
        w = WCS(hdul[sci_ext].header).celestial
    return sci, wht, w


def load_ctx(path: str | Path):
    """Drizzle context map (CTX): bit k set = input image k contributes to the pixel. None if absent
    or 3-D (more than 32 inputs)."""
    with fits.open(path, memmap=False) as hdul:
        names = [h.name for h in hdul]
        if "CTX" not in names or hdul["CTX"].data is None or hdul["CTX"].data.ndim != 2:
            return None
        return np.asarray(hdul["CTX"].data).astype(np.int64)


def valid_mask(sci, wht):
    m = np.isfinite(sci)
    if wht is not None:
        m &= wht > 0
    return m


def is_external(meta: dict, patterns) -> bool:
    text = (meta["name"] + " " + meta.get("targname", "")).lower()
    return any(p.lower() in text for p in patterns or [])


def _overlap_fraction(a: dict, b: dict) -> float:
    """Overlap of two footprints' RA/Dec boxes as a fraction of the SMALLER box (cos(dec)-corrected)."""
    pa, pb = np.asarray(a["footprint"]), np.asarray(b["footprint"])
    cosd = np.cos(np.deg2rad(0.5 * (pa[:, 1].mean() + pb[:, 1].mean())))
    w = min(pa[:, 0].max(), pb[:, 0].max()) - max(pa[:, 0].min(), pb[:, 0].min())
    h = min(pa[:, 1].max(), pb[:, 1].max()) - max(pa[:, 1].min(), pb[:, 1].min())
    if w <= 0 or h <= 0:
        return 0.0
    area = lambda p: np.ptp(p[:, 0]) * cosd * np.ptp(p[:, 1])
    return float(w * cosd * h / max(min(area(pa), area(pb)), 1e-12))


def group_pointings(metas: list[dict], radius_arcsec: float, min_overlap: float = 0.5) -> list[list[int]]:
    """Union-find: files whose centres are closer than radius_arcsec, OR whose footprints overlap by at
    least min_overlap of the smaller one, form one pointing. The overlap rule matters when the bands of
    one field come from different HST programmes, with centres far apart but the same sky."""
    n = len(metas)
    parent = list(range(n))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    ra = np.array([m["ra_centre"] for m in metas])
    dec = np.array([m["dec_centre"] for m in metas])
    for i in range(n):
        for j in range(i + 1, n):
            d = np.hypot((ra[i] - ra[j]) * np.cos(np.deg2rad(dec[i])), dec[i] - dec[j]) * 3600
            if d < radius_arcsec or _overlap_fraction(metas[i], metas[j]) >= min_overlap:
                parent[find(i)] = find(j)
    groups = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(i)
    # stable order: by RA of the group centre
    out = list(groups.values())
    out.sort(key=lambda g: (np.mean(ra[g]), np.mean(dec[g])))
    return out


def bbox_of(metas_in_group: list[dict]):
    pts = np.vstack([np.asarray(m["footprint"]) for m in metas_in_group])
    return float(pts[:, 0].min()), float(pts[:, 0].max()), float(pts[:, 1].min()), float(pts[:, 1].max())

In [ ]:
%%writefile src/p2p/inference.py
"""The supervised model of the pipeline (feature-level fusion) and its frozen, calibrated inference path.

    galaxy cutout (128 x 128, 3 bands)  -> ImageNet input (R, G, B = F775W, F606W, F435W; asinh; 224 x 224)
        -> ResNet-18 (ImageNet, frozen)        -> 512 features      \
        -> EfficientNet-B0 (ImageNet, frozen)  -> 1,280 features     >  ENSEMBLE (fusion): one 1,806-feature vector,
    + 14 physical features (Pass 2)            -> 14 features       /   block weights w1, w2, w3
        -> XGBoost (5 morphology classes)  +  XGBoost (merger yes / no)  -> calibration (Stage 4c)

Used by Stages 4a (feature extraction), 4b (training), 4c, 4d, 6a, 6c, 7b and 8.
"""
from __future__ import annotations

import time

import joblib
import numpy as np

from .context import Ctx, load_json
from .features import imagenet_input, load_cutouts, physical_matrix
from .nets import N_CLASSES, logit, softmax_np

CNN_BLOCKS = (("resnet18", 512), ("efficientnet_b0", 1280))
BLOCKS = ("resnet18", "efficientnet_b0", "physical")


# ================================================================ Stage 4a: frozen CNN features
def build_extractors(pretrained=True):
    """The two frozen ImageNet networks (no training: BatchNorm uses its ImageNet statistics, Dropout is off)."""
    from .nets import EfficientNetB0, ResNet18, frozen
    return {"resnet18": frozen(ResNet18(in_ch=3, pretrained=pretrained)),
            "efficientnet_b0": frozen(EfficientNetB0(pretrained=pretrained))}


def extract_cnn_features(ctx: Ctx, cutout_index, nets=None, log=None):
    """(n,) cutout rows of Stage 1d's cutouts.npy -> (n, 1792) float32: [ResNet-18 512 | EfficientNet-B0 1280]."""
    import torch
    cfg = ctx.cfg["supervised"]
    size, bs = int(cfg.get("input_size", 224)), int(cfg.get("feature_batch_size", 64))
    nets = nets or build_extractors(bool(cfg.get("pretrained", True)))
    dev = ctx.device()
    for m in nets.values():
        m.to(dev)
    cut = load_cutouts(ctx)
    idx = np.asarray(cutout_index, int)
    out = np.zeros((len(idx), sum(n for _, n in CNN_BLOCKS)), np.float32)
    t0 = time.time()
    with torch.no_grad():
        for s in range(0, len(idx), bs):
            rows = idx[s:s + bs]
            order = np.argsort(rows)                                   # sorted reads from the memory map
            x = imagenet_input(cut[rows[order]], size).to(dev)
            f = torch.cat([nets[k](x) for k, _ in CNN_BLOCKS], 1).cpu().numpy()
            out[s + order] = f
            if log and (s // bs) % 50 == 0:
                log(f"  [features] {min(s + bs, len(idx)):,} / {len(idx):,} cutouts ({time.time() - t0:.0f} s)")
    return out


def feature_names(physical):
    return ([f"resnet18_{i}" for i in range(512)] + [f"efficientnet_b0_{i}" for i in range(1280)] + list(physical))


def block_slices(n_physical):
    return {"resnet18": slice(0, 512), "efficientnet_b0": slice(512, 1792),
            "physical": slice(1792, 1792 + n_physical)}


def block_feature_weights(weights, n_physical):
    """ENSEMBLE weights w1, w2, w3 (sum 1) -> one sampling weight per feature: w_block / features in the block, so a
    block's share of the features each tree may use is w_block, whatever its size. (Multiplying the features
    themselves by w would do nothing: a tree split is a threshold, unchanged by scaling a feature.)"""
    sl = block_slices(n_physical)
    fw = np.zeros(1792 + n_physical)
    for b, w in zip(BLOCKS, weights):
        n = sl[b].stop - sl[b].start
        fw[sl[b]] = max(float(w), 1e-6) / n
    return fw


def fusion_matrix(ctx: Ctx, df, physical):
    """The 1,806-feature vector of every row of df: CNN features from Stage 4a + the physical columns."""
    ids = np.load(ctx.require("4a", "object_id.npy"))
    F = np.load(ctx.require("4a", "cnn_features.npy"), mmap_mode="r")
    pos = {int(o): i for i, o in enumerate(ids)}
    rows = np.array([pos.get(int(o), -1) for o in df["object_id"]])
    cnn = np.zeros((len(df), F.shape[1]), np.float32)
    have = rows >= 0
    cnn[have] = F[rows[have]]
    if (~have).any():                                  # a galaxy Stage 4a did not see: extract it now
        cnn[~have] = extract_cnn_features(ctx, df["cutout_index"].to_numpy()[~have])
    return np.hstack([cnn, physical_matrix(df, physical)])


# ================================================================ XGBoost members (Stage 4b)
class XGBMember:
    """One XGBoost classifier (5 morphology classes, or merger yes / no) with balanced sample weights.
    Runs in a separate process (p2p.xgb_worker): next to PyTorch on macOS XGBoost can crash Python.
    Classes missing from the training data are mapped back to all output columns."""

    def __init__(self, n_classes, seed=0, params=None, feature_weights=None):
        self.n_classes, self.seed, self.params = int(n_classes), seed, dict(params or {})
        self.feature_weights = None if feature_weights is None else np.asarray(feature_weights, float)
        self.classes_, self.raw, self.n_estimators = None, None, None

    @staticmethod
    def _prepare(y):
        classes = np.unique(y)
        remap = {c: i for i, c in enumerate(classes)}
        yy = np.array([remap[c] for c in y])
        counts = np.bincount(yy)
        w = (len(yy) / (len(counts) * np.maximum(counts, 1)))[yy]          # balanced sample weights
        return classes, remap, yy, w

    def cv_curve(self, X, y, Xva, yva, grid):
        """Held-out probabilities (len(grid), n_va, n_classes) after each tree count in grid + history."""
        from . import xgb_worker
        classes, remap, yy, w = self._prepare(y)
        yv = np.array([remap.get(c, -1) for c in yva])
        p, history = xgb_worker.run_isolated(xgb_worker.fit_curve, np.asarray(X, np.float32), yy, w,
                                             np.asarray(Xva, np.float32), yv, len(classes), list(grid), self.seed,
                                             self.params, self.feature_weights)
        out = np.zeros((len(grid), len(Xva), self.n_classes))
        out[:, :, classes] = p
        return out / out.sum(2, keepdims=True), history

    def fit(self, X, y, n_estimators):
        from . import xgb_worker
        self.classes_, _, yy, w = self._prepare(y)
        self.n_estimators = int(n_estimators)
        self.raw = xgb_worker.run_isolated(xgb_worker.fit_fixed, np.asarray(X, np.float32), yy, w, self.n_estimators,
                                           len(self.classes_), self.seed, self.params, self.feature_weights)
        return self

    def predict_proba(self, X):
        from . import xgb_worker
        out = np.zeros((len(X), self.n_classes))
        out[:, self.classes_] = xgb_worker.run_isolated(xgb_worker.predict, self.raw, np.asarray(X, np.float32))
        return out / out.sum(1, keepdims=True)

    def contribs(self, X):
        """TreeSHAP values (n, outputs, features)."""
        from . import xgb_worker
        return xgb_worker.run_isolated(xgb_worker.contribs, self.raw, np.asarray(X, np.float32))

    def summary(self, names):
        from . import xgb_worker
        return xgb_worker.run_isolated(xgb_worker.summary, self.raw, list(names), self.params)


def xgb_cv(X, y, folds, grid, n_classes, seed, params, feature_weights=None):
    """5-fold CV of one XGBoost configuration: OOF probabilities after every tree count in grid
    (len(grid), n, n_classes) and the per-fold histories."""
    oof = np.zeros((len(grid), len(y), n_classes))
    hist = []
    for tr, va in folds:
        m = XGBMember(n_classes, seed, params, feature_weights)
        oof[:, va], h = m.cv_curve(X[tr], y[tr], X[va], y[va], grid)
        hist.append(h)
    return oof, hist


# ================================================================ frozen inference path
def load_fusion(ctx: Ctx):
    """The trained Stage 4b models + Stage 4c calibration."""
    m = {"meta": load_json(ctx.require("4b", "fusion_meta.json")),
         "morph": joblib.load(ctx.require("4b", "xgb_morphology.joblib")),
         "merger": joblib.load(ctx.require("4b", "xgb_merger.joblib")) if ctx.path("4b", "xgb_merger.joblib").exists()
         else None}
    p = ctx.path("4c", "temperature.json")
    m["calibration"] = load_json(p) if p.exists() else None
    return m


def predict_galaxies(ctx: Ctx, df, models=None, calibrated=True):
    """Class probabilities (n, 5) and merger probability (n,) of the galaxies in df (rows of galaxy_table)."""
    models = models or load_fusion(ctx)
    X = fusion_matrix(ctx, df, models["meta"]["physical_columns"])
    P = models["morph"].predict_proba(X)
    pm = models["merger"].predict_proba(X)[:, 1] if models["merger"] is not None else \
        np.full(len(df), models["meta"].get("merger_base_rate", 0.0))
    cal = models["calibration"]
    if calibrated and cal:
        P = softmax_np(np.log(np.clip(P, 1e-12, 1)) / cal["T_morph"])
        pm = 1 / (1 + np.exp(-(cal.get("merger_platt_a", 1.0) * logit(pm) + cal.get("merger_platt_b", 0.0))))
    return {"morph": P, "merger": pm, "X": X}


__all__ = ["BLOCKS", "CNN_BLOCKS", "N_CLASSES", "XGBMember", "block_feature_weights", "block_slices",
           "build_extractors", "extract_cnn_features", "feature_names", "fusion_matrix", "load_fusion",
           "predict_galaxies", "xgb_cv"]

In [ ]:
%%writefile src/p2p/modes.py
"""Train vs apply mode of the supervised part (Stages 3a-7c).

train (default): the field has Galaxy Zoo: Hubble labels; Stages 4a-4d train the models on its train
                 split, calibrate on dev and test once on test (the original design).
apply          : the field has no (or too few) GZH labels - most HST fields outside the GZH surveys.
                 Stage 4a extracts the frozen CNN features as usual; Stages 4b-4c copy a MODEL BUNDLE (the XGBoost
                 models, ensemble weights and calibration) exported
                 from a training run (tools/export_model.py), Stage 4d is skipped, and every galaxy is
                 classified by that frozen model in Stage 8. If the field does have some GZH or CANDELS labels,
                 Stages 6a and 7a-7c use ALL of them as an out-of-field test (the model never saw this field).
"""
from __future__ import annotations

import json
import shutil
from pathlib import Path

# Stage 4a needs no bundle: its CNNs are the frozen ImageNet networks (only the band set must match, check_bands).
BUNDLE_FILES = {"4b": ["xgb_morphology.joblib", "xgb_merger.joblib", "fusion_meta.json", "dev_metrics.json",
                       "cv_metrics.json"],
                "4c": ["temperature.json"]}
BUNDLE_DIRS = {"4b": "stage4b_xgboost_fusion", "4c": "stage4c_calibration"}


def is_apply(ctx) -> bool:
    return str(ctx.cfg.get("supervised", {}).get("mode", "train")).lower() == "apply"


def bundle_dir(ctx) -> Path:
    b = ctx.cfg.get("supervised", {}).get("model_bundle")
    if not b:
        raise RuntimeError("supervised.mode is 'apply' but supervised.model_bundle is empty in config.yaml. "
                           "Export one from a training run: python tools/export_model.py --name <name>")
    p = Path(b)
    p = p if p.is_absolute() else ctx.root / p
    if not (p / "model_card.json").exists():
        raise RuntimeError(f"model bundle {p} not found (no model_card.json). "
                           "Export one from a training run: python tools/export_model.py --name <name>")
    return p


def card(ctx) -> dict:
    return json.loads((bundle_dir(ctx) / "model_card.json").read_text())


def install_bundle(ctx, key: str) -> dict:
    """Copy the bundle's files for stage `key` (4b or 4c) into this run's stage folder."""
    src = bundle_dir(ctx) / BUNDLE_DIRS[key]
    out = ctx.stage_dir(key)
    copied = []
    for f in BUNDLE_FILES[key]:
        if (src / f).exists():
            shutil.copy2(src / f, out / f)
            copied.append(f)
        elif (out / f).exists():
            (out / f).unlink()             # a leftover of an earlier training run must not mix with the bundle
    c = card(ctx)
    where = "; ".join(f"{p} at RA {v['centre'][0]:.3f} Dec {v['centre'][1]:+.3f}" for p, v in c.get("trained_on", {}).items()
                      if v.get("centre"))
    ctx.log.info(f"[Stage {key}] APPLY mode: frozen model bundle '{c.get('name')}' (trained on {where or '?'}, "
                 f"{c.get('n_train_labels', '?')} labels, bands {c.get('bands')}) -> {', '.join(copied)}")
    return {"mode": "apply", "bundle": c.get("name"), "files": copied}


def check_bands(ctx):
    """The CNN input (R, G, B) and the colours expect the bundle's band set, in the same order."""
    from .context import load_json
    bands = load_json(ctx.require("1a", "pointings.json"))["bands"]
    want = card(ctx).get("bands", [])
    if len(bands) != len(want):
        raise RuntimeError(f"apply mode: this field has {len(bands)} bands {bands}, the model was trained on "
                           f"{len(want)} bands {want}. Download the same number of filters (see tools/run_field.py).")
    if bands != want:
        ctx.log.info(f"[apply] NOTE: bands {bands} stand in for the model's {want} (channel by channel, blue to red). "
                     "Colours and the image branch see slightly different passbands: treat the result as approximate.")

In [ ]:
%%writefile src/p2p/morphology.py
"""Morphology measurement shared by Stage 1e (Pass 1) and Stage 5b (Pass 2).

statmorph (Rodriguez-Gomez et al. 2019) measures, for one source:
  non-parametric : Gini, M20, concentration C, asymmetry A, smoothness S,
                   multimode / intensity / deviation (MID), shape asymmetry
  parametric     : single Sersic fit (n, r_eff, ellipticity), convolved with
                   the PSF passed in -> this is where Pass 1 and Pass 2 differ.
If statmorph is missing or fails, simple NumPy estimators are used (flag = 4).
Masked pixels: other sources (segmentation stamp value 2) and cosmic-ray pixels
(value 3, Stage 1c). statmorph's -99 failure codes are stored as NaN, and a Sersic
fit that ends at a parameter limit (morphology.sersic_n_bounds,
sersic_rhalf_max_px) gets flag_sersic = 1.
"""
from __future__ import annotations

import warnings

import numpy as np
from scipy.signal import fftconvolve

try:
    import statmorph
    _HAVE_STATMORPH = True
except Exception:
    _HAVE_STATMORPH = False

FEATURES = ["gini", "m20", "concentration", "asymmetry", "smoothness", "sersic_n", "sersic_rhalf",
            "sersic_ellip", "r_half", "r20", "r80", "r_petro", "ellipticity_sm", "shape_asymmetry",
            "multimode", "intensity", "deviation", "gini_m20_merger", "sn_per_pixel", "flag", "flag_sersic"]

_STATMORPH_MAP = {"gini": "gini", "m20": "m20", "concentration": "concentration", "asymmetry": "asymmetry",
                  "smoothness": "smoothness", "sersic_n": "sersic_n", "sersic_rhalf": "sersic_rhalf",
                  "sersic_ellip": "sersic_ellip", "r_half": "rhalf_circ", "r20": "r20", "r80": "r80",
                  "r_petro": "rpetro_circ", "ellipticity_sm": "ellipticity_asymmetry",
                  "shape_asymmetry": "shape_asymmetry", "multimode": "multimode", "intensity": "intensity",
                  "deviation": "deviation", "gini_m20_merger": "gini_m20_merger", "sn_per_pixel": "sn_per_pixel",
                  "flag": "flag", "flag_sersic": "flag_sersic"}


def morph_image(cube):
    """Sum of the PSF-matched bands -> one image with the best signal-to-noise."""
    return np.asarray(cube, np.float32).sum(0)


def richardson_lucy(img, psf, iters=10):
    obs = np.clip(img, 0, None) + 1e-6
    est = np.full_like(obs, obs.mean())
    psf_m = psf[::-1, ::-1]
    for _ in range(iters):
        est *= fftconvolve(obs / (fftconvolve(est, psf, mode="same") + 1e-12), psf_m, mode="same")
    return est


def _numpy_fallback(img, target):
    pix = np.sort(np.abs(img[target]))
    n, tot = pix.size, pix.sum()
    gini = float(np.sum((2 * np.arange(1, n + 1) - n - 1) * pix) / (tot * (n - 1))) if n > 1 and tot > 0 else np.nan
    h, w = img.shape
    yy, xx = np.mgrid[:h, :w]
    r = np.hypot(yy - (h - 1) / 2, xx - (w - 1) / 2).ravel()
    o = np.argsort(r)
    cog = np.cumsum(np.clip(np.where(target, img, 0).ravel()[o], 0, None))
    out = {k: np.nan for k in FEATURES}
    if cog[-1] > 0:
        cog = cog / cog[-1]
        r20, r50, r80 = (r[o][np.searchsorted(cog, q)] for q in (0.2, 0.5, 0.8))
        out.update(r20=r20, r_half=r50, r80=r80, concentration=5 * np.log10(r80 / r20) if r20 > 0 else np.nan)
    rot = np.where(target, img, 0)[::-1, ::-1]
    den = np.abs(np.where(target, img, 0)).sum()
    out.update(gini=gini, asymmetry=float(np.abs(np.where(target, img, 0) - rot).sum() / (2 * den)) if den > 0 else np.nan,
               flag=4, flag_sersic=1)
    return out


def clean_result(res, n_bounds=(0.05, 8.0), rhalf_max=64.0):
    """-99 (statmorph's 'could not measure') -> NaN; a Sersic fit stuck at a limit -> flag_sersic = 1."""
    out = {k: (np.nan if (k not in ("flag", "flag_sersic") and v is not None and np.isfinite(v) and v <= -98.9)
               else v) for k, v in res.items()}
    n, rh = out.get("sersic_n", np.nan), out.get("sersic_rhalf", np.nan)
    stuck = (np.isfinite(n) and (n <= n_bounds[0] or n >= n_bounds[1])) or (np.isfinite(rh) and rh > rhalf_max)
    if stuck or not np.isfinite(n):
        out["flag_sersic"] = max(1.0, float(np.nan_to_num(out.get("flag_sersic", 1.0), nan=1.0)))
    return out


def measure_one(cube, seg, gain, psf=None, mode="psf_fit", rl_iters=10, sersic_maxiter=500,
                n_bounds=(0.05, 8.0), rhalf_max=64.0):
    return clean_result(_measure_one(cube, seg, gain, psf, mode, rl_iters, sersic_maxiter), n_bounds, rhalf_max)


def _measure_one(cube, seg, gain, psf=None, mode="psf_fit", rl_iters=10, sersic_maxiter=500):
    img = morph_image(cube)
    target = seg == 1                                   # cosmic-ray pixels (3) are NOT part of the target
    if target.sum() < 5:
        return {**{k: np.nan for k in FEATURES}, "flag": 4, "flag_sersic": 1}
    mask = (seg == 2) | (seg == 3) | ((img == 0) & ~target)
    use_psf = psf
    if mode == "deconvolve" and psf is not None:
        dil = np.zeros_like(target)
        from scipy.ndimage import binary_dilation
        dil = binary_dilation(target, iterations=3)
        img = np.where(dil, richardson_lucy(np.where(dil, img, 0), psf, rl_iters), img)
        use_psf = None
    if not _HAVE_STATMORPH:
        return _numpy_fallback(img, target)
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            m = statmorph.source_morphology(img, target.astype(int), mask=mask, gain=gain, psf=use_psf,
                                            sersic_maxiter=sersic_maxiter)[0]
        return {k: float(getattr(m, v)) if getattr(m, v, None) is not None else np.nan
                for k, v in _STATMORPH_MAP.items()}
    except Exception:
        return _numpy_fallback(img, target)


def measure_batch(cut_path, seg_path, indices, gains, psfs, mode, rl_iters, sersic_maxiter,
                  n_bounds=(0.05, 8.0), rhalf_max=64.0):
    """Worker for joblib: opens the memory-mapped arrays itself (nothing big is pickled)."""
    try:
        from astropy import log as _alog
        _alog.setLevel("ERROR")                    # statmorph reports through astropy's logger
    except Exception:
        pass
    warnings.simplefilter("ignore")
    cut = np.load(cut_path, mmap_mode="r")
    seg = np.load(seg_path, mmap_mode="r")
    out = []
    for i, g, p in zip(indices, gains, psfs):
        out.append(measure_one(cut[i].astype(np.float32), np.asarray(seg[i]), g, p, mode, rl_iters, sersic_maxiter,
                               n_bounds, rhalf_max))
    return out


def run_parallel(ctx, cut_path, seg_path, indices, gains, psfs, mode="psf_fit"):
    from joblib import Parallel, delayed
    cfg = ctx.cfg["morphology"]
    n_jobs = ctx.n_jobs
    chunks = np.array_split(np.arange(len(indices)), max(1, min(len(indices), n_jobs * 4)))
    jobs = [delayed(measure_batch)(str(cut_path), str(seg_path), [indices[j] for j in ch], [gains[j] for j in ch],
                                   [psfs[j] for j in ch], mode, cfg["rl_iterations"], cfg["sersic_maxiter"],
                                   tuple(cfg.get("sersic_n_bounds", (0.05, 8.0))),
                                   float(cfg.get("sersic_rhalf_max_px", 64.0)))
            for ch in chunks if len(ch)]
    res = Parallel(n_jobs=n_jobs, verbose=5)(jobs)
    return [r for part in res for r in part]

In [ ]:
%%writefile src/p2p/multiband.py
"""Multi-band checks shared by Stage 1c (and read by later stages through sources.csv).

Three problems of single-exposure HST products are handled here:

1. Band registration. Bands taken by different programmes can carry different
   astrometric solutions (e.g. an 'a priori' pointing-only WCS against one refined
   on a star catalogue). `registration_catalog` detects sources on a MEDIAN-FILTERED
   image: a single exposure holds tens of thousands of cosmic rays, which are
   brighter than most galaxies and would fill any 'brightest sources' list, but
   they are only 1-3 px wide and a 5x5 median erases them while galaxies survive.
   `measure_offset` finds the shift from the pairs of the two lists and reports how
   far the peak stands above the random-pair level; `shift_wcs` removes it.
2. Cosmic rays (source level). Cosmic rays hit each exposure at random places,
   so a real source is also seen in a second band of the same sky, a cosmic ray
   almost never. `segment_photometry` measures every detection segment in every
   band (S/N from the background rms) and the centroid of the band's significant
   pixels on a 3x3 MEDIAN-FILTERED copy (cosmic rays erased, compact sources kept).
   A band CONFIRMS a source when its S/N >= detection.confirm_snr AND its light is
   centred where the detection-band light is (within detection.confirm_max_shift_px):
   in a single exposure a cosmic ray of the other band often lands inside a large
   footprint (high S/N at the wrong place), and cosmic rays swallowed by a galaxy's
   detection footprint would pull a raw centroid away.
3. Cosmic rays (pixel level). Inside a cutout, a pixel that is far brighter in
   one band than the other bands predict (given the source's own colour) is a
   cosmic-ray hit in that band. `cr_pixel_mask` finds them; Stage 1c replaces
   them by the prediction and marks them 3 in the segmentation stamp, so
   statmorph masks them.
"""
from __future__ import annotations

import numpy as np
from scipy import ndimage
from scipy.spatial import cKDTree


# ---------------------------------------------------------------- 1. registration
def tangent_arcsec(ra, dec, ra0, dec0):
    """Small-field tangent-plane offsets (arcsec) from (ra0, dec0): x = dRA cos(dec), y = dDec."""
    ra, dec = np.asarray(ra, float), np.asarray(dec, float)
    return (ra - ra0) * np.cos(np.deg2rad(dec0)) * 3600.0, (dec - dec0) * 3600.0


def registration_catalog(sub, rms, valid, wcs, cfg, median_px=5, nsigma=5.0, npixels=25, n_max=3000, max_ellip=0.8):
    """RA/Dec of the brightest extended sources of a median-filtered image (cosmic rays removed)."""
    from .compat import SourceCatalog, n_labels, values
    from .detection import detect_and_deblend
    med = ndimage.median_filter(np.where(valid, sub, 0.0).astype(np.float32), size=median_px)
    sky = med[valid]
    if sky.size > 2_000_000:
        sky = sky[:: sky.size // 2_000_000]
    mad = 1.4826 * np.median(np.abs(sky - np.median(sky)))           # noise of the median image
    rms_m = rms * (mad / max(float(np.nanmedian(rms)), 1e-12))
    segm, conv = detect_and_deblend(med, rms_m, valid, {**cfg, "nsigma": nsigma, "npixels": npixels},
                                    deblend_sources=False)
    if n_labels(segm) == 0:
        return np.zeros(0), np.zeros(0)
    cat = SourceCatalog(med, segm, convolved_data=conv)
    x, y = values(cat, "x_centroid"), values(cat, "y_centroid")
    flux, ell = values(cat, "segment_flux"), values(cat, "ellipticity")
    ok = np.where(np.isfinite(x) & np.isfinite(flux) & (ell < max_ellip))[0]
    ok = ok[np.argsort(-flux[ok])][:n_max]
    return wcs.pixel_to_world_values(x[ok], y[ok])


def registration_sources(x, y, snr, semimajor, ellipticity, min_snr=20.0, min_sigma_px=1.2, max_ellip=0.7, n_max=3000):
    """Indices of sources good for registration: bright and EXTENDED (sharp cosmic rays and long
    thin tracks would match random partners in a crowded single exposure)."""
    ok = (np.isfinite(x) & np.isfinite(y) & (np.asarray(snr) >= min_snr) & (np.asarray(semimajor) >= min_sigma_px)
          & (np.asarray(ellipticity) < max_ellip))
    idx = np.where(ok)[0]
    return idx[np.argsort(-np.asarray(snr)[idx])][:n_max]


def measure_offset(ref_xy, band_xy, search_arcsec=3.0, bin_arcsec=0.05, refine_arcsec=0.15):
    """Median offset (band - reference, arcsec) from all pairs within `search_arcsec`.
    The peak of the 2-D histogram of pair offsets is robust to the random pairs of a crowded
    field; the median of the pairs within `refine_arcsec` of the peak refines it.
    Returns (dx, dy, n_pairs, scatter_arcsec, significance), significance = n_pairs / the number of
    random pairs expected in the same refine circle (a real offset gives >> 5)."""
    if len(ref_xy) == 0 or len(band_xy) == 0:
        return np.nan, np.nan, 0, np.nan, 0.0
    tree = cKDTree(band_xy)
    pairs = tree.query_ball_point(ref_xy, search_arcsec)
    d = np.array([band_xy[j] - ref_xy[i] for i, js in enumerate(pairs) for j in js], float).reshape(-1, 2)
    if len(d) < 5:
        return np.nan, np.nan, 0, np.nan, 0.0
    edges = np.arange(-search_arcsec, search_arcsec + bin_arcsec, bin_arcsec)
    H, ex, ey = np.histogram2d(d[:, 0], d[:, 1], bins=[edges, edges])
    H = ndimage.uniform_filter(H, 3)                     # 3x3 smoothing: the peak is a few bins wide
    i, j = np.unravel_index(np.argmax(H), H.shape)
    peak = np.array([0.5 * (ex[i] + ex[i + 1]), 0.5 * (ey[j] + ey[j + 1])])
    for _ in range(3):                                   # iterate the refinement around the median
        near = np.hypot(*(d - peak).T) <= refine_arcsec
        if near.sum() < 3:
            break
        peak = np.median(d[near], axis=0)
    near = np.hypot(*(d - peak).T) <= refine_arcsec
    scatter = float(1.4826 * np.median(np.hypot(*(d[near] - peak).T))) if near.any() else np.nan
    expected = len(d) * refine_arcsec ** 2 / search_arcsec ** 2          # random pairs in the refine circle
    return float(peak[0]), float(peak[1]), int(near.sum()), scatter, float(near.sum() / max(expected, 1e-9))


def shift_wcs(wcs, dx_arcsec, dy_arcsec):
    """Copy of `wcs` whose sky positions are moved by (-dx, -dy) arcsec: the band's positions were
    (dx, dy) too large (dx = dRA cos(dec)). Shifting CRVAL moves a small field rigidly."""
    w = wcs.deepcopy()
    dec0 = w.wcs.crval[1]
    w.wcs.crval = [w.wcs.crval[0] - dx_arcsec / 3600.0 / np.cos(np.deg2rad(dec0)), dec0 - dy_arcsec / 3600.0]
    w.wcs.set()
    return w


# ---------------------------------------------------------------- 2. source-level confirmation
def median_image(sub, valid, size=3):
    """Median-filtered copy and the factor by which its noise is lower than the input's (from the MAD)."""
    img = np.where(valid, sub, 0.0).astype(np.float32)
    med = ndimage.median_filter(img, size=size)
    a, m = img[valid], med[valid]
    step = max(1, a.size // 2_000_000)
    mad = lambda x: 1.4826 * np.median(np.abs(x - np.median(x)))
    return med, float(mad(m[::step]) / max(mad(a[::step]), 1e-12))


def segment_photometry(labels, det_wcs, band, det_pixarea, max_label, corr=1.0, sig_level=2.0):
    """Flux, error and coverage of every detection segment measured in another band.

    labels   : detection-band segmentation image (0 = sky)
    band     : dict with 'sub', 'rms', 'val', 'wcs', 'pixarea' of the band
    Each detection pixel samples the band at the same sky position (nearest pixel); the flux is
    scaled by the pixel-area ratio. The centroid (detection-band pixel units) is weighted by the band's
    MEDIAN-FILTERED pixels (band['med'], noise factor band['med_noise']) above sig_level sigma only, so
    cosmic rays inside the footprint cannot move it. Returns arrays indexed by label (0..max_label):
    flux, err, coverage, cx, cy (NaN where the band has no significant pixel)."""
    ys, xs = np.nonzero(labels)
    lab = labels[ys, xs]
    n_all = np.bincount(lab, minlength=max_label + 1).astype(float)
    if band is None:
        z = np.zeros(max_label + 1)
        nan = np.full(max_label + 1, np.nan)
        return z, nan, z, nan, nan
    ra, dec = det_wcs.pixel_to_world_values(xs, ys)
    bx, by = band["wcs"].world_to_pixel_values(ra, dec)
    del ra, dec
    xi, yi = np.round(bx).astype(np.int64), np.round(by).astype(np.int64)
    h, w = band["sub"].shape
    ok = (xi >= 0) & (yi >= 0) & (xi < w) & (yi < h)
    ok[ok] = band["val"][yi[ok], xi[ok]]
    scale = det_pixarea / band["pixarea"]
    v = np.zeros(len(lab))
    var = np.zeros(len(lab))
    v[ok] = band["sub"][yi[ok], xi[ok]] * scale
    var[ok] = (band["rms"][yi[ok], xi[ok]] * scale * corr) ** 2
    flux = np.bincount(lab, weights=v, minlength=max_label + 1)
    err = np.sqrt(np.bincount(lab, weights=var, minlength=max_label + 1))
    cov = np.bincount(lab, weights=ok.astype(float), minlength=max_label + 1) / np.maximum(n_all, 1)
    vm = np.zeros(len(lab))
    vm[ok] = band["med"][yi[ok], xi[ok]] * scale
    wgt = np.where(vm > sig_level * np.sqrt(var) * band["med_noise"], vm, 0.0)
    wsum = np.bincount(lab, weights=wgt, minlength=max_label + 1)
    with np.errstate(invalid="ignore", divide="ignore"):
        cx = np.where(wsum > 0, np.bincount(lab, weights=wgt * xs, minlength=max_label + 1) / wsum, np.nan)
        cy = np.where(wsum > 0, np.bincount(lab, weights=wgt * ys, minlength=max_label + 1) / wsum, np.nan)
    return flux, np.where(err > 0, err, np.nan), cov, cx, cy


# ---------------------------------------------------------------- 3. pixel-level cosmic-ray mask
_RING = np.array([[1, 1, 1], [1, 0, 1], [1, 1, 1]], bool)


def cr_pixel_mask(cube, valid, seg, sigma, flux, snr, nsigma=5.0, ratio=4.0, min_ref_snr=3.0, sharp_ratio=2.0):
    """Cosmic-ray pixels of one cutout, per band, and the replacement values.

    cube  : (bands, S, S) raw stamps (e-/s per output pixel)
    valid : (bands, S, S) bool coverage
    seg   : (S, S) 0 sky / 1 target / 2 other source
    sigma : (bands, S, S) per-pixel noise (background + source Poisson)
    flux, snr : per-band segment flux and S/N of the TARGET (fallback colour for faint targets)
    A pixel of band b is a hit when it is > nsigma above the value that EVERY usable other band c
    predicts (v_c x R_bc) and more than `ratio` times that prediction. R_bc, the target's colour, is the
    MEDIAN pixel ratio v_b / v_c over the target pixels bright in c (> 5 sigma): segment fluxes are no good
    here, because cosmic-ray hits inside a segment inflate them and the whole galaxy would then look too
    bright in the other band. Band c is usable when the target is detected in it (snr >= min_ref_snr) and
    it covers the pixel. Pixels of other sources (seg == 2) are not tested: their colour is not the
    target's, and they are masked anyway.
    A hit must also be SHARP in its own band: more than nsigma above the median of its 8 neighbours and
    more than sharp_ratio times that median. Without this, the smooth red halo of ACS F814W around bright
    stars (scattered light the blue bands do not have) is 'too bright for its colour' and was masked.
    Returns (mask (bands, S, S) bool, replacement (bands, S, S))."""
    nb = cube.shape[0]
    mask = np.zeros(cube.shape, bool)
    repl = np.zeros(cube.shape, np.float32)
    test_area = seg != 2
    target = seg == 1
    for b in range(nb):
        preds, errs = [], []
        for c in range(nb):
            if c == b or not (np.isfinite(snr[c]) and snr[c] >= min_ref_snr and flux[c] > 0 and np.isfinite(flux[b])):
                continue
            bright = target & valid[b] & valid[c] & (cube[c] > 5 * sigma[c])
            if bright.sum() >= 5:
                r = float(np.median(cube[b][bright] / cube[c][bright]))
            else:
                r = max(flux[b], 0.0) / flux[c]
            r = max(r, 0.0)
            preds.append(np.where(valid[c], cube[c] * r, np.nan))
            errs.append(np.where(valid[c], sigma[c] * r, np.nan))
        if not preds:
            continue
        P, E = np.stack(preds), np.stack(errs)
        excess = cube[b][None] - P
        tot = np.sqrt(sigma[b][None] ** 2 + E ** 2)
        hit_c = (excess > nsigma * tot) & (cube[b][None] > ratio * np.clip(P, 0, None))
        usable = np.isfinite(P)
        hit = np.all(hit_c | ~usable, axis=0) & usable.any(axis=0)       # every usable band disagrees
        hit &= valid[b] & test_area & (cube[b] > nsigma * sigma[b])
        if hit.any():
            ring = ndimage.median_filter(cube[b], footprint=_RING, mode="nearest")
            hit &= (cube[b] - ring > nsigma * sigma[b]) & (cube[b] > sharp_ratio * np.clip(ring, 0, None))
        if hit.any():
            hit = ndimage.binary_dilation(hit, iterations=1) & valid[b] & test_area   # CR wings
            mask[b] = hit
            repl[b] = np.nanmedian(np.where(usable, P, np.nan), axis=0)
    return mask, np.nan_to_num(repl)


# ---------------------------------------------------------------- exposures per pixel
def popcount(ctx_values):
    """Number of set bits of the drizzle context (CTX) = number of input images at that pixel.
    For ACS/WFC the two chips of ONE exposure never overlap, so this counts exposures."""
    v = np.asarray(ctx_values).astype(np.uint64)
    n = np.zeros(v.shape, np.int64)
    while np.any(v):
        n += (v & np.uint64(1)).astype(np.int64)
        v = v >> np.uint64(1)
    return n

In [ ]:
%%writefile src/p2p/nets.py
"""Neural networks: the autoencoders trained in Stages 1b and 2a, and the two frozen ImageNet feature extractors
of Stage 4a (ResNet-18, EfficientNet-B0); metrics shared by Stages 4-6."""
from __future__ import annotations

import copy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from sklearn.metrics import average_precision_score, balanced_accuracy_score, f1_score

MORPH_CLASSES = ("Smooth", "FaceOnSpiral", "EdgeOn", "Irregular", "Ambiguous")
N_CLASSES = len(MORPH_CLASSES)


# ================================================================ autoencoders
class ConvAutoencoder(nn.Module):
    """Under-complete convolutional autoencoder, 128x128 input.
    Conv C->16->32->64 (128 -> 16 px), Linear bottleneck to latent_dim, mirror decoder.
    Stage 1b uses C=1 (one tile of one band); Stage 2a uses C = number of bands."""

    def __init__(self, in_ch=1, latent_dim=64, size=128):
        super().__init__()
        s = size // 8
        self.s = s
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(in_ch, 16, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU())
        self.enc_fc = nn.Linear(64 * s * s, latent_dim)
        self.dec_fc = nn.Linear(latent_dim, 64 * s * s)
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, 2, 1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, 2, 1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(16, in_ch, 3, 2, 1, output_padding=1), nn.Sigmoid())

    def encode(self, x):
        return self.enc_fc(self.encoder_conv(x).flatten(1))

    def decode(self, z):
        return self.decoder_conv(self.dec_fc(z).view(-1, 64, self.s, self.s))

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z), z


def train_autoencoder(model, X, epochs, batch_size, device, lr=1e-3, weight_decay=1e-5, log=print, tag="CAE"):
    """X: float32 array (N, C, H, W) already scaled to [0, 1]."""
    model.to(device).train()
    data = torch.tensor(X)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    for ep in range(epochs):
        perm = torch.randperm(len(data))
        run = 0.0
        for i in range(0, len(data), batch_size):
            xb = data[perm[i:i + batch_size]].to(device)
            recon, _ = model(xb)
            loss = F.mse_loss(recon, xb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            run += loss.item() * len(xb)
        if epochs <= 30 or (ep + 1) % max(1, epochs // 15) == 0 or ep + 1 == epochs:   # ~15 log lines
            log(f"  [{tag}] epoch {ep + 1:2d}/{epochs}  recon MSE = {run / len(data):.5f}")
    return model


class RepAutoencoder(nn.Module):
    """Stage 2a representation autoencoder (cae_version 2).

    Input : bands data channels (field-noise asinh scale) + bands 0/1 "data present" channels.
    Encoder: 4 x [3x3 conv stride 2 + GroupNorm + SiLU] (128 -> 8 px), 1x1 conv to 32 channels,
             Linear(32*8*8 -> latent_dim).  The 1x1 reduction replaces the old 16,384 -> 64 dense layer
             (1.0 M weights -> 65 k) without throwing away WHERE the light is, which global average
             pooling would do (the cutouts are centred on the source, so position is information).
    Decoder: mirror image with nearest-neighbour upsampling + 3x3 conv; linear output (the input
             can be slightly negative), data channels only."""

    def __init__(self, bands=3, latent_dim=32, size=128, channels=(16, 32, 64, 64), mask_channels=True):
        super().__init__()
        self.bands, self.mask_channels = bands, mask_channels
        cin = bands * (2 if mask_channels else 1)
        layers, c = [], cin
        for ch in channels:
            layers += [nn.Conv2d(c, ch, 3, stride=2, padding=1), nn.GroupNorm(min(8, ch), ch), nn.SiLU()]
            c = ch
        self.encoder_conv = nn.Sequential(*layers, nn.Conv2d(c, 32, 1), nn.SiLU())
        self.s = size // 2 ** len(channels)
        self.enc_fc = nn.Linear(32 * self.s * self.s, latent_dim)
        self.dec_fc = nn.Linear(latent_dim, 32 * self.s * self.s)
        dec, c = [nn.Conv2d(32, channels[-1], 1), nn.SiLU()], channels[-1]
        for ch in list(channels[::-1][1:]) + [channels[0]]:
            dec += [nn.Upsample(scale_factor=2, mode="nearest"), nn.Conv2d(c, ch, 3, padding=1),
                    nn.GroupNorm(min(8, ch), ch), nn.SiLU()]
            c = ch
        dec += [nn.Conv2d(c, bands, 3, padding=1)]
        self.decoder_conv = nn.Sequential(*dec)

    def encode(self, x):
        return self.enc_fc(self.encoder_conv(x).flatten(1))

    def decode(self, z):
        return self.decoder_conv(self.dec_fc(z).view(-1, 32, self.s, self.s))

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z), z


def _rep_batch(X, P, W, idx, device, augment=False):
    """Float32 tensors for one batch; X data, P present mask, W loss weights (all (n, bands, S, S))."""
    x = torch.from_numpy(np.asarray(X[idx], np.float32))
    p = torch.from_numpy(np.asarray(P[idx], np.float32))
    w = torch.from_numpy(np.asarray(W[idx], np.float32)) if W is not None else torch.ones_like(x)
    if augment:                                   # morphology does not depend on orientation
        k = int(np.random.randint(4))
        if k:
            x, p, w = (torch.rot90(t, k, dims=(2, 3)) for t in (x, p, w))
        if np.random.rand() < 0.5:
            x, p, w = (torch.flip(t, dims=(3,)) for t in (x, p, w))
    return x.to(device), p.to(device), w.to(device)


def _rep_input(model, x, p):
    return torch.cat([x, p], 1) if model.mask_channels else x


def rep_loss(model, x, p, w):
    """Weighted MSE per cutout (weights already mean-normalised per cutout)."""
    recon, z = model(_rep_input(model, x, p))
    return (w * (recon - x) ** 2).mean(dim=(1, 2, 3)), z, recon


def train_rep_autoencoder(model, X, P, W, train_idx, val_idx, device, max_epochs=150, min_epochs=20, patience=12,
                          batch_size=64, lr=1e-3, weight_decay=1e-5, lr_patience=4, augment=True, min_delta=0.005,
                          checkpoint=None, no_stop_above=None, restore_lowest=False, log=print, tag="Rep-CAE"):
    """Adam + ReduceLROnPlateau on the validation loss; early stopping when the validation loss has not
    improved by at least `min_delta` (relative) for `patience` epochs; best weights kept (and written to
    `checkpoint` at every improvement, so an interrupted run keeps its best model). While the best validation
    loss is still above `no_stop_above` (e.g. 98 % of a trivial flat-image model) training is not stopped early:
    an autoencoder can sit on that plateau for many epochs before it starts to learn structure.
    restore_lowest=False keeps the last epoch that improved by min_delta (as Keras' restore_best_weights);
    True keeps the epoch with the LOWEST validation loss - min_delta then only drives the patience count.
    Returns the training history (train / validation loss per epoch)."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=lr_patience)
    best, bad, hist = np.inf, 0, {"train": [], "val": [], "lr": []}
    kept, best_state, best_ep = np.inf, None, 0             # the weights that are returned
    train_idx, val_idx = np.asarray(train_idx), np.asarray(val_idx)
    for ep in range(max_epochs):
        model.train()
        perm = np.random.permutation(train_idx)
        run = 0.0
        for i in range(0, len(perm), batch_size):
            b = np.sort(perm[i:i + batch_size])
            x, p, w = _rep_batch(X, P, W, b, device, augment)
            loss = rep_loss(model, x, p, w)[0].mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
            run += loss.item() * len(b)
        tr = run / len(train_idx)
        va = float(rep_errors(model, X, P, W, val_idx, device)[0].mean())
        sched.step(va)
        hist["train"].append(tr)
        hist["val"].append(va)
        hist["lr"].append(opt.param_groups[0]["lr"])
        improved = va < best * (1 - min_delta)                # counts for early stopping
        if improved:
            best, bad = va, 0
        else:
            bad += 1
        keep = va < kept if restore_lowest else improved
        if keep:
            kept, best_state, best_ep = va, copy.deepcopy(model.state_dict()), ep + 1
            if checkpoint is not None:
                torch.save(best_state, checkpoint)
        # "*" = weights kept (lowest validation loss so far); "no gain k/patience" = epochs since the last
        # improvement of at least min_delta - training stops early when k reaches patience
        log(f"  [{tag}] epoch {ep + 1:3d}  train {tr:.5f}  val {va:.5f}  lr {opt.param_groups[0]['lr']:.1e}"
            f"  no gain {bad}/{patience}" + ("  *" if keep else ""))
        on_plateau = no_stop_above is not None and best > no_stop_above
        if ep + 1 >= min_epochs and on_plateau and bad == patience:
            log(f"  [{tag}] no improvement for {patience} epochs, but still no better than the trivial model "
                f"({best:.5f} > {no_stop_above:.5f}) - continuing")
        if ep + 1 >= min_epochs and bad >= patience and not on_plateau:
            log(f"  [{tag}] early stop: validation loss not improved by {min_delta:.1%} for {patience} epochs")
            break
    if best_state is None:
        raise RuntimeError(f"[{tag}] the validation loss was never finite - no weights to keep")
    model.load_state_dict(best_state)
    hist["best_epoch"] = int(best_ep)
    hist["best_val"] = float(kept)
    return hist


@torch.no_grad()
def rep_errors(model, X, P, W, idx, device, batch=128):
    """Weighted reconstruction error and latent code for cutouts `idx` (no augmentation)."""
    model.to(device).eval()
    errs, zs = [], []
    idx = np.asarray(idx)
    for i in range(0, len(idx), batch):
        x, p, w = _rep_batch(X, P, W, idx[i:i + batch], device)
        e, z, _ = rep_loss(model, x, p, w)
        errs.append(e.cpu().numpy())
        zs.append(z.cpu().numpy())
    return np.concatenate(errs), np.concatenate(zs)


@torch.no_grad()
def ae_errors_and_latents(model, X, device, batch=256):
    model.to(device).eval()
    errs, zs = [], []
    for i in range(0, len(X), batch):
        xb = torch.tensor(X[i:i + batch]).to(device)
        r, z = model(xb)
        errs.append(((xb - r) ** 2).mean(dim=(1, 2, 3)).cpu().numpy())
        zs.append(z.cpu().numpy())
    return np.concatenate(errs), np.concatenate(zs)


# ================================================================ supervised
class ResidualBlock(nn.Module):
    """residual_block(x, filters, stride) - written as in Keras:
        shortcut = x
        x = Conv2D(filters, 3, strides=stride, padding='same')(x); x = BatchNormalization()(x); x = Activation('relu')(x)
        x = Conv2D(filters, 3, strides=1, padding='same')(x);      x = BatchNormalization()(x)
        if stride != 1 (or the channel count changes):
            shortcut = Conv2D(filters, 1, strides=stride)(shortcut); shortcut = BatchNormalization()(shortcut)
        x = Add()([x, shortcut]); return Activation('relu')(x)
    Layer names follow torchvision (conv1, bn1, conv2, bn2, downsample), so ImageNet weights load unchanged."""

    def __init__(self, in_filters, filters, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_filters, filters, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(filters)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(filters, filters, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(filters)
        self.downsample = None
        if stride != 1 or in_filters != filters:
            self.downsample = nn.Sequential(nn.Conv2d(in_filters, filters, 1, stride=stride, bias=False),
                                            nn.BatchNorm2d(filters))

    def forward(self, x):
        shortcut = x
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        if self.downsample is not None:
            shortcut = self.downsample(shortcut)
        return self.relu(x + shortcut)                      # Add()([x, shortcut]) + relu


class ResNet18(nn.Module):
    """ResNet-18 feature extractor (Stage 4a), 11.2 M parameters, written like a Keras model:
        inputs (N, in_ch, 128, 128)
        x = Conv2D(64, 7, strides=2) + BatchNormalization + relu      -> (N, 64, 64, 64)
        x = MaxPooling2D(3, strides=2)                                -> (N, 64, 32, 32)
        x = residual_block(x, 64);       x = residual_block(x, 64)    -> (N, 64, 32, 32)
        x = residual_block(x, 128, 2);   x = residual_block(x, 128)   -> (N, 128, 16, 16)
        x = residual_block(x, 256, 2);   x = residual_block(x, 256)   -> (N, 256, 8, 8)
        x = residual_block(x, 512, 2);   x = residual_block(x, 512)   -> (N, 512, 4, 4)
        x = GlobalAveragePooling2D()(x)                               -> (N, 512)
    (shapes for a 128 x 128 input; at 224 x 224 the last map is 7 x 7.) No classifier: the 512 pooled values are
    the features. pretrained=True loads torchvision's ImageNet weights (same layer names)."""

    def __init__(self, in_ch=3, pretrained=False):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(3, stride=2, padding=1)
        self.layer1 = nn.Sequential(ResidualBlock(64, 64), ResidualBlock(64, 64))
        self.layer2 = nn.Sequential(ResidualBlock(64, 128, stride=2), ResidualBlock(128, 128))
        self.layer3 = nn.Sequential(ResidualBlock(128, 256, stride=2), ResidualBlock(256, 256))
        self.layer4 = nn.Sequential(ResidualBlock(256, 512, stride=2), ResidualBlock(512, 512))
        self.avgpool = nn.AdaptiveAvgPool2d(1)               # GlobalAveragePooling2D
        self.flatten = nn.Flatten()
        if pretrained:
            w = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1).state_dict()
            self.load_state_dict({k: v for k, v in w.items() if not k.startswith("fc.")})
        if in_ch != 3:                                       # 6-channel input: a new first convolution
            self.conv1 = nn.Conv2d(in_ch, 64, 7, stride=2, padding=3, bias=False)
        self.out_dim = 512

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.flatten(self.avgpool(x))


class EfficientNetB0(nn.Module):
    """EfficientNet-B0 feature extractor (Stage 4a), torchvision's architecture with ImageNet weights.
        inputs (N, 3, 224, 224)
        x = Conv2D(32, 3, strides=2) + BatchNormalization + SiLU                    -> (N, 32, 112, 112)
        x = 16 MBConv blocks (depthwise conv + squeeze-excitation, SiLU)            -> (N, 320, 7, 7)
        x = Conv2D(1280, 1) + BatchNormalization + SiLU                             -> (N, 1280, 7, 7)
        x = GlobalAveragePooling2D()(x)                                             -> (N, 1280)
    The ImageNet classifier (Dropout + Dense 1280 -> 1000) is not used: the 1,280 pooled values are the features.
    4,007,548 parameters."""

    def __init__(self, pretrained=True):
        super().__init__()
        w = torchvision.models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        net = torchvision.models.efficientnet_b0(weights=w)
        self.features = net.features                         # convolution stages
        self.avgpool = net.avgpool                           # GlobalAveragePooling2D
        self.flatten = nn.Flatten()
        self.out_dim = 1280

    def forward(self, x):
        return self.flatten(self.avgpool(self.features(x)))


def frozen(model):
    """Feature extractor mode: no gradients, BatchNorm uses its ImageNet statistics, Dropout off."""
    for p in model.parameters():
        p.requires_grad_(False)
    return model.eval()


@torch.no_grad()
def model_summary(model, *inputs, name=None, max_depth=2):
    """Keras-style model.summary(): one row per layer (down to `max_depth` levels of nesting) with its
    output shape and weight count, from one forward pass of `inputs` (dummy tensors are fine)."""
    def shape(o):
        if torch.is_tensor(o):
            return str(list(o.shape))
        if isinstance(o, dict):
            return ", ".join(f"{k}: {shape(v)}" for k, v in o.items())
        if isinstance(o, (tuple, list)):
            return ", ".join(shape(v) for v in o)
        return "-"

    rows, hooks = [], []
    for path, mod in model.named_modules():
        depth = path.count(".") + 1 if path else 0
        if depth == 0 or depth > max_depth or (depth < max_depth and len(list(mod.children()))):
            continue
        n = sum(p.numel() for p in mod.parameters())
        hooks.append(mod.register_forward_hook(
            lambda m, i, o, path=path, n=n: rows.append((path, type(m).__name__, shape(o), n))))
    was_training = model.training
    model.eval()
    try:
        out = model(*inputs)
    finally:
        for h in hooks:
            h.remove()
        model.train(was_training)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    w1 = max([len(f"{p} ({t})") for p, t, _, _ in rows] + [20])
    w2 = max([len(s) for _, _, s, _ in rows] + [12])
    line = "=" * (w1 + w2 + 18)
    txt = [line, f'Model: "{name or type(model).__name__}"', line,
           f"{'Layer (type)':<{w1}}  {'Output shape':<{w2}}  {'Param #':>12}", "-" * len(line)]
    txt += [f"{f'{p} ({t})':<{w1}}  {s:<{w2}}  {n:>12,}" for p, t, s, n in rows]
    txt += [line, f"Input : {', '.join(str(list(x.shape)) for x in inputs)}", f"Output: {shape(out)}",
            f"Total params: {total:,} ({total * 4 / 2 ** 20:.2f} MB float32)",
            f"Trainable params: {trainable:,}", f"Non-trainable params: {total - trainable:,}", line]
    return "\n".join(txt)


def _balanced_acc(y, pred):
    import warnings
    if len(set(np.asarray(y).tolist())) < 2:
        return np.nan
    with warnings.catch_warnings():                     # "y_pred contains classes not in y_true"
        warnings.simplefilter("ignore", UserWarning)
        return round(float(balanced_accuracy_score(y, pred)), 4)


def morph_metrics(y, P, ym=None, pm=None):
    y = np.asarray(y)
    pred = P.argmax(1)
    present = sorted(set(y.tolist()) | set(pred.tolist()))      # classes absent from both are not scored
    out = {"macro_f1": round(float(f1_score(y, pred, average="macro", labels=present, zero_division=0)), 4),
           "accuracy": round(float((pred == y).mean()), 4) if len(y) else np.nan,
           "balanced_acc": _balanced_acc(y, pred),
           "n": int(len(y))}
    if pm is not None and ym is not None and 0 < np.asarray(ym).sum() < len(ym):
        out["merger_pr_auc"] = round(float(average_precision_score(ym, pm)), 4)
        out["merger_prevalence"] = round(float(np.mean(ym)), 4)
    return out


def nll(y, P):
    y = np.asarray(y)
    return float(-np.mean(np.log(np.clip(P[np.arange(len(y)), y], 1e-12, 1)))) if len(y) else np.inf


# ================================================================ calibration
def softmax_np(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))

In [ ]:
%%writefile src/p2p/representation.py
"""Stage 2a helpers: image normalisation for the representation autoencoder and its diagnostics.

Why a field-level normalisation (representation.normalisation: field_noise)
---------------------------------------------------------------------------
The original CAE input stretched every cutout band to its own 1st-99.5th percentile.
In single-exposure frames the 99.5th percentile is often a cosmic ray, so the apparent
background level changed from cutout to cutout, and the first two principal components
of the CAE latent tracked the mean level of F475W and F606W (Spearman rho 0.98 / 0.97)
instead of the source.  Here every band of every cutout is put on ONE scale per pointing:

    x' = (x - local background) / sigma_band(pointing)          [units of the background noise]
    y  = asinh(x' / a) / asinh(M / a)                             a = asinh_softening_sigma, M = asinh_max_sigma

* local background = median of the cutout's background pixels (segmentation code 0) in that band
* sigma_band       = 1.4826 x MAD of background pixels of up to 400 cutouts of the pointing
                     (an image-noise statistic: it uses no labels, so it cannot leak)
* pixels without data in a band (outside that band's footprint) are set to 0 and get a
  separate 0/1 "data present" channel, so a missing band is not mistaken for a dark source.

The loss weights (pixel_weights) make the autoencoder spend its capacity on the source:
target footprint (grown by 2 px) x target_weight, background 1, cosmic-ray pixels and
pixels without data 0.
"""
from __future__ import annotations

import numpy as np
from scipy import ndimage

SEG_BG, SEG_TARGET, SEG_NEIGHBOUR, SEG_CR = 0, 1, 2, 3


def band_sigmas(cut, seg, rows, max_stamps=400, seed=42):
    """Robust background sigma per band from the background pixels of up to `max_stamps` cutouts."""
    rng = np.random.default_rng(seed)
    rows = np.asarray(rows)
    pick = rows if len(rows) <= max_stamps else rng.choice(rows, max_stamps, replace=False)
    nb = cut.shape[1]
    vals = [[] for _ in range(nb)]
    for i in np.sort(pick):
        c = np.asarray(cut[i], np.float32)
        bg = np.asarray(seg[i]) == SEG_BG
        for b in range(nb):
            v = c[b][bg & (c[b] != 0)]
            if v.size > 50:
                vals[b].append(v - np.median(v))
    sig = []
    for b in range(nb):
        if vals[b]:
            v = np.concatenate(vals[b])
            sig.append(float(1.4826 * np.median(np.abs(v))) or 1.0)
        else:
            sig.append(np.nan)                                   # band never covered in this pointing
    return np.array(sig, np.float32)


def normalise_cutouts(cut, seg, idx, sigma_rows, a=3.0, m=1000.0):
    """Field-noise asinh normalisation of cutouts `idx`.

    sigma_rows: (len(idx), bands) background sigma for each cutout's pointing.
    Returns x (n, bands, S, S) float16 in about [-0.3, 1] and present (n, bands, S, S) bool."""
    idx = np.asarray(idx)
    n, nb, S = len(idx), cut.shape[1], cut.shape[-1]
    x = np.zeros((n, nb, S, S), np.float16)
    present = np.zeros((n, nb, S, S), bool)
    den = np.arcsinh(m / a)
    for k, i in enumerate(idx):
        c = np.asarray(cut[i], np.float32)
        bg = np.asarray(seg[i]) == SEG_BG
        for b in range(nb):
            p = c[b] != 0
            if not p.any() or not np.isfinite(sigma_rows[k, b]):
                continue
            ref = c[b][bg & p]
            lb = float(np.median(ref)) if ref.size > 50 else 0.0
            y = np.arcsinh((c[b] - lb) / (sigma_rows[k, b] * a)) / den
            x[k, b] = np.where(p, np.clip(y, -0.3, 1.0), 0.0)
            present[k, b] = p
    return x, present


def pixel_weights(seg, present, target_weight=10.0, ignore_cr=True, grow_px=2, chunk=512):
    """Loss weight per pixel and band: target footprint (grown) x target_weight, background 1,
    cosmic-ray pixels (segmentation code 3) and pixels without data 0. Normalised to mean 1 per cutout.
    seg (n, S, S), present (n, bands, S, S) -> float16 (n, bands, S, S); computed in chunks to save memory."""
    n = len(present)
    out = np.zeros(present.shape, np.float16)
    for s in range(0, n, chunk):
        sg = np.asarray(seg[s:s + chunk])
        w = np.ones(sg.shape, np.float32)
        tgt = sg == SEG_TARGET
        if grow_px > 0:
            tgt = np.stack([ndimage.binary_dilation(t, iterations=grow_px) for t in tgt])
        w[tgt] = float(target_weight)
        if ignore_cr:
            w[sg == SEG_CR] = 0.0
        w = w[:, None, :, :] * present[s:s + chunk].astype(np.float32)
        tot = w.sum(axis=(1, 2, 3), keepdims=True)
        out[s:s + chunk] = w * (w[0].size / np.maximum(tot, 1e-6))
    return out


# ------------------------------------------------------------------ diagnostics
def spearman(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 10:
        return np.nan
    ra = np.argsort(np.argsort(a[m])).astype(float)
    rb = np.argsort(np.argsort(b[m])).astype(float)
    return float(np.corrcoef(ra, rb)[0, 1])


def band_levels(cut, idx, per_stamp=True):
    """Mean level of each band of each cutout after the ORIGINAL per-stamp asinh stretch: the nuisance
    quantity that dominated the old latent. Used only to check that the new latent does not track it."""
    from .utils import asinh_stack
    out = []
    for s in range(0, len(idx), 256):
        X = asinh_stack(np.asarray(cut[np.asarray(idx[s:s + 256])], np.float32))
        out.append(X.mean(axis=(2, 3)))
    return np.concatenate(out)

In [ ]:
%%writefile src/p2p/sealing.py
"""Sealing of the test and Phase-7 galaxies during development (Stages 6a-7c).

Kept in its own module (not modes.py) so that a running pipeline picks it up at the first evaluation stage."""
from __future__ import annotations

from .modes import is_apply


def test_sealed(ctx) -> bool:
    """While the model is still being developed, the TEST pointing and the Phase-7 (external) pointings stay
    SEALED: no stage computes a score on their labels, so no design decision can be tuned on them, even by
    accident. Every decision is made on the cross-validation of train + dev (Stages 4a-4d). For the final run set
    evaluation.test_sealed: false and run `python run_pipeline.py --from 6a` - once; Stage 6a's ledger records it.
    A missing key counts as sealed. Apply mode is never sealed (the bundle never saw this field)."""
    if is_apply(ctx):
        return False
    return bool(ctx.cfg.get("evaluation", {}).get("test_sealed", True))


def require_unsealed(ctx, stage: str):
    if test_sealed(ctx):
        from .context import StageSkipped
        raise StageSkipped(f"test and Phase-7 galaxies are SEALED (evaluation.test_sealed: true) - stage {stage} scores "
                           "them only in the final run: set evaluation.test_sealed: false, then "
                           "python run_pipeline.py --from 6a")

In [ ]:
%%writefile src/p2p/supervised_cv.py
"""5-fold cross-validation of the supervised stages (4b, 4d) and its plots.

The TRAIN and DEV GZH-labelled galaxies are pooled (~1,100) and split into K folds (StratifiedGroupKFold: the class
mix is the same in every fold, and galaxies in the same 30" sky cell - which share image noise and can overlap in
the cutouts - stay in the same fold). Every model is trained K times on K-1 folds and predicts the held-out fold:
each galaxy gets one OUT-OF-FOLD (OOF) prediction from a model that never saw it. Settings (number of trees,
ensemble block weights) are chosen by the lowest OOF log-loss; the final model is retrained on all pooled galaxies.
The OOF predictions also feed the calibration (Stage 4c). The TEST split is never touched (Stage 6a, once).
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from .nets import morph_metrics, nll

SMOOTH = 3          # points in the moving average (metric macro_f1 only)


def pooled(splits):
    """train + dev labelled galaxies as one table (column 'orig_split' keeps where each came from)."""
    pool = pd.concat([splits["train"].assign(orig_split="train"), splits["dev"].assign(orig_split="dev")],
                     ignore_index=True)
    return pool


def make_folds(pool, n_splits=5, seed=42, cell_arcsec=30.0):
    from sklearn.model_selection import StratifiedGroupKFold
    y = pool.gz_label_id.astype(int).to_numpy()
    cell = cell_arcsec / 3600.0
    groups = (np.floor(pool.ra.to_numpy(float) / cell).astype(int).astype(str) + "_" +
              np.floor(pool.dec.to_numpy(float) / cell).astype(int).astype(str))
    n_splits = int(max(2, min(n_splits, len(set(groups)))))
    folds = list(StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed).split(pool, y, groups))
    fold_id = np.zeros(len(pool), int)
    for f, (_, va) in enumerate(folds):
        fold_id[va] = f
    return folds, fold_id, groups


def bootstrap_ci(y, P, n=1000, seed=0):
    """95% bootstrap interval of Macro-F1 (resampling galaxies)."""
    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        vals.append(morph_metrics(y[i], P[i])["macro_f1"])
    return [round(float(np.percentile(vals, 2.5)), 4), round(float(np.percentile(vals, 97.5)), 4)]


def choose_epoch(curve, min_epoch=3, metric="log_loss"):
    """curve: list of {'macro_f1', 'nll'} per epoch (or per tree count), out-of-fold.
    metric 'log_loss' (default): the minimum OOF log-loss - a proper scoring rule, smooth in the number of
    epochs / trees, so one lucky epoch cannot win (on ~200 galaxies Macro-F1 jumps by +-0.03 between
    neighbouring epochs). metric 'macro_f1': best 3-point moving-average Macro-F1, ties to the lower log-loss.
    Returns (1-based index, the smoothed Macro-F1 curve)."""
    f1 = np.array([c["macro_f1"] for c in curve], float)
    nl = np.array([c["nll"] for c in curve], float)
    sm = pd.Series(f1).rolling(SMOOTH, center=True, min_periods=1).mean().to_numpy()
    cand = list(range(max(0, min_epoch - 1), len(f1))) or list(range(len(f1)))
    if metric == "macro_f1":
        best = max(cand, key=lambda e: (round(sm[e], 3), -nl[e]))
    else:
        best = min(cand, key=lambda e: nl[e])
    return best + 1, sm.round(4).tolist()


def mean_history(hist):
    """Per-fold history (folds x epochs) -> one Keras-style history dict (mean over the folds per epoch)."""
    return {k: np.mean(np.asarray(v), axis=0).tolist() for k, v in hist.items() if k in HISTORY_KEYS}


HISTORY_KEYS = ("loss", "val_loss", "accuracy", "val_accuracy")


def plot_history(history, name, path, xlabel="Epochs", chosen=None):
    """The Keras plot: history = {'loss', 'val_loss', 'accuracy', 'val_accuracy'} lists (one value per epoch;
    here the mean over the CV folds, 'val' = the held-out fold). Loss = cross-entropy of the 5-class head."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    ep = np.arange(1, len(history["loss"]) + 1)

    plt.figure(figsize=(12, 5))

    # Plot Loss
    plt.subplot(1, 2, 1)
    plt.plot(ep, history["loss"], label="Train Loss")
    plt.plot(ep, history["val_loss"], label="Val Loss")
    if chosen:
        plt.axvline(chosen, color="0.5", ls="--", lw=1, label=f"chosen: {chosen}")
    plt.title(f"{name} Loss")
    plt.xlabel(xlabel)
    plt.ylabel("Loss")
    plt.legend()

    # Plot Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(ep, history["accuracy"], label="Train Acc")
    plt.plot(ep, history["val_accuracy"], label="Val Acc")
    if chosen:
        plt.axvline(chosen, color="0.5", ls="--", lw=1, label=f"chosen: {chosen}")
    plt.title(f"{name} Accuracy")
    plt.xlabel(xlabel)
    plt.ylabel("Accuracy")
    plt.legend()

    plt.tight_layout()
    plt.savefig(path, dpi=110, facecolor="white")
    plt.close()


def log_history(log, name, history, unit="epoch"):
    """Keras-style fit() lines: one per epoch (mean over the CV folds)."""
    log(f"{name} history (mean over the CV folds; val = held-out fold):\n" + "\n".join(
        f"  {unit} {e + 1:3d}  loss {history['loss'][e]:.4f}  accuracy {history['accuracy'][e]:.4f}  "
        f"val_loss {history['val_loss'][e]:.4f}  val_accuracy {history['val_accuracy'][e]:.4f}"
        for e in range(len(history["loss"]))))


def curve_metrics(y, oof, folds):
    out = []
    for e in range(oof.shape[0]):
        m = morph_metrics(y, oof[e])
        m["nll"] = round(nll(y, oof[e]), 4)
        m["fold_macro_f1"] = [morph_metrics(y[va], oof[e][va])["macro_f1"] for _, va in folds]
        out.append(m)
    return out


def summarise(y, P, folds, ym=None, pm=None):
    m = morph_metrics(y, P, ym, pm)
    fm = [morph_metrics(y[va], P[va])["macro_f1"] for _, va in folds]
    m.update({"nll": round(nll(y, P), 4), "fold_macro_f1": fm, "fold_mean": round(float(np.mean(fm)), 4),
              "fold_sd": round(float(np.std(fm, ddof=1)), 4) if len(fm) > 1 else None,
              "macro_f1_95ci": bootstrap_ci(y, P)})
    return m


def plot_curve(curves, chosen, path, title):
    """curves: {name: per-epoch metrics}; chosen: {name: epoch}."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    pal = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
    fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.7))
    for (name, cur), c in zip(curves.items(), pal):
        ep = np.arange(1, len(cur) + 1)
        f1 = np.array([x["macro_f1"] for x in cur])
        sd = np.array([np.std(x["fold_macro_f1"], ddof=1) / np.sqrt(len(x["fold_macro_f1"])) for x in cur])
        a.plot(ep, f1, "-", color=c, lw=2, label=f"{name} (chosen epoch {chosen[name]})")
        a.fill_between(ep, f1 - sd, f1 + sd, color=c, alpha=0.15, lw=0)
        a.axvline(chosen[name], color=c, ls="--", lw=1)
        b.plot(ep, [x["nll"] for x in cur], "-", color=c, lw=2, label=name)
    a.set_xlabel("epoch")
    a.set_ylabel("out-of-fold Macro-F1 (band = +-1 s.e. over folds)")
    a.set_title(title, fontsize=9.5)
    a.legend(fontsize=8, frameon=False)
    b.set_xlabel("epoch")
    b.set_ylabel("out-of-fold log-loss (lower = better)")
    b.set_title("Out-of-fold log-loss", fontsize=9.5)
    for ax in (a, b):
        ax.grid(alpha=0.25, lw=0.5)
        ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(path, dpi=120, facecolor="white")
    plt.close(fig)

In [ ]:
%%writefile src/p2p/utils.py
"""Numerical helpers shared by several stages."""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree


# ---------------------------------------------------------------- images
def asinh_normalise(img, q_lo=1.0, q_hi=99.5, soft=0.1):
    """Percentile-clipped asinh stretch to [0, 1]; robust to hot pixels."""
    p = np.nan_to_num(np.asarray(img, dtype=np.float32))
    lo, hi = np.percentile(p, [q_lo, q_hi])
    x = np.clip((p - lo) / max(hi - lo, 1e-8), 0, 1)
    return (np.arcsinh(x / soft) / np.arcsinh(1.0 / soft)).astype(np.float32)


def asinh_stack(cubes):
    """(N, C, H, W) -> asinh-normalised float32, each band of each object separately."""
    cubes = np.asarray(cubes, dtype=np.float32)
    out = np.empty_like(cubes)
    for n in range(cubes.shape[0]):
        for c in range(cubes.shape[1]):
            out[n, c] = asinh_normalise(cubes[n, c]) if np.any(cubes[n, c]) else 0.0
    return out


def gaussian_psf(fwhm_px: float, size: int = 25):
    ax = np.arange(size) - (size - 1) / 2
    xx, yy = np.meshgrid(ax, ax)
    s = fwhm_px / 2.3548
    g = np.exp(-(xx ** 2 + yy ** 2) / (2 * s * s))
    return g / g.sum()


def fwhm_halfmax(psf):
    """FWHM (pixels) from the area above half maximum: A = pi FWHM^2 / 4."""
    psf = np.asarray(psf, float)
    return float(2.0 * np.sqrt((psf >= 0.5 * psf.max()).sum() / np.pi))


def fwhm_fit(psf, half=7):
    """FWHM (pixels) of a source at the stamp centre from a 2-D Gaussian fit - smooth and not quantised to
    whole pixels like the half-maximum area. Robust for single-exposure stamps: only the central
    (2*half+1)^2 pixels are fitted (distant neighbours / cosmic-ray remnants cannot capture the fit), the
    fit starts at the brightest pixel within 2 px of the centre, and sigma (0.2-8 px) and the centre
    (+-2 px) are bounded, so one axis cannot collapse to zero. Falls back to the half-maximum area
    measured on a 4x upsampled image."""
    psf = np.asarray(psf, float)
    n0 = psf.shape[0]
    c0 = n0 // 2
    h = min(half, c0)
    img = np.nan_to_num(psf[c0 - h:c0 + h + 1, c0 - h:c0 + h + 1])
    n = img.shape[0]
    c = (n - 1) / 2
    try:
        import warnings
        from astropy.modeling import fitting, models
        yy, xx = np.mgrid[:n, :n]
        near = np.hypot(yy - c, xx - c) <= 2
        py, px = np.unravel_index(np.argmax(np.where(near, img, -np.inf)), img.shape)
        if not img[py, px] > 0:
            raise ValueError("no positive peak near the centre")
        img = img / img[py, px]                   # amplitude ~1: e-/s values of ~0.01 stall the optimiser
        g0 = models.Gaussian2D(1.0, px, py, 1.0, 1.0,
                               bounds={"x_stddev": (0.2, 8.0), "y_stddev": (0.2, 8.0),
                                       "x_mean": (c - 2, c + 2), "y_mean": (c - 2, c + 2), "amplitude": (0, None)})
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            g = fitting.TRFLSQFitter()(g0, xx, yy, img)
        f = 2.3548 * np.sqrt(abs(g.x_stddev.value * g.y_stddev.value))
        if 0.5 < f < 2.3548 * 7.9:
            return float(f)
    except Exception:
        pass
    from scipy.ndimage import zoom
    up = zoom(np.clip(img, 0, None), 4, order=1)
    return fwhm_halfmax(up) / 4.0


def half_light_radius(img):
    """Radius (px) enclosing half the positive flux, about the image centre."""
    img = np.clip(np.asarray(img, float), 0, None)
    h, w = img.shape
    yy, xx = np.mgrid[:h, :w]
    r = np.hypot(yy - (h - 1) / 2, xx - (w - 1) / 2).ravel()
    o = np.argsort(r)
    cog = np.cumsum(img.ravel()[o])
    if cog[-1] <= 0:
        return np.nan
    return float(r[o][np.searchsorted(cog, 0.5 * cog[-1])])


def cutout_snr(img, core_r=4, sky_r=40):
    """Peak SNR of a stamp: core mean over the MAD noise of the outer annulus."""
    img = np.asarray(img, float)
    h, w = img.shape
    yy, xx = np.mgrid[:h, :w]
    r = np.hypot(yy - h / 2, xx - w / 2)
    sky = img[r > sky_r]
    sigma = 1.4826 * np.median(np.abs(sky - np.median(sky))) + 1e-12
    return float((img[r <= core_r].mean() - np.median(sky)) / sigma)


def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))


# ---------------------------------------------------------------- sky matching
def unit_vectors(ra, dec):
    ra = np.deg2rad(np.asarray(ra, float))
    dec = np.deg2rad(np.asarray(dec, float))
    return np.column_stack([np.cos(dec) * np.cos(ra), np.cos(dec) * np.sin(ra), np.sin(dec)])


def sky_crossmatch(ra1, dec1, ra2, dec2, radius_arcsec=0.5):
    """Nearest neighbour of each catalogue-1 row in catalogue 2 (exact great-circle
    separations via a KD-tree on unit vectors). Returns idx2 (-1 = none), sep", matched."""
    n = len(ra1)
    if n == 0 or len(ra2) == 0:
        return np.full(n, -1), np.full(n, np.inf), np.zeros(n, bool)
    chord, idx = cKDTree(unit_vectors(ra2, dec2)).query(unit_vectors(ra1, dec1))
    sep = np.rad2deg(2.0 * np.arcsin(np.clip(chord / 2.0, 0, 1))) * 3600.0
    hit = sep <= radius_arcsec
    return np.where(hit, idx, -1), sep, hit


def match_with_offset(ra1, dec1, ra2, dec2, radius_arcsec=0.5, search_arcsec=1.5, estimate=True):
    """Remove the median catalogue-to-image offset before the final match
    (old catalogue coordinates can be off by a few tenths of an arcsecond).
    Matches are one-to-one: if two image objects claim one catalogue row,
    only the closer keeps it. Returns idx2, sep, hit, (dra", ddec")."""
    ra2 = np.asarray(ra2, float).copy()
    dec2 = np.asarray(dec2, float).copy()
    off = (0.0, 0.0)
    if estimate and len(ra1) and len(ra2):
        idx, sep, hit = sky_crossmatch(ra1, dec1, ra2, dec2, search_arcsec)
        if hit.sum() >= 5:
            cosd = np.cos(np.deg2rad(np.asarray(dec1)[hit]))
            dra = np.median((np.asarray(ra1)[hit] - ra2[idx[hit]]) * cosd) * 3600
            ddec = np.median((np.asarray(dec1)[hit] - dec2[idx[hit]]) * 3600)
            ra2 = ra2 + dra / 3600 / np.cos(np.deg2rad(dec2))
            dec2 = dec2 + ddec / 3600
            off = (float(dra), float(ddec))
    idx, sep, hit = sky_crossmatch(ra1, dec1, ra2, dec2, radius_arcsec)
    order = np.argsort(np.where(hit, sep, np.inf))
    taken = set()
    for i in order:
        if hit[i]:
            if idx[i] in taken:
                hit[i] = False
            else:
                taken.add(idx[i])
    idx = np.where(hit, idx, -1)
    return idx, sep, hit, off


# ---------------------------------------------------------------- metrics
def predictive_entropy(P):
    P = np.clip(np.asarray(P, float), 1e-12, 1)
    return -(P * np.log(P)).sum(1) / np.log(P.shape[1])


def expected_calibration_error(P, y, n_bins=15):
    P, y = np.asarray(P), np.asarray(y)
    conf, correct = P.max(1), (P.argmax(1) == y).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)


def bootstrap_ci(metric_fn, y, P, n=1000, seed=0, alpha=0.05):
    """Percentile bootstrap CI for a metric(y, P) - small test sets need error bars."""
    rng = np.random.default_rng(seed)
    y, P = np.asarray(y), np.asarray(P)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        try:
            vals.append(metric_fn(y[i], P[i]))
        except Exception:
            pass
    if not vals:
        return (np.nan, np.nan)
    vals = np.asarray(vals, float)
    if not np.isfinite(vals).any():
        return (np.nan, np.nan)
    return (float(np.nanpercentile(vals, 100 * alpha / 2)), float(np.nanpercentile(vals, 100 * (1 - alpha / 2))))


# ---------------------------------------------------------------- identity
def iau_designation(ra, dec, prefix="P2P J"):
    """IAU style: coordinates TRUNCATED, e.g. P2P J123456.7+621234."""
    ra_h = (np.asarray(ra, float) / 15.0) % 24
    h = np.floor(ra_h)
    m = np.floor((ra_h - h) * 60)
    s = np.floor((((ra_h - h) * 60 - m) * 60) * 10) / 10
    d = np.abs(np.asarray(dec, float))
    dd = np.floor(d)
    dm = np.floor((d - dd) * 60)
    ds = np.floor(((d - dd) * 60 - dm) * 60)
    sign = np.where(np.asarray(dec) >= 0, "+", "-")
    return [f"{prefix}{int(a):02d}{int(b):02d}{c:04.1f}{sg}{int(e):02d}{int(f):02d}{int(g):02d}"
            for a, b, c, sg, e, f, g in zip(h, m, s, sign, dd, dm, ds)]


def noise_evidence(df: pd.DataFrame, cfg_baseline: dict) -> np.ndarray:
    """Physical evidence that a detection is spurious: very low SNR, a profile sharper
    than the PSF (no real source can be), or a single-band detection. The single-band test is the
    S/N-based cr_candidate of Stage 1c (no other covering band sees the source at S/N >= confirm_snr);
    the older flux-ratio test (band_consistency < single_band_max) is used only when that column is
    missing - a ratio test calls a real red galaxy that is faint in a shallow blue band 'single-band'."""
    ev = (df["snr"].to_numpy(float) < cfg_baseline["noise_snr_max"]) | \
         (df["fwhm_over_psf"].to_numpy(float) < cfg_baseline["sub_psf_factor"])
    if "cr_candidate" in df:
        ev |= as_bool(df["cr_candidate"]).to_numpy()
    elif "band_consistency" in df:
        bc = df["band_consistency"].to_numpy(float)
        ev |= np.isfinite(bc) & (bc < cfg_baseline.get("single_band_max", 0.05))
    return ev


def as_bool(series) -> pd.Series:
    """CSV round-trips turn booleans into strings / NaN; normalise them."""
    return series.map(lambda v: str(v).strip().lower() in ("true", "1", "1.0")).astype(bool)

In [ ]:
%%writefile src/p2p/xgb_worker.py
"""XGBoost in its own process.

On macOS, PyTorch and XGBoost each load their own OpenMP runtime (libomp). Two OpenMP runtimes in one Python
process can crash it with a segmentation fault, which no try/except can catch. So every XGBoost call - fit,
predict, TreeSHAP - runs in a short-lived child process that never imports torch. Only the trained booster
(raw JSON bytes) travels back to the main process.

This module must NOT import torch (directly or through p2p.nets / p2p.inference).
"""
from __future__ import annotations

import numpy as np

# Defaults of the "compile" step; config.yaml supervised.xgboost overrides any of them.
#   max_depth / min_child_weight : size of each tree (shallow trees: ~1,100 galaxies, ~1,800 features)
#   learning_rate                : shrinkage of every tree
#   subsample / colsample_bytree : random rows / features per tree (the tree-model equivalent of dropout);
#                                  with feature_weights the features are drawn block by block (ensemble weights)
#   reg_lambda / reg_alpha       : L2 / L1 penalty on the leaf values (the equivalent of L2 weight regularisation)
XGB_PARAMS = dict(max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.3, min_child_weight=2,
                  reg_lambda=1.0, reg_alpha=0.0, tree_method="hist", n_jobs=4)


def run_isolated(fn, *args):
    """Run fn(*args) in a FRESH Python process (`python -m p2p.xgb_worker`) that imports nothing but
    this module - multiprocessing's 'spawn' would re-import the main script, and with it PyTorch.
    A crash of the child (e.g. a segmentation fault) raises RuntimeError here instead of killing the run."""
    import os
    import pickle
    import subprocess
    import sys
    import tempfile
    from pathlib import Path
    pkg_parent = str(Path(__file__).resolve().parents[1])            # the folder that holds p2p/
    with tempfile.TemporaryDirectory() as d:
        inp, out = Path(d, "in.pkl"), Path(d, "out.pkl")
        inp.write_bytes(pickle.dumps((fn.__name__, args)))
        env = dict(os.environ, PYTHONPATH=pkg_parent + os.pathsep + os.environ.get("PYTHONPATH", ""))
        r = subprocess.run([sys.executable, "-m", "p2p.xgb_worker", str(inp), str(out)], env=env,
                           capture_output=True, text=True)
        if r.returncode != 0 or not out.exists():
            raise RuntimeError(f"XGBoost process failed (exit code {r.returncode}): {r.stderr.strip()[-400:]}")
        return pickle.loads(out.read_bytes())


def loaded_modules():
    """Diagnostics: the heavy libraries loaded in the worker process (torch must not be one)."""
    import sys
    return sorted({m.split(".")[0] for m in sys.modules} & {"torch", "torchvision", "xgboost"})


def build_xgb(params, seed, n_estimators, n_classes, feature_weights=None, metrics=None):
    """model_xgb = XGBClassifier(...) - the XGBoost equivalent of building + compiling a Keras model.
    Objective: softmax over the classes ('multi:softprob'), or logistic for two classes (merger yes / no)."""
    from xgboost import XGBClassifier
    kw = dict(XGB_PARAMS, **(params or {}), random_state=seed, n_estimators=int(n_estimators))
    kw["objective"] = "multi:softprob" if n_classes > 2 else "binary:logistic"
    if feature_weights is not None:
        kw["feature_weights"] = np.asarray(feature_weights, float)
    if metrics is not None:
        kw["eval_metric"] = metrics
    return XGBClassifier(**kw)


def fit_curve(X, y, w, Xva, yva, n_classes, grid, seed, params=None, feature_weights=None):
    """Cross-validation helper: fit max(grid) trees on (X, y) and return
      * the class probabilities of Xva after each tree count in grid -> (len(grid), n_va, n_classes)
      * the Keras-style history per boosting round: loss / accuracy on the training rows and val_loss /
        val_accuracy on the held-out fold (rows of a class the training fold lacks are not scored)."""
    import xgboost as xgb
    loss, err = ("mlogloss", "merror") if n_classes > 2 else ("logloss", "error")
    model_xgb = build_xgb(params, seed, max(grid), n_classes, feature_weights, metrics=[loss, err])
    keep = np.asarray(yva) >= 0
    eval_set = [(X, y)] + ([(np.asarray(Xva)[keep], np.asarray(yva)[keep])] if keep.any() else [])
    model_xgb.fit(X, y, sample_weight=w, eval_set=eval_set, verbose=False)
    res = model_xgb.evals_result()
    history = {"loss": res["validation_0"][loss], "accuracy": [1 - e for e in res["validation_0"][err]]}
    if "validation_1" in res:
        history.update(val_loss=res["validation_1"][loss], val_accuracy=[1 - e for e in res["validation_1"][err]])
    b = model_xgb.get_booster()
    d = xgb.DMatrix(np.asarray(Xva, np.float32))
    out = []
    for g in grid:
        p = np.asarray(b.predict(d, iteration_range=(0, int(g))))
        out.append(np.column_stack([1 - p, p]) if p.ndim == 1 else p)
    return np.stack(out), history


def fit_fixed(X, y, w, n_estimators, n_classes, seed, params=None, feature_weights=None):
    """Final model: the number of trees chosen by cross-validation, trained on all pooled galaxies."""
    model_xgb = build_xgb(params, seed, n_estimators, n_classes, feature_weights)
    model_xgb.fit(X, y, sample_weight=w)
    return bytes(model_xgb.get_booster().save_raw("json"))


def _booster(raw):
    import xgboost as xgb
    b = xgb.Booster()
    b.load_model(bytearray(raw))
    return b, xgb


def predict(raw, X):
    b, xgb = _booster(raw)
    p = np.asarray(b.predict(xgb.DMatrix(np.asarray(X, np.float32))))
    return np.column_stack([1 - p, p]) if p.ndim == 1 else p          # binary -> two columns


def contribs(raw, X):
    """Exact TreeSHAP values, shape (n, classes, features) without the bias column (binary: classes = 1)."""
    b, xgb = _booster(raw)
    c = np.asarray(b.predict(xgb.DMatrix(np.asarray(X, np.float32)), pred_contribs=True))
    if c.ndim == 2:
        c = c[:, None, :]
    return c[:, :, :-1]


def summary(raw, feature_names, params=None):
    """model.summary() of a trained booster: trees, leaves, parameters and the gain importance of each input."""
    import json
    b, _ = _booster(raw)
    trees = b.get_dump(dump_format="json")
    leaves = [sum(1 for _ in _leaves(json.loads(t))) for t in trees]
    cfg = json.loads(b.save_config())["learner"]
    n_class = int(cfg["learner_model_param"]["num_class"]) or 1
    gain = b.get_score(importance_type="gain")
    imp = {feature_names[int(k[1:])] if k.startswith("f") and k[1:].isdigit() else k: v for k, v in gain.items()}
    return {"n_trees": len(trees), "boosting_rounds": len(trees) // n_class, "outputs": n_class,
            "leaves_total": int(sum(leaves)), "leaves_per_tree_mean": round(float(np.mean(leaves)), 2),
            "objective": cfg["objective"]["name"],
            "params": {k: v for k, v in dict(XGB_PARAMS, **(params or {})).items() if k != "n_jobs"},
            "gain_importance": dict(sorted(imp.items(), key=lambda t: -t[1]))}


def _leaves(node):
    if "leaf" in node:
        yield node
    for ch in node.get("children", []):
        yield from _leaves(ch)


if __name__ == "__main__":                     # the child process started by run_isolated
    import pickle
    import sys
    name, args = pickle.loads(open(sys.argv[1], "rb").read())
    result = globals()[name](*args)
    with open(sys.argv[2], "wb") as fh:
        pickle.dump(result, fh)

### 2.4 Stages — `src/p2p/stages/` (one file per stage)

In [ ]:
%%writefile src/p2p/stages/__init__.py
"""One module per pipeline stage. Each exposes run(ctx). See p2p.context.STAGES for the order."""

In [ ]:
%%writefile src/p2p/stages/stage0_partition.py
"""Stage 0 - Data governance: sky-region partition with a leakage collar.

Purpose : fix the train / dev / test / external split BY SKY REGION before any pixel is analysed. Neighbouring
          galaxies share pixels, background and PSF, so a random split of OBJECTS would leak; splitting the SKY
          does not. The rule is written once (split_rule.json) and every later stage only reads it.

Method  : 1. Each pointing (three files F435W / F606W / F775W, grouped as in Stage 1a) gets a REGION = its real
             data footprint: the convex hull of the pixels with data (WHT > 0) in ALL three bands. A drizzled ACS
             frame is a tilted quadrilateral inside a larger zero-weight array, so its RA/Dec box would be ~2x
             too big and overlap the neighbouring pointings.
          2. partition.pointing_splits (config.yaml) sets the split of each pointing, matched by file-name
             substrings: either the whole pointing (split: train | dev | test | external) or stripes along the
             pointing's longer axis (stripes: {train: 0.25, dev: 0.50, external: 0.25}). Stripe sizes follow
             the share of the pointing's GZH galaxies (partition.stripe_share: gzh_galaxies - POSITIONS only,
             never classes), so "20 %" means 20 % of the labelled galaxies, not 20 % of the area (an area stripe
             can fall outside the GZH coverage and hold no labels). Stripes are clipped to the footprint.
             With an empty pointing_splits list (e.g. a new field in apply mode) every pointing listed in
             partition.external_patterns is external and every other one is cut into fractions (70/15/15)
             by data area.
          3. A source gets the split of the region(s) it lies in. It gets NO split ('buffer') if it lies outside
             every region, or inside / within the collar (partition.collar_arcmin) of regions with DIFFERENT
             splits - the leakage guard. Overlapping regions of the SAME split keep the source.

Plan (Sangeeta, 28 Oct). Each split mixes GOODS-South and GOODS-North pointings from different HST
programmes, epochs and depths, so train, dev and test see the same variety of data. GZH catalogue entries
(positions only) per split, from the real footprints of the 24 files:

    Pointing    train   dev   test   external   no split
    GOODS-S        61   121      -         68         49   stripes 25 / 50 / 25 %
    GOODS-S-2       -    30    129          -         14   stripes 80 (test) / 20 (dev) %
    GOODS-S-3     342     -      -          -         12
    GOODS-S-4       -     -      -        319         34   Phase 7 (Kartaltepe human labels)
    GOODS-N-1     383     -      -          -         12
    GOODS-N-2       -     -      -        320         29   held-out field
    GOODS-N-3     198   168      -          -         58   stripes 50 / 50 %
    GOODS-N-4       -     -    294          -         11
    Total         984   319    423        707        219   (no split = collar or regions of different splits)

Cross-validation (Stages 4a-4d) trains on train + dev. Test and external stay sealed until the model is final
(evaluation.test_sealed). Every run writes this table to split_rule.json ("gzh_per_pointing") and the log.

Output  : outputs/stage0_partition/split_rule.json, split_regions.png
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..context import Ctx, load_json, save_json
from ..fitsio import find_products, group_pointings, is_external, read_meta

SPLITS = ["train", "dev", "test", "external"]


class SkyRegionPartitioner:
    """Polygon regions with a split tag; assigns a split (or None = buffer) to sky positions."""

    def __init__(self, collar_arcmin: float):
        self.buffer_deg = collar_arcmin / 60.0
        self.regions: dict[str, dict] = {}

    def define_region(self, name, polygon, split, pointing):
        poly = np.asarray(polygon, float)
        self.regions[name] = {"split_tag": split, "pointing": pointing,
                              "ra_range": [float(poly[:, 0].min()), float(poly[:, 0].max())],
                              "dec_range": [float(poly[:, 1].min()), float(poly[:, 1].max())],
                              "polygon": poly.tolist()}

    def assign_many(self, ra, dec):
        """Split of every position: the region's tag if all regions it lies in, or is within the collar of,
        share one tag; otherwise (or outside every region) None."""
        from matplotlib.path import Path as MPath
        ra, dec = np.atleast_1d(np.asarray(ra, float)), np.atleast_1d(np.asarray(dec, float))
        inside = [set() for _ in ra]
        near = [set() for _ in ra]
        buf = self.buffer_deg * 60.0                                       # arcmin
        for reg in self.regions.values():
            poly = np.asarray(reg.get("polygon") or _box_polygon(reg), float)
            P, X = _tangent_arcmin(poly, ra, dec)
            idx = np.where(np.hypot(*X.T) < np.hypot(*P.T).max() + buf + 1e-9)[0]   # cheap pre-selection
            if not len(idx):
                continue
            ins = MPath(P).contains_points(X[idx])
            close = ins | (_dist_to_edges(X[idx], P) < buf)
            for i, a, b in zip(idx, ins, close):
                if a:
                    inside[i].add(reg["split_tag"])
                if b:
                    near[i].add(reg["split_tag"])
        return [next(iter(n)) if i and len(n) == 1 else None for i, n in zip(inside, near)]

    def assign_split(self, ra, dec):
        return self.assign_many([ra], [dec])[0]

    def to_dict(self):
        return {"buffer_deg": self.buffer_deg, "regions": self.regions}

    @classmethod
    def from_file(cls, path):
        d = load_json(path)
        p = cls(d["buffer_deg"] * 60.0)
        p.regions = d["regions"]
        return p


# ---------------------------------------------------------------- geometry
def _box_polygon(reg):
    """Regions of split_rule.json files written before 28 Oct have only an RA/Dec box."""
    (r0, r1), (d0, d1) = reg["ra_range"], reg["dec_range"]
    return [[r0, d0], [r1, d0], [r1, d1], [r0, d1]]


def _tangent_arcmin(poly, ra, dec):
    """Polygon and points in a flat (arcmin) frame centred on the polygon (RA wrap and cos(dec) handled)."""
    ra0, dec0 = poly[:, 0].mean(), poly[:, 1].mean()
    cosd = np.cos(np.deg2rad(dec0))
    to = lambda r, d: np.column_stack([((r - ra0 + 180) % 360 - 180) * cosd * 60, (d - dec0) * 60])
    return to(poly[:, 0], poly[:, 1]), to(ra, dec)


def _dist_to_edges(X, P):
    """Distance from points X (n, 2) to the outline of the closed polygon P (m, 2), same units."""
    AB = np.roll(P, -1, axis=0) - P
    t = np.clip(((X[:, None] - P[None]) * AB[None]).sum(2) / np.maximum((AB ** 2).sum(1), 1e-12), 0, 1)
    return np.sqrt(((X[:, None] - (P[None] + t[..., None] * AB[None])) ** 2).sum(2)).min(1)


def coverage_points(ms, stride=8, log=print):
    """Sky positions of pixels with data (WHT > 0) in EVERY file of a pointing, every `stride` pixels.
    Only coverage is read here, never sources."""
    from ..fitsio import load_image, valid_mask
    ref = max(ms, key=lambda m: m["naxis1"] * m["naxis2"])
    sci, wht, w = load_image(ref["file"])
    yy, xx = np.mgrid[0:sci.shape[0]:stride, 0:sci.shape[1]:stride]
    ok = valid_mask(sci, wht)[yy, xx]
    ra, dec = w.pixel_to_world_values(xx[ok], yy[ok])
    keep = np.ones(len(ra), bool)
    for m in ms:
        if m is not ref:
            s2, w2, wcs2 = load_image(m["file"])
            xi, yi = (np.round(v).astype(int) for v in wcs2.world_to_pixel_values(ra, dec))
            inb = (xi >= 0) & (yi >= 0) & (xi < s2.shape[1]) & (yi < s2.shape[0])
            v = np.zeros(len(ra), bool)
            v[inb] = valid_mask(s2, w2)[yi[inb], xi[inb]]
            keep &= v
    if keep.sum() < 0.05 * len(keep):            # bands hardly overlap: use the largest file's coverage
        log(f"[Stage 0] WARNING: the bands of {ref['name']} overlap on only {keep.mean():.1%} of its area; "
            "its region is that file's coverage alone.")
        keep[:] = True
    return np.asarray(ra)[keep], np.asarray(dec)[keep]


def footprint_polygon(ra, dec):
    """Convex hull of the covered pixels (the intersection of the bands' convex data areas is convex)."""
    from scipy.spatial import ConvexHull
    hull = ConvexHull(np.column_stack([(ra - ra.mean()) * np.cos(np.deg2rad(dec.mean())), dec]))
    return np.column_stack([ra, dec])[hull.vertices]


def stripe_boxes(ra, dec, fractions, wra=None, wdec=None):
    """Consecutive stripes along the longer axis of the data area (ra, dec = covered pixels), in the order of
    `fractions`. Cuts at the quantiles of the weight points (wra, wdec = GZH positions) if given, else of the
    covered pixels (equal data area). The outer stripes reach the edge of the data area."""
    along_ra = np.ptp(ra) * np.cos(np.deg2rad(np.median(dec))) >= np.ptp(dec)
    t = ra if along_ra else dec
    w = t if wra is None else (wra if along_ra else wdec)
    cum = np.cumsum(list(fractions.values()))
    cuts = np.concatenate([[t.min()], np.quantile(w, cum[:-1] / cum[-1]), [t.max()]])
    r0, r1, d0, d1 = ra.min(), ra.max(), dec.min(), dec.max()
    return [(tag, (a, b, d0, d1) if along_ra else (r0, r1, a, b))
            for tag, a, b in zip(fractions, cuts[:-1], cuts[1:])]


def clip_to_box(poly, box):
    """Sutherland-Hodgman: the part of a convex polygon inside an RA/Dec box (one stripe of a pointing)."""
    r0, r1, d0, d1 = box
    out = [tuple(p) for p in np.asarray(poly, float)]
    for axis, lim, keep_greater in ((0, r0, True), (0, r1, False), (1, d0, True), (1, d1, False)):
        inp, out = out, []
        inside = (lambda q: q[axis] >= lim) if keep_greater else (lambda q: q[axis] <= lim)
        for prev, cur in zip(inp[-1:] + inp[:-1], inp):
            if inside(cur) != inside(prev):                       # the edge crosses the limit
                t = (lim - prev[axis]) / (cur[axis] - prev[axis])
                out.append(tuple(prev[j] + t * (cur[j] - prev[j]) for j in range(2)))
            if inside(cur):
                out.append(cur)
    return np.asarray(out, float)


# ---------------------------------------------------------------- the split plan
def pointing_rules(metas, groups, cfg):
    """One rule per pointing: {'name', 'split'} or {'name', 'stripes'}. From partition.pointing_splits, or, if
    that list is empty, external_patterns -> external and every other pointing -> fractions (70/15/15)."""
    plan, ext_pat = cfg.get("pointing_splits") or [], cfg.get("external_patterns") or []
    rules, problems = [], []
    for gi, g in enumerate(groups):
        ms = [metas[i] for i in g]
        pid = f"P{gi + 1:02d}"
        is_ext = any(is_external(m, ext_pat) for m in ms)
        if not plan:
            rules.append({"name": pid, "split": "external"} if is_ext else {"name": pid, "stripes": cfg["fractions"]})
            continue
        text = " ".join(m["name"] + " " + m.get("targname", "") for m in ms).lower()
        hit = [r for r in plan if any(str(x).lower() in text for x in r["match"])]
        if len(hit) != 1:
            problems.append(f"{pid} ({', '.join(m['name'] for m in ms)}) matches {len(hit)} entries (needs 1)")
        elif (hit[0].get("split") == "external") != is_ext:
            problems.append(f"{pid} ({hit[0]['name']}): split '{hit[0].get('split', 'stripes')}', but external_"
                            f"patterns {'match' if is_ext else 'do not match'} it - external_patterns must list "
                            "exactly the wholly external pointings (Stage 1a uses them)")
        else:
            rules.append(hit[0])
    unused = [r["name"] for r in plan if not any(r is q for q in rules)]
    if unused:
        problems.append(f"entries match no file in data/raw: {unused}")
    if problems:
        raise RuntimeError("[Stage 0] partition.pointing_splits does not fit the files in data/raw:\n  - "
                           + "\n  - ".join(problems))
    return rules


def build_partition(metas, groups, cfg, gzh_radec=None, log=print):
    from matplotlib.path import Path as MPath
    part = SkyRegionPartitioner(cfg["collar_arcmin"])
    by_gzh = cfg.get("stripe_share", "gzh_galaxies") == "gzh_galaxies" and gzh_radec is not None
    for gi, (g, rule) in enumerate(zip(groups, pointing_rules(metas, groups, cfg))):
        pid = f"P{gi + 1:02d}"
        name = f"{pid}_{rule['name']}".replace(" ", "_") if rule["name"] != pid else pid
        ra, dec = coverage_points([metas[i] for i in g], log=log)
        poly = footprint_polygon(ra, dec)
        if "split" in rule:
            part.define_region(name, poly, rule["split"], name)
            continue
        w, share = (None, None), "data area"
        if by_gzh:
            ins = MPath(poly).contains_points(np.column_stack(gzh_radec))
            if ins.sum() >= 20:                  # too few labelled galaxies: equal data area instead
                w, share = (gzh_radec[0][ins], gzh_radec[1][ins]), f"GZH galaxies ({int(ins.sum())} in the pointing)"
        log(f"[Stage 0] {name}: stripes {rule['stripes']} by share of {share}")
        for tag, box in stripe_boxes(ra, dec, rule["stripes"], *w):
            sp = clip_to_box(poly, box)
            if len(sp) >= 3:
                part.define_region(f"{name}_{tag}", sp, tag, name)
    mode = "planned (partition.pointing_splits)" if cfg.get("pointing_splits") else "automatic (fractions)"
    return part, mode


# ---------------------------------------------------------------- stage entry
def run(ctx: Ctx):
    cfg = ctx.cfg["partition"]
    out = ctx.stage_dir("0")
    files = find_products(ctx.raw, ctx.cfg["acquisition"]["file_patterns"])
    if not files:
        raise FileNotFoundError(f"No *_drc.fits / *_drz.fits files in {ctx.raw}. Put your HST files there "
                                "(python tools/get_fits.py ...).")
    metas = [m for m in (read_meta(f) for f in files) if m["wcs_ok"] and m["has_sci"]]   # as Stage 1a
    acq = ctx.cfg["acquisition"]
    groups = group_pointings(metas, acq["group_radius_arcsec"], acq.get("group_min_overlap", 0.5))
    for gi, g in enumerate(groups):
        ctx.log.info(f"[Stage 0] pointing P{gi + 1:02d}: {[metas[i]['filter'] for i in g]} "
                     f"({', '.join(metas[i]['name'] for i in g)})")

    from ..catalogs import load_gzh               # POSITIONS only (no classes): where the labelled galaxies are
    g = load_gzh(ctx, metas, required=False)
    gzh_radec = (g.ra.to_numpy(float), g.dec.to_numpy(float)) if g is not None and len(g) else None

    part, mode = build_partition(metas, groups, cfg, gzh_radec, log=ctx.log.info)
    rule = {**part.to_dict(), "mode": mode}
    if gzh_radec is not None:
        table = _gzh_per_pointing(part, gzh_radec)
        rule["gzh_per_pointing"] = table.to_dict(orient="index")
        ctx.log.info("[Stage 0] GZH galaxies (positions, no classes) per pointing and split:\n" + table.to_string())
    save_json(rule, out / "split_rule.json")
    _plot(part, metas, out / "split_regions.png")
    tags = pd.Series([r["split_tag"] for r in part.regions.values()]).value_counts().to_dict()
    ctx.log.info(f"[Stage 0] {len(part.regions)} regions ({mode}); collar {cfg['collar_arcmin']}'; tags {tags}")
    ctx.log.info(f"[Stage 0] Ruleset frozen -> {ctx.rel(out / 'split_rule.json')}")
    return {"mode": mode, "n_regions": len(part.regions)}


def _gzh_per_pointing(part, gzh_radec):
    """GZH galaxies per pointing and split (a galaxy inside two pointings of the same split counts once, for
    the first); 'no split' = inside a pointing but in a collar or in regions of different splits."""
    from matplotlib.path import Path as MPath
    ra, dec = gzh_radec
    split = pd.Series(part.assign_many(ra, dec), dtype=object).fillna("no split").to_numpy()
    where = np.full(len(ra), None, dtype=object)
    for reg in part.regions.values():
        P, X = _tangent_arcmin(np.asarray(reg["polygon"]), ra, dec)
        where[(where == None) & MPath(P).contains_points(X)] = reg["pointing"]    # noqa: E711
    keep = where != None                                                      # noqa: E711
    t = pd.crosstab(where[keep], split[keep]).rename_axis(index="pointing", columns="split")
    t = t.reindex(columns=[s for s in SPLITS + ["no split"] if s in t.columns], fill_value=0)
    t.loc["Total"] = t.sum()
    return t


def _plot(part, metas, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.patches import Polygon
    colours = {"train": "#2E5496", "dev": "#E69F00", "test": "#009E73", "external": "#CC79A7"}
    fields = []                                   # one panel per sky field (regions > 1 degree apart)
    for r in part.regions.values():
        c = (np.mean(r["ra_range"]), np.mean(r["dec_range"]))
        f = next((f for f in fields if abs(f["c"][0] - c[0]) * np.cos(np.deg2rad(c[1])) < 1
                  and abs(f["c"][1] - c[1]) < 1), None)
        if f is None:
            fields.append(f := {"c": c, "regs": []})
        f["regs"].append(r)
    fig, axes = plt.subplots(1, len(fields), figsize=(5.2 * len(fields), 5), squeeze=False)
    for ax, f in zip(axes[0], fields):
        for m in metas:
            fp = np.asarray(m["footprint"])
            if abs(fp[:, 1].mean() - f["c"][1]) < 1:
                ax.add_patch(Polygon(fp, fill=False, ec="0.6", lw=0.6))
        pointings = {}
        for r in f["regs"]:
            poly = np.asarray(r.get("polygon") or _box_polygon(r))
            ax.add_patch(Polygon(poly, alpha=0.35, color=colours.get(r["split_tag"], "grey")))
            ax.text(poly[:, 0].mean(), poly[:, 1].mean(), r["split_tag"], ha="center", va="center", fontsize=7)
            pointings.setdefault(r.get("pointing", "?"), []).append(poly)
        for name, polys in pointings.items():    # pointing name once, above its footprint
            p = np.vstack(polys)
            ax.text(p[:, 0].mean(), p[:, 1].max(), name, ha="center", va="bottom", fontsize=7, weight="bold")
        ax.autoscale()
        ax.invert_xaxis()
        ax.set_aspect(1 / np.cos(np.deg2rad(f["c"][1])))
        ax.set_xlabel("RA [deg]")
        ax.set_ylabel("Dec [deg]")
    for tag, c in colours.items():
        axes[0][0].fill([], [], color=c, alpha=0.35, label=tag)
    axes[0][0].legend(fontsize=7, loc="lower left", frameon=False)
    fig.suptitle("Stage 0: sky regions by split (grey outlines = file footprints)")
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage1a_acquisition.py
"""Stage 1a - Archive acquisition: verify and catalogue the input files.

Purpose : know exactly which science products the run uses, and check them before any pixel is processed.
Method  : scan data/raw (files come from tools/get_fits.py), check each file has a SCI extension and a
          celestial WCS (rejected otherwise) and a WHT map, read the photometric keywords (PHOTFLAM/PHOTPLAM ->
          AB zero-point), pixel scale, exposure and epoch; group the files into pointings exactly as Stage 0
          does; choose the band set (blue -> red) and, per pointing, one file per band (_drc before _drz, then
          the deepest) and the detection band (reddest).
Output  : manifest.json (one entry per file) and pointings.json (pointing -> role, splits from Stage 0, bands,
          files, zero-points, detection band). Read by Stages 1b-9.
"""
from __future__ import annotations

from collections import Counter

import numpy as np

from ..context import Ctx, save_json
from ..fitsio import find_products, group_pointings, is_external, read_meta
from .stage0_partition import SkyRegionPartitioner


def choose_bands(pointings, metas, cfg):
    want = cfg["bands"]
    if want != "auto":
        return [b.upper() for b in want]
    counts = Counter(metas[i]["filter"] for p in pointings for i in p)
    lam = {m["filter"]: float(m["photplam"]) for m in metas if m["photplam"]}
    best = [b for b, _ in counts.most_common(cfg.get("max_bands", 3))]
    return sorted(best, key=lambda b: lam.get(b, 1e9))          # blue -> red


def run(ctx: Ctx):
    acfg = ctx.cfg["acquisition"]
    out = ctx.stage_dir("1a")
    files = find_products(ctx.raw, acfg["file_patterns"])
    if not files:
        raise FileNotFoundError(f"No HST products found in {ctx.raw} (download them with tools/get_fits.py).")

    metas, rejected = [], []
    for f in files:
        m = read_meta(f)
        problems = [k for k in ("has_sci", "wcs_ok") if not m[k]]
        if not m["has_wht"]:
            problems.append("no WHT (coverage will be taken from finite SCI pixels)")
        m["problems"] = problems
        (rejected if ("has_sci" in problems or "wcs_ok" in problems) else metas).append(m)
    if not metas:
        raise RuntimeError(f"All files failed the structural check: {[r['name'] for r in rejected]}")

    groups = group_pointings(metas, acfg["group_radius_arcsec"], acfg.get("group_min_overlap", 0.5))
    bands = choose_bands(groups, metas, acfg)
    rules = SkyRegionPartitioner.from_file(ctx.require("0", "split_rule.json"))
    pointings = {}
    for gi, g in enumerate(groups):
        pid = f"P{gi + 1:02d}"
        ms = [metas[i] for i in g]
        by_band = {}
        for m in sorted(ms, key=lambda m: (m["product_type"] != "drc", -m["exptime"])):
            if m["filter"] in by_band:
                ctx.log.info(f"[Stage 1a] WARNING {pid}: {m['name']} not used - {by_band[m['filter']]['name']} "
                             f"is the {m['filter']} file of this pointing (_drc first, then deepest)")
            by_band.setdefault(m["filter"], m)
        used = {b: by_band[b]["file"] for b in bands if b in by_band}
        if not used:
            ctx.log.info(f"[Stage 1a] {pid}: none of the bands {bands} present - pointing skipped")
            continue
        if acfg["detection_band"] != "auto" and acfg["detection_band"] in used:
            det = acfg["detection_band"]
        else:
            det = max(used, key=lambda b: by_band[b]["photplam"] or 0)          # reddest
        role = "external" if any(is_external(m, ctx.cfg["partition"].get("external_patterns")) for m in ms) else "gzh"
        cen = (float(np.mean([m["ra_centre"] for m in ms])), float(np.mean([m["dec_centre"] for m in ms])))
        for m in ms:
            m["pointing"] = pid
        regs = {k: r for k, r in rules.regions.items() if k == pid or k.startswith(pid + "_")}
        names = {r.get("pointing") or (k[:-len(r["split_tag"]) - 1] if k.endswith("_" + r["split_tag"]) else k)
                 for k, r in regs.items()}                                   # e.g. P01_GOODS-S-2
        name = names.pop() if len(names) == 1 else pid
        pointings[pid] = {"name": name[len(pid) + 1:] if name.startswith(pid + "_") else name, "role": role,
                          "splits": sorted({r["split_tag"] for r in regs.values()}),
                          "bands": used, "missing_bands": [b for b in bands if b not in used],
                          "detection_band": det, "centre": cen,
                          "split_at_centre": rules.assign_split(*cen),
                          "instrument": by_band[det]["instrument"], "epoch": by_band[det]["epoch"],
                          "exptime_detection": by_band[det]["exptime"], "zp_ab": {b: by_band[b]["zp_ab"] for b in used},
                          "pixscale": {b: by_band[b]["pixscale_arcsec"] for b in used}}

    manifest = {m["name"]: dict(m) for m in metas}
    save_json(manifest, out / "manifest.json")
    save_json({"bands": bands, "pointings": pointings, "rejected": [r["name"] for r in rejected]},
              out / "pointings.json")
    for pid, p in pointings.items():
        ctx.log.info(f"[Stage 1a] {pid} {p['name']} ({p['role']}, splits {p['splits']}): bands {list(p['bands'])} "
                     f"detection={p['detection_band']} "
                     f"missing={p['missing_bands']} instrument={p['instrument']}")
    ctx.log.info(f"[Stage 1a] {len(metas)} verified file(s), {len(rejected)} rejected; band set = {bands}")
    return {"files": len(metas), "pointings": len(pointings), "bands": bands}

In [ ]:
%%writefile src/p2p/stages/stage1b_frame_qc.py
"""Stage 1b - Frame quality control with a convolutional autoencoder.

Purpose : flag detector artefacts (satellite trails, bad drizzle regions, chip
          edges, hot-pixel clusters) that survive drizzle combination.
Method  : 128x128 sub-tiles of every SCI image are asinh-normalised; an
          under-complete CAE is trained on a random sample of well-covered
          tiles (artefacts are rare, so the model learns 'normal sky'); every
          tile is then scored with a 64-px stride (50% overlap, so an artefact
          on a tile edge is not split). The score is the reconstruction error
          with COMPACT sources masked (galaxies are rare too and would
          otherwise dominate); tiles above the 99th percentile of the scores of
          20 % HELD-OUT tiles (never trained on) are flagged.

          Which tiles: only tiles in the Stage 0 train / dev sky (frame_qc.train_splits) fit, stop and
          calibrate the model; test, external and collar sky is only scored. The tiles are split into fit /
          validation / calibration by SKY CELL (frame_qc.split_cell_arcsec), not tile by tile: the three bands
          of a pointing (and overlapping files) see the same sky, so a per-tile shuffle put near-copies of a
          validation tile into training and made the validation loss and the threshold optimistic.

          qc_version 2 (default): a further 10 % of the tiles is a VALIDATION set. Training stops when the
          validation loss has not improved by min_delta for `patience` epochs (never before min_epochs, never
          while the model is < min_gain_vs_flat better than a flat tile, at most max_epochs) and keeps the
          epoch with the LOWEST validation loss (a fixed epoch count stopped the original model on its "flat
          tile" plateau: < 1 % better than a tile filled with its own mean), with the GroupNorm autoencoder of
          Stage 2a (no plateau), 90-degree rotations / flips, the reconstruction gain over a flat tile, and a
          bootstrap interval for the threshold and the flagged share. qc_version 1 reproduces the original
          (fixed steps).
Output  : tile_qc.csv (file, x0, y0, mse, flagged), qc_cae.pt, qc_threshold.json,
          qc_error_hist.png, qc_training.png / qc_training.json (v2) and one flag-map PNG per file.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import torch

from ..context import Ctx, load_json, save_json
from ..fitsio import load_image, valid_mask
from ..nets import ConvAutoencoder, RepAutoencoder, train_autoencoder, train_rep_autoencoder
from ..utils import asinh_normalise
from .stage0_partition import SkyRegionPartitioner


def frame_sigma(sci, valid):
    d = sci[valid]
    if d.size > 2_000_000:
        d = d[:: d.size // 2_000_000]
    return float(1.4826 * np.median(np.abs(d - np.median(d)))) or 1.0


def qc_normalise(tile, sigma, cap=200.0):
    """ONE fixed stretch for every tile of a file (in units of the sky noise).
    A per-tile percentile stretch would blow empty-sky noise up to full range and
    squash a satellite trail - exactly the opposite of what an anomaly detector needs."""
    x = np.arcsinh(np.clip(tile / (3.0 * sigma), 0, None)) / np.arcsinh(cap)
    return np.clip(x, 0, 1).astype(np.float32)


def source_masked_mse(x, recon, tile, level=0.2, min_extent_frac=0.5, max_fill=0.25):
    """Reconstruction error with SOURCE-LIKE structure masked.

    Galaxies are rare too, so a plain MSE mostly flags bright galaxies. Every
    connected structure above `level` is masked unless it looks like an artefact:
      * long and thin  (extent >= half a tile, filling < 25% of its box) -> trail
      * very large     (area >= a quarter of the tile)                   -> bad patch
    so the score measures what the CAE cannot explain OUTSIDE ordinary sources."""
    from scipy import ndimage
    errs = np.empty(len(x), np.float32)
    for i in range(len(x)):
        img = x[i, 0]
        lab, n = ndimage.label(img > level)
        keep = np.ones_like(img, bool)
        for k, sl in enumerate(ndimage.find_objects(lab), start=1):
            if sl is None:
                continue
            h, w = sl[0].stop - sl[0].start, sl[1].stop - sl[1].start
            area = int((lab[sl] == k).sum())
            thin_long = max(h, w) >= min_extent_frac * tile and area / (h * w) < max_fill
            big = area >= 0.25 * tile * tile
            if not (thin_long or big):
                keep[sl] &= lab[sl] != k
        d = (img - recon[i, 0]) ** 2
        errs[i] = d[keep].mean() if keep.any() else 0.0
    return errs


@torch.no_grad()
def reconstruct(model, X, device, batch=256):
    model.to(device).eval()
    out = []
    for i in range(0, len(X), batch):
        r, _ = model(torch.tensor(X[i:i + batch]).to(device))
        out.append(r.cpu().numpy())
    return np.concatenate(out)


def tiles(sci, valid, size, stride):
    ny, nx = sci.shape
    for y0 in range(0, ny - size + 1, stride):
        for x0 in range(0, nx - size + 1, stride):
            cov = valid[y0:y0 + size, x0:x0 + size].mean()
            yield x0, y0, cov


def split_by_sky_cell(cells, cal_frac, val_frac, rng):
    """Whole sky cells (not single tiles) go to calibration / validation / fit, so the same patch of sky seen
    in another band, or by an overlapping file, never sits on both sides. Cells are taken in random order
    until each role holds its share of the tiles. Returns the role of every tile: 'cal', 'val' or 'fit'."""
    keys, inv, counts = np.unique(cells, axis=0, return_inverse=True, return_counts=True)
    order = rng.permutation(len(keys))
    start = (np.cumsum(counts[order]) - counts[order]) / counts.sum()    # tile share before each cell
    role = np.empty(len(keys), object)
    role[order] = np.where(start < cal_frac, "cal", np.where(start < cal_frac + val_frac, "val", "fit"))
    return role[inv.ravel()]


def run(ctx: Ctx):
    cfg = ctx.cfg["frame_qc"]
    out = ctx.stage_dir("1b")
    pts = load_json(ctx.require("1a", "pointings.json"))["pointings"]
    rules = SkyRegionPartitioner.from_file(ctx.require("0", "split_rule.json"))
    files = sorted({f for p in pts.values() for f in p["bands"].values()})
    size, rng = cfg["tile"], np.random.default_rng(ctx.seed)
    device = ctx.device()
    fit_splits = set(cfg.get("train_splits", ["train", "dev"]))
    cell_deg = float(cfg.get("split_cell_arcsec", 30.0)) / 3600.0

    # ---- 1. well-covered tiles from the Stage 0 train / dev sky only (test, external and collar sky is
    #         scored in step 2 but never used to fit, stop or calibrate the QC model) ----
    train, cells, tags = [], [], []
    for f in files:
        sci, wht, w = load_image(f)
        val = valid_mask(sci, wht)
        sig = frame_sigma(sci, val)
        sci = np.where(val, sci - np.median(sci[val]), 0.0)
        pos = [(x0, y0) for x0, y0, cov in tiles(sci, val, size, cfg["train_stride"])
               if cov >= cfg["min_tile_coverage"]]
        if not pos:
            continue
        ra, dec = w.pixel_to_world_values(*(np.asarray(pos, float) + (size - 1) / 2).T)
        for (x0, y0), r, d, t in zip(pos, ra, dec, rules.assign_many(ra, dec)):
            tags.append(t or "collar")
            train.append(qc_normalise(sci[y0:y0 + size, x0:x0 + size], sig))
            cells.append((int(np.floor(r * np.cos(np.deg2rad(d)) / cell_deg)), int(np.floor(d / cell_deg))))
    tag_counts = pd.Series(tags).value_counts().to_dict()
    use = np.isin(tags, list(fit_splits))
    if not use.any():          # e.g. a field that is wholly external: nothing else to learn 'normal sky' from
        ctx.log.info(f"[Stage 1b] WARNING: no well-covered tile in the Stage 0 {sorted(fit_splits)} sky "
                     f"(tiles per split: {tag_counts}); the QC model uses every tile of the field")
        fit_splits, use = set(tag_counts), np.ones(len(tags), bool)
    train, cells = [x for x, u in zip(train, use) if u], [c for c, u in zip(cells, use) if u]
    if len(train) > cfg["max_train_tiles"]:
        pick = rng.choice(len(train), cfg["max_train_tiles"], replace=False)
        train, cells = [train[i] for i in pick], [cells[i] for i in pick]
    X = np.stack(train)[:, None]
    # 20 % of the tiles are HELD OUT: the threshold is set on tiles the CAE never trained on. A threshold
    # from its own training tiles is biased low (they are reconstructed best) and flags ordinary sky.
    version = int(cfg.get("qc_version", 2))
    role = split_by_sky_cell(np.asarray(cells), float(cfg.get("cal_fraction", 0.2)),
                             float(cfg.get("val_fraction", 0.1)) if version == 2 else 0.0, rng)
    n_cells = len(set(cells))
    if min((role == r).sum() for r in (("cal", "val", "fit") if version == 2 else ("cal", "fit"))) < 20:
        raise RuntimeError(f"[Stage 1b] {len(X)} tiles in {n_cells} sky cells cannot fill the fit / validation / "
                           "calibration sets (>= 20 tiles each); lower frame_qc.split_cell_arcsec")
    ctx.log.info(f"[Stage 1b] tiles per Stage 0 split: {tag_counts}; QC model uses {sorted(fit_splits)} only: "
                 f"{len(X)} tiles in {n_cells} sky cells of {cell_deg * 3600:g}\" (whole cells go to one role)")
    split_info = {"train_splits": sorted(fit_splits), "split_cell_arcsec": cell_deg * 3600, "n_sky_cells": n_cells,
                  "tiles_per_stage0_split": tag_counts}
    if version == 1:
        Xcal, Xfit = X[role == "cal"], X[role != "cal"]
        ctx.log.info(f"[Stage 1b] training QC-CAE v1 on {len(Xfit)} tiles from {len(files)} file(s) ({device}); "
                     f"{len(Xcal)} held-out tiles set the threshold")
        model = ConvAutoencoder(in_ch=1, latent_dim=cfg["latent_dim"], size=size)
        steps_per_epoch = int(np.ceil(len(Xfit) / cfg["batch_size"]))
        epochs = max(int(cfg.get("epochs", 8)), int(np.ceil(cfg.get("min_steps", 0) / steps_per_epoch)))
        ctx.log.info(f"[Stage 1b] {epochs} epochs x {steps_per_epoch} steps = {epochs * steps_per_epoch} optimisation steps")
        train_autoencoder(model, Xfit, epochs, cfg["batch_size"], device, log=ctx.log.info, tag="QC-CAE")
        extra = {"qc_version": 1, "epochs_run": epochs}
    else:
        # a further 10 % VALIDATION tiles decide when training stops; the 20 % calibration tiles still set the
        # threshold and are never used for training or stopping
        Xcal, Xval, Xfit = X[role == "cal"], X[role == "val"], X[role == "fit"]
        model, extra = _train_v2(ctx, cfg, Xfit, Xval, size, device, out)
        ctx.log.info(f"[Stage 1b] {len(Xfit)} training, {len(Xval)} validation and {len(Xcal)} calibration tiles "
                     f"from {len(files)} file(s)")
    train_err = source_masked_mse(Xfit, reconstruct(model, Xfit, device), size)
    cal_err = source_masked_mse(Xcal, reconstruct(model, Xcal, device), size)
    thr = float(np.percentile(cal_err, cfg["threshold_percentile"]))
    # the 99th percentile of a few hundred tiles rests on a handful of them: report its uncertainty
    boot = np.random.default_rng(ctx.seed).choice(cal_err, (1000, len(cal_err)), replace=True)
    thr_ci = np.percentile(np.percentile(boot, cfg["threshold_percentile"], axis=1), [2.5, 97.5])
    torch.save(model.state_dict(), out / "qc_cae.pt")
    qc_info = {"qc_version": version, "threshold_mse": thr, "threshold_ci95": [float(v) for v in thr_ci],
               "percentile": cfg["threshold_percentile"], "n_train_tiles": len(Xfit), "n_heldout_tiles": len(Xcal),
               "median_mse_train": float(np.median(train_err)), "median_mse_heldout": float(np.median(cal_err)),
               **split_info, **extra}
    ctx.log.info(f"[Stage 1b] threshold {thr:.5f} (95 % bootstrap interval {thr_ci[0]:.5f}-{thr_ci[1]:.5f})")

    # ---- 2. score every tile (overlapping) of every file ----
    rows = []
    for f in files:
        sci, wht, _ = load_image(f)
        val = valid_mask(sci, wht)
        sig = frame_sigma(sci, val)
        sci = np.where(val, sci - np.median(sci[val]), 0.0)
        pos, batch = [], []
        for x0, y0, cov in tiles(sci, val, size, cfg["score_stride"]):
            if cov < 0.05:            # empty border: nothing to judge
                continue
            pos.append((x0, y0, cov))
            batch.append(qc_normalise(sci[y0:y0 + size, x0:x0 + size], sig))
        Xs = np.stack(batch)[:, None]
        err = source_masked_mse(Xs, reconstruct(model, Xs, device), size)
        for (x0, y0, cov), e in zip(pos, err):
            rows.append({"file": f, "x0": x0, "y0": y0, "size": size, "coverage": round(cov, 3),
                         "mse": float(e), "flagged": bool(e > thr)})
        _flag_map(f, sci, pos, err, thr, size, out)
    df = pd.DataFrame(rows)
    df.to_csv(out / "tile_qc.csv", index=False)
    qc_info["flagged_fraction"] = float(df["flagged"].mean())
    qc_info["flagged_fraction_ci95"] = [float((df["mse"] > t).mean()) for t in thr_ci[::-1]]
    save_json(qc_info, out / "qc_threshold.json")
    _hist(cal_err, df["mse"].to_numpy(), thr, out / "qc_error_hist.png")
    frac = df.groupby("file")["flagged"].mean().round(4).to_dict()
    ctx.log.info(f"[Stage 1b] threshold MSE = {thr:.5f}; flagged tile fraction per file: "
                 f"{ {Path(k).name: v for k, v in frac.items()} }; overall {qc_info['flagged_fraction']:.2%} "
                 f"(threshold interval -> {qc_info['flagged_fraction_ci95'][0]:.2%}-{qc_info['flagged_fraction_ci95'][1]:.2%})")
    return {"threshold": thr, "flagged_fraction": float(df["flagged"].mean()), "qc_version": version,
            **({"epochs_run": extra.get("epochs_run"), "best_epoch": extra.get("best_epoch"),
                "gain_vs_flat_tile": extra.get("gain_vs_flat_tile")} if version == 2 else {})}


def _hist(cal_err, all_err, thr, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5.5, 3.2))
    ax.hist(all_err, bins=60, alpha=0.6, label="all scored tiles")
    ax.hist(cal_err, bins=60, alpha=0.6, label="held-out tiles (set the threshold)")
    ax.axvline(thr, color="r", ls="--", label=f"threshold {thr:.4f}")
    ax.set_xlabel("reconstruction MSE")
    ax.set_yscale("log")
    ax.legend(fontsize=8)
    ax.set_title("Stage 1b: frame QC")
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)


def _flag_map(f, sci, pos, err, thr, size, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle
    step = max(1, max(sci.shape) // 1024)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(asinh_normalise(sci[::step, ::step]), origin="lower", cmap="gray")
    for (x0, y0, _), e in zip(pos, err):
        if e > thr:
            ax.add_patch(Rectangle((x0 / step, y0 / step), size / step, size / step, fill=False, ec="red", lw=0.8))
    ax.set_title(f"{Path(f).name}: flagged tiles (red)", fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.tight_layout()
    fig.savefig(out / (Path(f).name.replace(".gz", "").replace(".fits", "") + "_qc.png"), dpi=100)
    plt.close(fig)


def _train_v2(ctx, cfg, Xfit, Xval, size, device, out):
    """QC autoencoder v2: GroupNorm encoder (as Stage 2a), plain MSE, early stopping on the validation tiles."""
    torch.manual_seed(ctx.seed)
    np.random.seed(ctx.seed)
    model = RepAutoencoder(bands=1, latent_dim=int(cfg["latent_dim"]), size=size,
                           channels=tuple(cfg.get("channels", [16, 32, 64, 64])), mask_channels=False)
    X = np.concatenate([Xfit, Xval]).astype(np.float16)
    P = np.ones(X.shape, bool)
    tr, va = np.arange(len(Xfit)), np.arange(len(Xfit), len(X))
    ctx.log.info(f"[Stage 1b] training QC-CAE v2 ({sum(p.numel() for p in model.parameters()):,} weights) on "
                 f"{len(tr)} tiles, validation {len(va)} ({device}); at most {cfg.get('max_epochs', 150)} epochs, "
                 f"early stopping after {cfg.get('patience', 15)} epochs without a "
                 f"{float(cfg.get('min_delta', 0.005)):.1%} improvement")
    v = np.asarray(Xval, np.float32)
    flat = float(((v - v.mean(axis=(2, 3), keepdims=True)) ** 2).mean())     # a tile filled with its own mean
    hist = train_rep_autoencoder(model, X, P, None, tr, va, device, max_epochs=int(cfg.get("max_epochs", 150)),
                                 min_epochs=int(cfg.get("min_epochs", 20)), patience=int(cfg.get("patience", 15)),
                                 batch_size=int(cfg["batch_size"]), lr=float(cfg.get("lr", 1e-3)),
                                 lr_patience=int(cfg.get("lr_patience", 5)), augment=bool(cfg.get("augment", True)),
                                 min_delta=float(cfg.get("min_delta", 0.005)), checkpoint=out / "qc_cae.pt",
                                 no_stop_above=flat * (1 - float(cfg.get("min_gain_vs_flat", 0.02))),
                                 restore_lowest=True, log=ctx.log.info, tag="QC-CAE")
    gain = 1 - hist["best_val"] / flat
    ctx.log.info(f"[Stage 1b] best validation loss {hist['best_val']:.5f} at epoch {hist['best_epoch']} of "
                 f"{len(hist['val'])}; a flat tile (each tile = its mean) scores {flat:.5f} -> gain {gain:.1%}")
    if gain < 0.02:
        ctx.log.info("[Stage 1b] WARNING: the QC autoencoder is hardly better than a flat tile - its flags then mark "
                     "high-variance tiles rather than what the model cannot explain.")
    save_json(hist, out / "qc_training.json")
    _training_plot(hist, flat, out / "qc_training.png")
    model.to(device)
    return model, {"epochs_run": len(hist["val"]), "best_epoch": hist["best_epoch"],
                   "best_val_loss": hist["best_val"], "flat_tile_val_loss": flat, "gain_vs_flat_tile": round(gain, 4)}


def _training_plot(hist, flat, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    ep = np.arange(1, len(hist["train"]) + 1)
    fig, ax = plt.subplots(figsize=(5.5, 3.4))
    ax.plot(ep, hist["train"], label="train (augmented)", color="#2E5496")
    ax.plot(ep, hist["val"], label="validation tiles", color="#E69F00")
    ax.axhline(flat, color="0.4", ls=":", lw=1, label="flat tile (each tile = its mean)")
    ax.axvline(hist["best_epoch"], color="0.5", ls="--", lw=0.8, label=f"best epoch {hist['best_epoch']}")
    ax.set_xlabel("epoch")
    ax.set_ylabel("reconstruction MSE")
    ax.set_title("Stage 1b: QC autoencoder training")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage1c_detection.py
"""Stage 1c - Detection, band registration, cosmic-ray tests and aligned cutouts (no learned model).

Per pointing:
  1. Background-subtract every band; detect and deblend sources on the detection band (photutils).
  2. Register each band to the detection band: offsets measured on median-filtered images (cosmic rays erased)
     are removed from the band's WCS; the stage stops if the offset cannot be measured or a residual remains.
  3. Measure every source in every band. A source that no other covering band confirms (S/N >= confirm_snr,
     light at the same position) is a cosmic-ray candidate. Confirmed sources and candidates are capped separately.
  4. Cut a north-up 128 x 128 stamp per source from every band at the same RA/Dec (affine resampling).
     Cosmic-ray pixels (far brighter in one band than the source colour predicts) are replaced and marked.
  5. Flags: edge (WHT coverage), frame QC (Stage 1b tiles), saturation, exposures (CTX); split from Stage 0.
Output  : sources.csv, cutouts_raw.npy (N, bands, 128, 128, float16), segstamps.npy (N, 128, 128, uint8:
          1 target, 2 neighbour, 3 cosmic-ray pixel), registration.json, summary.json, preview PNGs.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from photutils.utils import calc_total_error
from scipy import ndimage

from ..compat import SourceCatalog, n_labels, values
from ..context import Ctx, load_json, save_json
from ..detection import detect_and_deblend, subtract_background
from ..fitsio import load_ctx, load_image, read_meta, valid_mask
from ..multiband import (cr_pixel_mask, measure_offset, median_image, popcount, registration_catalog,
                         segment_photometry, shift_wcs, tangent_arcsec)
from .stage0_partition import SkyRegionPartitioner


def affine_for(wcs, ra0, dec0, size, scale_arcsec):
    """ndimage.affine_transform matrix/offset from a north-up cutout (row, col) to the band's pixels, and the
    pixel-area ratio."""
    c = (size - 1) / 2.0
    cosd = np.cos(np.deg2rad(dec0))
    # cutout pixel (u, v) -> sky: east is left (RA grows to -u), north is up
    def sky(du, dv):
        return ra0 - du * scale_arcsec / 3600.0 / cosd, dec0 + dv * scale_arcsec / 3600.0
    pts = [sky(0, 0), sky(1, 0), sky(0, 1)]
    xy = np.array([wcs.world_to_pixel_values(r, d) for r, d in pts], float)
    x0, y0 = xy[0]
    J = np.array([[xy[1, 0] - x0, xy[2, 0] - x0],          # dx/du dx/dv
                  [xy[1, 1] - y0, xy[2, 1] - y0]])         # dy/du dy/dv
    M = np.array([[J[1, 1], J[1, 0]],                      # (row, col) order
                  [J[0, 1], J[0, 0]]])
    offset = np.array([y0, x0]) - M @ np.array([c, c])
    return M, offset, abs(np.linalg.det(J))


def make_stamp(img, M, offset, size, order=1):
    return ndimage.affine_transform(img, M, offset=offset, output_shape=(size, size),
                                    order=order, mode="constant", cval=0.0, prefilter=False)


def flagged_tiles(ctx: Ctx):
    """Stage 1b flagged tiles as their central stride x stride cell (tiles overlap, so each pixel is judged by
    exactly one tile per band)."""
    p = ctx.path("1b", "tile_qc.csv")
    if not p.exists():
        return {}
    t = pd.read_csv(p)
    t = t[t["flagged"].astype(str).str.lower() == "true"]
    stride = int(ctx.cfg["frame_qc"].get("score_stride", 64))
    m = (t["size"] - stride) / 2.0
    t = t.assign(cx0=t.x0 + m, cy0=t.y0 + m, cs=np.minimum(stride, t["size"]))
    return {f: g[["cx0", "cy0", "cs"]].to_numpy(float) for f, g in t.groupby("file")}


def in_flagged(x, y, boxes):
    if boxes is None or len(boxes) == 0:
        return np.zeros(len(x), bool)
    x, y = np.asarray(x)[:, None], np.asarray(y)[:, None]
    x0, y0, s = boxes[:, 0][None], boxes[:, 1][None], boxes[:, 2][None]
    return np.any((x >= x0) & (x < x0 + s) & (y >= y0) & (y < y0 + s), axis=1)


def saturated_cores(sub, rms, x, y, half=4, tol=0.02, min_peak_sigma=50.0):
    """Flat-topped cores on native pixels: the 5 brightest pixels within +-4 px agree to 2 %, peak >= 50 sigma."""
    h, w = sub.shape
    out = np.zeros(len(x), bool)
    for k, (xc, yc) in enumerate(zip(np.round(x).astype(int), np.round(y).astype(int))):
        y0, y1, x0, x1 = max(0, yc - half), min(h, yc + half + 1), max(0, xc - half), min(w, xc + half + 1)
        box = sub[y0:y1, x0:x1]
        if box.size < 5:
            continue
        top = np.sort(box.ravel())[-5:]
        out[k] = top[-1] > min_peak_sigma * rms[min(h - 1, max(0, yc)), min(w - 1, max(0, xc))] and \
            (top[-1] - top[0]) / top[-1] < tol
    return out


def register_bands(ctx, pid, imgs, det_b, dcfg):
    """Measure and remove each band's sky offset against the detection band (sources from median-filtered
    images, so cosmic rays do not pair at random). Returns the per-band report."""
    det = imgs[det_b]
    ra0, dec0 = det["wcs"].wcs.crval
    limit = float(dcfg.get("max_band_offset_arcsec", 0.05))
    search = float(dcfg.get("register_search_arcsec", 3.0))
    min_sig = float(dcfg.get("register_min_significance", 5.0))
    manual = {str(k).upper(): v for k, v in (dcfg.get("band_offsets_arcsec") or {}).items()}
    rr, rd = registration_catalog(det["sub"], det["rms"], det["val"], det["wcs"], dcfg)
    ref_xy = np.column_stack(tangent_arcsec(rr, rd, ra0, dec0))
    report = {}
    for b, im in imgs.items():
        if b == det_b:
            continue
        br, bd = registration_catalog(im["sub"], im["rms"], im["val"], im["wcs"], dcfg)
        bpx = im["wcs"].world_to_pixel_values(br, bd)        # pixel positions: fixed, only the WCS changes

        def offset(wcs):
            r, d = wcs.pixel_to_world_values(*bpx)
            return measure_offset(ref_xy, np.column_stack(tangent_arcsec(r, d, ra0, dec0)), search)

        dx, dy, n, scat, sig = offset(im["wcs"])
        reliable = n >= 20 and sig >= min_sig
        rep = {"n_reference": len(ref_xy), "n_band": len(br), "n_pairs": n, "significance": round(sig, 1),
               "offset_measured_arcsec": [round(dx, 4), round(dy, 4)] if n else None,
               "scatter_arcsec": round(scat, 4) if n else None, "reliable": bool(reliable), "applied": None}
        if b in manual:                                  # offset given in config.yaml
            ox, oy = (float(v) for v in manual[b])
            rep["applied"] = {"source": "config detection.band_offsets_arcsec", "offset_arcsec": [ox, oy]}
        elif reliable and dcfg.get("register_bands", True) and np.hypot(dx, dy) > 0.01:
            ox, oy = dx, dy
            rep["applied"] = {"source": "measured", "offset_arcsec": [round(ox, 4), round(oy, 4)]}
        if rep["applied"]:
            im["wcs"] = shift_wcs(im["wcs"], ox, oy)
            dx, dy, n2, _, sig2 = offset(im["wcs"])
            reliable = n2 >= 20 and sig2 >= min_sig
            rep.update(n_pairs_after=n2, significance_after=round(sig2, 1))
        rep["offset_after_arcsec"] = [round(dx, 4), round(dy, 4)] if reliable else None
        rep["residual_arcsec"] = round(float(np.hypot(dx, dy)), 4) if reliable else None
        report[b] = rep
        ctx.log.info(f"[Stage 1c] {pid}: {b} vs {det_b}: {rep['n_pairs']} matched sources (peak {rep['significance']}x the "
                     f"random-pair level), offset {rep['offset_measured_arcsec']}\" -> "
                     f"{'corrected (' + rep['applied']['source'] + ')' if rep['applied'] else 'no correction'}, "
                     f"residual {rep['offset_after_arcsec']}\" (dRA cos(dec), dDec)")
        if not reliable:
            raise RuntimeError(
                f"[Stage 1c] {pid}: the offset of {b} against {det_b} cannot be measured reliably ({rep['n_pairs']} "
                f"pairs, {rep['significance']}x the random level; need >= 20 pairs and >= {min_sig}x). Colours and "
                f"two-band tests would use unchecked positions. If you know the offset, give it in config.yaml, e.g. "
                f"detection.band_offsets_arcsec: {{{b}: [dRA_cos_dec, dDec]}} (arcsec, band minus {det_b}).")
        if np.hypot(dx, dy) > limit:
            raise RuntimeError(
                f"[Stage 1c] {pid}: {b} is still offset by {np.hypot(dx, dy):.3f}\" from {det_b} after registration "
                f"(limit detection.max_band_offset_arcsec = {limit}\"). "
                + ("Turn on detection.register_bands." if not dcfg.get("register_bands", True) else
                   "Check detection.band_offsets_arcsec, or use a file whose WCS is aligned to a star catalogue."))
    return report


def process_pointing(ctx, pid, p, bands, rules, qc_boxes, pidx):
    dcfg = ctx.cfg["detection"]
    S = dcfg["cutout_size"]
    corr = dcfg.get("correlated_noise_factor", 1.0)
    imgs = {}
    for b, f in p["bands"].items():
        sci, wht, w = load_image(f)
        val = valid_mask(sci, wht)
        sub, rms = subtract_background(sci, val, dcfg["background_box"], dcfg["background_filter"])
        imgs[b] = {"sub": sub, "rms": rms, "val": val, "valf": val.astype(np.float32), "wcs": w, "file": f,
                   "scale": p["pixscale"][b], "pixarea": p["pixscale"][b] ** 2, "zp": p["zp_ab"][b],
                   "exptime": max(float(read_meta(Path(f))["exptime"]), 1.0)}
        del sci, wht
    det_b = p["detection_band"]
    det = imgs[det_b]
    segm, conv = detect_and_deblend(det["sub"], det["rms"], det["val"], dcfg)
    if n_labels(segm) == 0:
        ctx.log.info(f"[Stage 1c] {pid}: no sources detected")
        return None, None, None, None
    err = calc_total_error(det["sub"], det["rms"] * corr, effective_gain=max(p["exptime_detection"], 1.0))
    cat = SourceCatalog(det["sub"], segm, convolved_data=conv, error=err)
    x, y = values(cat, "x_centroid"), values(cat, "y_centroid")
    kron = values(cat, "kron_flux")
    kerr = values(cat, "kron_flux_err")
    segf = values(cat, "segment_flux")
    flux = np.where(np.isfinite(kron) & (kron > 0), kron, segf)
    ferr = np.where(np.isfinite(kerr) & (kerr > 0), kerr, values(cat, "segment_flux_err"))
    ra, dec = det["wcs"].pixel_to_world_values(x, y)
    tab = pd.DataFrame({
        "label": np.asarray(cat.labels, int), "x": x, "y": y, "ra": ra, "dec": dec,
        "area_px": values(cat, "area"), "semimajor_px": values(cat, "semimajor_axis"),
        "semiminor_px": values(cat, "semiminor_axis"), "ellipticity": values(cat, "ellipticity"),
        "orientation_deg": values(cat, "orientation"), "fwhm_px": values(cat, "fwhm"),
        "flux": flux, "flux_err": ferr, "snr": flux / np.where(ferr > 0, ferr, np.nan)})
    tab = tab[np.isfinite(tab.x) & np.isfinite(tab.ra) & (tab.flux > 0)]
    tab["saturated"] = saturated_cores(det["sub"], det["rms"], tab.x.to_numpy(), tab.y.to_numpy())

    # ---- 1. register every band to the detection band (fails on a residual offset) ----
    registration = register_bands(ctx, pid, imgs, det_b, dcfg)

    # ---- 2. every segment measured in every band -> cosmic-ray candidates (no second band confirms) ----
    lab_img = segm.data
    max_lab = int(lab_img.max())
    li = tab.label.to_numpy()
    confirm = float(dcfg.get("confirm_snr", 3.0))
    # allowed centroid shift between bands: 1.5 px, or half the equivalent radius for large galaxies
    max_shift = np.maximum(float(dcfg.get("confirm_max_shift_px", 1.5)),
                           float(dcfg.get("confirm_max_shift_frac", 0.5)) * np.sqrt(tab.area_px.to_numpy() / np.pi))
    best_other = np.full(len(tab), np.nan)
    n_cover_other = np.zeros(len(tab), int)
    for b, im in imgs.items():                       # cosmic-ray-free copies for the centroid test
        im["med"], im["med_noise"] = median_image(im["sub"], im["val"], 3)
    phot = {b: segment_photometry(lab_img, det["wcs"], imgs.get(b), det["pixarea"], max_lab, corr) for b in bands}
    dcx, dcy = phot[det_b][3][li], phot[det_b][4][li]
    for b in bands:
        fl, er, cov, cx, cy = phot[b]
        tab[f"segflux_{b}"] = fl[li]
        tab[f"segsnr_{b}"] = (fl / er)[li]
        tab[f"segcov_{b}"] = cov[li]
        tab[f"segshift_{b}"] = np.hypot(cx[li] - dcx, cy[li] - dcy)       # centroid shift vs detection band [px]
        if b != det_b and b in imgs:
            covered = cov[li] >= 0.9
            n_cover_other += covered
            # a band confirms only with light at the same position
            snr_here = np.where(tab[f"segshift_{b}"].to_numpy() <= max_shift, (fl / er)[li], np.nan)
            best_other = np.fmax(best_other, np.where(covered, snr_here, np.nan))
    tab["confirm_snr_best"] = best_other
    # True = no other covering band sees it; False = confirmed; None = no other band covers it
    use_cr = str(dcfg.get("cr_rejection", "two_band")).lower() != "none"
    tab["cr_candidate"] = [None if (not use_cr or n == 0) else bool(not (np.isfinite(s) and s >= confirm))
                           for n, s in zip(n_cover_other, best_other)]
    is_cr = tab.cr_candidate.map(lambda v: v is True).astype(bool)
    ctx.log.info(f"[Stage 1c] {pid}: {len(tab)} detections | confirmed by another band {int((~is_cr & (n_cover_other > 0)).sum())} "
                 f"| cosmic-ray candidates {int(is_cr.sum())} | no other band covers {int((n_cover_other == 0).sum())}")

    # ---- 3. caps: real (confirmed / unknown) sources and cosmic-ray candidates separately ----
    cap, cap_cr = int(dcfg["max_sources_per_pointing"]), int(dcfg.get("max_cr_candidates_per_pointing", 1000))
    real, crs = tab[~is_cr], tab[is_cr]
    if len(real) > cap:
        ctx.log.info(f"[Stage 1c] {pid}: {len(real)} confirmed sources, keeping the brightest {cap} "
                     "(detection.max_sources_per_pointing)")
        real = real.nlargest(cap, "flux")
    if len(crs) > cap_cr:
        ctx.log.info(f"[Stage 1c] {pid}: {len(crs)} cosmic-ray candidates, keeping the brightest {cap_cr} "
                     "(detection.max_cr_candidates_per_pointing; they stay in the catalogue as Noise)")
        crs = crs.nlargest(cap_cr, "flux")
    tab = pd.concat([real, crs]).sort_values("label").reset_index(drop=True)

    # ---- 4. aligned stamps; cosmic-ray pixels replaced and marked 3 ----
    seg_f = lab_img.astype(np.float32)          # converted once, sampled per source
    scale = det["scale"] if dcfg["cutout_pixscale_arcsec"] == "auto" else float(dcfg["cutout_pixscale_arcsec"])
    cubes = np.zeros((len(tab), len(bands), S, S), np.float16)
    segs = np.zeros((len(tab), S, S), np.uint8)
    yy, xx = np.mgrid[:S, :S]
    core = np.hypot(yy - (S - 1) / 2, xx - (S - 1) / 2) <= dcfg["core_radius_px"]
    grow = int(dcfg.get("cr_mask_grow_px", 1))
    mag_snr = float(dcfg.get("mag_min_snr", 2.0))
    rows = []
    for i, r in enumerate(tab.itertuples()):
        n_nb = 0
        raw = np.zeros((len(bands), S, S), np.float32)
        sig = np.full((len(bands), S, S), np.inf, np.float32)
        val = np.zeros((len(bands), S, S), bool)
        for bi, b in enumerate(bands):
            if b not in imgs:
                continue
            im = imgs[b]
            M, off, area_ratio = affine_for(im["wcs"], r.ra, r.dec, S, scale)
            raw[bi] = make_stamp(im["sub"], M, off, S) * area_ratio
            val[bi] = make_stamp(im["valf"], M, off, S, order=0) > 0.5
            bx, by = im["wcs"].world_to_pixel_values(r.ra, r.dec)
            yi = int(np.clip(round(float(by)), 0, im["rms"].shape[0] - 1))
            xi = int(np.clip(round(float(bx)), 0, im["rms"].shape[1] - 1))
            rms_px = float(im["rms"][yi, xi]) * area_ratio * corr
            sig[bi] = np.sqrt(rms_px ** 2 + np.clip(raw[bi], 0, None) * area_ratio / im["exptime"])
            if b == det_b:
                lab = make_stamp(seg_f, M, off, S, order=0).astype(np.int64)
                segs[i][(lab > 0) & (lab != r.label)] = 2
                segs[i][lab == r.label] = 1
                n_nb = len(set(np.unique(lab).tolist()) - {0, r.label})
        valid_det = val[bands.index(det_b)]
        target = segs[i] == 1                     # full target footprint, before the cosmic-ray mask
        sflux = np.array([getattr(r, f"segflux_{b}") for b in bands], float)
        ssnr = np.array([getattr(r, f"segsnr_{b}") for b in bands], float)
        n_cr_target = 0
        if use_cr and len(imgs) > 1:
            crm, repl = cr_pixel_mask(raw, val, segs[i], sig, sflux, ssnr, float(dcfg.get("cr_pixel_nsigma", 5.0)),
                                      float(dcfg.get("cr_pixel_ratio", 4.0)))
            raw = np.where(crm, repl, raw)
            anyc = crm.any(axis=0)
            if anyc.any():
                if grow > 0:                      # PSF matching (Stage 1d) spreads a residual a little
                    anyc = ndimage.binary_dilation(anyc, iterations=grow)
                anyc &= segs[i] != 2
                n_cr_target = int((anyc & target).sum())
                segs[i][anyc] = 3
        cubes[i] = raw.astype(np.float16)
        row = {"valid_frac": float(valid_det.mean()), "core_ok": bool(valid_det[core].all()), "n_neighbours": n_nb,
               "cr_pixels_masked": n_cr_target}
        for bi, b in enumerate(bands):
            covered = b in imgs and bool(val[bi][core].all())          # this band has data on the source core
            row[f"covered_{b}"] = covered
            # flux over the whole target footprint, cosmic-ray pixels replaced
            fl = float(raw[bi][target & val[bi]].astype(np.float64).sum()) if covered else np.nan
            row[f"flux_{b}"] = fl
            ok = covered and fl > 0 and np.isfinite(imgs[b]["zp"]) and np.nan_to_num(ssnr[bi]) >= mag_snr
            row[f"mag_{b}"] = -2.5 * np.log10(fl) + imgs[b]["zp"] if ok else np.nan
            e = sflux[bi] / ssnr[bi] if np.isfinite(ssnr[bi]) else np.nan
            row[f"magerr_{b}"] = 1.0857 * e / fl if ok and np.isfinite(e) else np.nan
        rows.append(row)
    tab = pd.concat([tab, pd.DataFrame(rows)], axis=1)
    tab["object_id"] = (pidx + 1) * 100000 + tab["label"]
    tab["pointing"] = pid
    tab["role"] = p["role"]
    tab["detection_band"] = det_b
    tab["pixscale_out"] = scale
    tab["exptime"] = p["exptime_detection"]
    tab["mag"] = -2.5 * np.log10(tab["flux"]) + (det["zp"] if np.isfinite(det["zp"]) else 0.0)
    # statistical centroid error = source size / SNR per axis (absolute WCS error not included)
    pos_err = tab.fwhm_px / 2.3548 / tab.snr * det["scale"]
    tab["ra_err"] = pos_err.round(5)
    tab["dec_err"] = pos_err.round(5)
    ctx_img = load_ctx(det["file"])
    if ctx_img is not None:
        xi = np.clip(np.round(tab.x).astype(int), 0, ctx_img.shape[1] - 1)
        yi = np.clip(np.round(tab.y).astype(int), 0, ctx_img.shape[0] - 1)
        tab["n_exposures"] = popcount(ctx_img[yi, xi])
    else:
        tab["n_exposures"] = np.nan
    tab["edge_flag"] = (tab["valid_frac"] < dcfg["min_valid_fraction"]) | (~tab["core_ok"])
    bad = np.zeros(len(tab), bool)
    bad_bands = [[] for _ in range(len(tab))]
    for b, im in imgs.items():                 # a source is QC-flagged if ANY band's tile is flagged
        bx, by = im["wcs"].world_to_pixel_values(tab.ra.to_numpy(), tab.dec.to_numpy())
        hit = in_flagged(bx, by, qc_boxes.get(im["file"]))
        bad |= hit
        for j in np.where(hit)[0]:
            bad_bands[j].append(b)
    tab["frame_quality_flag"] = (~bad).astype(int)          # 1 = clean, 0 = in a flagged tile
    tab["qc_flagged_bands"] = [",".join(v) for v in bad_bands]
    tab["bands_missing"] = ",".join(p["missing_bands"])
    split = rules.assign_many(tab.ra, tab.dec)
    if p["role"] == "external":
        split = ["external" if s == "external" else None for s in split]
    tab["split_tag"] = [s if s is not None else "buffer" for s in split]     # buffer = no split (collar)
    hit = tab.cr_pixels_masked[tab.cr_pixels_masked > 0]
    ctx.log.info(f"[Stage 1c] {pid}: cosmic-ray pixels masked in {len(hit)} of {len(tab)} target footprints "
                 f"(median {hit.median() if len(hit) else 0} px)")
    _preview(ctx, pid, det, tab, cubes, bands)
    return tab, cubes, segs, registration


def run(ctx: Ctx):
    out = ctx.stage_dir("1c")
    info = load_json(ctx.require("1a", "pointings.json"))
    bands, pts = info["bands"], info["pointings"]
    rules = SkyRegionPartitioner.from_file(ctx.require("0", "split_rule.json"))
    qc = flagged_tiles(ctx)
    parts, registration = [], {}
    for pidx, (pid, p) in enumerate(pts.items()):
        ctx.log.info(f"[Stage 1c] {pid}: detecting on {p['detection_band']} ...")
        tab, cubes, segs, reg = process_pointing(ctx, pid, p, bands, rules, qc, pidx)
        if tab is None:
            continue
        registration[pid] = reg
        tmp = out / f"_tmp_{pid}"
        np.save(f"{tmp}_cut.npy", cubes)
        np.save(f"{tmp}_seg.npy", segs)
        parts.append((tab, tmp))
        ctx.log.info(f"[Stage 1c] {pid}: {len(tab)} sources | median SNR {tab.snr.median():.1f} | "
                     f"edge-flagged {int(tab.edge_flag.sum())} | QC-flagged {int((tab.frame_quality_flag == 0).sum())} | "
                     f"splits {tab.split_tag.value_counts(dropna=False).to_dict()}")
    if not parts:
        raise RuntimeError("No sources detected in any pointing.")

    # ---- merge pointings into single arrays ----
    n = sum(len(t) for t, _ in parts)
    S = ctx.cfg["detection"]["cutout_size"]
    cut = np.lib.format.open_memmap(out / "cutouts_raw.npy", mode="w+", dtype=np.float16, shape=(n, len(bands), S, S))
    seg = np.lib.format.open_memmap(out / "segstamps.npy", mode="w+", dtype=np.uint8, shape=(n, S, S))
    k, tabs = 0, []
    for tab, tmp in parts:
        c, s = np.load(f"{tmp}_cut.npy"), np.load(f"{tmp}_seg.npy")
        cut[k:k + len(tab)], seg[k:k + len(tab)] = c, s
        tab["cutout_index"] = np.arange(k, k + len(tab))
        tabs.append(tab)
        k += len(tab)
        Path(f"{tmp}_cut.npy").unlink()
        Path(f"{tmp}_seg.npy").unlink()
    cut.flush()
    seg.flush()
    sources = pd.concat(tabs, ignore_index=True)
    bl = list(bands)
    sources["colour_1"] = sources[f"mag_{bl[0]}"] - sources[f"mag_{bl[1]}"] if len(bl) > 1 else np.nan
    sources["colour_2"] = sources[f"mag_{bl[1]}"] - sources[f"mag_{bl[2]}"] if len(bl) > 2 else np.nan
    # band consistency: weakest other-band flux / detection-band flux, over the bands that cover the source
    # (a real source shows in every filter, a cosmic ray in one)
    det_flux = sources.apply(lambda r: r[f"flux_{r.detection_band}"], axis=1)
    ratios = [(sources[f"flux_{b}"] / det_flux).where((sources.detection_band != b) & sources[f"covered_{b}"])
              for b in bl]
    sources["band_consistency"] = np.clip(pd.concat(ratios, axis=1).min(axis=1), 0, 2) if len(bl) > 1 else np.nan
    front = ["object_id", "pointing", "role", "split_tag", "ra", "dec", "x", "y", "mag", "snr", "flux", "flux_err"]
    sources = sources[front + [c for c in sources.columns if c not in front]]
    sources.to_csv(out / "sources.csv", index=False)
    save_json(registration, out / "registration.json")
    save_json({"bands": bands, "cutout_size": S, "n_sources": n,
               "split_counts": sources["split_tag"].value_counts(dropna=False).to_dict()}, out / "summary.json")
    ctx.log.info(f"[Stage 1c] {n} sources total -> sources.csv, cutouts_raw.npy {cut.shape}")
    return {"n_sources": n}


def _preview(ctx, pid, det, tab, cubes, bands):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from ..utils import asinh_normalise
    out = ctx.stage_dir("1c")
    step = max(1, max(det["sub"].shape) // 1024)
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(asinh_normalise(det["sub"][::step, ::step]), origin="lower", cmap="gray")
    sc = ax.scatter(tab.x / step, tab.y / step, s=8, c=np.log10(tab.snr.clip(1)), cmap="viridis",
                    edgecolors="none")
    fig.colorbar(sc, ax=ax, fraction=0.046, label="log10 SNR")
    ax.set_title(f"Stage 1c {pid}: {len(tab)} detections on {tab.detection_band.iloc[0]}")
    ax.set_xticks([])
    ax.set_yticks([])
    fig.tight_layout()
    fig.savefig(out / f"{pid}_detections.png", dpi=100)
    plt.close(fig)
    # 16 brightest stamps as colour composites (red = reddest band)
    pick = tab.sort_values("flux", ascending=False).index[:16]
    fig, axes = plt.subplots(4, 4, figsize=(8, 8))
    for ax, i in zip(axes.ravel(), pick):
        cube = cubes[i].astype(np.float32)
        rgb = np.stack([asinh_normalise(cube[min(j, len(bands) - 1)]) for j in (2, 1, 0)], -1)
        ax.imshow(rgb, origin="lower")
        ax.set_title(f"id {tab.loc[i, 'label']}  SNR {tab.loc[i, 'snr']:.0f}", fontsize=7)
        ax.axis("off")
    for ax in axes.ravel()[len(pick):]:
        ax.axis("off")
    fig.suptitle(f"{pid}: brightest cutouts (aligned, {len(bands)} bands)")
    fig.tight_layout()
    fig.savefig(out / f"{pid}_cutouts.png", dpi=100)
    plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage1d_psf_match.py
"""Stage 1d - Cross-filter PSF matching (no learned model).

Purpose : give every band of a cutout the same resolution, so colour differences in the PSF are not read as
          galaxy structure.
Method  : 1. PSF stars per pointing: S/N >= star_min_snr, round (ellipticity < star_max_ellipticity), not
             saturated, not a cosmic-ray candidate, seen in a second band, Gaussian-fit FWHM within 15 % of the
             stellar locus (median of the Gaia stars, else the peak of the size distribution near the expected
             width), core free of neighbours (star_isolation_px).
          2. Per band, stars with S/N >= star_min_snr_per_band are recentred on the detection-band core and
             median-stacked; the band FWHM = median of the stars' 2-D Gaussian fits. With fewer than min_stars
             the band BORROWS the stack of the pointing with most stars in that band (same camera, filter, pixel
             scale); a Gaussian of the instrument width only if no pointing has stars. No stars anywhere -> stop
             (require_stars).
          3. The broadest band is the target; every sharper band is convolved with a Gaussian kernel,
             sigma_K^2 = sigma_target^2 - sigma_band^2 (or photutils' Fourier-ratio kernel: kernel: photutils).
Output  : cutouts.npy (PSF-matched, used by all later stages), psf_match.json, psf_<pointing>_<band>.npy,
          placeholder_psf_<pointing>.npy (Gaussian of the target FWHM for Stage 1e; the empirical PSF is Stage 5a)
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import ndimage
from scipy.signal import fftconvolve

from ..context import Ctx, load_json, save_json
from ..utils import as_bool, fwhm_fit, gaussian_psf

def crop_center(img, n):
    c = img.shape[0] // 2
    h = n // 2
    return img[c - h:c + h + 1, c - h:c + h + 1]


def core_centroid(stamp, half=3):
    """Flux centroid of the central (2*half+1)^2 pixels only (neighbours cannot pull it)."""
    s = np.clip(np.asarray(stamp, float), 0, None)
    c = s.shape[0] // 2
    box = s[c - half:c + half + 1, c - half:c + half + 1]
    if box.sum() <= 0:
        return float(c), float(c)
    cy, cx = ndimage.center_of_mass(box)
    return float(cy + c - half), float(cx + c - half)


def stack_stars(stamps, centers=None, masks=None):
    """Recentre each star (cubic sub-pixel shift) on centers (detection-band cores, so every band is shifted
    identically), normalise, median; masks (True = other source) are left out."""
    n = stamps[0].shape[0]
    c = (n - 1) / 2.0
    out = []
    for k, s in enumerate(stamps):
        s = np.asarray(s, float)
        cy, cx = centers[k] if centers is not None else core_centroid(s)
        bad = np.asarray(masks[k], bool) if masks is not None else np.zeros_like(s, bool)
        s = ndimage.shift(np.where(bad, 0.0, s), (c - cy, c - cx), order=3, mode="constant")
        bad = ndimage.shift(bad.astype(float), (c - cy, c - cx), order=0, mode="constant", cval=1.0) > 0.5
        tot = s[~bad].sum()
        if tot > 0:
            out.append(np.where(bad, np.nan, s / tot))
    psf = np.nan_to_num(np.clip(np.nanmedian(out, axis=0), 0, None))
    return psf / psf.sum()


def star_masks(segs, idx, n):
    """Pixels of other sources (2) in each star stamp. Cosmic-ray pixels (3) are not masked: Stage 1c already
    replaced them in the band they hit."""
    return [crop_center(np.asarray(segs[i]), n) == 2 for i in idx]


def flat_top(stamp, tol=0.02):
    top = np.sort(np.asarray(stamp, float).ravel())[-5:]
    return (top[-1] - top[0]) / max(abs(top[-1]), 1e-12) < tol


def matching_kernel(psf_sharp, psf_broad, beta, method="gaussian", fwhm_sharp=None, fwhm_broad=None):
    """gaussian (default): Gaussian K with sigma_K^2 = sigma_broad^2 - sigma_sharp^2 (robust for noisy stacks);
    photutils: Fourier-ratio kernel with a TopHat window (needs clean PSFs)."""
    if method == "gaussian":
        fs = fwhm_sharp if fwhm_sharp is not None else fwhm_fit(psf_sharp)
        fb = fwhm_broad if fwhm_broad is not None else fwhm_fit(psf_broad)
        fk = np.sqrt(max(fb ** 2 - fs ** 2, 0.0))
        if fk < 0.3:
            k = np.zeros_like(psf_sharp)
            k[k.shape[0] // 2, k.shape[1] // 2] = 1.0
            return k
        return gaussian_psf(fk, psf_sharp.shape[0])
    from photutils.psf.matching import TopHatWindow, create_matching_kernel
    return create_matching_kernel(psf_sharp, psf_broad, window=TopHatWindow(beta))


def convolve_stamp(stamp, K):
    pad = K.shape[0] // 2
    padded = np.pad(stamp, pad, mode="reflect")
    return fftconvolve(padded, K, mode="same")[pad:-pad, pad:-pad]


def gaia_flags(ctx, src, gaia):
    """True for sources within gaia_radius_arcsec of a Gaia DR3 star."""
    if gaia is None or gaia.empty:
        return np.zeros(len(src), bool)
    from ..utils import sky_crossmatch
    return sky_crossmatch(src.ra, src.dec, gaia.ra, gaia.dec, ctx.cfg["semantic_audit"]["gaia_radius_arcsec"])[2]


def load_gaia_once(ctx):
    """Gaia DR3 stars of the field (data/catalogs), read once for all pointings; None if unavailable."""
    from ..catalogs import load_gaia
    try:
        return load_gaia(ctx, list(load_json(ctx.require("1a", "manifest.json")).values()))
    except Exception as e:
        ctx.log.info(f"[Stage 1d] WARNING: Gaia catalogue not usable ({e!r}); stellar locus from the size peak.")
        return None


def select_stars(src: pd.DataFrame, segs, raw, bands, present, cfg, expected_px, gaia=None):
    """Bright, round, isolated, compact sources detected in every band that COVERS them and in at least
    one band besides the detection band (a cosmic ray is as compact as a star but appears in one band only)."""
    bright = src[(src.snr >= cfg["star_min_snr"]) & (~src.edge_flag.astype(bool))
                 & (src.ellipticity < cfg.get("star_max_ellipticity", 0.3))]
    if "cr_candidate" in bright:                # never a cosmic ray (no second band confirms it)
        bright = bright[~as_bool(bright.cr_candidate)]
    if "saturated" in bright:
        bright = bright[~as_bool(bright.saturated)]
    det = src.detection_band.iloc[0] if len(src) else None
    others = [b for b in present if b != det]
    seen_in_other = np.zeros(len(bright), bool)
    for b in others:
        cov = bright[f"covered_{b}"].astype(bool) if f"covered_{b}" in bright else pd.Series(True, index=bright.index)
        ok_b = bright[f"flux_{b}"] > 0.2 * bright[f"flux_{det}"]
        keep = ~cov | ok_b                      # detected in every band that COVERS it ...
        seen_in_other = seen_in_other[keep.to_numpy()] | ok_b[keep].to_numpy()
        bright = bright[keep]
    if others:
        bright = bright[seen_in_other]          # ... and in at least one other band (cosmic rays are single-band)
    if len(bright) == 0:
        return bright, np.nan
    # size from a 2-D Gaussian fit of the detection-band stamp (the segment-moment FWHM depends on the isophote)
    det_i = bands.index(det)
    size = []
    for i in bright.cutout_index:
        img = np.where(np.asarray(segs[i]) == 2, 0.0, np.asarray(raw[i, det_i], np.float32))
        size.append(fwhm_fit(crop_center(img, cfg["stamp"])))
    bright = bright.assign(fit_fwhm_px=size)
    # stellar locus: median size of >= 2 Gaia stars, else the peak of the size distribution near the expected
    # PSF width (cosmic-ray coincidences are sharper, compact galaxies broader)
    g = bright.index.isin(src.index[gaia]) if gaia is not None else np.zeros(len(bright), bool)
    if g.sum() >= 2:
        locus = float(np.median(bright.fit_fwhm_px[g]))
    else:
        lo, hi = 0.85 * expected_px, 1.5 * expected_px
        inwin = bright.fit_fwhm_px.between(lo, hi)
        if inwin.sum() < 3:
            return bright.iloc[:0], np.nan
        h, e = np.histogram(bright.fit_fwhm_px[inwin], bins=np.arange(lo, hi + 0.1, 0.1))
        h = np.convolve(h, np.ones(3) / 3, mode="same")
        locus = float(0.5 * (e[np.argmax(h)] + e[np.argmax(h) + 1]))
    cand = bright[np.abs(bright.fit_fwhm_px - locus) <= 0.15 * locus]
    S = segs.shape[1]
    yy, xx = np.mgrid[:S, :S]
    # only the core must be free of other segments; neighbours further out are masked in the stack
    near = np.hypot(yy - (S - 1) / 2, xx - (S - 1) / 2) <= cfg.get("star_isolation_px", 5)
    iso = [not np.any(segs[i][near] == 2) for i in cand.cutout_index]
    cand = cand[np.asarray(iso, bool)]
    return cand.nlargest(cfg["max_stars"], "snr"), float(locus)


def _fallback_fw(cfg, p):
    fb = cfg["fallback_fwhm_arcsec"]
    return fb.get(p["instrument"], fb.get(p["instrument"].split("_")[0], fb["default"]))


def measure_pointing(ctx, cfg, pid, p, rows, bands, raw, segs, gaia):
    """PSF stars and per-band stacked PSFs of one pointing (bands with too few stars are left out)."""
    expected = _fallback_fw(cfg, p) / float(rows.pixscale_out.iloc[0])
    stars, locus = select_stars(rows, segs, raw, bands, list(p["bands"]), cfg, expected, gaia_flags(ctx, rows, gaia))
    det_i = bands.index(p["detection_band"])
    stars = stars[np.array([not flat_top(raw[i, det_i]) for i in stars.cutout_index], dtype=bool)]
    if len(stars):                              # Gaussian-fit size of the PSF stars (same estimator as the PSFs)
        locus = float(np.median(stars.fit_fwhm_px))
    psfs, method, star_fwhm, n_per_band = {}, {}, {}, {}
    for bi, b in enumerate(bands):
        if b not in p["bands"]:
            continue
        in_b = stars[stars[f"covered_{b}"].astype(bool)] if f"covered_{b}" in stars else stars
        if f"segsnr_{b}" in in_b:              # a red star is too faint in a blue band to shape its PSF
            in_b = in_b[in_b[f"segsnr_{b}"].fillna(0) >= cfg.get("star_min_snr_per_band", 30)]
        n_per_band[b] = int(len(in_b))
        if len(in_b) >= cfg["min_stars"]:
            idx = in_b.cutout_index.tolist()
            cen = [core_centroid(crop_center(raw[i, det_i].astype(np.float32), cfg["stamp"])) for i in idx]
            masks = star_masks(segs, idx, cfg["stamp"])
            crops = [crop_center(raw[i, bi].astype(np.float32), cfg["stamp"]) for i in idx]
            psfs[b] = stack_stars(crops, cen, masks)
            star_fwhm[b] = float(np.median([fwhm_fit(np.where(m, 0.0, c)) for c, m in zip(crops, masks)]))
            method[b] = f"stacked {len(in_b)} stars"
    if psfs:                                    # a stack far broader than the other bands = too few clean stars
        fw = {b: star_fwhm.get(b, fwhm_fit(v)) for b, v in psfs.items()}
        med = float(np.median(list(fw.values())))
        for b in [b for b, f in fw.items() if f > 1.6 * med and len(fw) > 1]:
            ctx.log.info(f"[Stage 1d] {pid}: WARNING - the {b} PSF stack ({fw[b]:.2f} px) is > 1.6 x the median of the "
                         f"bands ({med:.2f} px): too few clean stars in {b}; a borrowed / fallback PSF is used for it.")
            psfs.pop(b)
            method.pop(b)
            star_fwhm.pop(b, None)
    return {"stars": stars, "locus": locus, "psfs": psfs, "method": method, "star_fwhm": star_fwhm,
            "n_per_band": n_per_band, "scale": float(rows.pixscale_out.iloc[0])}


def run(ctx: Ctx):
    cfg = ctx.cfg["psf_match"]
    out = ctx.stage_dir("1d")
    info = load_json(ctx.require("1a", "pointings.json"))
    bands, pts = info["bands"], info["pointings"]
    src = pd.read_csv(ctx.require("1c", "sources.csv"))
    raw = np.load(ctx.require("1c", "cutouts_raw.npy"), mmap_mode="r")
    segs = np.load(ctx.require("1c", "segstamps.npy"), mmap_mode="r")
    matched = np.lib.format.open_memmap(out / "cutouts.npy", mode="w+", dtype=np.float16, shape=raw.shape)

    # pass 1: measure every pointing
    gaia = load_gaia_once(ctx)
    meas = {}
    for pid, p in pts.items():
        rows = src[src.pointing == pid]
        if not rows.empty:
            meas[pid] = measure_pointing(ctx, cfg, pid, p, rows, bands, raw, segs, gaia)

    # library of measured PSFs per (instrument, pixel scale, band): the pointing with most stars in that band
    library = {}
    for pid, m in meas.items():
        for b, psf in m["psfs"].items():
            key = (pts[pid]["instrument"], round(m["scale"], 4), b)
            n = m["n_per_band"][b]
            if key not in library or n > library[key][2]:
                library[key] = (psf, m["star_fwhm"].get(b, fwhm_fit(psf)), n, pid)
    if not library and cfg.get("require_stars", True):
        raise RuntimeError(
            f"[Stage 1d] no PSF star passed the cuts in ANY pointing (SNR >= {cfg['star_min_snr']}, round, "
            "isolated, unsaturated, confirmed in a second band). The PSFs would be the configured Gaussian, not a "
            "measurement. Check the Stage 1c registration / cosmic-ray log lines; to continue with the Gaussian "
            "fallback anyway set psf_match.require_stars: false (the catalogue then stores no measured matched FWHM).")

    # pass 2: fill missing bands (borrow, else Gaussian), match, write cutouts
    report = {}
    for pid, m in meas.items():
        p = pts[pid]
        rows = src[src.pointing == pid]
        stars, locus, scale = m["stars"], m["locus"], m["scale"]
        psfs, method, star_fwhm = dict(m["psfs"]), dict(m["method"]), dict(m["star_fwhm"])
        for b in bands:
            if b not in p["bands"] or b in psfs:
                continue
            key = (p["instrument"], round(scale, 4), b)
            if key in library:
                psf, fw_px, n, src_pid = library[key]
                psfs[b], star_fwhm[b] = psf, fw_px
                method[b] = f"borrowed from {src_pid} ({n} stars; only {m['n_per_band'].get(b, 0)} here)"
            else:
                fw = _fallback_fw(cfg, p)
                psfs[b] = gaussian_psf(fw / scale, cfg["stamp"])
                method[b] = f"Gaussian fallback ({fw}\" FWHM; only {m['n_per_band'].get(b, 0)} stars cover {b})"
        if any(v.startswith("borrowed") for v in method.values()):
            ctx.log.info(f"[Stage 1d] {pid}: {len(stars)} PSF stars here - PSFs borrowed from a pointing with the same "
                         f"camera, filter and pixel scale: { {b: v for b, v in method.items() if v.startswith('borrowed')} }")
        for b, v in psfs.items():
            np.save(out / f"psf_{pid}_{b}.npy", v)
        fwhm = {b: star_fwhm.get(b, fwhm_fit(v)) for b, v in psfs.items()}
        target = max(fwhm, key=fwhm.get)
        kernels = {b: matching_kernel(psfs[b], psfs[target], cfg["window"], cfg.get("kernel", "gaussian"),
                                      fwhm[b], fwhm[target]) for b in psfs if b != target}
        # FWHM after matching: Gaussian kernel -> widths add in quadrature (no kernel if f_K < 0.3 px);
        # photutils kernel -> fit of the convolved stack
        def after_matching(b, K):
            if cfg.get("kernel", "gaussian") != "gaussian":
                return fwhm_fit(fftconvolve(psfs[b], K, mode="same"))
            fk = np.sqrt(max(fwhm[target] ** 2 - fwhm[b] ** 2, 0.0))
            return float(np.hypot(fwhm[b], fk)) if fk >= 0.3 else fwhm[b]
        check = {b: round(after_matching(b, K), 3) for b, K in kernels.items()}
        for i in rows.cutout_index:
            cube = raw[i].astype(np.float32)
            for bi, b in enumerate(bands):
                if b in kernels:
                    cube[bi] = convolve_stamp(cube[bi], kernels[b])
            matched[i] = cube.astype(np.float16)
        np.save(out / f"placeholder_psf_{pid}.npy", gaussian_psf(fwhm[target], cfg["stamp"]))
        if not np.isfinite(locus):          # no bright compact sources: use the PSF width
            locus = fwhm[target]
        measured = method[target].startswith(("stacked", "borrowed"))
        if not measured:
            ctx.log.info(f"[Stage 1d] {pid}: WARNING - the target band {target} uses the Gaussian fallback; "
                         "no measured matched FWHM is stored.")
        report[pid] = {"n_stars": int(len(stars)), "star_ids": stars.object_id.tolist(), "method": method,
                       "target_psf_measured": bool(measured),
                       "target_psf_borrowed": method[target].startswith("borrowed"),
                       "matched_fwhm_arcsec": round(fwhm[target] * scale, 4) if measured else None,
                       "stellar_locus_fwhm_px": round(float(locus), 3),
                       "fwhm_px": {b: round(v, 3) for b, v in fwhm.items()}, "target_band": target,
                       "matched_fwhm_px": check, "pixscale_arcsec": scale}
        ctx.log.info(f"[Stage 1d] {pid}: stars={len(stars)} FWHM(px)={report[pid]['fwhm_px']} -> target {target}; "
                     f"after matching {check}")
    matched.flush()
    save_json(report, out / "psf_match.json")
    _plot(report, out)
    return {"pointings": len(report)}


def _plot(report, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    for pid in report:
        files = sorted(out.glob(f"psf_{pid}_*.npy"))
        if not files:
            continue
        fig, axes = plt.subplots(1, len(files), figsize=(3 * len(files), 3))
        for ax, f in zip(np.atleast_1d(axes), files):
            ax.imshow(np.log10(np.load(f) + 1e-5), origin="lower", cmap="magma")
            b = f.stem.split("_")[-1]
            ax.set_title(f"{b}: FWHM {report[pid]['fwhm_px'][b]:.2f} px", fontsize=9)
            ax.axis("off")
        fig.suptitle(f"Stage 1d {pid}: per-band PSFs (log)")
        fig.tight_layout()
        fig.savefig(out / f"{pid}_psfs.png", dpi=100)
        plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage1e_pass1_morphology.py
"""Stage 1e - Pass-1 photometry and morphology.

Purpose : a provisional table of physical parameters for every source.
Method  : statmorph on the sum of the PSF-matched bands, with the target
          segment from Stage 1c and every neighbouring segment masked. The
          Sersic fit is convolved with a generic PLACEHOLDER PSF (Gaussian with
          the Stage-1d target FWHM); Stage 5b re-runs the galaxy track with the
          empirical PSF. Sources below morphology.min_snr are not measured
          (NaN, flag 4). Gain = detection-band EXPTIME (images are in e-/s).
          fwhm_over_psf = source FWHM / PSF FWHM (1 = point-like), both from a
          2-D Gaussian fit to the detection-band cutout (neighbours removed,
          cosmic-ray pixels already replaced by Stage 1c) and to the Stage-1d PSF of that band. A fit
          does not shrink with S/N the way a segment-moment FWHM does (faint
          galaxies only show their core above the threshold and looked
          point-like or even sharper than the PSF).
Output  : features_pass1.csv  (psf_pass = 1)
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..context import Ctx, load_json
from ..morphology import FEATURES, _HAVE_STATMORPH, run_parallel
from ..utils import fwhm_fit
from .stage1d_psf_match import crop_center


def fit_fwhm_ratio(ctx, src):
    """Gaussian-fit FWHM of every source / of its pointing's detection-band PSF (Stage 1d)."""
    raw = np.load(ctx.require("1c", "cutouts_raw.npy"), mmap_mode="r")
    seg = np.load(ctx.require("1c", "segstamps.npy"), mmap_mode="r")
    bands = load_json(ctx.require("1a", "pointings.json"))["bands"]
    stamp = ctx.cfg["psf_match"]["stamp"]
    locus = {}
    rep = load_json(ctx.require("1d", "psf_match.json"))
    for pid, det in src.groupby("pointing").detection_band.first().items():
        locus[pid] = rep[pid]["fwhm_px"].get(det) or fwhm_fit(np.load(ctx.require("1d", f"psf_{pid}_{det}.npy")))
    out = np.full(len(src), np.nan)
    for k, (i, p, det) in enumerate(zip(src.cutout_index, src.pointing, src.detection_band)):
        img = np.asarray(raw[int(i), bands.index(det)], np.float32)
        img = np.where(np.asarray(seg[int(i)]) == 2, 0.0, img)    # neighbours (cosmic-ray pixels: already replaced)
        out[k] = fwhm_fit(crop_center(img, stamp)) / locus[p]
    return out, locus


def run(ctx: Ctx):
    out = ctx.stage_dir("1e")
    src = pd.read_csv(ctx.require("1c", "sources.csv"))
    cut_path = ctx.require("1d", "cutouts.npy")
    seg_path = ctx.require("1c", "segstamps.npy")
    min_snr = ctx.cfg["morphology"]["min_snr"]
    todo = src[src.snr >= min_snr]
    psf = {pid: np.load(ctx.require("1d", f"placeholder_psf_{pid}.npy")) for pid in todo.pointing.unique()}
    ctx.log.info(f"[Stage 1e] measuring {len(todo)} of {len(src)} sources (SNR >= {min_snr}) with "
                 f"{'statmorph' if _HAVE_STATMORPH else 'NumPy fallback'} on {ctx.n_jobs} worker(s) ...")
    res = run_parallel(ctx, cut_path, seg_path, todo.cutout_index.tolist(), todo.exptime.tolist(),
                       [psf[p] for p in todo.pointing], mode="psf_fit")
    feats = pd.DataFrame(res, index=todo.index)
    table = pd.DataFrame(index=src.index, columns=FEATURES, dtype=float)
    table.loc[feats.index, FEATURES] = feats[FEATURES].to_numpy()
    table["flag"] = table["flag"].fillna(4)
    table.insert(0, "object_id", src.object_id)
    table["ellipticity"] = src["ellipticity"]           # from the Stage-1c segment moments
    # size relative to the PSF, measured with the SAME estimator (2-D Gaussian fit) on source and PSF
    ratio, locus = fit_fwhm_ratio(ctx, src)
    table["fwhm_over_psf"] = np.round(ratio, 4)
    ctx.log.info(f"[Stage 1e] PSF FWHM (Gaussian fit, detection band) per pointing: "
                 f"{ {p: round(v, 3) for p, v in locus.items()} } px | fwhm_over_psf median {np.nanmedian(ratio):.2f}, "
                 f"share point-like (0.8-1.2) {np.mean(np.abs(ratio - 1) <= 0.2):.1%}")
    table["psf_pass"] = 1
    table["morph_engine"] = "statmorph" if _HAVE_STATMORPH else "numpy_fallback"
    table.to_csv(out / "features_pass1.csv", index=False)
    good = table[table.flag <= 1]
    ctx.log.info(f"[Stage 1e] flags: {table.flag.value_counts().sort_index().to_dict()} | "
                 f"median Gini {good.gini.median():.3f}, C {good.concentration.median():.2f}, "
                 f"A {good.asymmetry.median():.3f}, n {good.sersic_n.median():.2f}")
    _plot(table, out)
    return {"measured": int(len(todo)), "good_flag": int(len(good))}


def _plot(t, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    g = t[t.flag <= 1]
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].scatter(g.m20, g.gini, s=4, alpha=0.5)
    xs = np.array([-3.0, 0.0])
    ax[0].plot(xs, -0.14 * xs + 0.33, "k--", lw=1, label="merger line (Lotz+08)")
    ax[0].plot(xs, 0.14 * xs + 0.80, "k:", lw=1, label="E/S0 vs Sb line")
    ax[0].invert_xaxis()
    ax[0].set_xlabel("M20")
    ax[0].set_ylabel("Gini")
    ax[0].legend(fontsize=7)
    ax[1].scatter(g.asymmetry, g.concentration, s=4, alpha=0.5)
    ax[1].set_xlabel("asymmetry A")
    ax[1].set_ylabel("concentration C")
    fig.suptitle("Stage 1e: Pass-1 morphology (flag <= 1)")
    fig.tight_layout()
    fig.savefig(out / "pass1_morphology.png", dpi=110)
    plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage2a_representation.py
"""Stage 2a - Representation learning and feature fusion.

Purpose : one feature space that holds both pixel structure and physics.
Method  : a convolutional autoencoder (CAE) encodes every cutout to a 32-d latent code. Input: each band on one
          noise scale per pointing (background-subtracted, / band sigma, asinh) plus a "data present" mask per
          band; loss: MSE weighted x target_weight on the source, 0 on cosmic-ray and no-data pixels; rotations /
          flips; early stopping on the dev split. PCA keeps the components above chance (Horn's parallel analysis).
          The tabular features are median-imputed, signed-log transformed and RobustScaled. Both blocks are
          weighted (spatial_weight / 1 - spatial_weight) and concatenated. Everything is fitted on the TRAIN split
          only (dev only decides when to stop) and applied to all other objects.
Output  : rep_cae.pt, latent.npy, recon_error.npy, hybrid_features.npy (row = sources.csv row), object_id.npy,
          fusion.joblib, pca_scree.png, cae_training.png / .json, cae_diagnostics.json, summary.json
"""
from __future__ import annotations

import joblib
import numpy as np
import torch
from sklearn.decomposition import PCA

from ..context import Ctx, load_json, load_objects, save_json
from ..features import TabularPrep, load_cutouts
from ..nets import RepAutoencoder, rep_errors, train_rep_autoencoder
from ..representation import band_levels, band_sigmas, normalise_cutouts, pixel_weights, spearman


def run(ctx: Ctx):
    cfg = ctx.cfg["representation"]
    out = ctx.stage_dir("2a")
    df = load_objects(ctx, ["sources", "pass1"])
    cut = load_cutouts(ctx)
    is_train = (df.split_tag == "train").to_numpy()
    if is_train.sum() < 20:
        raise RuntimeError(f"Only {int(is_train.sum())} train-split sources - check Stage 0 regions / collar size.")
    latent, errs, diag = _train_cae(ctx, cfg, df, cut, is_train, ctx.device(), out)

    # ---- PCA of the latent (train split), scree + what each component tracks ----
    full = PCA(svd_solver="full", random_state=ctx.seed).fit(latent[is_train])
    scree = full.explained_variance_ratio_
    if scree[0] > 0.9:
        ctx.log.info(f"[Stage 2a] WARNING: PC1 explains {scree[0]:.1%} of the latent variance - the CAE latent has "
                     "collapsed to one direction (under-trained).")
    min_var = float(cfg.get("pca_min_variance", 0.0))
    pa = parallel_analysis(latent[is_train], n_iter=int(cfg.get("pca_parallel_iterations", 50)), seed=ctx.seed)
    diag["parallel_analysis"] = pa
    if str(cfg["n_pca"]).lower() == "auto":
        # Horn's parallel analysis: components with more variance than the same data with every column shuffled
        k = int(min(pa["n_components"], int(cfg.get("n_pca_max", 12)), latent.shape[1]))
        ctx.log.info(f"[Stage 2a] parallel analysis: {pa['n_components']} components above the 95th percentile of "
                     f"shuffled-data eigenvalues -> n_pca = {k} (cap n_pca_max {cfg.get('n_pca_max', 12)})")
    else:
        k = int(min(cfg["n_pca"], latent.shape[1]))
        ctx.log.info(f"[Stage 2a] n_pca fixed at {k} (parallel analysis would keep {pa['n_components']})")
    k_used = max(2, min(k, int(np.sum(scree[:k] >= min_var))))
    if k_used < k:
        ctx.log.info(f"[Stage 2a] PC{k_used + 1}..PC{k} explain < {min_var:.0%} each "
                     f"({np.round(scree[k_used:k] * 100, 2).tolist()} %) -> keeping {k_used} components")
    pca = PCA(n_components=k_used, svd_solver="full", random_state=ctx.seed).fit(latent[is_train])
    Z = pca.transform(latent)
    pca_all = full.transform(latent)                   # all components (same axes as pca)
    corr = _component_correlations(ctx, df, cut, pca_all[:, :8], cfg["tabular_features"])
    diag["component_correlations"] = corr
    diag["max_abs_rho_kept_pcs_vs_band_levels"] = round(float(np.nanmax(np.abs(
        np.array([[corr[f"PC{i + 1}"][c] for c in corr["band_level_columns"]] for i in range(k_used)])))), 3)
    _scree_plot(scree, k_used, corr, out, pa=pa)
    ctx.log.info(f"[Stage 2a] largest |Spearman rho| between a kept component and a band background level: "
                 f"{diag['max_abs_rho_kept_pcs_vs_band_levels']} (the original CAE: 0.98)")

    z_scale = float(Z[is_train].std()) or 1.0          # one scale for the image block (not whitened)
    Z = Z / z_scale
    prep = TabularPrep(cfg["tabular_features"]).fit(df[is_train])
    w = cfg["spatial_weight"]
    hybrid = np.hstack([Z * w, prep.transform(df) * (1 - w)]).astype(np.float32)
    diag["probe"] = _probe(df, Z, is_train)
    diag["probe_vs_n_components"] = {str(n): _probe(df, pca_all[:, :n], is_train)
                                     for n in sorted({2, 4, k_used, min(12, pca_all.shape[1]), pca_all.shape[1]})}
    ctx.log.info(f"[Stage 2a] linear probe vs number of components (information kept): {diag['probe_vs_n_components']}")
    ctx.log.info(f"[Stage 2a] linear probe on the kept components (train -> dev, balanced accuracy): {diag['probe']}")

    np.save(out / "latent.npy", latent)
    np.save(out / "recon_error.npy", errs)
    np.save(out / "hybrid_features.npy", hybrid)
    np.save(out / "object_id.npy", df.object_id.to_numpy())
    joblib.dump({"pca": pca, "z_scale": z_scale, "prep": prep, "spatial_weight": w,
                 "cae_version": 2, "band_sigma": diag.get("band_sigma")}, out / "fusion.joblib")
    save_json(diag, out / "cae_diagnostics.json")
    save_json({"n_objects": len(df), "n_train": int(is_train.sum()), "hybrid_dim": hybrid.shape[1],
               "tabular_kept": prep.kept, "cae_version": 2, "n_pca_used": k_used,
               "pca_explained_variance": float(pca.explained_variance_ratio_.sum()),
               "scree_first10_percent": np.round(scree[:10] * 100, 2).tolist(),
               "n_pca_rule": "parallel analysis (95th pct of shuffled eigenvalues)" if str(cfg["n_pca"]).lower() == "auto"
               else "fixed", "n_pca_parallel_analysis": pa["n_components"],
               "max_abs_rho_kept_pcs_vs_band_levels": diag["max_abs_rho_kept_pcs_vs_band_levels"],
               "recon_gain_vs_flat_dev": diag.get("recon_gain_vs_flat", {}).get("dev_median")},
              out / "summary.json")
    ctx.log.info(f"[Stage 2a] hybrid matrix {hybrid.shape} | tabular kept {prep.kept} | "
                 f"PCA keeps {pca.explained_variance_ratio_.sum():.1%} of latent variance")
    return {"hybrid_dim": int(hybrid.shape[1]), "n_pca": k_used,
            "max_rho_band_levels": diag["max_abs_rho_kept_pcs_vs_band_levels"]}


# ===================================================================== CAE
def _train_cae(ctx, cfg, df, cut, is_train, device, out):
    seg = np.load(ctx.require("1c", "segstamps.npy"), mmap_mode="r")
    rows = df.cutout_index.to_numpy()
    a, m = float(cfg.get("asinh_softening_sigma", 3.0)), float(cfg.get("asinh_max_sigma", 1000.0))
    # background sigma per band and pointing (image-noise statistic, no labels)
    sig_rows = np.zeros((len(df), cut.shape[1]), np.float32)
    band_sigma = {}
    for pid, sub in df.groupby("pointing"):
        s = band_sigmas(cut, seg, sub.cutout_index.to_numpy(), seed=ctx.seed)
        band_sigma[str(pid)] = [None if not np.isfinite(v) else round(float(v), 6) for v in s]
        sig_rows[(df.pointing == pid).to_numpy()] = s
    ctx.log.info(f"[Stage 2a] background sigma per band (data units) {band_sigma}")
    X, P = normalise_cutouts(cut, seg, rows, sig_rows, a, m)
    W = pixel_weights(np.asarray(seg)[rows], P, float(cfg.get("target_weight", 10.0)),
                      bool(cfg.get("ignore_cr_pixels", True)))
    missing = (~P.any(axis=(2, 3))).mean(axis=0)
    ctx.log.info(f"[Stage 2a] cutouts without data per band: {np.round(missing * 100, 1).tolist()} % "
                 f"(fed to the CAE as 'data present' channels)")

    tr = np.where(is_train)[0]
    va = np.where((df.split_tag == "dev").to_numpy())[0]
    if len(va) < 50:                                   # no usable dev region: hold out 10 % of train for stopping
        rng = np.random.default_rng(ctx.seed)
        va = rng.choice(tr, max(50, len(tr) // 10), replace=False)
        tr = np.setdiff1d(tr, va)
    torch.manual_seed(ctx.seed)
    np.random.seed(ctx.seed)
    model = RepAutoencoder(bands=cut.shape[1], latent_dim=int(cfg["latent_dim"]), size=cut.shape[-1],
                           channels=tuple(cfg.get("channels", [16, 32, 64, 64])),
                           mask_channels=bool(cfg.get("band_mask_channels", True)))
    ctx.log.info(f"[Stage 2a] training representation CAE v2 ({cut.shape[1]} bands + presence masks -> "
                 f"{cfg['latent_dim']}-d, {sum(p.numel() for p in model.parameters()):,} weights) on {len(tr)} "
                 f"train cutouts, validation {len(va)} ({device}); at most {cfg.get('max_epochs', 150)} epochs, "
                 f"early stopping after {cfg.get('patience', 10)} epochs without a {float(cfg.get('min_delta', 0.005)):.1%} improvement")
    ck, hj = out / "rep_cae.pt", out / "cae_training.json"
    if cfg.get("reuse_trained_cae", False) and ck.exists() and hj.exists():
        # re-run only the PCA / fusion with the CAE trained by an earlier run (no retraining)
        model.load_state_dict(torch.load(ck, map_location="cpu"))
        model.to(device)
        hist = load_json(hj)
        ctx.log.info(f"[Stage 2a] reuse_trained_cae: loaded {ctx.rel(ck)} (best epoch {hist['best_epoch']}) - not retrained")
    else:
        hist = train_rep_autoencoder(model, X, P, W, tr, va, device, max_epochs=int(cfg.get("max_epochs", 100)),
                                     min_epochs=int(cfg.get("min_epochs", 20)), patience=int(cfg.get("patience", 10)),
                                     batch_size=int(cfg["batch_size"]), lr=float(cfg.get("lr", 1e-3)),
                                     lr_patience=int(cfg.get("lr_patience", 4)), augment=bool(cfg.get("augment", True)),
                                     min_delta=float(cfg.get("min_delta", 0.005)), checkpoint=ck,
                                     log=ctx.log.info)              # best weights are written to rep_cae.pt
    ctx.log.info(f"[Stage 2a] best validation loss {hist['best_val']:.5f} at epoch {hist['best_epoch']} "
                 f"of {len(hist['val'])}")
    errs, latent = rep_errors(model, X, P, W, np.arange(len(df)), device)

    # reconstruction gain over a flat image (each band filled with its weighted mean)
    gains = {}
    for name, idx in (("train", tr), ("dev", va)):
        e_flat = _flat_error(X, P, W, idx)
        g = 1 - errs[idx] / np.maximum(e_flat, 1e-12)
        gains[f"{name}_median"] = round(float(np.median(g)), 3)
        gains[f"{name}_fraction_better"] = round(float((g > 0).mean()), 3)
    ctx.log.info(f"[Stage 2a] reconstruction gain over a flat per-band image: {gains}")
    _training_plot(hist, out)
    save_json(hist, out / "cae_training.json")
    diag = {"cae_version": 2, "band_sigma": band_sigma, "missing_band_fraction": np.round(missing, 4).tolist(),
            "epochs_run": len(hist["val"]), "best_epoch": hist["best_epoch"], "best_val_loss": hist["best_val"],
            "recon_gain_vs_flat": gains, "n_weights": int(sum(p.numel() for p in model.parameters()))}
    return latent, errs, diag


def _flat_error(X, P, W, idx, batch=256):
    out = []
    for s in range(0, len(idx), batch):
        b = np.sort(idx[s:s + batch])
        x, p, w = (np.asarray(A[b], np.float32) for A in (X, P, W))
        mean = (x * w).sum(axis=(2, 3), keepdims=True) / np.maximum(w.sum(axis=(2, 3), keepdims=True), 1e-6)
        e = (w * (x - mean) ** 2).mean(axis=(1, 2, 3))
        order = np.argsort(np.argsort(idx[s:s + batch]))       # back to the order of idx
        out.append(e[order])
    return np.concatenate(out)


# ===================================================================== diagnostics
def _component_correlations(ctx, df, cut, Zall, physical, n_sample=1500):
    """Spearman rho of each principal component with the band background levels (a component that tracks them
    encodes sky level, not structure) and with the tabular features, on a random sample of objects."""
    rng = np.random.default_rng(ctx.seed)
    s = np.sort(rng.choice(len(df), min(n_sample, len(df)), replace=False))
    lev = band_levels(cut, df.cutout_index.to_numpy()[s])
    bands = list(load_json(ctx.require("1a", "pointings.json"))["bands"])
    level_cols = [f"level_{b}" for b in bands][:lev.shape[1]]
    cols = {c: lev[:, i] for i, c in enumerate(level_cols)}
    physical = [c for c in physical if c in df]
    cols.update({c: df[c].to_numpy(float)[s] for c in physical})
    res = {"band_level_columns": level_cols, "physical_columns": physical}
    for i in range(Zall.shape[1]):
        res[f"PC{i + 1}"] = {c: round(spearman(Zall[s, i], v), 3) for c, v in cols.items()}
    return res


def _probe(df, Z, is_train):
    """How much the kept image components alone say about (a) cosmic-ray candidates and (b) point-like
    vs extended sources: logistic regression fitted on train, balanced accuracy on dev."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import balanced_accuracy_score
    dev = (df.split_tag == "dev").to_numpy()
    out = {}
    targets = {}
    if "cr_candidate" in df:
        targets["cosmic_ray_candidate"] = df.cr_candidate.astype(float).to_numpy()
    f = df.get("fwhm_over_psf")
    if f is not None:
        y = np.where(f < 1.15, 0.0, np.where(f > 1.4, 1.0, np.nan))
        y[(df.snr < 10).to_numpy()] = np.nan                   # size is only meaningful at S/N >= 10
        targets["extended_vs_pointlike"] = y
    for name, y in targets.items():
        tr, te = is_train & np.isfinite(y), dev & np.isfinite(y)
        if tr.sum() < 30 or te.sum() < 10 or len(np.unique(y[tr])) < 2:
            continue
        clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Z[tr], y[tr])
        out[name] = round(float(balanced_accuracy_score(y[te], clf.predict(Z[te]))), 3)
    return out


def parallel_analysis(Xtr, n_iter=50, q=95, seed=42, max_rows=5000):
    """Horn (1965) parallel analysis. Eigenvalues of the train latent vs the q-th percentile of eigenvalues of
    the same matrix with each column permuted independently (marginals kept, correlations destroyed).
    Returns the number of leading components above that null and both spectra (as variance fractions)."""
    rng = np.random.default_rng(seed)
    X = np.asarray(Xtr, np.float64)
    if len(X) > max_rows:
        X = X[rng.choice(len(X), max_rows, replace=False)]
    X = X - X.mean(0)
    ev = np.linalg.eigvalsh(np.cov(X, rowvar=False))[::-1]
    null = np.array([np.linalg.eigvalsh(np.cov(np.column_stack([rng.permutation(c) for c in X.T]),
                                                rowvar=False))[::-1] for _ in range(n_iter)])
    thr = np.percentile(null, q, axis=0)
    above = ev > thr
    n = int(np.argmin(above)) if not above.all() else len(ev)
    tot = ev.sum()
    return {"n_components": n, "eigen_fraction": np.round(ev / tot, 5).tolist(),
            "null_q_fraction": np.round(thr / tot, 5).tolist(), "percentile": q, "iterations": n_iter}


def _scree_plot(ratio, k_used, corr, out, pa=None):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    n = min(len(ratio), 15)
    x = np.arange(1, n + 1)
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(12.5, 3.9), gridspec_kw={"width_ratios": [1, 1.4]})
    ax.bar(x, ratio[:n] * 100, color=["#2E5496" if i < k_used else "#BBBBBB" for i in range(n)])
    ax.plot(x, np.cumsum(ratio[:n]) * 100, "o-", color="#E69F00", ms=3, label="cumulative")
    if pa is not None:
        ax.plot(x, np.asarray(pa["null_q_fraction"][:n]) * 100, "s--", color="#C00000", ms=3, lw=1,
                label=f"shuffled data, {pa['percentile']}th pct (parallel analysis: {pa['n_components']})")
    ax.set_xticks(x)
    ax.set_xlabel("principal component of the CAE latent")
    ax.set_ylabel("explained variance [%]")
    ax.set_title(f"Stage 2a scree (train split) - {k_used} kept")
    ax.legend(loc="center right", fontsize=7)
    cols = corr["band_level_columns"] + corr["physical_columns"]
    pcs = [k for k in corr if k.startswith("PC")]
    M = np.array([[abs(corr[p][c]) if np.isfinite(corr[p][c]) else np.nan for c in cols] for p in pcs])
    im = bx.imshow(M, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    bx.set_xticks(range(len(cols)))
    bx.set_xticklabels([c.replace("level_", "bg level ") for c in cols], rotation=60, ha="right", fontsize=7.5)
    bx.set_yticks(range(len(pcs)))
    bx.set_yticklabels([f"{p} ({ratio[i] * 100:.1f} %)" for i, p in enumerate(pcs)], fontsize=8)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            if np.isfinite(M[i, j]):
                bx.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=6.5,
                        color="white" if M[i, j] > 0.6 else "black")
    nb = len(corr["band_level_columns"])
    bx.axvline(nb - 0.5, color="#C00000", lw=1)
    bx.axhline(k_used - 0.5, color="#555555", lw=0.8, ls="--")
    bx.set_title("|Spearman rho| of each component with band background levels (left of red line)\n"
                 "and physical features; dashed line = last kept component", fontsize=8.5)
    fig.colorbar(im, ax=bx, fraction=0.03, pad=0.01)
    fig.tight_layout()
    fig.savefig(out / "pca_scree.png", dpi=110)
    plt.close(fig)


def _training_plot(hist, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    ep = np.arange(1, len(hist["train"]) + 1)
    fig, ax = plt.subplots(figsize=(5.5, 3.4))
    ax.plot(ep, hist["train"], label="train (augmented)", color="#2E5496")
    ax.plot(ep, hist["val"], label="validation (dev region)", color="#E69F00")
    ax.axvline(hist["best_epoch"], color="0.5", ls="--", lw=0.8, label=f"best epoch {hist['best_epoch']}")
    ax.set_yscale("log")
    ax.set_xlabel("epoch")
    ax.set_ylabel("weighted reconstruction MSE")
    ax.set_title("Stage 2a: representation CAE training")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(out / "cae_training.png", dpi=110)
    plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage2b_clustering.py
"""Stage 2b - Unsupervised clustering (GMM + HDBSCAN), fitted on the TRAIN split only.

GMM     : full covariance (the features are correlated; a diagonal mixture never reaches a BIC minimum), k from
          the minimum BIC over [k_min, k_max] (BIC elbow if the minimum is on the boundary). Checks: ICL, AIC,
          held-out (dev) log-likelihood, BIC minimum with other seeds, bootstrap stability (ARI).
HDBSCAN : density-based; best DBCV over a small min_cluster_size grid (fits calling > hdbscan_max_noise of the
          sources noise are rejected). Every object gets the GLOSH outlier score; density_outlier = HDBSCAN noise
          or GLOSH above the train hdbscan_outlier_percentile (used by Stage 2e).
Other objects are assigned with GMM.predict / hdbscan.approximate_predict (never refitted). External validation
against Gaia / GZH / noise evidence is in Stage 2d (clustering_validation).
Output  : clusters.csv, gmm.joblib, hdbscan.joblib, gmm_k_selection.png / .csv, summary.json (incl. evaluation)
"""
from __future__ import annotations

import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.mixture import GaussianMixture

from ..context import Ctx, save_json

try:
    import hdbscan as _hdb
except Exception:
    _hdb = None


def _fit_gmm(X, k, cov, seed, reg0, n_init=3):
    """GMM fit; reg_covar is raised x10 (up to 1e-1) if a component collapses onto identical points."""
    reg = reg0
    while reg <= 1e-1:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                return GaussianMixture(n_components=k, covariance_type=cov, random_state=seed, n_init=n_init,
                                       reg_covar=reg, max_iter=300).fit(X), reg
        except ValueError:
            reg *= 10
    return None, reg


def icl(g, X):
    """Integrated Completed Likelihood (lower is better, same scale as sklearn's BIC)."""
    t = g.predict_proba(X)
    return float(g.bic(X) + 2 * -np.sum(t * np.log(np.clip(t, 1e-300, 1))))


def knee(ks, vals):
    """Kneedle elbow: the point of the (normalised) decreasing curve farthest below the chord first -> last."""
    ks, v = np.asarray(ks, float), np.asarray(vals, float)
    if len(ks) < 3:
        return int(ks[int(np.argmin(v))])
    x = (ks - ks[0]) / (ks[-1] - ks[0])
    y = (v - v.min()) / max(v.max() - v.min(), 1e-12)
    chord = y[0] + (y[-1] - y[0]) * x
    return int(ks[int(np.argmax(chord - y))])


def run(ctx: Ctx):
    cfg = ctx.cfg["clustering"]
    out = ctx.stage_dir("2b")
    X = np.load(ctx.require("2a", "hybrid_features.npy"))
    oid = np.load(ctx.require("2a", "object_id.npy"))
    src = pd.read_csv(ctx.require("1c", "sources.csv")).set_index("object_id").loc[oid]
    tr = (src.split_tag == "train").to_numpy()
    dev = (src.split_tag == "dev").to_numpy()
    held = ~tr

    # float64: in float32 the variance of a tight component can round to <= 0 ("ill-defined covariance")
    X = X.astype(np.float64)
    Xtr = X[tr]
    f = pd.read_csv(ctx.require("1e", "features_pass1.csv"), usecols=["object_id", "flag"]).set_index("object_id")
    n_faint = int((f.reindex(oid)["flag"].to_numpy()[tr] == 4).sum())
    ctx.log.info(f"[Stage 2b] {n_faint} of {int(tr.sum())} train sources are too faint for morphology "
                 "(their shape features are median-filled, so they sit on top of each other)")
    cov = cfg.get("covariance_type", "full")
    reg0 = float(cfg.get("reg_covar", 1e-3))
    n_init = int(cfg.get("n_init", 3))

    # ---------------- GMM: sweep k ----------------
    rows, models = [], {}
    for k in range(int(cfg["k_min"]), int(cfg["k_max"]) + 1):
        g, reg = _fit_gmm(Xtr, k, cov, ctx.seed, reg0, n_init)
        if g is None:
            ctx.log.info(f"  [GMM] k={k}  failed (collapsed component even with reg_covar 1e-1) - skipped")
            continue
        models[k] = g
        r = {"k": k, "bic": g.bic(Xtr), "aic": g.aic(Xtr), "icl": icl(g, Xtr),
             "heldout_loglik": float(g.score(X[dev])) if dev.any() else np.nan, "reg_covar": reg}
        rows.append(r)
        ctx.log.info(f"  [GMM {cov}] k={k:2d}  BIC={r['bic']:.1f}  ICL={r['icl']:.1f}  AIC={r['aic']:.1f}  "
                     f"dev log-lik/object={r['heldout_loglik']:.3f}" + (f"  (reg_covar {reg:g})" if reg > reg0 else ""))
    if not rows:
        raise RuntimeError("Every GMM fit collapsed. Lower clustering.k_max or raise clustering.reg_covar.")
    sel = pd.DataFrame(rows)
    k_bic = int(sel.k[sel.bic.idxmin()])
    k_icl = int(sel.k[sel.icl.idxmin()])
    k_knee = knee(sel.k, sel.bic)
    rule = "minimum BIC"
    best_k = k_bic
    if k_bic == int(sel.k.max()):
        best_k, rule = k_knee, "elbow (knee) of BIC - the BIC minimum is at the upper boundary of the range"
        ctx.log.info(f"[Stage 2b] WARNING: BIC still falling at k_max={k_bic}; using the BIC elbow k={k_knee}. "
                     "Raise clustering.k_max or check the features (correlated / tied values).")

    # robustness 1: does the minimum move with other seeds?
    seed_check = {}
    ks_near = [k for k in range(max(int(cfg["k_min"]), best_k - 2), min(int(cfg["k_max"]), best_k + 2) + 1)]
    for s in cfg.get("robustness_seeds", [7, 1]):
        b = {}
        for k in ks_near:
            g, _ = _fit_gmm(Xtr, k, cov, int(s), reg0, n_init)
            if g is not None:
                b[k] = g.bic(Xtr)
        seed_check[str(s)] = int(min(b, key=b.get)) if b else None
    # robustness 2: bootstrap stability of the chosen model (ARI of held-out labels)
    gmm = models[best_k]
    lab_all = gmm.predict(X)
    rng = np.random.default_rng(ctx.seed)
    aris = []
    for b in range(int(cfg.get("bootstrap_refits", 3))):
        bi = rng.choice(np.where(tr)[0], int(tr.sum()), replace=True)
        g2, _ = _fit_gmm(X[bi], best_k, cov, 100 + b, reg0, n_init)
        if g2 is not None:
            aris.append(float(adjusted_rand_score(lab_all[held], g2.predict(X[held]))))
    ctx.log.info(f"[Stage 2b] GMM k = {best_k} ({rule}); ICL minimum k = {k_icl}; BIC elbow k = {k_knee}; "
                 f"BIC minimum with other seeds: {seed_check}; bootstrap stability ARI {np.round(aris, 3).tolist()}")

    probs = gmm.predict_proba(X)
    res = pd.DataFrame({"object_id": oid, "cluster_id_gmm": probs.argmax(1)})
    for j in range(best_k):
        res[f"gmm_p{j}"] = probs[:, j].round(5)
    joblib.dump(gmm, out / "gmm.joblib")

    # comparison curve: the diagonal-covariance BIC (why the covariance type matters)
    diag_curve = {}
    if cfg.get("compare_diag", True) and cov != "diag":
        for k in sel.k:
            g, _ = _fit_gmm(Xtr, int(k), "diag", ctx.seed, reg0, n_init=1)
            if g is not None:
                diag_curve[int(k)] = float(g.bic(Xtr))
    sel["bic_diag_covariance"] = sel.k.map(diag_curve)
    sel.to_csv(out / "gmm_k_selection.csv", index=False)

    # ---------------- HDBSCAN ----------------
    mcs = cfg["hdbscan_min_cluster_size"]
    mcs = max(10, len(Xtr) // 50) if mcs == "auto" else int(mcs)
    hdb_status, hdb_trials, n_h, hdb_internal = "not run", [], 0, {}
    max_noise = float(cfg.get("hdbscan_max_noise", 0.5))
    if _hdb is not None:
        grid = sorted({mcs, max(5, mcs // 2), max(5, mcs // 4), 10}, reverse=True)
        best = None
        for m in grid:
            h = _hdb.HDBSCAN(min_cluster_size=m, min_samples=cfg.get("hdbscan_min_samples", 5),
                             prediction_data=True, gen_min_span_tree=True).fit(Xtr)
            n_c = len(set(h.labels_) - {-1})
            try:
                dbcv = float(h.relative_validity_) if n_c >= 2 else np.nan
            except Exception:
                dbcv = np.nan
            noise = float((h.labels_ == -1).mean())
            hdb_trials.append({"min_cluster_size": m, "clusters": n_c, "noise_fraction": round(noise, 4),
                               "dbcv": None if not np.isfinite(dbcv) else round(dbcv, 4),
                               "largest_cluster_fraction": round(float(np.max(np.bincount(h.labels_[h.labels_ >= 0]))
                                                                       / len(h.labels_)), 4) if n_c else None})
            ctx.log.info(f"  [HDBSCAN] min_cluster_size={m}: {n_c} clusters, noise {noise:.1%}, DBCV {hdb_trials[-1]['dbcv']}, "
                         f"largest cluster {hdb_trials[-1]['largest_cluster_fraction']}")
            if n_c >= 2 and noise <= max_noise and (best is None or np.nan_to_num(dbcv, nan=-1) > best[2]):
                best = (h, m, np.nan_to_num(dbcv, nan=-1))
        if best is not None:
            h, mcs = best[0], best[1]
            lab, strength = _hdb.approximate_predict(h, X)
            glosh = np.asarray(_hdb.approximate_predict_scores(h, X), float)
            lab[tr], glosh[tr] = h.labels_, h.outlier_scores_          # training points keep their fitted values
            glosh = np.nan_to_num(glosh, nan=1.0)
            thr = float(np.percentile(glosh[tr], float(cfg.get("hdbscan_outlier_percentile", 95))))
            res["cluster_id_hdbscan"] = lab
            res["hdbscan_strength"] = np.round(strength, 4)
            res["hdbscan_glosh"] = np.round(glosh, 4)
            res["density_outlier"] = (lab == -1) | (glosh > thr)
            joblib.dump(h, out / "hdbscan.joblib")
            n_h = len(set(h.labels_) - {-1})
            sizes = np.bincount(h.labels_[h.labels_ >= 0])
            hdb_internal = {"dbcv": round(float(best[2]), 4), "clusters": n_h,
                            "noise_fraction_train": round(float((h.labels_ == -1).mean()), 4),
                            "largest_cluster_fraction_train": round(float(sizes.max() / len(h.labels_)), 4),
                            "glosh_threshold": round(thr, 4),
                            "density_outlier_fraction": round(float(res.density_outlier.mean()), 4),
                            "ari_vs_gmm_train": round(float(adjusted_rand_score(lab_all[tr], h.labels_)), 4),
                            "nmi_vs_gmm_train": round(float(normalized_mutual_info_score(lab_all[tr], h.labels_)), 4)}
            if hdb_internal["largest_cluster_fraction_train"] > 0.8:
                ctx.log.info(f"[Stage 2b] NOTE: one HDBSCAN cluster holds {hdb_internal['largest_cluster_fraction_train']:.0%} "
                             "of the train sources - the feature space is one density continuum with small islands, "
                             "not separate populations; HDBSCAN is used for density outliers (GLOSH), the GMM for the populations.")
            hdb_status = "ok"
        else:
            res["cluster_id_hdbscan"] = np.nan
            hdb_status = "failed"
            ctx.log.info(f"[Stage 2b] HDBSCAN FAILED: no min_cluster_size in {grid} gives >= 2 clusters with noise <= "
                         f"{max_noise:.0%}. cluster_id_hdbscan is left empty and Stage 2e gating runs without it.")
    else:
        res["cluster_id_hdbscan"] = np.nan
        ctx.log.info("[Stage 2b] hdbscan package missing (pip install hdbscan) - HDBSCAN skipped")
    res.to_csv(out / "clusters.csv", index=False)
    evaluation = _evaluate(ctx, sel, best_k, seed_check, aris, hdb_internal, res, src.split_tag.to_numpy())
    summary = {"covariance_type": cov, "k_range": [int(cfg["k_min"]), int(cfg["k_max"])], "best_k": best_k,
               "k_rule": rule, "k_min_bic": k_bic, "k_min_icl": k_icl, "k_bic_elbow": k_knee,
               "k_min_bic_other_seeds": seed_check, "bootstrap_stability_ari": [round(a, 3) for a in aris],
               "bic": {int(r.k): float(r.bic) for r in sel.itertuples()},
               "icl": {int(r.k): float(r.icl) for r in sel.itertuples()},
               "heldout_loglik": {int(r.k): float(r.heldout_loglik) for r in sel.itertuples()},
               "bic_diag_covariance": diag_curve,
               "hdbscan_status": hdb_status, "hdbscan_trials": hdb_trials, "hdbscan_min_cluster_size": mcs,
               "hdbscan_clusters": n_h, "hdbscan_internal": hdb_internal,
               "gmm_sizes": {int(k): int(v) for k, v in res.cluster_id_gmm.value_counts().items()},
               "evaluation": evaluation}
    save_json(summary, out / "summary.json")
    _plot(sel, best_k, k_icl, k_knee, rule, cov, out)
    ctx.log.info(f"[Stage 2b] GMM k = {best_k}; sizes {summary['gmm_sizes']}; HDBSCAN clusters = {n_h} "
                 f"(min_cluster_size {mcs}); density outliers {float(res.get('density_outlier', pd.Series([0])).mean()):.1%}")
    return {"best_k": best_k, "hdbscan_clusters": n_h}


def _evaluate(ctx, sel, best_k, seed_check, aris, hdb, res, split):
    """Internal checks of both clusterings, logged in plain words (external check vs evidence: Stage 2d)."""
    b = sel.set_index("k").bic
    others = b.drop(best_k)
    runner = int(others.idxmin()) if len(others) else None
    ari = float(np.mean(aris)) if aris else np.nan
    seeds_agree = all(v == best_k for v in seed_check.values())
    ev = {"gmm": {"k": best_k, "bic_margin_to_next_best_k": None if runner is None else
                  {"k": runner, "delta_bic": round(float(others.min() - b[best_k]), 1)},
                  "heldout_loglik_best_k": int(sel.k[sel.heldout_loglik.idxmax()]),
                  "seeds_agree_on_k": seeds_agree, "bootstrap_ari_mean": round(ari, 3),
                  "stability": "stable" if ari >= 0.8 else "moderately stable" if ari >= 0.5 else "unstable"}}
    g = ev["gmm"]
    ctx.log.info(f"[Stage 2b] GMM check: k = {best_k}; next-best k = {runner} (BIC +{g['bic_margin_to_next_best_k']['delta_bic'] if runner else '-'}); "
                 f"dev log-likelihood peaks at k = {g['heldout_loglik_best_k']}; other seeds "
                 f"{'agree' if seeds_agree else 'disagree: ' + str(seed_check)}; bootstrap ARI {g['bootstrap_ari_mean']} "
                 f"-> {g['stability']} (>= 0.8 stable, 0.5-0.8 moderate). The exact k matters little: Stage 2d "
                 "names clusters by evidence and several clusters can share a name.")
    if hdb:
        rate = pd.Series(res.density_outlier.to_numpy()).groupby(split).mean().round(3).to_dict()
        ev["hdbscan"] = {"dbcv": hdb["dbcv"], "clusters": hdb["clusters"],
                         "largest_cluster_fraction_train": hdb["largest_cluster_fraction_train"],
                         "density_outlier_rate_by_split": rate,
                         "structure": "none (one density continuum)" if hdb["largest_cluster_fraction_train"] > 0.8
                         or hdb["dbcv"] < 0.1 else "clusters found"}
        ctx.log.info(f"[Stage 2b] HDBSCAN check: DBCV {hdb['dbcv']} (-1..1; < 0.1 = no density structure), "
                     f"{hdb['clusters']} clusters, largest {hdb['largest_cluster_fraction_train']:.0%} of train -> "
                     f"{ev['hdbscan']['structure']}; used only for density outliers. Outlier rate by split: {rate} "
                     f"(train ~{100 - float(ctx.cfg['clustering'].get('hdbscan_outlier_percentile', 95)):g} % by "
                     "construction; buffer = collars and sources outside the three-band area)")
    return ev


def _plot(sel, best, k_icl, k_knee, rule, cov, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    blue, orange, aqua, ink, muted = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#52514e"
    ks = sel.k.to_numpy()
    fig, axs = plt.subplots(1, 3, figsize=(14, 3.9))
    ax = axs[0]
    ax.plot(ks, sel.bic / 1e3, "o-", color=blue, lw=2, ms=4, label="BIC")
    ax.plot(ks, sel.icl / 1e3, "s-", color=orange, lw=2, ms=4, label="ICL (BIC + assignment entropy)")
    ax.plot(ks, sel.aic / 1e3, "^-", color=aqua, lw=1.5, ms=4, label="AIC")
    ax.axvline(best, color=ink, ls="--", lw=1)
    ax.annotate(f"chosen k = {best}", (best, sel.bic.min() / 1e3), xytext=(14, -22), textcoords="offset points",
                fontsize=8.5, color=ink, arrowprops={"arrowstyle": "-", "color": muted, "lw": 0.8})
    ax.set_xlabel("number of GMM components k")
    ax.set_ylabel("criterion / 1000 (lower is better)")
    ax.set_title(f"Elbow / information criteria ({cov} covariance)\n{rule}; ICL min k = {k_icl}, BIC knee k = {k_knee}",
                 fontsize=9)
    ax.legend(fontsize=7.5, frameon=False)
    ax = axs[1]
    ax.plot(ks, sel.heldout_loglik, "o-", color=blue, lw=2, ms=4)
    ax.axvline(best, color=ink, ls="--", lw=1)
    ax.set_xlabel("number of GMM components k")
    ax.set_ylabel("mean log-likelihood per object")
    ax.set_title("Held-out (dev region) log-likelihood\nhigher = generalises better", fontsize=9)
    ax = axs[2]
    if sel.bic_diag_covariance.notna().any():
        ax.plot(ks, sel.bic_diag_covariance / 1e3, "o-", color=muted, lw=1.5, ms=4, label="diagonal covariance")
        ax.plot(ks, sel.bic / 1e3, "o-", color=blue, lw=2, ms=4, label=f"{cov} covariance (used)")
        ax.legend(fontsize=7.5, frameon=False)
        ax.set_title("BIC: diagonal vs full covariance\n(diagonal keeps falling - no minimum)", fontsize=9)
    else:
        ax.axis("off")
    ax.set_xlabel("number of GMM components k")
    ax.set_ylabel("BIC / 1000")
    from matplotlib.ticker import MaxNLocator
    for a in axs:
        a.xaxis.set_major_locator(MaxNLocator(integer=True))
        a.grid(alpha=0.25, lw=0.5)
        a.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(out / "gmm_k_selection.png", dpi=120, facecolor="white")
    plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage2c_baseline.py
"""Stage 2c - Baseline segregation (+ the direct Gaia DR3 cross-match).

Purpose : a simple, transparent Star / Galaxy / Noise separator - the yardstick
          the learned segregation (Stages 2a-2d) must beat (Objective 2).
Method  : (1) Gaia DR3 match within 0.5" after propagating Gaia positions to the
          HST epoch with proper motions -> Star;
          (2) noise: SNR < 3, a profile SHARPER than the PSF (FWHM < 0.7 x the
              stellar locus - no real source can be sharper; cosmic rays can),
              or a single-band detection (band_consistency < 0.05);
          (3) stellar locus: FWHM within 20% of the stellar locus and a
              CLASS_STAR-like stellarity > 0.5 -> Star;
          (4) everything else -> Galaxy.
          The stellarity is a stand-in for SExtractor's CLASS_STAR built from
          how close the source FWHM is to the PSF FWHM.
Output  : baseline.csv, gaia_match.csv
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..catalogs import load_gaia, propagate_gaia
from ..context import Ctx, load_json, load_objects
from ..utils import noise_evidence, sky_crossmatch


def gaia_crossmatch(ctx: Ctx, df: pd.DataFrame, pts: dict, metas):
    cfg = ctx.cfg["semantic_audit"]
    gaia = load_gaia(ctx, metas)
    out = pd.DataFrame({"object_id": df.object_id, "gaia_hit": False})
    if gaia is None or gaia.empty:
        ctx.log.info("[Stage 2c] no Gaia table (data/catalogs/*gaia*.csv) - Gaia leg disabled")
        return out, False
    ctx.log.info(f"[Stage 2c] Gaia: {len(gaia)} sources in the footprint")
    cols = {k: np.full(len(df), np.nan) for k in ("gaia_sep_arcsec", "parallax", "pmra", "pmdec", "gaia_g", "gaia_teff")}
    sid = np.full(len(df), -1, dtype=np.int64)
    hit_all = np.zeros(len(df), bool)
    for pid, p in pts.items():
        m = (df.pointing == pid).to_numpy()
        if not m.any():
            continue
        g = propagate_gaia(gaia, p["epoch"], ctx.cfg["catalogs"]["gaia"].get("epoch", 2016.0))
        idx, sep, hit = sky_crossmatch(df.ra[m], df.dec[m], g.ra, g.dec, cfg["gaia_radius_arcsec"])
        rows = np.where(m)[0][hit]
        j = idx[hit]
        hit_all[rows] = True
        sid[rows] = pd.to_numeric(g["source_id"], errors="coerce").fillna(-1).astype(np.int64).to_numpy()[j]
        cols["gaia_sep_arcsec"][rows] = sep[hit]
        for c, gc in (("parallax", "parallax"), ("pmra", "pmra"), ("pmdec", "pmdec"), ("gaia_g", "g_mag"), ("gaia_teff", "teff")):
            cols[c][rows] = g[gc].to_numpy(float)[j]
    out["gaia_hit"] = hit_all
    out["gaia_source_id"] = np.where(hit_all, sid, -1)
    for c, v in cols.items():
        out[c] = v
    return out, True


def run(ctx: Ctx):
    cfg = ctx.cfg["baseline"]
    out = ctx.stage_dir("2c")
    df = load_objects(ctx, ["sources", "pass1"])
    pts = load_json(ctx.require("1a", "pointings.json"))["pointings"]
    metas = list(load_json(ctx.require("1a", "manifest.json")).values())
    gm, have_gaia = gaia_crossmatch(ctx, df, pts, metas)
    gm.to_csv(out / "gaia_match.csv", index=False)

    ratio = df.fwhm_over_psf.to_numpy(float)                       # 1 = as compact as a star (Stage 1e)
    stellarity = np.exp(-((ratio - 1.0) ** 2) / (2 * 0.15 ** 2))   # CLASS_STAR-like, 1 = PSF-like
    stellarity = np.where(np.isfinite(stellarity), stellarity, 0.0)
    noise = noise_evidence(df, cfg)
    locus = np.abs(ratio - 1.0) <= cfg["point_source_tolerance"]
    star = locus & (stellarity > cfg["class_star_threshold"]) & ~noise
    label = np.where(noise, "Noise", np.where(star, "Star", "Galaxy")).astype(object)
    label[gm.gaia_hit.to_numpy()] = "Star"
    res = pd.DataFrame({"object_id": df.object_id, "class_star_proxy": stellarity.round(4),
                        "fwhm_over_psf": ratio.round(4), "baseline_semantic": label,
                        "baseline_semantic_nogaia": np.where(noise, "Noise", np.where(star, "Star", "Galaxy"))})
    res.to_csv(out / "baseline.csv", index=False)
    ctx.log.info(f"[Stage 2c] baseline: {pd.Series(label).value_counts().to_dict()} | Gaia matches "
                 f"{int(gm.gaia_hit.sum())} (Gaia leg {'on' if have_gaia else 'off'})")
    return {"baseline_counts": pd.Series(label).value_counts().to_dict(), "gaia_matches": int(gm.gaia_hit.sum())}

In [ ]:
%%writefile src/p2p/stages/stage2d_semantic_audit.py
"""Stage 2d - Semantic audit: naming the clusters from INDEPENDENT evidence.

Purpose : turn abstract GMM cluster IDs into Star / Galaxy / Noise, and
          measure how good the learned segregation is against Gaia.
Method  : three evidence signals per object, none of them used to cluster:
            Star   - Gaia DR3 match (Stage 2c), OR on the stellar locus of the
                   size-magnitude plane: point-like (|FWHM / stellar locus - 1|
                   <= baseline.point_source_tolerance) with SNR >=
                   baseline.star_locus_min_snr, or saturated, AND confirmed in a
                   second band (a cosmic ray is as compact as a star but
                   single-band). Below that S/N the point-like band fills up with
                   compact distant galaxies that size cannot separate from stars
                   (check size_magnitude.png). Deep fields hold only a handful of
                   Gaia stars, too few to name a cluster alone;
            Galaxy - position matched to the Galaxy Zoo: Hubble catalogue, OR
                     resolved (FWHM / PSF > 1 + point_source_tolerance), SNR >= 5
                     and confirmed in a second band: GZH stops near mag 23.5, and
                     without this a cluster of fainter galaxies would carry no
                     galaxy evidence and be named Noise;
            Noise  - SNR < 3, a profile sharper than the PSF, or a single-band
                     detection (utils.noise_evidence).
          Each signal's per-cluster rate - measured on TRAIN-split members
          only - is divided by its maximum over the clusters (so a rare signal
          like Gaia is not swamped) and the strongest one names the cluster;
          several clusters may share a name.
          A cluster with no evidence takes the majority of the baseline rule.
          Gaia confirms stars but cannot reject them (it stops at G~21), so
          precision/recall are reported for STARS ONLY, inside Gaia's magnitude
          range, on HELD-OUT (non-train) objects, so naming and scoring never
          use the same Gaia matches; the baseline is scored WITHOUT its own Gaia leg.
          25-object montages per cluster support a visual check.
Output  : evidence.csv, cluster_names.json, semantic_audit.json, montage_cluster_*.png,
          size_magnitude.png (check the stellar locus before trusting the names)
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..catalogs import load_gzh
from ..context import Ctx, load_json, load_objects, save_json
from ..features import load_cutouts
from ..utils import as_bool, asinh_normalise, match_with_offset, noise_evidence

NAMES = ("Star", "Galaxy", "Noise")


def star_metrics(pred, truth, in_range):
    p = (np.asarray(pred) == "Star") & in_range
    t = np.asarray(truth, bool) & in_range
    tp, fp, fn = int((p & t).sum()), int((p & ~t).sum()), int((~p & t).sum())
    prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return {"precision": round(prec, 3), "recall": round(rec, 3),
            "f1": round(2 * prec * rec / max(prec + rec, 1e-9), 3), "n_gaia_stars": int(t.sum())}


def run(ctx: Ctx):
    cfg = ctx.cfg["semantic_audit"]
    out = ctx.stage_dir("2d")
    df = load_objects(ctx, ["sources", "pass1", "clusters", "baseline", "gaia"])
    metas = list(load_json(ctx.require("1a", "manifest.json")).values())

    gzh = load_gzh(ctx, metas, required=False)
    if gzh is not None and len(gzh):
        _, _, hit, off = match_with_offset(df.ra, df.dec, gzh.ra, gzh.dec, cfg["gzh_radius_arcsec"],
                                           estimate=ctx.cfg["catalogs"]["gzh"].get("estimate_offset", True))
        ctx.log.info(f"[Stage 2d] GZH positions: {int(hit.sum())} matches (offset removed: "
                     f"dRA {off[0]:+.3f}\", dDec {off[1]:+.3f}\")")
    else:
        hit = np.zeros(len(df), bool)
        ctx.log.info("[Stage 2d] no GZH table - galaxy evidence falls back to the baseline rule")
    gaia = df.gaia_hit.astype(str).str.lower().eq("true").to_numpy()
    noise = noise_evidence(df, ctx.cfg["baseline"])
    locus_star = stellar_locus(df, ctx.cfg["baseline"])
    star_ev = gaia | locus_star
    resolved_real = resolved_two_band(df, ctx.cfg["baseline"])
    gal_ev = hit | resolved_real
    ctx.log.info(f"[Stage 2d] galaxy evidence: {int(hit.sum())} GZH + {int((resolved_real & ~hit).sum())} resolved and "
                 "confirmed in a second band")
    ctx.log.info(f"[Stage 2d] star evidence: {int(gaia.sum())} Gaia + {int((locus_star & ~gaia).sum())} on the stellar "
                 f"locus (point-like or saturated, confirmed in a second band)")
    ev = pd.DataFrame({"cluster": df.cluster_id_gmm, "Star": star_ev, "Galaxy": gal_ev, "Noise": noise})
    # Clusters are NAMED from TRAIN-split evidence only and SCORED on held-out objects: naming and
    # scoring on the same Gaia matches would be circular, and test objects must not shape the labels.
    tr = (df.split_tag == "train").to_numpy()
    rate = ev[tr].groupby("cluster")[list(NAMES)].mean()
    rate["n_train"] = ev[tr].groupby("cluster").size()
    no_train = sorted(set(ev.cluster) - set(rate.index))
    if no_train:                                   # a cluster with no train member: use all its members
        extra = ev[ev.cluster.isin(no_train)].groupby("cluster")[list(NAMES)].mean()
        extra["n_train"] = 0
        rate = pd.concat([rate, extra])
    rate["n"] = ev.groupby("cluster").size()
    score = rate[list(NAMES)] / rate[list(NAMES)].max().replace(0, 1)
    # a signal can only name a cluster if enough TRAIN members carry it (3 Gaia stars in a cluster of 400 are
    # not evidence that the cluster is stars - the max-normalisation would otherwise inflate a rare signal)
    min_n = int(cfg.get("min_evidence_members", 5))
    support = ev[tr].groupby("cluster")[list(NAMES)].sum().reindex(rate.index).fillna(0)
    score = score.where(support >= min_n, 0.0)
    names = {}
    for c, row in score.iterrows():
        if row.max() > 0:
            names[int(c)] = NAMES[int(np.argmax(row.values))]
        else:
            names[int(c)] = df.loc[df.cluster_id_gmm == c, "baseline_semantic_nogaia"].mode().iloc[0]
    rate["semantic"] = pd.Series(names)
    semantic = df.cluster_id_gmm.map(names)

    in_range = (df.mag < cfg["gaia_mag_limit"]).to_numpy()
    held = in_range & ~tr
    metrics = {"learned_star_vs_gaia": star_metrics(semantic, gaia, held),
               "baseline_star_vs_gaia": star_metrics(df.baseline_semantic_nogaia, gaia, held),
               "scored_on": "held-out objects (not train), detection-band mag < gaia_mag_limit",
               "learned_star_vs_gaia_all_objects": star_metrics(semantic, gaia, in_range),
               "baseline_star_vs_gaia_all_objects": star_metrics(df.baseline_semantic_nogaia, gaia, in_range),
               "learned_galaxy_recall_vs_gzh": round(float((semantic[hit & ~tr] == "Galaxy").mean()), 3)
               if (hit & ~tr).any() else None,
               "baseline_galaxy_recall_vs_gzh": round(float((df.baseline_semantic_nogaia[hit & ~tr] == "Galaxy").mean()), 3)
               if (hit & ~tr).any() else None,
               "cluster_names": {str(k): v for k, v in names.items()},
               "cluster_evidence": rate.round(3).reset_index().to_dict(orient="records")}
    metrics["clustering_validation"] = clustering_validation(df, ev, tr, names)
    ctx.log.info(f"[Stage 2d] external validation of the clusterings (held-out objects with exactly one evidence "
                 f"signal): {metrics['clustering_validation']['summary']}")
    l, b = metrics["learned_star_vs_gaia"], metrics["baseline_star_vs_gaia"]
    metrics["objective2_minimum_met"] = bool(l["n_gaia_stars"] > 0 and l["f1"] > b["f1"])
    save_json(metrics, out / "semantic_audit.json")
    save_json({str(k): v for k, v in names.items()}, out / "cluster_names.json")
    pd.DataFrame({"object_id": df.object_id, "gzh_position_hit": hit, "noise_evidence": noise,
                  "stellar_locus": locus_star, "cluster_semantic_2d": semantic}).to_csv(out / "evidence.csv", index=False)
    _montages(ctx, df, names, cfg["montage_n"], out)
    _size_magnitude(df, gaia, locus_star, noise, semantic, ctx.cfg["baseline"], out)
    ctx.log.info("[Stage 2d] cluster evidence (share of members with each signal):\n" + rate.round(3).to_string())
    ctx.log.info(f"[Stage 2d] stars vs Gaia (held-out, in range, n={l['n_gaia_stars']}): learned P={l['precision']} "
                 f"R={l['recall']} F1={l['f1']} | baseline P={b['precision']} R={b['recall']} F1={b['f1']} | "
                 f"Objective-2 minimum met: {metrics['objective2_minimum_met']}")
    return {"learned_f1": l["f1"], "baseline_f1": b["f1"]}


def clustering_validation(df, ev, tr, gmm_names):
    """EXTERNAL validation of GMM and HDBSCAN against the independent evidence (Gaia / stellar locus, GZH /
    resolved-two-band, noise signals), on HELD-OUT objects that carry exactly ONE signal:
      purity  = share of objects whose own evidence equals their cluster's majority evidence (cluster majority
                taken from TRAIN members), compared with the trivial 'everything is the majority class' baseline;
      NMI / ARI between cluster label and evidence label (0 = no information, 1 = identical partitions).
    A clustering is only useful if its purity beats the majority baseline."""
    from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
    E = ev[list(NAMES)].astype(bool)
    one = (E.sum(axis=1) == 1).to_numpy()
    own = E.idxmax(axis=1).to_numpy()
    held = ~tr & one
    res = {"n_heldout_single_evidence": int(held.sum()),
           "majority_class_share": round(float(pd.Series(own[held]).value_counts(normalize=True).iloc[0]), 3)
           if held.any() else None}
    cols = {"gmm": df.cluster_id_gmm.to_numpy()}
    if "cluster_id_hdbscan" in df and np.isfinite(df.cluster_id_hdbscan.to_numpy(float)).any():
        cols["hdbscan"] = df.cluster_id_hdbscan.to_numpy(float)
    for name, lab in cols.items():
        lab = np.asarray(lab)
        ok = held & (lab.astype(float) >= 0)                      # HDBSCAN noise (-1) is not a cluster
        if not ok.any():
            continue
        maj = pd.Series(own[tr & one]).groupby(lab[tr & one]).agg(lambda s: s.value_counts().index[0])
        pred = pd.Series(lab[ok]).map(maj).fillna(pd.Series(own[tr & one]).mode().iloc[0])  # no train member: majority
        res[name] = {"n_scored": int(ok.sum()), "n_clusters": int(len(set(lab[lab.astype(float) >= 0]))),
                     "purity": round(float((pred.to_numpy() == own[ok]).mean()), 3),
                     "nmi_vs_evidence": round(float(normalized_mutual_info_score(own[ok], lab[ok])), 3),
                     "ari_vs_evidence": round(float(adjusted_rand_score(own[ok], lab[ok])), 3)}
        res[name]["purity_gain_over_majority"] = round(res[name]["purity"] - res["majority_class_share"], 3)
    sem = df.cluster_id_gmm.map(gmm_names).to_numpy()
    for c in NAMES:                                                # recall of each named population
        m = held & (own == c)
        if m.any():
            res.setdefault("gmm_named_recall", {})[c] = round(float((sem[m] == c).mean()), 3)
    res["summary"] = {k: {kk: v[kk] for kk in ("purity", "purity_gain_over_majority", "nmi_vs_evidence")}
                      for k, v in res.items() if isinstance(v, dict) and "purity" in v}
    res["summary"]["majority_class_share"] = res["majority_class_share"]
    return res


def two_band_confirmed(df):
    return df.cr_candidate.map(lambda v: str(v).strip().lower() in ("false", "0", "0.0")).to_numpy() \
        if "cr_candidate" in df else np.ones(len(df), bool)


def resolved_two_band(df, cfg_b, min_snr=5.0):
    """A real extended source: broader than a point source, SNR >= 5, confirmed in a second band."""
    ratio = np.nan_to_num(df.fwhm_over_psf.to_numpy(float), nan=0)
    return (ratio > 1 + cfg_b["point_source_tolerance"]) & (df.snr.to_numpy(float) >= min_snr) & two_band_confirmed(df)


def stellar_locus(df, cfg_b):
    """Point-like (or saturated) AND confirmed in a second band AND bright enough (SNR >= star_locus_min_snr)
    that size separates a star from a compact galaxy: the stellar sequence of the size-magnitude plane."""
    min_snr = float(cfg_b.get("star_locus_min_snr", 150))
    ratio = df.fwhm_over_psf.to_numpy(float)
    two_band = df.cr_candidate.map(lambda v: str(v).strip().lower() in ("false", "0", "0.0")).to_numpy() \
        if "cr_candidate" in df else np.ones(len(df), bool)
    bright = df.snr.to_numpy(float) >= min_snr
    point = np.abs(ratio - 1.0) <= cfg_b["point_source_tolerance"]
    sat = as_bool(df.saturated).to_numpy() if "saturated" in df else np.zeros(len(df), bool)
    return ((point & bright) | sat) & two_band


def _size_magnitude(df, gaia, locus, noise, semantic, cfg_b, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
    y = df.fwhm_over_psf.to_numpy(float)
    m = df.mag.to_numpy(float)
    for a, title, groups in (
            (ax[0], "evidence", [(noise, "noise evidence", "0.75", 3), (~noise & ~locus & ~gaia, "other", "tab:blue", 4),
                                 (locus & ~gaia, "stellar locus", "tab:orange", 10), (gaia, "Gaia", "red", 30)]),
            (ax[1], "cluster names (Stage 2d)", [(np.asarray(semantic) == n, n, c, 4) for n, c in
                                                 (("Noise", "0.75"), ("Galaxy", "tab:blue"), ("Star", "tab:orange"))])):
        for mask, lab, col, s in groups:
            mask = np.asarray(mask, bool)
            a.scatter(m[mask], y[mask], s=s, c=col, label=f"{lab} ({int(mask.sum())})", edgecolors="none", alpha=0.7)
        a.axhspan(1 - cfg_b["point_source_tolerance"], 1 + cfg_b["point_source_tolerance"], color="k", alpha=0.08)
        lim = np.nanmedian(m[np.abs(df.snr.to_numpy(float) / cfg_b.get("star_locus_min_snr", 150) - 1) < 0.1])
        if np.isfinite(lim):
            a.axvline(lim, color="k", ls=":", lw=1)
            a.text(lim, a.get_ylim()[1] if a.get_yscale() == "linear" else 8, f" SNR {cfg_b.get('star_locus_min_snr', 150):g}",
                   fontsize=7, va="top")
        a.set_yscale("log")
        a.set_xlabel("detection-band AB magnitude")
        a.set_title(title)
        a.legend(fontsize=7, markerscale=1.5)
    ax[0].set_ylabel("FWHM / stellar-locus FWHM")
    fig.suptitle("Stage 2d: size-magnitude plane (grey band = point-like); stars sit on a horizontal sequence at 1")
    fig.tight_layout()
    fig.savefig(out / "size_magnitude.png", dpi=110)
    plt.close(fig)


def _montages(ctx, df, names, n, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    cut = load_cutouts(ctx)
    rng = np.random.default_rng(ctx.seed)
    for c in sorted(df.cluster_id_gmm.unique()):
        members = df[df.cluster_id_gmm == c]
        pick = members.sample(min(n, len(members)), random_state=int(rng.integers(1e6)))
        fig, axes = plt.subplots(5, 5, figsize=(6, 6))
        for ax in axes.ravel():
            ax.axis("off")
        for ax, i in zip(axes.ravel(), pick.cutout_index):
            ax.imshow(asinh_normalise(np.asarray(cut[i], np.float32).sum(0)), origin="lower", cmap="gray")
        fig.suptitle(f"GMM cluster {c} -> {names[int(c)]}  (n = {len(members)})")
        fig.tight_layout()
        fig.savefig(out / f"montage_cluster_{c}.png", dpi=80)
        plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage2e_continuum_gating.py
"""Stage 2e - Continuum handling and outlier gating.

Purpose : do not throw away real galaxies that sit between populations, and do
          not silently drop true anomalies either.
Method  : GMM components are summed into Star / Galaxy / Noise posteriors using
          the Stage-2d names. An object in a LOW-DENSITY region (HDBSCAN noise -1, or a
          GLOSH outlier score above the train percentile - Stage 2b density_outlier) but whose
          galaxy posterior is >= 0.80 is KEPT as a galaxy and tagged
          is_transitional. A low-density object with no posterior above 0.50
          AND a CAE reconstruction error above the 99th percentile of the train
          split is an extreme outlier -> anomaly_log.csv (anomaly_flag = 1).
          Physical consistency gate (gating.physical_gate): a GMM cluster is
          named as a whole, but one cluster can mix stars with compact galaxies.
          Each object is then checked against evidence the cluster name must
          not contradict:
            cosmic-ray candidate (no second band, Stage 1c)  -> Noise
            Gaia star                                         -> Star
            in a Star cluster without star evidence (Gaia, or the bright stellar
              locus of Stage 2d): sharper than the PSF -> Noise, otherwise -> Galaxy
              (faint point-like sources in a deep field are mostly compact galaxies)
            in a Galaxy cluster but on the stellar locus (Stage 2d)                    -> Star
            in a Noise cluster but confirmed in two bands, SNR >= 5, not sharper than
              the PSF (a real source): on the stellar locus -> Star, otherwise -> Galaxy
          The reason is kept in classification_override (empty = cluster name
          kept); cluster_semantic_2d keeps the pure cluster name, which is what
          Stage 2d scores for Objective 2.
          If HDBSCAN failed (Stage 2b), the transitional rule cannot be applied:
          is_transitional is left EMPTY (unknown, not False), and the outlier
          rule runs without the HDBSCAN condition.
Output  : segregation.csv (final cluster_semantic for every object), anomaly_log.csv
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..context import Ctx, load_json, load_objects
from ..utils import as_bool

NAMES = ("Star", "Galaxy", "Noise")


def physical_gate(df, sem, override, cfg_b):
    """Object-level checks a cluster name must not contradict (see the module docstring)."""
    tol = cfg_b["point_source_tolerance"]
    ratio = df.fwhm_over_psf.to_numpy(float)
    cr = as_bool(df.cr_candidate).to_numpy() if "cr_candidate" in df else np.zeros(len(df), bool)
    gaia = as_bool(df.gaia_hit).to_numpy() if "gaia_hit" in df else np.zeros(len(df), bool)
    locus = as_bool(df.stellar_locus).to_numpy() if "stellar_locus" in df else np.zeros(len(df), bool)
    two_band = df.cr_candidate.map(lambda v: str(v).strip().lower() in ("false", "0", "0.0")).to_numpy() \
        if "cr_candidate" in df else np.ones(len(df), bool)
    resolved = np.nan_to_num(ratio, nan=0) > 1 + 2 * tol
    real_noise_cluster = two_band & (df.snr.to_numpy(float) >= 5) & (np.nan_to_num(ratio, nan=0) >= cfg_b["sub_psf_factor"])
    sem, override = sem.copy(), override.copy()

    def move(mask, to, why):
        mask = mask & (sem != to)
        sem[mask] = to
        override[mask] = why

    move((sem == "Noise") & real_noise_cluster & locus, "Star", "Noise cluster, but confirmed in 2 bands on the stellar locus")
    move((sem == "Noise") & real_noise_cluster & ~locus, "Galaxy",
         "Noise cluster, but a real source (confirmed in 2 bands, SNR >= 5, not sharper than the PSF)")
    move((sem == "Galaxy") & locus & ~resolved, "Star", "Galaxy cluster, but on the stellar locus")
    broader = np.nan_to_num(ratio, nan=0) > 1 + tol
    sharper = np.nan_to_num(ratio, nan=1) < cfg_b["sub_psf_factor"]
    no_star_evidence = (sem == "Star") & ~locus & ~gaia
    move(no_star_evidence & broader, "Galaxy", "Star cluster, but broader than a point source")
    move(no_star_evidence & sharper, "Noise", "Star cluster, but sharper than the PSF")
    move(no_star_evidence & ~broader & ~sharper, "Galaxy",
         "Star cluster, but too faint for size to separate a star from a compact galaxy")
    move(gaia, "Star", "Gaia star")
    move(cr & ~gaia, "Noise", "cosmic-ray candidate (no second band confirms it)")
    return sem, override


def run(ctx: Ctx):
    cfg = ctx.cfg["gating"]
    out = ctx.stage_dir("2e")
    df = load_objects(ctx, ["sources", "pass1", "clusters", "gaia", "evidence"])
    names = {int(k): v for k, v in load_json(ctx.require("2d", "cluster_names.json")).items()}
    err = pd.Series(np.load(ctx.require("2a", "recon_error.npy")),
                    index=np.load(ctx.require("2a", "object_id.npy"))).reindex(df.object_id).to_numpy()

    post = {n: np.zeros(len(df)) for n in NAMES}
    for k, n in names.items():
        col = f"gmm_p{k}"
        if col in df and n in post:
            post[n] += df[col].to_numpy(float)
    hdb = df.cluster_id_hdbscan.to_numpy(float)
    hdb_ok = np.isfinite(hdb).any() and (hdb >= 0).any()
    if not hdb_ok:
        ctx.log.info("[Stage 2e] WARNING: HDBSCAN failed (Stage 2b) - is_transitional left empty; the outlier "
                     "rule uses GMM posterior + reconstruction error only.")
    if hdb_ok and "density_outlier" in df:
        # HDBSCAN noise OR a GLOSH outlier score above the train percentile (Stage 2b): in a deep field the
        # sources form one density continuum, so label -1 alone would mark almost nothing
        hnoise = as_bool(df.density_outlier).to_numpy()
    else:
        hnoise = (hdb == -1) if hdb_ok else np.ones(len(df), bool)
    ctx.log.info(f"[Stage 2e] low-density objects (HDBSCAN noise / GLOSH outliers): {int(hnoise.sum())}")
    P = np.column_stack([post[n] for n in NAMES])
    thr = float(np.nanpercentile(err[(df.split_tag == "train").to_numpy()], cfg["recon_percentile"]))
    transitional = hnoise & (post["Galaxy"] >= cfg["transitional_p"]) if hdb_ok else np.zeros(len(df), bool)
    anomaly = hnoise & (P.max(1) < cfg["outlier_max_p"]) & (np.nan_to_num(err, nan=-1) > thr)

    sem = df.cluster_semantic_2d.to_numpy(object).copy()
    sem[transitional] = "Galaxy"
    override = np.full(len(df), None, dtype=object)
    override[transitional] = "transitional (low density in HDBSCAN, GMM galaxy posterior >= transitional_p)"
    if cfg.get("physical_gate", True):
        sem, override = physical_gate(df, sem, override, ctx.cfg["baseline"])
        ctx.log.info(f"[Stage 2e] physical gate: {pd.Series(override).value_counts().to_dict()}")
    res = pd.DataFrame({"object_id": df.object_id, "cluster_semantic": sem,
                        "gmm_p_star": post["Star"].round(4), "gmm_p_galaxy": post["Galaxy"].round(4),
                        "gmm_p_noise": post["Noise"].round(4), "reconstruction_error": err,
                        "is_transitional": transitional if hdb_ok else None, "anomaly_flag": anomaly.astype(int),
                        "classification_override": override})
    res.to_csv(out / "segregation.csv", index=False)
    log = res[res.anomaly_flag == 1].merge(df[["object_id", "ra", "dec", "pointing", "cluster_id_gmm",
                                               "cluster_id_hdbscan"]], on="object_id")
    log["reason"] = "low density (HDBSCAN noise / GLOSH) + no GMM population > 0.5 + reconstruction error above train 99th pct"
    log.to_csv(out / "anomaly_log.csv", index=False)
    ctx.log.info(f"[Stage 2e] populations {res.cluster_semantic.value_counts().to_dict()} | transitional kept "
                 f"{int(transitional.sum())} | anomalies logged {int(anomaly.sum())} (recon threshold {thr:.5f})")
    return {"populations": res.cluster_semantic.value_counts().to_dict(), "anomalies": int(anomaly.sum())}

In [ ]:
%%writefile src/p2p/stages/stage3a_gzh_labels.py
"""Stage 3a - Galaxies: Galaxy Zoo: Hubble label anchoring.

Purpose : attach crowd-sourced morphology labels to the galaxy track.
Method  : galaxy positions are matched one-to-one to GZH within 0.5" after the
          median catalogue-to-image offset is removed. GZH vote fractions are
          conditional on the earlier answers of the decision tree, so:
            Smooth        f_smooth >= 0.6
            featured      f_features >= 0.5 (and not Smooth), then
              EdgeOn        f_edgeon >= 0.5
              FaceOnSpiral  not edge-on and f_spiral >= 0.5
              Irregular     not edge-on, not spiral, and (f_clumpy >= 0.5 or
                            [f_odd >= 0.4 and max(irregular, disturbed) >= 0.5])
            Ambiguous     everything else
          merger flag: f_odd >= 0.4 and f_merger >= 0.5.
          Human agreement = the WEAKEST vote fraction along the path to the
          assigned class; matched galaxies with agreement < 0.5 form the
          Ambiguous Boundary set used by Stage 6b. Unmatched galaxies are kept
          for inference only (is_pseudo_label = True).
Output  : labels.csv, label_summary.json
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..catalogs import load_gzh
from ..context import Ctx, load_json, load_objects, save_json
from ..nets import MORPH_CLASSES
from ..utils import match_with_offset


def map_labels(g: pd.DataFrame, t: dict) -> pd.DataFrame:
    f = lambda c: g[c].fillna(0).to_numpy(float)
    smooth, feat, edge, spiral = f("smooth"), f("features"), f("edgeon"), f("spiral")
    odd, clumpy = f("odd"), f("clumpy")
    irr_odd = np.maximum(f("odd_irregular"), f("odd_disturbed"))
    irregular_votes = np.maximum(clumpy, np.where(odd >= t["odd"], irr_odd, 0.0))
    lab = np.full(len(g), 4)
    featured = (smooth < t["smooth"]) & (feat >= t["features"])
    lab[smooth >= t["smooth"]] = 0
    lab[featured & (edge >= t["edgeon"])] = 2
    lab[featured & (edge < t["edgeon"]) & (spiral >= t["spiral"])] = 1
    lab[featured & (edge < t["edgeon"]) & (spiral < t["spiral"]) & (irregular_votes >= t["irregular"])] = 3
    # agreement = weakest link along the decision path
    A = np.column_stack([smooth,
                         np.minimum.reduce([feat, 1 - edge, spiral]),
                         np.minimum(feat, edge),
                         np.minimum.reduce([feat, 1 - edge, 1 - spiral, irregular_votes])])
    A = np.column_stack([A, 1.0 - A.max(1)])
    agree = A[np.arange(len(lab)), lab]
    return pd.DataFrame({"gz_label_id": lab, "gz_label": np.array(MORPH_CLASSES)[lab],
                         "gz_vote_frac": agree.round(4), "human_disagreement": (1 - agree).round(4),
                         "gz_merger_flag": ((odd >= t["odd"]) & (f("odd_merger") >= t["merger"])).astype(int)})


def run(ctx: Ctx):
    cfg = ctx.cfg["catalogs"]["gzh"]
    out = ctx.stage_dir("3a")
    df = load_objects(ctx, ["sources", "segregation"])
    metas = list(load_json(ctx.require("1a", "manifest.json")).values())
    from ..modes import is_apply
    gzh = load_gzh(ctx, metas, required=not is_apply(ctx))
    gal = df[(df.cluster_semantic == "Galaxy") & (df.anomaly_flag == 0)].copy()
    if gzh is None or gzh.empty:
        if not is_apply(ctx):
            raise RuntimeError("No Galaxy Zoo: Hubble galaxies cover this field, so the models cannot be trained here. "
                               "Run it in apply mode with a model bundle (supervised.mode: apply, "
                               "supervised.model_bundle: models/<name>; tools/run_field.py does this for you).")
        res = pd.DataFrame({"object_id": gal.object_id.to_numpy(), "gz_label_id": np.nan, "gz_label": None,
                            "gz_vote_frac": np.nan, "human_disagreement": np.nan, "gz_merger_flag": np.nan,
                            "gzh_sep_arcsec": np.nan, "gzh_row": -1, "is_pseudo_label": True, "in_boundary_set": False})
        res.to_csv(out / "labels.csv", index=False)
        summary = {"galaxies": len(gal), "matched": 0, "unmatched": len(gal), "mode": "apply",
                   "note": "no Galaxy Zoo: Hubble coverage - every galaxy is classified by the frozen model bundle"}
        save_json(summary, out / "label_summary.json")
        ctx.log.info(f"[Stage 3a] no GZH galaxies in this field (apply mode): {len(gal)} galaxies, all model-labelled")
        return {"matched": 0}
    idx, sep, hit, off = match_with_offset(gal.ra, gal.dec, gzh.ra, gzh.dec, cfg["match_radius_arcsec"],
                                           estimate=cfg.get("estimate_offset", True))
    lab = map_labels(gzh, cfg["thresholds"])
    res = pd.DataFrame({"object_id": gal.object_id.to_numpy()})
    for c in lab.columns:
        res[c] = np.where(hit, lab[c].to_numpy()[np.clip(idx, 0, None)], np.nan)
    res["gz_label"] = np.where(hit, lab["gz_label"].to_numpy()[np.clip(idx, 0, None)], None)
    res["gzh_sep_arcsec"] = np.where(hit, sep, np.nan).round(4)
    res["gzh_row"] = np.where(hit, gzh.gzh_row.to_numpy()[np.clip(idx, 0, None)], -1)
    res["is_pseudo_label"] = ~hit
    res["in_boundary_set"] = hit & (res.gz_vote_frac.fillna(1).to_numpy() < cfg["boundary_agreement"])
    res.to_csv(out / "labels.csv", index=False)

    lab_rows = res[~res.is_pseudo_label].merge(gal[["object_id", "split_tag"]], on="object_id")
    table = pd.crosstab(lab_rows.split_tag.fillna("collar"), lab_rows.gz_label)
    from ..sealing import test_sealed
    if test_sealed(ctx):
        # sealed: the class mix of the test / external galaxies is not shown either - only how many there are
        sealed_rows = [r for r in table.index if r in ("test", "external")]
        counts = {r: int(table.loc[r].sum()) for r in sealed_rows}
        table = table.drop(index=sealed_rows)
        for r, n in counts.items():
            table.loc[f"{r} (sealed: {n} galaxies)"] = np.nan
    summary = {"galaxies": len(gal), "matched": int(hit.sum()), "unmatched": int((~hit).sum()),
               "offset_arcsec": {"dra": off[0], "ddec": off[1]},
               "per_split_class": table.to_dict(), "boundary_set": int(res.in_boundary_set.sum()),
               "merger_prevalence_train": float(lab_rows[lab_rows.split_tag == "train"].gz_merger_flag.mean())
               if (lab_rows.split_tag == "train").any() else None}
    save_json(summary, out / "label_summary.json")
    ctx.log.info(f"[Stage 3a] galaxy track {len(gal)} | GZH-matched {int(hit.sum())} | unmatched {int((~hit).sum())} "
                 f"| offset removed dRA {off[0]:+.3f}\" dDec {off[1]:+.3f}\"")
    ctx.log.info("[Stage 3a] labelled galaxies per split / class:\n" + table.to_string())
    return {"matched": int(hit.sum())}

In [ ]:
%%writefile src/p2p/stages/stage3b_star_routing.py
"""Stage 3b - Stars: calibration routing.

Purpose : point sources skip morphology classification and become PSF tracers.
Method  : candidates are the star track PLUS every Gaia-matched object (a Gaia
          star mis-assigned to another track is still a star). A candidate is
          routed to the empirical-PSF builder if it is a Gaia star or on the
          bright stellar locus of Stage 2d (point-like, SNR >=
          baseline.star_locus_min_snr, confirmed in a second band), and it has SNR >= 20, is not saturated (no flat-topped
          core in the detection band), has no other segment within
          psf_match.star_isolation_px (5 px) of its centre (Stage 5a leaves
          neighbour and cosmic-ray pixels out of the stack) and is not at an edge. Only Gaia stars are used
          whenever at least 5 of them in its pointing pass those cuts (in deep
          fields most Gaia stars saturate).
Output  : star_routing.csv (one row per star with each test), routing_summary.json
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..context import Ctx, load_json, load_objects, save_json
from ..utils import as_bool
from .stage1d_psf_match import flat_top


def run(ctx: Ctx, min_snr=20.0, min_epsf_stars=5):
    iso_radius = ctx.cfg["psf_match"].get("star_isolation_px", 5)   # Stage 5a leaves neighbour pixels out
    out = ctx.stage_dir("3b")
    df = load_objects(ctx, ["sources", "pass1", "gaia", "evidence", "segregation"])
    tol = ctx.cfg["baseline"]["point_source_tolerance"]
    bands = load_json(ctx.require("1a", "pointings.json"))["bands"]
    raw = np.load(ctx.require("1c", "cutouts_raw.npy"), mmap_mode="r")
    segs = np.load(ctx.require("1c", "segstamps.npy"), mmap_mode="r")
    stars = df[((df.cluster_semantic == "Star") | as_bool(df.gaia_hit)) & (df.anomaly_flag == 0)].copy()
    S = segs.shape[1]
    yy, xx = np.mgrid[:S, :S]
    near = np.hypot(yy - (S - 1) / 2, xx - (S - 1) / 2) <= iso_radius
    det_i = stars.detection_band.map(bands.index).to_numpy() if len(stars) else []
    stars["epsf_snr_ok"] = stars.snr >= min_snr
    stars["epsf_unsaturated"] = [not flat_top(raw[i, d]) for i, d in zip(stars.cutout_index, det_i)]
    stars["epsf_isolated"] = [not np.any(segs[i][near] == 2) for i in stars.cutout_index]
    stars["epsf_not_edge"] = ~as_bool(stars.edge_flag)
    stars["epsf_unsaturated"] &= ~as_bool(stars.saturated) if "saturated" in stars else True
    stars["epsf_point_like"] = (stars.fwhm_over_psf.astype(float) - 1.0).abs() <= tol
    stars["epsf_two_band"] = stars.cr_candidate.map(lambda v: str(v).strip().lower() in ("false", "0", "0.0")) \
        if "cr_candidate" in stars else True
    gaia = as_bool(stars.gaia_hit)
    locus = as_bool(stars.stellar_locus) if "stellar_locus" in stars else (stars.epsf_point_like & stars.epsf_two_band)
    stars["epsf_star_evidence"] = gaia | (locus & stars.epsf_point_like)
    ok = (stars.epsf_star_evidence & stars.epsf_snr_ok & stars.epsf_unsaturated & stars.epsf_isolated
          & stars.epsf_not_edge)
    # Gaia confirmation is required only when enough confirmed stars survive the cuts: in a deep field
    # most Gaia stars (G < 21) saturate a long ACS exposure, and Stage 5a needs >= 5 stars per pointing
    gaia_required = {}
    for pid, idx in stars.groupby("pointing").groups.items():
        with_gaia = ok.loc[idx] & gaia.loc[idx]
        gaia_required[pid] = bool(with_gaia.sum() >= min_epsf_stars)
        if gaia_required[pid]:
            ok.loc[idx] = with_gaia
        elif gaia.loc[idx].any():
            ctx.log.info(f"[Stage 3b] {pid}: only {int(with_gaia.sum())} Gaia-confirmed clean stars - using all "
                         f"{int(ok.loc[idx].sum())} clean stars of the star track for the ePSF")
    stars["used_in_epsf"] = ok
    cols = ["object_id", "pointing", "snr", "gaia_hit", "cluster_semantic", "epsf_point_like", "epsf_two_band",
            "epsf_star_evidence", "epsf_snr_ok", "epsf_unsaturated", "epsf_isolated", "epsf_not_edge", "used_in_epsf"]
    stars[cols].to_csv(out / "star_routing.csv", index=False)
    per = stars.groupby("pointing").used_in_epsf.sum().astype(int).to_dict()
    save_json({"stars": len(stars), "routed": int(ok.sum()), "per_pointing": per,
               "gaia_required_per_pointing": gaia_required}, out / "routing_summary.json")
    ctx.log.info(f"[Stage 3b] candidates {len(stars)} (star track + Gaia) | Gaia {int(gaia.sum())} | point-like "
                 f"{int(stars.epsf_point_like.sum())} | confirmed in 2 bands {int(stars.epsf_two_band.sum())} | "
                 f"SNR ok {int(stars.epsf_snr_ok.sum())} | unsaturated "
                 f"{int(stars.epsf_unsaturated.sum())} | isolated {int(stars.epsf_isolated.sum())} | routed to ePSF "
                 f"{int(ok.sum())} {per}")
    return {"routed": int(ok.sum())}

In [ ]:
%%writefile src/p2p/stages/stage3c_noise_model.py
"""Stage 3c - Noise: false-positive modelling.

Purpose : characterise spurious detections - the Noise population has no
          external catalogue, so this model is also its validation.
Method  : two kinds of spurious detection are modelled separately:
          (1) sky-noise peaks - the NEGATIVE-IMAGE test on each detection frame.
              Sky noise is symmetric, so the number of 'sources' found on -SCI
              at nu x sigma estimates how many detections on +SCI at the same
              threshold are spurious. nu is each source's PEAK significance in
              the smoothed detection image (the quantity the detection threshold
              is applied to), not its total SNR. The grid of thresholds extends
              to the brightest source, so bright sources are not clamped to the
              last grid value. The FPR cannot be resolved below 1 / n_pos(nu)
              (one negative detection); that floor is reported, and an FPR of 0
              means 'below the floor'.
          (2) cosmic rays - positive-only, so the negative-image test cannot see
              them (single exposures are full of them, at HIGH S/N). Their term
              is p_cr = sigmoid((confirm_snr - best other-band S/N) / 0.5) from
              the Stage-1c two-band test (0 when no other band covers the source).
          (3) a profile sharper than the PSF: p_sub = sigmoid((sub_psf_factor - FWHM/locus) / 0.05).
          noise_score = 1 - (1 - FPR(nu)) (1 - p_sub) (1 - p_cr).
Output  : noise_model.json (FPR curve per pointing, operating threshold, resolution floor),
          noise_scores.csv, fpr_curve.png
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..compat import detect, n_labels
from ..context import Ctx, load_json, load_objects, save_json
from ..detection import smooth, subtract_background
from ..fitsio import load_image, valid_mask


def threshold_grid(nu_max, lo=1.5, fine_to=6.0, step=0.25, n_log=20):
    """Fine linear steps to 6 sigma, then logarithmic steps up to the brightest source."""
    fine = np.arange(lo, fine_to + 1e-9, step)
    if not np.isfinite(nu_max) or nu_max <= fine_to:
        return fine
    return np.unique(np.concatenate([fine, np.geomspace(fine_to, nu_max * 1.05, n_log)]))


def run(ctx: Ctx, target_fpr=0.01):
    out = ctx.stage_dir("3c")
    dcfg = ctx.cfg["detection"]
    pts = load_json(ctx.require("1a", "pointings.json"))["pointings"]
    df = load_objects(ctx, ["sources", "baseline", "segregation"])
    nu = np.full(len(df), np.nan)
    curves = {}
    for pid, p in pts.items():
        sci, wht, _ = load_image(p["bands"][p["detection_band"]])
        val = valid_mask(sci, wht)
        sub, rms = subtract_background(sci, val, dcfg["background_box"], dcfg["background_filter"])
        conv = smooth(sub, dcfg["kernel_fwhm_px"])                 # same filter as Stage 1c, done once
        base = rms * dcfg.get("correlated_noise_factor", 1.0)
        m = (df.pointing == pid).to_numpy()
        xi = np.clip(np.round(df.x[m].to_numpy()).astype(int), 0, conv.shape[1] - 1)
        yi = np.clip(np.round(df.y[m].to_numpy()).astype(int), 0, conv.shape[0] - 1)
        nu[m] = conv[yi, xi] / base[yi, xi]                        # peak significance at the centroid
        grid = threshold_grid(np.nanmax(nu[m]) if m.any() else np.nan)
        rows = []
        for k in grid:
            n_pos = n_labels(detect(conv, k * base, dcfg["npixels"], mask=~val))
            n_neg = n_labels(detect(-conv, k * base, dcfg["npixels"], mask=~val))
            rows.append({"nsigma": float(k), "n_pos": n_pos, "n_neg": n_neg,
                         "fpr": float(min(n_neg / max(n_pos, 1), 1.0)),
                         "fpr_floor": float(1.0 / max(n_pos, 1))})
            if n_pos == 0:
                break
        cur = pd.DataFrame(rows)
        ok = cur[cur.fpr <= target_fpr]
        last_neg = cur[cur.n_neg > 0]
        floor_nu = float(last_neg.nsigma.max()) if len(last_neg) else float(cur.nsigma.min())
        curves[pid] = {"curve": cur.to_dict(orient="list"),
                       "operating_nsigma": float(ok.nsigma.iloc[0]) if len(ok) else float(cur.nsigma.iloc[-1]),
                       "fpr_at_configured_nsigma": float(np.interp(dcfg["nsigma"], cur.nsigma, cur.fpr)),
                       "last_threshold_with_a_negative_detection": floor_nu,
                       "resolution_note": "FPR = n_neg / n_pos; above the last threshold with a negative detection "
                                          "the FPR is only known to be below 1 / n_pos at that threshold"}
        ctx.log.info(f"[Stage 3c] {pid}: FPR <= {target_fpr} from {curves[pid]['operating_nsigma']:.2f} sigma; "
                     f"FPR at the detection threshold ({dcfg['nsigma']} sigma) = "
                     f"{curves[pid]['fpr_at_configured_nsigma']:.3f}; no negative detection above "
                     f"{floor_nu:.1f} sigma (FPR resolution there: 1/{int(cur.n_pos[cur.nsigma >= floor_nu].iloc[0])})")

    fpr = np.full(len(df), np.nan)
    for pid, c in curves.items():
        m = (df.pointing == pid).to_numpy()
        fpr[m] = np.interp(nu[m], c["curve"]["nsigma"], c["curve"]["fpr"], right=0.0)
    sub_psf = 1.0 / (1.0 + np.exp((df.fwhm_over_psf.to_numpy(float) - ctx.cfg["baseline"]["sub_psf_factor"]) / 0.05))
    confirm = float(dcfg.get("confirm_snr", 3.0))
    best = pd.to_numeric(df.get("confirm_snr_best", pd.Series(np.nan, index=df.index)), errors="coerce").to_numpy(float)
    p_cr = np.where(np.isfinite(best), 1.0 / (1.0 + np.exp((best - confirm) / 0.5)), 0.0)
    score = 1 - (1 - np.nan_to_num(fpr)) * (1 - np.nan_to_num(sub_psf)) * (1 - p_cr)
    res = pd.DataFrame({"object_id": df.object_id, "peak_significance": np.round(nu, 3),
                        "fpr_at_threshold": np.round(fpr, 5), "p_cosmic_ray": np.round(p_cr, 4),
                        "noise_score": np.round(score, 4)})
    res.to_csv(out / "noise_scores.csv", index=False)
    validation = res.groupby(df.cluster_semantic.to_numpy())["noise_score"].mean().round(3).to_dict()
    snr_by_track = df.groupby("cluster_semantic").snr.median().round(1).to_dict()
    save_json({"target_fpr": target_fpr, "pointings": curves, "validation_mean_noise_score_by_track": validation,
               "median_snr_by_track": snr_by_track,
               "terms": "noise_score = 1 - (1 - FPR(peak significance)) (1 - p_sub_psf) (1 - p_cosmic_ray)"},
              out / "noise_model.json")
    _plot(curves, out)
    ctx.log.info(f"[Stage 3c] mean noise_score by track (validation): {validation} | median SNR by track "
                 f"{snr_by_track} (cosmic rays are high-S/N: the p_cosmic_ray term, not the FPR, scores them)")
    return {"validation": validation}


def _plot(curves, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5.5, 3.5))
    for pid, c in curves.items():
        ax.plot(c["curve"]["nsigma"], c["curve"]["fpr"], "o-", ms=3, label=f"{pid} FPR")
        ax.plot(c["curve"]["nsigma"], c["curve"]["fpr_floor"], ":", lw=1, label=f"{pid} resolution 1/n_pos")
    ax.axhline(0.01, color="k", ls=":", lw=1)
    ax.set_xscale("log")
    ax.set_yscale("symlog", linthresh=1e-4)
    ax.set_xlabel("peak significance threshold [sigma]")
    ax.set_ylabel("false-positive rate (negative image)")
    ax.legend(fontsize=7)
    ax.set_title("Stage 3c: false-positive curve")
    fig.tight_layout()
    fig.savefig(out / "fpr_curve.png", dpi=110)
    plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage4a_cnn_features.py
"""Stage 4a - CNN feature extraction: ResNet-18 + EfficientNet-B0 (ImageNet weights, frozen).

Purpose : describe every galaxy's image by the visual traits two pretrained CNNs see (edges, curves, textures,
          gradients). No CNN is trained here - with ~1,100 labelled galaxies a trained CNN memorises them; the
          only model that learns from the labels is XGBoost (Stage 4b).
Input   : every Galaxy-track cutout (Stage 2e; Stage 1d's PSF-matched 128 x 128 x 3 cutouts):
          one asinh stretch per cutout (same for all bands, colours kept) -> R, G, B = F775W, F606W, F435W
          -> ImageNet mean / std -> bilinear resize to supervised.input_size (224).
Models  : ResNet-18       conv 7x7 -> 4 x 2 residual blocks (64-128-256-512) -> GlobalAveragePooling -> 512
          EfficientNet-B0 conv 3x3 -> 16 MBConv blocks -> conv 1x1 (1280)  -> GlobalAveragePooling -> 1,280
          Activations ReLU (ResNet) / SiLU (EfficientNet), BatchNorm with ImageNet statistics, Dropout off;
          the ImageNet classifiers (1,000 classes) are not used.
Output  : cnn_features.npy (n_galaxies x 1,792: [ResNet-18 512 | EfficientNet-B0 1,280]), object_id.npy,
          feature_info.json, model_summary.txt
"""
from __future__ import annotations

import time

import numpy as np

from ..context import Ctx, save_json
from ..features import galaxy_table, imagenet_input, load_cutouts
from ..inference import CNN_BLOCKS, build_extractors, extract_cnn_features
from ..nets import model_summary


def run(ctx: Ctx):
    from ..modes import check_bands, is_apply
    cfg = ctx.cfg["supervised"]
    out = ctx.stage_dir("4a")
    if is_apply(ctx):                       # the frozen CNNs need no bundle - only the same band set
        check_bands(ctx)
    size, pretrained = int(cfg.get("input_size", 224)), bool(cfg.get("pretrained", True))
    gal = galaxy_table(ctx)
    ctx.log.info(f"[Stage 4a] {len(gal):,} galaxies -> ImageNet input {size} x {size} -> frozen ResNet-18 + "
                 f"EfficientNet-B0 ({'ImageNet weights' if pretrained else 'RANDOM weights (pretrained: false)'}) "
                 f"on {ctx.device()}")

    # ---- 1. build the two frozen networks  2. summary ----
    nets = build_extractors(pretrained)
    x = imagenet_input(load_cutouts(ctx)[gal.cutout_index.to_numpy()[:1]], size)
    summ = "\n\n".join(model_summary(nets[k], x, name=f"Stage 4a - {k} (ImageNet, frozen)", max_depth=2)
                       for k, _ in CNN_BLOCKS)
    (out / "model_summary.txt").write_text(summ + "\n")
    ctx.log.info("[Stage 4a] model summaries:\n" + summ)

    # ---- 3. extract: one forward pass per cutout, no training ----
    t0 = time.time()
    F = extract_cnn_features(ctx, gal.cutout_index.to_numpy(), nets, log=ctx.log.info)
    np.save(out / "cnn_features.npy", F)
    np.save(out / "object_id.npy", gal.object_id.to_numpy())
    blocks, s = {}, 0
    for k, n in CNN_BLOCKS:
        blocks[k] = [s, s + n]
        s += n
    info = {"n_galaxies": int(len(gal)), "n_features": int(F.shape[1]), "blocks": blocks, "input_size": size,
            "pretrained": pretrained, "channels": "R, G, B = reddest, middle, bluest band (asinh, ImageNet mean/std)",
            "feature_mean_by_block": {k: round(float(F[:, a:b].mean()), 4) for k, (a, b) in blocks.items()},
            "seconds": round(time.time() - t0, 1)}
    save_json(info, out / "feature_info.json")
    ctx.log.info(f"[Stage 4a] features {F.shape} ({info['seconds']} s): ResNet-18 columns 0-511, EfficientNet-B0 "
                 f"columns 512-1791 -> {ctx.rel(out / 'cnn_features.npy')}")
    return {"n_galaxies": int(len(gal)), "n_features": int(F.shape[1]), "input_size": size}

In [ ]:
%%writefile src/p2p/stages/stage4b_xgboost_fusion.py
"""Stage 4b - ENSEMBLE (feature-level fusion) + XGBoost classifier.

Ensemble: every labelled galaxy becomes ONE vector of 1,806 features =
              ResNet-18 (512) + EfficientNet-B0 (1,280)   [Stage 4a, frozen ImageNet CNNs]
            + 14 physical features (Gini, M20, concentration, asymmetry, colours, ... ; Pass 2, Stage 5b).
          Ensemble weights w1 (ResNet-18), w2 (EfficientNet-B0), w3 (physical): the share of the features each
          tree may draw from each block (XGBoost feature_weights; scaling the features themselves would not
          change a tree). The candidate sets in supervised.fusion_block_weights are compared by 5-fold CV and the
          set with the lowest out-of-fold log-loss is used (weight_search.png).
XGBoost : "compile" in supervised.xgboost: softmax over the 5 GZH classes (multi:softprob), shallow trees
          (max_depth 4), learning rate 0.05, 80 % of the galaxies and 30 % of the features per tree,
          L2 penalty reg_lambda on the leaf values, balanced class weights.
Fit     : 5-fold cross-validation on the pooled TRAIN + DEV galaxies (grouped by 30" sky cell); the number of
          trees is chosen by the lowest out-of-fold log-loss; the final model is trained on all pooled galaxies.
          Keras-style history per boosting round (loss / val_loss / accuracy / val_accuracy).
Merger  : a second XGBoost (merger yes / no, logistic) on the same 1,806 features and ensemble weights.
Output  : xgb_morphology.joblib, xgb_merger.joblib, fusion_meta.json, model_summary.txt, weight_search.png / .csv,
          training_history_xgboost.png / .json, cv_curve.png, cv_folds.csv, cv_metrics.json, oof_fusion.npz,
          dev_metrics.json
"""
from __future__ import annotations

import joblib
import numpy as np
import pandas as pd

from ..context import Ctx, save_json
from ..features import supervised_splits
from ..inference import BLOCKS, XGBMember, block_feature_weights, block_slices, feature_names, fusion_matrix, xgb_cv
from ..nets import MORPH_CLASSES, morph_metrics, nll
from ..supervised_cv import (choose_epoch, curve_metrics, make_folds, mean_history, plot_curve, plot_history, pooled,
                             summarise)


def tree_grid(cfg):
    return [10] + list(range(25, int(cfg.get("xgb_max_trees", 400)) + 1, 25))


def run(ctx: Ctx):
    from ..modes import install_bundle, is_apply
    if is_apply(ctx):                       # no training: copy the frozen model bundle
        return install_bundle(ctx, "4b")
    cfg = ctx.cfg["supervised"]
    out = ctx.stage_dir("4b")
    params = dict(cfg.get("xgboost") or {})
    metric = str(cfg.get("selection_metric", "log_loss"))
    grid = tree_grid(cfg)

    # ---- data: GZH-labelled galaxies of train + dev, pooled for 5-fold cross-validation ----
    splits = supervised_splits(ctx)
    pool = pooled(splits)
    folds, fold_id, groups = make_folds(pool, int(cfg.get("cv_folds", 5)), ctx.seed,
                                        float(cfg.get("cv_group_arcsec", 30.0)))
    pool.assign(fold=fold_id, sky_cell=groups)[["object_id", "orig_split", "fold", "sky_cell", "gz_label"]] \
        .to_csv(out / "cv_folds.csv", index=False)
    physical = [c for c in ctx.cfg["representation"]["tabular_features"] if c in pool.columns]
    X = fusion_matrix(ctx, pool, physical)
    names = feature_names(physical)
    y = pool.gz_label_id.astype(int).to_numpy()
    ym = pool.gz_merger_flag.fillna(0).astype(int).to_numpy()
    counts = pd.crosstab(pool.orig_split, pool.gz_label)
    ctx.log.info(f"[Stage 4b] ENSEMBLE input: {X.shape[0]} galaxies x {X.shape[1]} features "
                 f"(ResNet-18 512 + EfficientNet-B0 1,280 + physical {len(physical)})\n{counts.to_string()}\n"
                 f"[Stage 4b] compile: XGBClassifier(objective multi:softprob, {params}, n_estimators <= {max(grid)}, "
                 f"balanced class weights) | metrics: log-loss, accuracy, Macro-F1 | {len(folds)}-fold CV "
                 f"(folds of {[len(v) for _, v in folds]}, grouped by {cfg.get('cv_group_arcsec', 30)}\" sky cells)")

    # ---- 1. ENSEMBLE weights: each candidate (w1, w2, w3) cross-validated, lowest OOF log-loss wins ----
    rows, runs = [], {}
    for w in cfg.get("fusion_block_weights") or [[1 / 3, 1 / 3, 1 / 3]]:
        w = np.asarray(w, float) / np.sum(w)
        fw = block_feature_weights(w, len(physical))
        oof, hist = xgb_cv(X, y, folds, grid, len(MORPH_CLASSES), ctx.seed, params, fw)
        cur = curve_metrics(y, oof, folds)
        k = choose_epoch(cur, 1, metric)[0] - 1
        key = tuple(float(v) for v in np.round(w, 3))
        runs[key] = {"w": w, "fw": fw, "oof": oof, "hist": hist, "curve": cur, "k": k}
        rows.append({"w_resnet18": key[0], "w_efficientnet_b0": key[1], "w_physical": key[2], "trees": grid[k],
                     "oof_log_loss": cur[k]["nll"], "oof_macro_f1": cur[k]["macro_f1"],
                     "oof_accuracy": cur[k]["accuracy"]})
        ctx.log.info(f"  [ensemble] w = {list(key)}: best {grid[k]} trees, OOF log-loss {cur[k]['nll']:.4f}, "
                     f"Macro-F1 {cur[k]['macro_f1']:.4f}, accuracy {cur[k]['accuracy']:.4f}")
    search = pd.DataFrame(rows)
    search.to_csv(out / "weight_search.csv", index=False)
    best_row = search.loc[search.oof_log_loss.idxmin() if metric != "macro_f1" else search.oof_macro_f1.idxmax()]
    best = runs[(best_row.w_resnet18, best_row.w_efficientnet_b0, best_row.w_physical)]
    w, fw, k, n_trees = best["w"], best["fw"], best["k"], grid[best["k"]]
    ctx.log.info(f"[Stage 4b] ENSEMBLE weights chosen: w1 (ResNet-18) = {w[0]:.3f}, w2 (EfficientNet-B0) = "
                 f"{w[1]:.3f}, w3 (physical) = {w[2]:.3f} -> {n_trees} trees per class")
    _plot_search(search, best_row, out)

    # ---- 2. history of the chosen configuration (Keras-style, per boosting round) ----
    history = mean_history({key: [h[key] for h in best["hist"]] for key in best["hist"][0]})
    plot_history(history, "XGBoost (fusion)", out / "training_history_xgboost.png",
                 xlabel="Boosting rounds (trees per class)", chosen=n_trees)
    save_json(dict(history, chosen_trees=n_trees), out / "training_history_xgboost.json")
    r = sorted({i for i in (9, 49, 99, 199, 299, max(grid) - 1, n_trees - 1) if i < len(history["loss"])})
    ctx.log.info("[Stage 4b] XGBoost history (mean over the CV folds; val = held-out fold):\n" + "\n".join(
        f"  round {i + 1:3d}  loss {history['loss'][i]:.4f}  accuracy {history['accuracy'][i]:.4f}  "
        f"val_loss {history['val_loss'][i]:.4f}  val_accuracy {history['val_accuracy'][i]:.4f}" for i in r))
    plot_curve({"XGBoost fusion": best["curve"]}, {"XGBoost fusion": k + 1}, out / "cv_curve.png",
               f"Stage 4b XGBoost fusion: out-of-fold Macro-F1 per tree count index (grid {grid[0]}, {grid[1]}, ...)")
    P = best["oof"][k]
    res = summarise(y, P, folds)

    # ---- 3. merger: second XGBoost (yes / no) on the same features and ensemble weights ----
    merger_trees, pm = None, np.full(len(y), float(ym.mean()))
    if 0 < ym.sum() < len(ym):
        oof_m, _ = xgb_cv(X, ym, folds, grid, 2, ctx.seed, params, fw)
        lls = [nll(ym, o) for o in oof_m]
        km = int(np.argmin(lls))
        merger_trees, pm = grid[km], oof_m[km][:, 1]
        res.update({k_: v for k_, v in morph_metrics(y, P, ym, pm).items() if k_.startswith("merger")})
        ctx.log.info(f"[Stage 4b] merger XGBoost: {int(ym.sum())} mergers of {len(ym)}; {merger_trees} trees "
                     f"(OOF log-loss {lls[km]:.4f}); OOF merger PR-AUC {res.get('merger_pr_auc')} "
                     f"(prevalence {res.get('merger_prevalence')})")

    # ---- 4. final models: all pooled galaxies ----
    model_xgb = XGBMember(len(MORPH_CLASSES), ctx.seed, params, fw).fit(X, y, n_trees)
    joblib.dump(model_xgb, out / "xgb_morphology.joblib")
    if merger_trees:
        model_merger = XGBMember(2, ctx.seed, params, fw).fit(X, ym, merger_trees)
        joblib.dump(model_merger, out / "xgb_merger.joblib")
    else:
        (out / "xgb_merger.joblib").unlink(missing_ok=True)

    # ---- 5. summary (trees, parameters, which block the model uses) ----
    s = model_xgb.summary(names)
    shares = _block_shares(s["gain_importance"], physical)
    txt = _summary_text(s, shares, w, physical)
    (out / "model_summary.txt").write_text(txt + "\n")
    ctx.log.info("[Stage 4b] model summary:\n" + txt)

    meta = {"physical_columns": physical, "n_features": int(X.shape[1]), "blocks": {b: [sl.start, sl.stop] for b, sl
                                                                                    in block_slices(len(physical)).items()},
            "ensemble_weights": {b: round(float(v), 4) for b, v in zip(BLOCKS, w)}, "trees": n_trees,
            "merger_trees": merger_trees, "merger_base_rate": round(float(ym.mean()), 4),
            "input_size": int(cfg.get("input_size", 224)), "xgboost": params, "classes": list(MORPH_CLASSES),
            "n_pool": int(len(pool)), "selection": f"5-fold CV, minimum out-of-fold {metric}",
            "gain_share_by_block": shares}
    save_json(meta, out / "fusion_meta.json")
    np.savez(out / "oof_fusion.npz", object_id=pool.object_id.to_numpy(), y=y, ym=ym, morph=P, merger=pm, fold=fold_id)
    save_json({"summary": res, "curve": best["curve"], "weight_search": rows}, out / "cv_metrics.json")
    save_json(dict(res, note="out-of-fold (cross-validated) metrics on the pooled train + dev galaxies"),
              out / "dev_metrics.json")
    ctx.log.info(f"[Stage 4b] out-of-fold result: accuracy {res['accuracy']} | Macro-F1 {res['macro_f1']} "
                 f"(95% CI {res['macro_f1_95ci']}) | balanced accuracy {res['balanced_acc']} | log-loss {res['nll']} "
                 f"| per fold {res['fold_macro_f1']} | importance by block {shares}")
    return {"oof_macro_f1": res["macro_f1"], "oof_accuracy": res["accuracy"], "trees": n_trees,
            "ensemble_weights": meta["ensemble_weights"]}


def _block_shares(gain, physical):
    tot = sum(gain.values()) or 1.0
    share = {b: 0.0 for b in BLOCKS}
    for name, v in gain.items():
        b = "resnet18" if name.startswith("resnet18_") else "efficientnet_b0" if name.startswith("efficientnet_b0_") \
            else "physical"
        share[b] += v / tot
    return {b: round(v, 4) for b, v in share.items()}


def _summary_text(s, shares, w, physical):
    line = "=" * 66
    rows = [line, 'Model: "Stage 4b - XGBoost (feature fusion)"', line,
            f"input            1,806 features = ResNet-18 512 + EfficientNet-B0 1,280 + physical {len(physical)}",
            f"ensemble weights w1 ResNet-18 {w[0]:.3f} | w2 EfficientNet-B0 {w[1]:.3f} | w3 physical {w[2]:.3f}",
            f"objective        {s['objective']} ({s['outputs']} classes)",
            f"boosting rounds  {s['boosting_rounds']} (x {s['outputs']} classes = {s['n_trees']} trees)",
            f"leaves           {s['leaves_total']:,} in total, {s['leaves_per_tree_mean']} per tree",
            "parameters       " + ", ".join(f"{k}={v}" for k, v in s["params"].items()), "-" * 66,
            "importance (total gain) by block: " + ", ".join(f"{b} {v:.1%}" for b, v in shares.items()),
            "top 15 features:"]
    tot = sum(s["gain_importance"].values()) or 1.0
    rows += [f"  {k:<22s} {v / tot:6.2%}" for k, v in list(s["gain_importance"].items())[:15]]
    rows.append(line)
    return "\n".join(rows)


def _plot_search(search, best_row, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    lab = [f"{r.w_resnet18:.2f} / {r.w_efficientnet_b0:.2f} / {r.w_physical:.2f}" for r in search.itertuples()]
    col = ["#eb6834" if (r.w_resnet18, r.w_efficientnet_b0, r.w_physical) ==
           (best_row.w_resnet18, best_row.w_efficientnet_b0, best_row.w_physical) else "#2a78d6"
           for r in search.itertuples()]
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
    axs[0].barh(lab, search.oof_log_loss, color=col)
    axs[0].set_xlim(search.oof_log_loss.min() * 0.97, search.oof_log_loss.max() * 1.01)
    axs[0].set_xlabel("out-of-fold log-loss (lower = better)")
    axs[1].barh(lab, search.oof_macro_f1, color=col)
    axs[1].set_xlabel("out-of-fold Macro-F1")
    for ax in axs:
        ax.set_ylabel("w1 ResNet-18 / w2 EfficientNet-B0 / w3 physical")
        ax.grid(alpha=0.25, lw=0.5)
    fig.suptitle("Stage 4b: ENSEMBLE weight search (5-fold CV; orange = chosen)")
    fig.tight_layout()
    fig.savefig(out / "weight_search.png", dpi=110, facecolor="white")
    plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage4c_calibration.py
"""Stage 4c - Probability calibration of the Stage 4b XGBoost models (no network trained here).

Model   : 3 parameters. Morphology: temperature scaling p = softmax(log P / T) - one T for all 5 classes
          (T < 1 sharpens an under-confident average, T > 1 softens; the predicted class, hence the accuracy,
          does not change). Merger: Platt scaling p = sigmoid(a z + b) on the merger logit z (the positive-class
          weight used in training shifts every logit; a temperature cannot undo a shift).
Fit     : minimum negative log-likelihood on the OUT-OF-FOLD predictions of the pooled train + dev galaxies
          (Stage 4b; each galaxy predicted by a model that never saw it).
Metrics : accuracy, log-loss and Expected Calibration Error (ECE) before / after; merger flag threshold by
          prevalence matching. A negative Platt slope (the head ranks mergers below non-mergers) -> base rate.
Output  : temperature.json, reliability_dev.png
"""
from __future__ import annotations

import numpy as np
from scipy.optimize import minimize_scalar

from ..context import Ctx, save_json
from ..nets import logit, nll, softmax_np
from ..utils import expected_calibration_error


def fit_platt(z, y):
    """p = sigmoid(a z + b): a re-scales AND shifts the logit. The merger head is
    trained with a positive-class weight, which shifts every logit up; a single
    temperature cannot undo a shift, Platt scaling can."""
    from scipy.optimize import minimize
    y = np.asarray(y, float)

    def nll(ab):
        p = 1 / (1 + np.exp(-(ab[0] * z + ab[1])))
        return -np.mean(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12))
    r = minimize(nll, x0=[1.0, 0.0], method="Nelder-Mead")
    return float(r.x[0]), float(r.x[1])


def fit_temperature(logits, y):
    y = np.asarray(y, int)
    nll = lambda lt: -np.mean(np.log(softmax_np(logits / np.exp(lt))[np.arange(len(y)), y] + 1e-12))
    return float(np.exp(minimize_scalar(nll, bounds=(-3, 3), method="bounded").x))


def run(ctx: Ctx):
    from ..modes import install_bundle, is_apply
    if is_apply(ctx):                       # no training: copy the frozen model bundle
        return install_bundle(ctx, "4c")
    out = ctx.stage_dir("4c")
    d = np.load(ctx.require("4b", "oof_fusion.npz"))           # out-of-fold predictions, pooled train + dev
    ens = {"morph": d["morph"], "merger": d["merger"]}
    y, ym = d["y"].astype(int), d["ym"].astype(int)
    source = "out-of-fold predictions, pooled train + dev"
    L = np.log(np.clip(ens["morph"], 1e-12, 1))
    T_m = fit_temperature(L, y)
    zm = logit(ens["merger"])
    merger_note = None
    if not ctx.path("4b", "xgb_merger.joblib").exists():        # no merger model: the base rate is the prediction
        a, b = 0.0, float(logit(np.array([max(ym.mean(), 1e-6)]))[0])
        merger_note = f"no merger model (too few mergers): p_merger = base rate {ym.mean():.3f}"
    elif 0 < ym.sum() < len(ym):
        a, b = fit_platt(zm, ym)
    else:
        a, b = 1.0, 0.0
    if merger_note is None and a <= 0 and 0 < ym.sum() < len(ym):
        # a negative slope would INVERT the merger score: the head has no usable ranking signal on these
        # objects (typically a handful of mergers). Report the base rate instead of an inverted score.
        merger_note = (f"Platt slope {a:.3f} <= 0: the merger head does not rank mergers above non-mergers on the "
                       f"calibration set ({int(ym.sum())} mergers) - p_merger set to the base rate {ym.mean():.3f}")
        a, b = 0.0, float(logit(np.array([ym.mean()]))[0])
        ctx.log.info(f"[Stage 4c] WARNING: {merger_note}")
    Pc = softmax_np(L / T_m)
    two = lambda p: np.column_stack([1 - p, p])
    pm_c = 1 / (1 + np.exp(-(a * zm + b)))
    rep = {"T_morph": round(T_m, 4), "merger_platt_a": round(a, 4), "merger_platt_b": round(b, 4),
           "accuracy": round(float((Pc.argmax(1) == y).mean()), 4),
           "log_loss_before": round(nll(y, ens["morph"]), 4), "log_loss_after": round(nll(y, Pc), 4),
           "dev_ece_morph_before": round(expected_calibration_error(ens["morph"], y), 4),
           "dev_ece_morph_after": round(expected_calibration_error(Pc, y), 4),
           "dev_ece_merger_before": round(expected_calibration_error(two(ens["merger"]), ym), 4),
           "dev_ece_merger_after": round(expected_calibration_error(two(pm_c), ym), 4), "n_dev": int(len(y)),
           "dev_merger_rate_true": round(float(ym.mean()), 4),
           "dev_merger_rate_predicted_before": round(float((ens["merger"] >= 0.5).mean()), 4),
           "dev_merger_rate_predicted_after": round(float((pm_c >= 0.5).mean()), 4),
           "fitted_on": source, "merger_note": merger_note}
    # decision threshold for the binary merger FLAG: flag the same fraction of dev galaxies as are
    # true mergers (prevalence matching). The probability p_merger itself stays calibrated.
    if merger_note:
        rep["merger_threshold"] = 1.0               # no usable signal: raise no merger flags
    elif 0 < ym.sum() < len(ym):
        thr = float(np.quantile(pm_c, 1 - ym.mean()))
        flag = pm_c >= thr
        rep["merger_threshold"] = round(thr, 4)
        rep["dev_merger_precision_at_threshold"] = round(float((flag & (ym == 1)).sum() / max(flag.sum(), 1)), 4)
        rep["dev_merger_recall_at_threshold"] = round(float((flag & (ym == 1)).sum() / ym.sum()), 4)
    else:
        rep["merger_threshold"] = 0.5
    save_json(rep, out / "temperature.json")
    _plot(ens["morph"], Pc, y, out)
    ctx.log.info(f"[Stage 4c] calibration model: T = {rep['T_morph']} (morphology), Platt a = {rep['merger_platt_a']}, "
                 f"b = {rep['merger_platt_b']} (merger) - fitted on {len(y)} out-of-fold galaxies")
    ctx.log.info(f"[Stage 4c] accuracy {rep['accuracy']} (unchanged by T) | log-loss {rep['log_loss_before']} -> "
                 f"{rep['log_loss_after']} | ECE morphology {rep['dev_ece_morph_before']} -> {rep['dev_ece_morph_after']}"
                 f" | ECE merger {rep['dev_ece_merger_before']} -> {rep['dev_ece_merger_after']} | merger threshold "
                 f"{rep['merger_threshold']}")
    return rep


def _plot(P0, P1, y, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(4.5, 4))
    for P, lab in ((P0, "before"), (P1, "after")):
        conf, acc = P.max(1), (P.argmax(1) == y)
        idx = np.digitize(conf, np.linspace(0, 1, 11)) - 1
        xs = [conf[idx == b].mean() for b in range(10) if (idx == b).any()]
        ys = [acc[idx == b].mean() for b in range(10) if (idx == b).any()]
        ax.plot(xs, ys, "o-", label=f"{lab} T-scaling")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("confidence")
    ax.set_ylabel("accuracy")
    ax.set_title("Stage 4c: reliability (calibration set)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out / "reliability_dev.png", dpi=110)
    plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage4d_ablation.py
"""Stage 4d - Controlled ablation: what does each part of the fusion add? (Objective 3)

Rows    : every row is XGBoost with the Stage 4b settings, cross-validated on the SAME 5 folds (number of trees
          chosen from its own out-of-fold curve), on a subset of the 1,806 features:
            ResNet-18 only (512) | EfficientNet-B0 only (1,280) | physical only (14) | CNN features only (1,792)
            | full fusion (1,806, Stage 4b) | full fusion + calibration (the final model, Stage 4c)
            | full fusion with 128 px input (no resize; supervised.ablation_input_size)
            | the previous three-model ensemble (supervised.compare_previous_ensemble, same galaxies).
          Star segregation vs Gaia: the Stage 2c baseline against the learned clustering (Stage 2d).
Tier    : full fusion minus the better single source (CNN features only vs physical only):
          >= 0 Minimum, >= 0.03 Target, >= 0.05 Stretch.
Output  : ablation_table.csv, ablation.png, summary.json
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..context import Ctx, load_json, save_json
from ..utils import expected_calibration_error


def run(ctx: Ctx):
    from ..context import StageSkipped
    from ..modes import is_apply
    if is_apply(ctx):
        raise StageSkipped("apply mode: the models come from a bundle - the ablation belongs to the training run")
    from ..features import supervised_splits
    from ..inference import block_feature_weights, block_slices, extract_cnn_features, fusion_matrix, xgb_cv
    from ..nets import MORPH_CLASSES, softmax_np
    from ..supervised_cv import choose_epoch, curve_metrics, pooled, summarise
    from .stage4b_xgboost_fusion import tree_grid
    cfg = ctx.cfg["supervised"]
    out = ctx.stage_dir("4d")
    meta = load_json(ctx.require("4b", "fusion_meta.json"))
    physical = meta["physical_columns"]
    w = np.array([meta["ensemble_weights"][b] for b in ("resnet18", "efficientnet_b0", "physical")])
    params, grid = meta.get("xgboost") or {}, tree_grid(cfg)

    # the same galaxies and folds as Stage 4b
    pool = pooled(supervised_splits(ctx))
    fdf = pd.read_csv(ctx.require("4b", "cv_folds.csv")).set_index("object_id")
    fold_id = fdf.fold.reindex(pool.object_id).to_numpy()
    if np.isnan(fold_id.astype(float)).any():
        raise RuntimeError("Stage 4b cv_folds.csv does not match the current labelled galaxies - rerun Stage 4b.")
    folds = [(np.where(fold_id != f)[0], np.where(fold_id == f)[0]) for f in sorted(set(fold_id))]
    y = pool.gz_label_id.astype(int).to_numpy()
    ym = pool.gz_merger_flag.fillna(0).astype(int).to_numpy()
    X = fusion_matrix(ctx, pool, physical)
    sl = block_slices(len(physical))
    fw_all = block_feature_weights(w, len(physical))
    rows = []

    def add(name, rep, P, pm=None, n_trees=None, n_features=None):
        s = summarise(y, P, folds, ym, pm)
        rows.append({"task": "morphology (out-of-fold, train + dev)", "configuration": name, "representation": rep,
                     "n_features": n_features, "trees": n_trees,
                     "accuracy": s["accuracy"], "macro_f1": s["macro_f1"], "macro_f1_ci_low": s["macro_f1_95ci"][0],
                     "macro_f1_ci_high": s["macro_f1_95ci"][1], "fold_sd": s["fold_sd"],
                     "balanced_acc": s["balanced_acc"], "log_loss": s["nll"],
                     "merger_pr_auc": s.get("merger_pr_auc", np.nan), "ece": round(expected_calibration_error(P, y), 4)})
        ctx.log.info(f"  [ablation] {name:52s} accuracy {s['accuracy']:.3f}  Macro-F1 {s['macro_f1']:.3f}  "
                     f"log-loss {s['nll']:.3f}")

    def cv_subset(cols, XX=None):
        XX = X if XX is None else XX
        fw = fw_all[cols]
        oof, _ = xgb_cv(XX[:, cols], y, folds, grid, len(MORPH_CLASSES), ctx.seed, params, fw / fw.sum())
        k = choose_epoch(curve_metrics(y, oof, folds), 1, str(cfg.get("selection_metric", "log_loss")))[0] - 1
        return oof[k], grid[k]

    r = lambda b: np.arange(sl[b].start, sl[b].stop)
    for name, rep, cols in (("ResNet-18 features only", "resnet18", r("resnet18")),
                            ("EfficientNet-B0 features only", "efficientnet_b0", r("efficientnet_b0")),
                            ("Physical features only (classical)", "physical", r("physical")),
                            ("CNN features only (ResNet-18 + EfficientNet-B0)", "resnet18+efficientnet_b0",
                             np.r_[r("resnet18"), r("efficientnet_b0")])):
        P, n = cv_subset(cols)
        add(name, rep, P, n_trees=n, n_features=int(len(cols)))
    d = np.load(ctx.require("4b", "oof_fusion.npz"))
    add("Full fusion (Stage 4b)", "resnet18+efficientnet_b0+physical", d["morph"], d["merger"], meta["trees"],
        int(X.shape[1]))
    T = load_json(ctx.require("4c", "temperature.json"))
    add("Full fusion + calibration (final model)", "resnet18+efficientnet_b0+physical",
        softmax_np(np.log(np.clip(d["morph"], 1e-12, 1)) / T["T_morph"]), d["merger"], meta["trees"], int(X.shape[1]))

    small = int(cfg.get("ablation_input_size") or 0)
    if small and small != int(meta.get("input_size", 224)):
        ctx.log.info(f"[Stage 4d] CNN features at {small} px for the {len(pool)} labelled galaxies ...")
        X2 = X.copy()
        X2[:, :1792] = _features_at(ctx, pool.cutout_index.to_numpy(), small, extract_cnn_features)
        P, n = cv_subset(np.arange(X.shape[1]), X2)
        add(f"Full fusion, {small} px input (no resize)", "resnet18+efficientnet_b0+physical", P, n_trees=n,
            n_features=int(X.shape[1]))

    prev = cfg.get("compare_previous_ensemble")
    pf = ctx.root / str(prev) / "stage4b_ensemble" / "oof_members.npz" if prev else None
    if pf is not None and pf.exists():
        o = np.load(pf)
        pos = {int(v): i for i, v in enumerate(o["object_id"])}
        rows_prev = np.array([pos.get(int(v), -1) for v in pool.object_id])
        if (rows_prev >= 0).all():
            add("Previous three-model ensemble (dual-branch + EfficientNet + XGBoost)", "ensemble",
                o["ensemble"][rows_prev], o["merger"][rows_prev])
            rows[-1]["note"] = f"from {prev}: weighted vote of three classifiers, out-of-fold"
        else:
            ctx.log.info(f"[Stage 4d] {pf} covers {int((rows_prev >= 0).sum())} of {len(pool)} galaxies - "
                         "different labelled set, row skipped")
    elif prev:
        ctx.log.info(f"[Stage 4d] supervised.compare_previous_ensemble: {pf} not found - row skipped")

    audit = load_json(ctx.require("2d", "semantic_audit.json"))
    for name, key in (("Baseline segregator (Stage 2c, no Gaia leg)", "baseline_star_vs_gaia"),
                      ("CAE + GMM/HDBSCAN (Stage 2d)", "learned_star_vs_gaia")):
        m = audit[key]
        rows.append({"task": "star segregation vs Gaia", "configuration": name, "star_precision": m["precision"],
                     "star_recall": m["recall"], "star_f1": m["f1"]})
    tab = pd.DataFrame(rows)
    tab.to_csv(out / "ablation_table.csv", index=False)
    f = tab.set_index("configuration").macro_f1
    delta = float(f["Full fusion (Stage 4b)"] - max(f["CNN features only (ResNet-18 + EfficientNet-B0)"],
                                                     f["Physical features only (classical)"]))
    tier = "Stretch" if delta >= 0.05 else "Target" if delta >= 0.03 else "Minimum" if delta >= 0 else "Below minimum"
    save_json({"fusion_delta_macro_f1": round(delta, 4), "tier": tier,
               "scored_on": "out-of-fold predictions, pooled train + dev galaxies, Stage 4b folds"}, out / "summary.json")
    _plot(tab[tab.task.str.startswith("morphology")], out)
    ctx.log.info("[Stage 4d]\n" + tab.drop(columns=["task"]).to_string(index=False))
    ctx.log.info(f"[Stage 4d] full fusion minus the better single source = {delta:+.4f} -> {tier}")
    return {"fusion_delta": round(delta, 4), "tier": tier}


def _features_at(ctx, cutout_index, size, extract):
    """CNN features with another input size (the config is changed for this call only)."""
    old = ctx.cfg["supervised"].get("input_size", 224)
    ctx.cfg["supervised"]["input_size"] = size
    try:
        return extract(ctx, cutout_index)
    finally:
        ctx.cfg["supervised"]["input_size"] = old


def _plot(t, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    t = t.sort_values("macro_f1")
    col = ["#eb6834" if "final model" in c else "#2E5496" if c.startswith("Full fusion") else
           "#9AA3B5" if "only" in c else "#D3D9E5" for c in t.configuration]
    fig, ax = plt.subplots(figsize=(9, 0.42 * len(t) + 1.2))
    ax.barh(t.configuration, t.macro_f1, color=col)
    for i, (v, a) in enumerate(zip(t.macro_f1, t.accuracy)):
        ax.text(v + 0.01, i, f"Macro-F1 {v:.3f} | acc {a:.3f}", va="center", fontsize=8)
    ax.set_xlim(0, 1.0)
    ax.set_xlabel("out-of-fold Macro-F1 (pooled train + dev, same 5 folds)")
    ax.set_title("Stage 4d ablation - orange: final model; blue: fusion; grey: single feature sources", fontsize=9.5)
    fig.tight_layout()
    fig.savefig(out / "ablation.png", dpi=110)
    plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage5a_empirical_psf.py
"""Stage 5a - Empirical PSF construction (photutils EPSFBuilder).

Purpose : a localised, high-quality PSF built from the pipeline's own stars.
Method  : the stars routed by Stage 3b are cut to 31x31 stamps from the image
          that is actually measured (the sum of the PSF-matched bands) and
          stacked with photutils.psf.EPSFBuilder (oversampling 2, sub-pixel
          recentring, sigma-clipped). Pointings with too few stars pool the
          stars of all pointings with the same instrument; if that is still too
          few, the Pass-1 placeholder PSF is kept (and the report says so).
Output  : epsf_<pointing>.npy (+ .fits), epsf_report.json, epsf.png
"""
from __future__ import annotations

import warnings

import numpy as np
import pandas as pd
from astropy.io import fits

from ..context import Ctx, load_json, save_json
from ..features import load_cutouts
from ..morphology import morph_image
from ..utils import as_bool, fwhm_fit
from .stage1d_psf_match import core_centroid, crop_center, stack_stars, star_masks

try:
    from photutils.psf import EPSFBuilder, EPSFStar, EPSFStars
    _HAVE_EPSF = True
except Exception:
    _HAVE_EPSF = False


def build_epsf(stamps, masks=None, oversampling=2, maxiters=5):
    """masks: True = neighbour / cosmic-ray pixel (weight 0). Stars are centred on their core centroid."""
    masks = masks if masks is not None else [np.zeros_like(s, bool) for s in stamps]
    if _HAVE_EPSF:
        try:
            stars = EPSFStars([EPSFStar(np.where(m, 0.0, s).astype(float), weights=(~m).astype(float),
                                        cutout_center=core_centroid(np.where(m, 0.0, s))[::-1])
                               for s, m in zip(stamps, masks)])
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                res = EPSFBuilder(oversampling=oversampling, maxiters=maxiters, progress_bar=False)(stars)
            epsf = res.epsf if hasattr(res, "epsf") else res[0]
            n = stamps[0].shape[0]
            yy, xx = np.mgrid[:n, :n]
            psf = np.clip(epsf.evaluate(xx, yy, flux=1.0, x_0=(n - 1) / 2, y_0=(n - 1) / 2), 0, None)
            return psf / psf.sum(), "photutils.EPSFBuilder"
        except Exception as e:
            print(f"[Stage 5a] EPSFBuilder failed ({e}); median stack used")
    return stack_stars(stamps, masks=masks), "median stack"


def run(ctx: Ctx, size=31, min_stars=5):
    out = ctx.stage_dir("5a")
    pts = load_json(ctx.require("1a", "pointings.json"))["pointings"]
    routing = pd.read_csv(ctx.require("3b", "star_routing.csv"))
    src = pd.read_csv(ctx.require("1c", "sources.csv"))[["object_id", "cutout_index", "pointing"]]
    routed = routing[as_bool(routing.used_in_epsf)].merge(src, on=["object_id", "pointing"])
    cut = load_cutouts(ctx)
    segs = np.load(ctx.require("1c", "segstamps.npy"), mmap_mode="r")
    match = load_json(ctx.require("1d", "psf_match.json"))
    stamps = {r.object_id: crop_center(morph_image(cut[r.cutout_index]), size) for r in routed.itertuples()}
    report = {}
    for pid, p in pts.items():
        own = routed[routed.pointing == pid].object_id.tolist()
        use, how = own, "own stars"
        if len(own) < min_stars:
            same = [q for q, v in pts.items() if v["instrument"] == p["instrument"]]
            use = routed[routed.pointing.isin(same)].object_id.tolist()
            how = f"pooled over {len(same)} pointing(s) of {p['instrument']}"
        if len(use) >= min_stars:
            idx = routed.set_index("object_id").loc[use, "cutout_index"].tolist()
            psf, method = build_epsf([stamps[o] for o in use], star_masks(segs, idx, size))
        else:
            ph = np.load(ctx.require("1d", f"placeholder_psf_{pid}.npy"))
            psf = np.zeros((size, size))
            h = (size - ph.shape[0]) // 2
            psf[h:h + ph.shape[0], h:h + ph.shape[1]] = ph
            method, how = "Pass-1 placeholder kept", f"only {len(use)} star(s) available"
        np.save(out / f"epsf_{pid}.npy", psf.astype(np.float32))
        fits.writeto(out / f"epsf_{pid}.fits", psf.astype(np.float32), overwrite=True)
        report[pid] = {"n_stars": len(use), "source": how, "method": method, "fwhm_px": round(fwhm_fit(psf), 3)}
        # the ePSF is built on PSF-MATCHED cutouts, so it cannot be sharper than the matched PSF, and a much
        # broader one means galaxies or cosmic rays got into the stack
        m = match.get(pid, {})
        ref = m.get("fwhm_px", {}).get(m.get("target_band")) if m.get("target_psf_measured") else None
        if ref:
            ratio = report[pid]["fwhm_px"] / ref
            report[pid]["fwhm_over_matched"] = round(ratio, 3)
            if ratio > 1.5 or ratio < 0.85:
                ctx.log.info(f"[Stage 5a] {pid}: WARNING - ePSF FWHM {report[pid]['fwhm_px']} px is {ratio:.2f} x the "
                             f"Stage-1d matched FWHM {ref} px (expected ~1): check the routed stars "
                             "(outputs/stage3b_star_routing/star_routing.csv).")
        ctx.log.info(f"[Stage 5a] {pid}: {method} from {len(use)} stars ({how}); FWHM {report[pid]['fwhm_px']} px")
    save_json(report, out / "epsf_report.json")
    _plot(report, out)
    return {"pointings": len(report)}


def _plot(report, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    pids = list(report)
    fig, axes = plt.subplots(1, len(pids), figsize=(2.6 * len(pids), 2.8))
    for ax, pid in zip(np.atleast_1d(axes), pids):
        ax.imshow(np.log10(np.load(out / f"epsf_{pid}.npy") + 1e-5), origin="lower", cmap="magma")
        ax.set_title(f"{pid}: {report[pid]['n_stars']} stars\nFWHM {report[pid]['fwhm_px']} px", fontsize=8)
        ax.axis("off")
    fig.suptitle("Stage 5a: empirical PSFs (log scale)")
    fig.tight_layout()
    fig.savefig(out / "epsf.png", dpi=100)
    plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage5b_pass2_refinement.py
"""Stage 5b - Deterministic Pass-2 refinement.

Purpose : physics-calibrated morphology for the galaxy track.
Method  : the empirical PSF from Stage 5a is fed back into the SAME statmorph
          measurement (same image, same segmentation, same gain) and the galaxy
          track is re-measured exactly once:
            psf_fit    (default) the Sersic model is convolved with the ePSF, so
                       n and r_eff are PSF-corrected; non-parametric indices are
                       re-measured on the same image;
            deconvolve Richardson-Lucy deconvolution (10 fixed iterations)
                       inside the dilated footprint, then statmorph.
          Split tags and cluster IDs are hashed before and after: Pass 2 is not
          allowed to change them.
Output  : features_pass2.csv (psf_pass = 2, galaxy rows only - load_objects()
          lets these overwrite the Pass-1 values), morph_delta_pass.csv
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..context import Ctx, load_objects, save_json
from ..morphology import FEATURES, run_parallel

FROZEN = ["object_id", "split_tag", "cluster_semantic", "cluster_id_gmm", "is_transitional", "anomaly_flag"]


def _hash(df):
    return int(pd.util.hash_pandas_object(df[FROZEN].astype(str), index=False).sum())


def run(ctx: Ctx):
    out = ctx.stage_dir("5b")
    mode = ctx.cfg["morphology"]["pass2_mode"]
    df = load_objects(ctx, ["sources", "pass1", "clusters", "segregation"])
    before = _hash(df)
    gal = df[(df.cluster_semantic == "Galaxy") & (df.anomaly_flag == 0) & (df.snr >= ctx.cfg["morphology"]["min_snr"])]
    psf = {p: np.load(ctx.require("5a", f"epsf_{p}.npy")) for p in gal.pointing.unique()}
    ctx.log.info(f"[Stage 5b] Pass-2 ({mode}) on {len(gal)} galaxies with {ctx.n_jobs} worker(s) ...")
    res = run_parallel(ctx, ctx.require("1d", "cutouts.npy"), ctx.require("1c", "segstamps.npy"),
                       gal.cutout_index.tolist(), gal.exptime.tolist(), [psf[p] for p in gal.pointing], mode=mode)
    p2 = pd.DataFrame(res)[FEATURES]
    p2.insert(0, "object_id", gal.object_id.to_numpy())
    p2["psf_pass"] = 2
    p2["pass2_mode"] = mode
    p2.to_csv(out / "features_pass2.csv", index=False)
    assert _hash(df) == before, "Pass 2 changed a frozen column (split / cluster)!"

    p1 = gal.set_index("object_id")[FEATURES]
    d = (p2.set_index("object_id")[FEATURES] - p1).add_suffix("_delta")
    d.reset_index().to_csv(out / "morph_delta_pass.csv", index=False)
    med = d[[c for c in d.columns if c.split("_delta")[0] in ("gini", "m20", "concentration", "asymmetry",
                                                               "sersic_n", "sersic_rhalf", "r_half")]].median()
    save_json({"mode": mode, "galaxies": len(gal), "median_delta": med.round(4).to_dict(),
               "frozen_columns_unchanged": True}, out / "summary.json")
    ctx.log.info("[Stage 5b] median Pass-2 minus Pass-1:\n" + med.round(4).to_string())
    return {"galaxies": len(gal)}

In [ ]:
%%writefile src/p2p/stages/stage6a_test_eval.py
"""Stage 6a - In-domain test (run once).

Purpose : the final, honest in-domain score.
Method  : the frozen, calibrated model (Stage 4b fusion XGBoost + Stage 4c calibration) is run on the untouched GZH test
          split: Macro-F1, balanced accuracy, per-class precision/recall,
          one-vs-rest PR-AUC, merger PR-AUC and ECE, each with a 95% bootstrap
          interval (small test sets need error bars), scored against the tiered
          success criteria. Every run is appended to a ledger, so the report can
          state how many times the test split was looked at.
Output  : test_metrics.json, test_predictions.csv, confusion_matrix_test.png, test_split_ledger.json
"""
from __future__ import annotations

import datetime

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix, f1_score

from ..context import Ctx, load_json, save_json
from ..features import supervised_splits
from ..inference import predict_galaxies
from ..nets import MORPH_CLASSES, N_CLASSES, morph_metrics
from ..utils import bootstrap_ci, expected_calibration_error


def tier(v, minimum, target, stretch, lower_better=False):
    if v is None or not np.isfinite(v):
        return "n/a"
    if lower_better:
        v, minimum, target, stretch = -v, -minimum, -target, -stretch
    return "Stretch" if v >= stretch else "Target" if v >= target else "Minimum" if v > minimum else "Below minimum"


def _ci(c):
    return f"{c[0]:.3f}-{c[1]:.3f}" if c and np.isfinite(c[0]) else "n/a"


def run(ctx: Ctx):
    from ..sealing import require_unsealed
    require_unsealed(ctx, "6a")
    out = ctx.stage_dir("6a")
    ledger_p = out / "test_split_ledger.json"
    ledger = load_json(ledger_p) if ledger_p.exists() else {"evaluations": []}
    if ledger["evaluations"]:
        ctx.log.info(f"[Stage 6a] WARNING: the test split was already evaluated {len(ledger['evaluations'])} time(s). "
                     "Report the FIRST result and do not tune on these numbers.")
    splits = supervised_splits(ctx)
    tdf = splits["test"]
    res = predict_galaxies(ctx, tdf)
    y, ym, P = tdf.gz_label_id.astype(int).to_numpy(), tdf.gz_merger_flag.fillna(0).astype(int).to_numpy(), res["morph"]
    pred = P.argmax(1)
    from ..modes import card, is_apply
    if is_apply(ctx) and card(ctx).get("train_class_counts"):
        cc = card(ctx)["train_class_counts"]                      # the bundle's own training set
        maj = int(np.argmax([cc.get(c, 0) for c in MORPH_CLASSES]))
    else:
        maj = np.bincount(splits["train"].gz_label_id, minlength=N_CLASSES).argmax()
    maj_f1 = morph_metrics(y, np.eye(N_CLASSES)[np.full(len(y), maj)])["macro_f1"]
    m = morph_metrics(y, P, ym, res["merger"])
    f1 = lambda yy, PP: f1_score(yy, PP.argmax(1), average="macro",
                                 labels=sorted(set(yy.tolist()) | set(PP.argmax(1).tolist())), zero_division=0)
    m["macro_f1_ci95"] = bootstrap_ci(f1, y, P, seed=ctx.seed)
    present = np.eye(N_CLASSES)[y].sum(0) > 0
    m["morph_pr_auc_ovr"] = round(float(average_precision_score(np.eye(N_CLASSES)[y][:, present], P[:, present],
                                                                 average="macro")), 4)
    m["ece"] = round(expected_calibration_error(P, y), 4)
    m["ece_ci95"] = bootstrap_ci(lambda yy, PP: expected_calibration_error(PP, yy), y, P, seed=ctx.seed)
    if m.get("merger_pr_auc") is not None and 0 < ym.sum() < len(ym):
        ap = lambda a, p: average_precision_score(a, p) if 0 < a.sum() < len(a) else float("nan")
        lo, hi = bootstrap_ci(ap, ym, np.asarray(res["merger"], float), seed=ctx.seed)
        m["merger_pr_auc_ci95"] = (lo, hi)
        m["n_merger_test"] = int(ym.sum())
    m["majority_class_macro_f1"] = maj_f1
    m["per_class"] = classification_report(y, pred, labels=list(range(N_CLASSES)), target_names=list(MORPH_CLASSES),
                                           output_dict=True, zero_division=0)
    m["tiers"] = {"Morphology Macro-F1": tier(m["macro_f1"], maj_f1, 0.65, 0.75),
                  "Merger PR-AUC": tier(m.get("merger_pr_auc"), m.get("merger_prevalence", 0), 0.50, 0.65),
                  "Calibration ECE": tier(m["ece"], 0.10, 0.05, 0.03, lower_better=True)}
    save_json(m, out / "test_metrics.json")
    ledger["evaluations"].append({"utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
                                  "macro_f1": m["macro_f1"], "n_test": int(len(y))})
    save_json(ledger, ledger_p)
    pr = pd.DataFrame({"object_id": tdf.object_id.to_numpy(), "y_true": y, "y_pred": pred, "p_merger": res["merger"]})
    for k, c in enumerate(MORPH_CLASSES):
        pr[f"p_{c}"] = P[:, k]
    pr.to_csv(out / "test_predictions.csv", index=False)
    _confusion(y, pred, out)
    ctx.log.info(f"[Stage 6a] test n={len(y)}: accuracy {m['accuracy']} | Macro-F1 {m['macro_f1']} (95% CI {m['macro_f1_ci95'][0]:.3f}-"
                 f"{m['macro_f1_ci95'][1]:.3f}) | balanced acc {m['balanced_acc']} | merger PR-AUC "
                 f"{m.get('merger_pr_auc')} (95% CI {_ci(m.get('merger_pr_auc_ci95'))}, "
                 f"{m.get('n_merger_test', 0)} mergers) | ECE {m['ece']} (95% CI {_ci(m['ece_ci95'])}) | "
                 f"majority-class F1 {maj_f1}")
    ctx.log.info(f"[Stage 6a] tiers: {m['tiers']}")
    return {"macro_f1": m["macro_f1"], "tiers": m["tiers"]}


def _confusion(y, pred, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    cm = confusion_matrix(y, pred, labels=list(range(N_CLASSES)))
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(N_CLASSES))
    ax.set_xticklabels(MORPH_CLASSES, rotation=35, ha="right")
    ax.set_yticks(range(N_CLASSES))
    ax.set_yticklabels(MORPH_CLASSES)
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j, i, f"{cmn[i, j]:.2f}\n({cm[i, j]})", ha="center", va="center", fontsize=7,
                    color="white" if cmn[i, j] > 0.5 else "black")
    ax.set_xlabel("predicted")
    ax.set_ylabel("GZH label")
    ax.set_title("Stage 6a: in-domain test (row-normalised, counts)")
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(out / "confusion_matrix_test.png", dpi=110)
    plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage6b_uncertainty.py
"""Stage 6b - Uncertainty audit.

Purpose : does the model hesitate where humans disagree?
Method  : normalised predictive entropy of each test galaxy is compared with
          human disagreement (1 - the weakest GZH vote fraction along the path
          to its label): Spearman correlation, plus a one-sided Mann-Whitney U
          test that the Ambiguous Boundary set (agreement < 0.5) has higher
          entropy than confidently labelled galaxies.
Output  : uncertainty_audit.json, uncertainty_audit.png
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, spearmanr

from ..context import Ctx, save_json
from ..features import supervised_splits
from ..nets import MORPH_CLASSES
from ..utils import as_bool, predictive_entropy


def run(ctx: Ctx):
    from ..sealing import require_unsealed
    require_unsealed(ctx, "6b")
    out = ctx.stage_dir("6b")
    if not ctx.path("6a", "test_predictions.csv").exists():
        from ..context import StageSkipped
        raise StageSkipped("Stage 6a has no test predictions (no labelled galaxies to evaluate)")
    pred = pd.read_csv(ctx.require("6a", "test_predictions.csv"))
    test = supervised_splits(ctx)["test"][["object_id", "human_disagreement", "in_boundary_set"]]
    d = pred.merge(test, on="object_id")
    P = d[[f"p_{c}" for c in MORPH_CLASSES]].to_numpy()
    e = predictive_entropy(P)
    h = d.human_disagreement.to_numpy(float)
    b = as_bool(d.in_boundary_set).to_numpy()
    rho, p = spearmanr(e, h) if len(d) > 2 else (np.nan, np.nan)
    acc = (d.y_true == d.y_pred).to_numpy()
    res = {"n_test": int(len(d)), "spearman_entropy_vs_disagreement": round(float(rho), 4),
           "spearman_p": float(p), "n_boundary": int(b.sum()), "n_confident": int((~b).sum()),
           "mean_entropy_boundary": float(e[b].mean()) if b.any() else None,
           "mean_entropy_confident": float(e[~b].mean()) if (~b).any() else None,
           "accuracy_boundary": float(acc[b].mean()) if b.any() else None,
           "accuracy_confident": float(acc[~b].mean()) if (~b).any() else None}
    if b.sum() >= 3 and (~b).sum() >= 3:
        res["mannwhitney_p_boundary_gt_confident"] = float(mannwhitneyu(e[b], e[~b], alternative="greater").pvalue)
    else:
        res["note"] = "fewer than 3 boundary-set galaxies in the test split - the U-test is not run"
    save_json(res, out / "uncertainty_audit.json")
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(4.8, 4))
    ax.scatter(h[~b], e[~b], s=12, alpha=0.6, label="confident (humans agree)")
    ax.scatter(h[b], e[b], s=18, marker="x", label="Ambiguous Boundary set")
    ax.set_xlabel("human disagreement (1 - GZH path agreement)")
    ax.set_ylabel("model predictive entropy")
    ax.set_title(f"Stage 6b: Spearman rho = {rho:.2f}")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(out / "uncertainty_audit.png", dpi=110)
    plt.close(fig)
    ctx.log.info(f"[Stage 6b] {res}")
    return res

In [ ]:
%%writefile src/p2p/stages/stage6c_xai.py
"""Stage 6c - Explainable AI: which evidence drives each prediction (exact TreeSHAP of the fusion XGBoost).

Method  : XGBoost's built-in pred_contribs gives the exact SHAP value of every one of the 1,806 inputs for every
          test galaxy and class. They are summarised (mean |SHAP| over the test galaxies)
            * per ENSEMBLE block - how much of each class decision rests on ResNet-18 features, EfficientNet-B0
              features and the physical features;
            * per physical feature - which measured quantity (Gini, colour, ellipticity, ...) matters per class;
          and the most confident correctly classified test galaxy of each class is shown as a colour cutout.
Output  : shap_blocks.csv, shap_physical.csv, shap_summary.png, examples.png, summary.json
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..context import Ctx, StageSkipped, save_json
from ..features import load_cutouts, supervised_splits
from ..inference import BLOCKS, block_slices, load_fusion, predict_galaxies
from ..nets import MORPH_CLASSES, N_CLASSES


def run(ctx: Ctx):
    from ..sealing import require_unsealed
    require_unsealed(ctx, "6c")
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    out = ctx.stage_dir("6c")
    if not ctx.path("6a", "test_predictions.csv").exists():
        raise StageSkipped("Stage 6a has no test predictions (no labelled galaxies to evaluate)")
    pred = pd.read_csv(ctx.require("6a", "test_predictions.csv"))
    test = supervised_splits(ctx)["test"].set_index("object_id").loc[pred.object_id].reset_index()
    models = load_fusion(ctx)
    physical = models["meta"]["physical_columns"]
    X = predict_galaxies(ctx, test, models)["X"]
    c = models["morph"].contribs(X)                      # (n, classes present in training, 1806)
    cls = [MORPH_CLASSES[k] for k in models["morph"].classes_]
    A = np.abs(c).mean(0)                                # (classes, features)
    sl = block_slices(len(physical))
    blocks = pd.DataFrame({b: A[:, sl[b]].sum(1) for b in BLOCKS}, index=cls)
    shares = blocks.div(blocks.sum(1), axis=0)
    phys = pd.DataFrame(A[:, sl["physical"]], index=cls, columns=physical)
    shares.to_csv(out / "shap_blocks.csv")
    phys.to_csv(out / "shap_physical.csv")

    fig, axs = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1, 1.5]})
    left = np.zeros(len(cls))
    for b, col in zip(BLOCKS, ("#2a78d6", "#1baf7a", "#eb6834")):
        axs[0].barh(cls, shares[b], left=left, color=col, label=b)
        left += shares[b].to_numpy()
    axs[0].set_xlabel("share of total |SHAP| per class")
    axs[0].set_title("Which ENSEMBLE block drives each class")
    axs[0].legend(fontsize=8, loc="lower right")
    phys.T.plot.barh(ax=axs[1], width=0.8)
    axs[1].set_xlabel("mean |SHAP| (log-odds)")
    axs[1].set_title("Physical features per class")
    fig.tight_layout()
    fig.savefig(out / "shap_summary.png", dpi=110)
    plt.close(fig)

    # most confident correct test galaxy per class, as a colour cutout (R, G, B = reddest .. bluest band)
    P = pred[[f"p_{c_}" for c_ in MORPH_CLASSES]].to_numpy()
    picks = [ok[np.argmax(P[ok, k])] for k in range(N_CLASSES)
             if (ok := np.where((pred.y_pred == k) & (pred.y_true == k))[0]).size]
    if picks:
        cut = load_cutouts(ctx)
        fig, axes = plt.subplots(1, len(picks), figsize=(2.6 * len(picks), 2.9), squeeze=False)
        for ax, i in zip(axes[0], picks):
            img = np.asarray(cut[int(test.cutout_index[i])], np.float32)[::-1].transpose(1, 2, 0)
            lo, hi = np.percentile(img, [1, 99.5])
            v = np.clip((img - lo) / max(hi - lo, 1e-8), 0, 1)
            ax.imshow(np.arcsinh(v / 0.1) / np.arcsinh(10), origin="lower")
            k = int(pred.y_pred[i])
            ax.set_title(f"{MORPH_CLASSES[k]}\np = {P[i, k]:.2f}", fontsize=9)
            ax.axis("off")
        fig.suptitle("Stage 6c: most confident correct test galaxy per class")
        fig.tight_layout()
        fig.savefig(out / "examples.png", dpi=100)
        plt.close(fig)
    save_json({"method": "XGBoost TreeSHAP (pred_contribs), fusion model", "n_test": int(len(test)),
               "block_share_by_class": shares.round(4).to_dict(orient="index"),
               "example_classes": [MORPH_CLASSES[int(pred.y_pred[i])] for i in picks]}, out / "summary.json")
    ctx.log.info("[Stage 6c] share of |SHAP| by ENSEMBLE block:\n" + shares.round(3).to_string()
                 + "\n[Stage 6c] physical features (mean |SHAP|):\n" + phys.round(3).to_string())
    return {"classes": len(cls)}

In [ ]:
%%writefile src/p2p/stages/stage7a_taxonomy.py
"""Stage 7a - Taxonomy alignment and the human-agreement ceiling.

Purpose : separate label-scheme differences from model failure.
Method  : galaxies in the EXTERNAL (CANDELS) pointings - or, with a single field,
          in the held-out TEST split (partition.phase7_region: auto) - are matched to the
          Kartaltepe et al. (2015) expert catalogue. Both schemes are mapped to
          a shared coarse taxonomy (Spheroid, Disk, Irregular, Unclassifiable;
          face-on and edge-on discs both become Disk; expert point sources are
          dropped). On galaxies labelled by BOTH GZH and the experts, the
          alignment matrix and the Macro-F1 of GZH vs expert labels give the
          Taxonomy Ceiling - the best any GZH-trained model could score.
          Leakage guards: no evaluated object may lie within 1" of an object
          on the training side (train/dev, plus test when external fields exist).
Output  : candels_objects.csv, taxonomy_alignment_matrix.csv, taxonomy_ceiling.json
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score, f1_score

from ..catalogs import load_candels
from ..context import Ctx, StageSkipped, load_json, save_json
from ..features import galaxy_table
from ..utils import match_with_offset, sky_crossmatch

K15 = ("spheroid", "disk", "irregular", "point_source", "unclassifiable")
COMMON = ("Spheroid", "Disk", "Irregular", "Unclassifiable")
GZH_TO_COMMON = {"Smooth": "Spheroid", "FaceOnSpiral": "Disk", "EdgeOn": "Disk", "Irregular": "Irregular",
                 "Ambiguous": "Unclassifiable"}
K15_TO_COMMON = {"spheroid": "Spheroid", "disk": "Disk", "irregular": "Irregular", "unclassifiable": "Unclassifiable",
                 "point_source": None}


def phase7_region(ctx: Ctx, df) -> str:
    """Which galaxies Phase 7 evaluates on. 'external' = separate CANDELS pointings (proposal design).
    With ONE field there are none, so 'auto' falls back to the TEST split: those galaxies were never
    used for training, tuning or calibration (sky-region split), so the evaluation is still leakage-free;
    it measures transfer to the expert label scheme on held-out sky, with the same depth and PSF."""
    want = str(ctx.cfg["partition"].get("phase7_region", "auto")).lower()
    if want in ("external", "test"):
        return want
    return "external" if (df.split_tag == "external").any() else "test"


def run(ctx: Ctx):
    from ..sealing import require_unsealed
    require_unsealed(ctx, "7a")
    out = ctx.stage_dir("7a")
    df = galaxy_table(ctx)
    from ..modes import is_apply
    region = "all" if is_apply(ctx) else phase7_region(ctx, df)
    ext = df.copy() if region == "all" else df[df.split_tag == region].copy()
    if region == "all":
        ctx.log.info("[Stage 7a] apply mode: the frozen model never saw this field - every galaxy is out-of-field")
    if ext.empty:
        raise StageSkipped(f"No galaxies in the '{region}' region for Phase 7. Add a CANDELS field to data/raw and "
                           "list a filename/TARGNAME substring in partition.external_patterns.")
    if region == "test":
        ctx.log.info("[Stage 7a] no external pointing: Phase 7 uses the held-out TEST split "
                     "(partition.phase7_region: auto). Galaxies there were never trained on.")
    metas = list(load_json(ctx.require("1a", "manifest.json")).values())
    k15 = load_candels(ctx, metas)
    if k15 is None or k15.empty:
        raise StageSkipped("No CANDELS (Kartaltepe et al. 2015) table in data/catalogs covering the field.")
    inner = df.iloc[0:0] if region == "all" else df[df.split_tag.isin([s for s in ("train", "dev", "test") if s != region])]
    _, _, leak = sky_crossmatch(ext.ra, ext.dec, inner.ra, inner.dec, 1.0)
    assert not leak.any(), f"{region} object within 1 arcsec of a training-side object!"

    idx, sep, hit, off = match_with_offset(ext.ra, ext.dec, k15.ra, k15.dec, 0.5)
    F = k15[list(K15)].to_numpy(float)
    main = np.array(K15)[np.nanargmax(np.nan_to_num(F, nan=-1), axis=1)]
    ext["k15_class"] = np.where(hit, main[np.clip(idx, 0, None)], None)
    ext["k15_common"] = ext.k15_class.map(K15_TO_COMMON)
    ext["k15_merger_frac"] = np.where(hit, k15.merger.to_numpy(float)[np.clip(idx, 0, None)], np.nan)
    ext["gzh_common"] = ext.gz_label.map(GZH_TO_COMMON)
    ext["in_gzh_overlap"] = ~ext.is_pseudo_label & ext.k15_common.notna()
    keep = ["object_id", "ra", "dec", "snr", "mag", "k15_class", "k15_common", "k15_merger_frac", "gz_label",
            "gzh_common", "in_gzh_overlap"]
    ext[keep].to_csv(out / "candels_objects.csv", index=False)

    ov = ext[ext.in_gzh_overlap & ext.gzh_common.notna()]
    if len(ov) < 5:
        raise StageSkipped(f"Only {len(ov)} galaxies are labelled by both GZH and CANDELS - no ceiling possible.")
    mat = pd.crosstab(pd.Series(ov.gzh_common.to_numpy(), name="GZH (mapped)"),
                      pd.Series(ov.k15_common.to_numpy(), name="CANDELS expert"))
    mat.to_csv(out / "taxonomy_alignment_matrix.csv")
    present = sorted(set(ov.k15_common) | set(ov.gzh_common))        # only classes that occur are scored
    ceil = {"ceiling_macro_f1": round(float(f1_score(ov.k15_common, ov.gzh_common, labels=present,
                                                     average="macro", zero_division=0)), 4),
            "classes_scored": present,
            "cohen_kappa": round(float(cohen_kappa_score(ov.k15_common, ov.gzh_common)), 4),
            "n_overlap": int(len(ov)), "n_external_galaxies": int(len(ext)), "phase7_region": region,
            "n_k15_matched": int(hit.sum()), "k15_offset_arcsec": off}
    save_json(ceil, out / "taxonomy_ceiling.json")
    ctx.log.info("[Stage 7a] alignment matrix (overlap set):\n" + mat.to_string())
    ctx.log.info(f"[Stage 7a] {ceil}")
    return ceil

In [ ]:
%%writefile src/p2p/stages/stage7b_crosscat.py
"""Stage 7b - Cross-catalogue evaluation (run once).

Purpose : how well does the frozen model generalise to an independent expert scheme?
Method  : the frozen, calibrated model (Stage 4b + 4c) predicts every external galaxy with a
          CANDELS label; galaxies in the GZH overlap (used for the ceiling) and
          expert point sources are excluded. Predictions are mapped to the
          shared taxonomy and scored with Macro-F1 (+ 95% bootstrap interval).
Output  : crosscat_predictions.csv, cross_catalogue_metrics.json
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, f1_score

from ..context import Ctx, StageSkipped, save_json
from ..features import galaxy_table
from ..inference import predict_galaxies
from ..nets import MORPH_CLASSES
from ..utils import bootstrap_ci
from .stage7a_taxonomy import COMMON, GZH_TO_COMMON


def run(ctx: Ctx):
    from ..sealing import require_unsealed
    require_unsealed(ctx, "7b")
    out = ctx.stage_dir("7b")
    p = ctx.path("7a", "candels_objects.csv")
    if not p.exists():
        raise StageSkipped("Stage 7a produced no CANDELS objects.")
    c = pd.read_csv(p)
    c["in_gzh_overlap"] = c.in_gzh_overlap.astype(str).str.lower().eq("true")
    test = c[~c.in_gzh_overlap & c.k15_common.notna()]
    if test.empty:
        raise StageSkipped("No CANDELS-labelled external galaxies outside the GZH overlap.")
    g = galaxy_table(ctx).set_index("object_id").loc[test.object_id].reset_index()
    res = predict_galaxies(ctx, g)
    pred = np.array(MORPH_CLASSES)[res["morph"].argmax(1)]
    test = test.assign(pred_class=pred, pred_common=pd.Series(pred).map(GZH_TO_COMMON).to_numpy(),
                       p_merger=res["merger"])
    test.to_csv(out / "crosscat_predictions.csv", index=False)
    labs = {c_: i for i, c_ in enumerate(COMMON)}
    yt = test.k15_common.map(labs).to_numpy()
    yp = test.pred_common.map(labs).to_numpy()
    f1 = lambda a, b: f1_score(a, b, labels=sorted(set(np.asarray(a).tolist()) | set(np.asarray(b).tolist())),
                               average="macro", zero_division=0)
    m = {"xcat_macro_f1": round(float(f1(yt, yp)), 4),
         "xcat_macro_f1_ci95": bootstrap_ci(lambda a, P: f1(a, P), yt, yp, seed=ctx.seed),
         "n_test": int(len(test)), "n_excluded_overlap": int(c.in_gzh_overlap.sum()),
         "n_excluded_point_source": int(c.k15_common.isna().sum()),
         "per_class": classification_report(test.k15_common, test.pred_common, labels=list(COMMON),
                                            output_dict=True, zero_division=0)}
    save_json(m, out / "cross_catalogue_metrics.json")
    ctx.log.info(f"[Stage 7b] cross-catalogue Macro-F1 = {m['xcat_macro_f1']} on n = {m['n_test']}\n" +
                 pd.crosstab(test.k15_common, test.pred_common, rownames=["CANDELS"], colnames=["model"]).to_string())
    return {"xcat_macro_f1": m["xcat_macro_f1"]}

In [ ]:
%%writefile src/p2p/stages/stage7c_generalisation.py
"""Stage 7c - Generalisation disentanglement.

Purpose : report true degradation, not a raw score drop.
Method  : true degradation = Taxonomy Ceiling - Cross-Catalogue Macro-F1.
          The in-domain test score is recomputed on the same coarse taxonomy so
          all three numbers share one scale. The external sample is also split
          into SNR tertiles, and the covariate shift (SNR, size, morphology)
          between the GZH training set and the external test set is listed.
Output  : generalisation_report.json, generalisation_gap.png
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

from ..context import Ctx, StageSkipped, load_json, save_json
from ..features import galaxy_table, supervised_splits
from ..nets import MORPH_CLASSES
from .stage7a_taxonomy import COMMON, GZH_TO_COMMON


def run(ctx: Ctx):
    from ..sealing import require_unsealed
    require_unsealed(ctx, "7c")
    out = ctx.stage_dir("7c")
    if not ctx.path("7b", "cross_catalogue_metrics.json").exists():
        raise StageSkipped("Stage 7b has no result.")
    ceil = load_json(ctx.require("7a", "taxonomy_ceiling.json"))["ceiling_macro_f1"]
    xcat = load_json(ctx.require("7b", "cross_catalogue_metrics.json"))["xcat_macro_f1"]
    names = np.array(MORPH_CLASSES)
    f1 = lambda a, b: float(f1_score(a, b, labels=sorted(set(a) | set(b)), average="macro", zero_division=0))
    if ctx.path("6a", "test_predictions.csv").exists():
        tp = pd.read_csv(ctx.path("6a", "test_predictions.csv"))
        indomain = round(f1(pd.Series(names[tp.y_true]).map(GZH_TO_COMMON), pd.Series(names[tp.y_pred]).map(GZH_TO_COMMON)), 4)
    else:
        indomain = None                     # no labelled in-field test galaxies (apply mode without GZH)
    xp = pd.read_csv(ctx.require("7b", "crosscat_predictions.csv"))
    tert = pd.qcut(xp.snr, 3, labels=["low SNR", "mid SNR", "high SNR"]) if len(xp) >= 9 else None
    by_snr = {str(k): round(f1(g.k15_common, g.pred_common), 4) for k, g in xp.groupby(tert, observed=True)} if tert is not None else {}
    ext = galaxy_table(ctx).set_index("object_id").loc[xp.object_id]
    SHIFT_COLS = ("snr", "mag", "r_half", "gini", "asymmetry", "concentration")
    num_median = lambda s: float(pd.to_numeric(s, errors="coerce").median())   # robust to text-typed columns
    from ..modes import card, is_apply
    if is_apply(ctx):           # the training set is the bundle's (another field): use its stored medians
        tmed = card(ctx).get("train_feature_medians", {})
    else:
        train = supervised_splits(ctx)["train"]
        tmed = {c: num_median(train[c]) for c in SHIFT_COLS if c in train.columns}
    shift = {c: {"gzh_train_median": round(float(tmed[c]), 3), "external_median": round(num_median(ext[c]), 3)}
             for c in SHIFT_COLS if c in tmed and c in ext}
    rep = {"indomain_macro_f1_coarse": indomain, "taxonomy_ceiling_macro_f1": ceil, "cross_catalogue_macro_f1": xcat,
           "true_degradation": round(ceil - xcat, 4),
           "naive_drop": round(indomain - xcat, 4) if indomain is not None else None,
           "label_scheme_component": round(indomain - ceil, 4) if indomain is not None else None,
           "xcat_macro_f1_by_snr_tertile": by_snr,
           "covariate_shift": shift,
           "phase7_region": load_json(ctx.require("7a", "taxonomy_ceiling.json")).get("phase7_region", "external")}
    if rep["phase7_region"] == "all":
        rep["note"] = ("apply mode: the frozen model was trained on another field, so every galaxy here is "
                       "out-of-field; the gap mixes the field change (depth, PSF, bands) and the label-scheme change")
    if rep["phase7_region"] == "test":
        rep["note"] = ("single field: Phase 7 used the held-out test split, so depth and PSF match training; the gap "
                       "measures the label-scheme change and the fainter CANDELS-only galaxies, not a new field")
    save_json(rep, out / "generalisation_report.json")
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5.5, 3.4))
    vals = [indomain if indomain is not None else 0.0, ceil, xcat]
    ax.bar(["In-domain\n(coarse)", "Taxonomy\nceiling", "Cross-\ncatalogue"], vals, color=["#9AA3B5", "#D3D9E5", "#2E5496"])
    for i, v in enumerate(vals):
        ax.text(i, v + 0.02, "n/a" if (i == 0 and indomain is None) else f"{v:.3f}", ha="center", fontsize=9)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Macro-F1 (shared taxonomy)")
    ax.set_title(f"Stage 7c: true degradation = {ceil - xcat:+.3f}")
    fig.tight_layout()
    fig.savefig(out / "generalisation_gap.png", dpi=110)
    plt.close(fig)
    ctx.log.info(f"[Stage 7c] {rep}")
    return {"true_degradation": rep["true_degradation"]}

In [ ]:
%%writefile src/p2p/stages/stage8_catalogue.py
"""Stage 8 - Production compilation: the unified, provenance-tracked catalog.csv.

Purpose : one deliverable that a scientist can filter with a single column.
Method  : all tracks (Star, Galaxy, Noise; GZH and external pointings) are merged
          into one row per detected source. Every galaxy gets calibrated class
          probabilities and a merger probability from the frozen fusion model (Stage 4b XGBoost + 4c calibration).
          Duplicates from overlapping pointings (closer than 0.2") are merged,
          keeping the higher-SNR detection. Each row gets an IAU-style
          designation (truncated coordinates, a letter appended when two would
          collide). `classification` (C18) is the final Star / Galaxy / Noise class
          after clustering and gating. Columns are written in the fixed order of
          COLUMNS (C1 object_id ... C75 baseline_semantic); the merger result is the
          probability p_merger only (no thresholded duplicate). Columns added
          after C75 (BW) follow EXTRA_COLUMNS: positional errors, per-band
          magnitudes, exposures, the cosmic-ray tests, gaia_match for every row
          and the CANDELS (Kartaltepe 2015) match for every galaxy. Every column has its own rule in COLUMNS
          (group, filled_for): detection measurements and the segregation evidence
          are kept for every row; magnitudes and colours for Star and Galaxy;
          morphology, labels and classification for Galaxy; Gaia and PSF
          calibration for Star; the noise model for Noise - null everywhere else.
          Before writing, the delivery checks of Objective 5 are asserted,
          including that rule and unique designations.
Output  : catalog.csv, catalog_columns.csv (data dictionary), delivery_checks.json
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from ..context import Ctx, load_json, load_objects, save_json
from ..inference import predict_galaxies
from ..nets import MORPH_CLASSES
from ..catalogs import load_candels
from ..utils import as_bool, fwhm_fit, iau_designation, match_with_offset, predictive_entropy, unit_vectors

# One row per catalogue column: (name, group, filled_for, description).
# filled_for = the classifications whose rows MAY hold a value; every other row is null (checked before writing).
# A note in brackets narrows it further, e.g. "Star (Gaia-matched)": unmatched stars are null too.
ALL, SG = "all rows", "Star, Galaxy"
COLUMNS = [
    # Order = Sangeeta's column sequence (catalog_columns_Sangeeta.xlsx, 24 Oct 2026), pred_merger removed.
    # Stage 8 asserts that catalog.csv is written in exactly this order (C1 object_id ... C75 baseline_semantic).
    ("object_id", "identity", ALL, "unique source id (Stage 1c)"),  # C1
    ("designation", "identity", ALL, "IAU-style name from truncated RA/Dec; a letter is appended when two detections would share a name"),  # C2
    ("product_type", "provenance", ALL, "drc or drz"),  # C3
    ("ra", "identity", ALL, "right ascension [deg, ICRS]"),  # C4
    ("dec", "identity", ALL, "declination [deg, ICRS]"),  # C5
    ("pointing", "identity", ALL, "pointing id (Stage 1a)"),  # C6
    ("detection_band", "identity", ALL, "filter the source was detected in"),  # C7
    ("tile_id", "identity", ALL, "detection-band _drc file"),  # C8
    ("frame_quality_flag", "provenance", ALL, "1 = clean, 0 = in a QC-flagged tile (Stage 1b)"),  # C9
    ("edge_flag", "provenance", ALL, "cutout truncated by the image edge"),  # C10
    ("anomaly_flag", "provenance", ALL, "1 = fits no population (Stage 2e)"),  # C11
    ("cluster_id_gmm", "segregation", ALL, "GMM component (Stage 2b)"),  # C12
    ("cluster_id_hdbscan", "segregation", ALL, "HDBSCAN cluster, -1 = density noise"),  # C13
    ("gmm_p_star", "segregation", ALL, "GMM posterior of the Star population"),  # C14
    ("gmm_p_galaxy", "segregation", ALL, "GMM posterior of the Galaxy population"),  # C15
    ("gmm_p_noise", "segregation", ALL, "GMM posterior of the Noise population"),  # C16
    ("split_tag", "provenance", ALL, "train / dev / test / external; buffer = inside a collar (no split)"),  # C17
    ("classification", "identity", ALL, "final population after clustering + gating: Star / Galaxy / Noise"),  # C18
    ("morph_flag", "morphology", "Galaxy", "statmorph flag: 0 good, 1 suspect, 2 bad, 4 too faint to measure"),  # C19
    ("pred_class", "classification", "Galaxy (anomaly_flag 0)", "predicted morphology class"),  # C20
    ("p_smooth", "classification", "Galaxy (anomaly_flag 0)", "calibrated probability Smooth"),  # C21
    ("p_spiral", "classification", "Galaxy (anomaly_flag 0)", "calibrated probability FaceOnSpiral"),  # C22
    ("p_edgeon", "classification", "Galaxy (anomaly_flag 0)", "calibrated probability EdgeOn"),  # C23
    ("p_irregular", "classification", "Galaxy (anomaly_flag 0)", "calibrated probability Irregular"),  # C24
    ("p_ambiguous", "classification", "Galaxy (anomaly_flag 0)", "calibrated probability Ambiguous"),  # C25
    ("pred_entropy", "classification", "Galaxy (anomaly_flag 0)", "predictive entropy of the class probabilities"),  # C26
    ("p_merger", "classification", "Galaxy (anomaly_flag 0)", "calibrated merger probability (merger XGBoost on the fused features)"),  # C27
    ("star_fwhm_arcsec", "psf_calibration", "Star", "this star's FWHM, 2-D Gaussian fit, detection band [arcsec]"),  # C28
    ("used_in_psf_match", "psf_calibration", "Star", "star used for the per-band PSFs in cross-filter matching (Stage 1d)"),  # C29
    ("used_in_epsf", "psf_calibration", "Star", "star stacked into the empirical PSF (Stage 5a)"),  # C30
    ("epsf_isolated", "psf_calibration", "Star", "no neighbour near the star"),  # C31
    ("pointing_epsf_fwhm_arcsec", "psf_calibration", "Star", "FWHM of this pointing's empirical PSF [arcsec]"),  # C32
    ("pointing_matched_fwhm_arcsec", "psf_calibration", "Star", "MEASURED FWHM of the PSF all bands were matched to (Stage 1d) [arcsec]; empty = no stars, Gaussian fallback"),  # C33
    ("is_pseudo_label", "labels", "Galaxy", "False = a GZH label exists; True = model prediction only"),  # C34
    ("gz_label", "labels", "Galaxy (GZH-matched)", "Galaxy Zoo: Hubble class"),  # C35
    ("gz_vote_frac", "labels", "Galaxy (GZH-matched)", "GZH agreement along the decision path"),  # C36
    ("gz_merger_flag", "labels", "Galaxy (GZH-matched)", "GZH merger flag"),  # C37
    ("gaia_source_id", "gaia", "Star (Gaia-matched)", "Gaia DR3 source_id (exact integer)"),  # C38
    ("gaia_sep_arcsec", "gaia", "Star (Gaia-matched)", "separation to the Gaia position [arcsec]"),  # C39
    ("mag", "photometry", SG, "AB magnitude, detection band"),  # C40
    ("colour_1", "photometry", SG, "colour {c1} [mag]"),  # C41
    ("colour_2", "photometry", SG, "colour {c2} [mag]"),  # C42
    ("gini", "morphology", "Galaxy (measured)", "Gini coefficient"),  # C43
    ("m20", "morphology", "Galaxy (measured)", "M20 moment"),  # C44
    ("concentration", "morphology", "Galaxy (measured)", "concentration C"),  # C45
    ("asymmetry", "morphology", "Galaxy (measured)", "asymmetry A"),  # C46
    ("smoothness", "morphology", "Galaxy (measured)", "smoothness / clumpiness S"),  # C47
    ("is_transitional", "segregation", "Galaxy", "density-noise object kept as a galaxy (posterior >= 0.8)"),  # C48
    ("psf_pass", "morphology", "Galaxy", "2 = morphology measured with the empirical PSF (Stage 5b)"),  # C49
    ("morph_flag_sersic", "morphology", "Galaxy", "statmorph Sersic-fit flag (0 = good fit)"),  # C50
    ("sersic_n", "morphology", "Galaxy (measured)", "Sersic index"),  # C51
    ("sersic_rhalf", "morphology", "Galaxy (measured)", "Sersic half-light radius [px]"),  # C52
    ("r_half", "morphology", "Galaxy (measured)", "non-parametric half-light radius [px]"),  # C53
    ("ellipticity", "morphology", "Galaxy (measured)", "ellipticity"),  # C54
    ("shape_asymmetry", "morphology", "Galaxy (measured)", "shape asymmetry"),  # C55
    ("multimode", "morphology", "Galaxy (measured)", "multimode statistic"),  # C56
    ("intensity", "morphology", "Galaxy (measured)", "intensity statistic"),  # C57
    ("deviation", "morphology", "Galaxy (measured)", "deviation statistic"),  # C58
    ("parallax", "gaia", "Star (Gaia-matched)", "Gaia parallax [mas]"),  # C59
    ("pmra", "gaia", "Star (Gaia-matched)", "Gaia proper motion RA [mas/yr]"),  # C60
    ("pmdec", "gaia", "Star (Gaia-matched)", "Gaia proper motion Dec [mas/yr]"),  # C61
    ("gaia_g", "gaia", "Star (Gaia-matched)", "Gaia G magnitude"),  # C62
    ("gaia_teff", "gaia", "Star (Gaia-matched)", "Gaia effective temperature [K]"),  # C63
    ("noise_score", "noise", "Noise", "probability-like score that the detection is spurious (Stage 3c)"),  # C64
    ("fpr_at_threshold", "noise", "Noise", "false-positive rate of detections at this source's SNR (Stage 3c)"),  # C65
    ("qc_flagged_bands", "provenance", ALL, "bands whose QC tile was flagged"),  # C66
    ("pipeline_version", "provenance", ALL, "code version"),  # C67
    ("snr", "detection", ALL, "signal-to-noise ratio of the detection, detection band"),  # C68
    ("ndrizim", "provenance", ALL, "NDRIZIM header: input images drizzled into the detection file (ACS/WFC: 2 per exposure, one per chip) - NOT exposures, see n_exposures"),  # C69
    ("flux", "detection", ALL, "flux of the detection [e-/s], detection band"),  # C70
    ("flux_err", "detection", ALL, "flux error [e-/s]"),  # C71
    ("fwhm_over_psf", "segregation", ALL, "source FWHM / PSF FWHM, both 2-D Gaussian fits in the detection band (1 = point-like)"),  # C72
    ("band_consistency", "segregation", ALL, "weakest other-band flux / detection-band flux (~0 = single-band, e.g. a cosmic ray); empty if no other band covers it"),  # C73
    ("reconstruction_error", "segregation", ALL, "Stage 2a autoencoder reconstruction error"),  # C74
    ("baseline_semantic", "segregation", ALL, "rule-based baseline class (Stage 2c)"),  # C75
]
# Columns after C75 (BW), in this order. "{band}" entries are repeated for every band, blue -> red.
EXTRA_COLUMNS = [
    ("ra_err", "astrometry", ALL, "statistical centroid error, RA cos(dec): FWHM / (2.355 SNR) [arcsec] (absolute WCS error not included)"),
    ("dec_err", "astrometry", ALL, "statistical centroid error, Dec: FWHM / (2.355 SNR) [arcsec]"),
    ("mag_{band}", "photometry", SG, "AB magnitude in {band} (detection footprint, cosmic-ray pixels replaced)"),
    ("magerr_{band}", "photometry", SG, "error of mag_{band} [mag]"),
    ("n_exposures", "provenance", ALL, "exposures covering the source in the detection band (drizzle CTX map); 1 = no cosmic-ray rejection possible"),
    ("cr_candidate", "segregation", ALL, "True = no other covering band confirms the source at S/N >= detection.confirm_snr (cosmic-ray candidate); empty = no other band covers it"),
    ("cr_pixels_masked", "segregation", ALL, "cosmic-ray pixels replaced and masked inside the source footprint (Stage 1c)"),
    ("gaia_match", "gaia", ALL, "True = a Gaia DR3 source within semantic_audit.gaia_radius_arcsec, whatever the classification"),
    ("classification_override", "segregation", ALL, "why classification differs from the GMM cluster name (Stage 2e physical gate / transitional rule); empty = cluster name kept"),
    ("k15_class", "cross_catalogue", "Galaxy (CANDELS-matched)", "CANDELS (Kartaltepe et al. 2015) dominant expert class"),
    ("k15_merger_frac", "cross_catalogue", "Galaxy (CANDELS-matched)", "CANDELS expert merger fraction f(M)"),
    ("k15_sep_arcsec", "cross_catalogue", "Galaxy (CANDELS-matched)", "separation to the CANDELS position after the median offset is removed [arcsec]"),
]


def catalogue_columns(bands):
    """The full column list: C1-C75 (COLUMNS), then EXTRA_COLUMNS with {band} expanded."""
    extra = []
    for n, g, f, d in EXTRA_COLUMNS:
        if "{band}" in n:
            extra += [(n.format(band=b), g, f, d.format(band=b)) for b in bands]
        else:
            extra.append((n, g, f, d))
    return COLUMNS + extra

RENAME = {"flag": "morph_flag", "flag_sersic": "morph_flag_sersic"}          # internal name -> catalogue name
ORDER = [n for n, *_ in COLUMNS]                           # the fixed 75-column sequence, C1 ... C75
assert len(ORDER) == 75 and len(set(ORDER)) == 75, "COLUMNS must hold 75 distinct columns"
MORPH_COLS = ["gini", "m20", "concentration", "asymmetry", "smoothness", "sersic_n", "sersic_rhalf", "r_half",
              "shape_asymmetry", "multimode", "intensity", "deviation"]


def allowed(filled_for: str) -> set | None:
    """Classifications a column may be filled for (None = every row)."""
    base = filled_for.split("(")[0].strip()
    return None if base == ALL else {c.strip() for c in base.split(",")}


PCOLS = ["p_smooth", "p_spiral", "p_edgeon", "p_irregular", "p_ambiguous"]


def unique_designations(names: pd.Series, ra: pd.Series) -> pd.Series:
    """Truncated IAU names collide for detections closer than ~1.5": append A, B, C ... (ordered by RA)."""
    names = pd.Series(np.asarray(names, dtype=object), index=ra.index)
    dup = names.duplicated(keep=False)
    if dup.any():
        order = ra[dup].groupby(names[dup]).rank(method="first").astype(int) - 1
        letters = [chr(65 + k % 26) + ("" if k < 26 else str(k // 26)) for k in order]
        names.loc[order.index] = names.loc[order.index] + np.array(letters, dtype=object)
    return names


def deduplicate(df, radius_arcsec):
    pairs = cKDTree(unit_vectors(df.ra, df.dec)).query_pairs(2 * np.sin(np.deg2rad(radius_arcsec / 3600) / 2))
    snr = df.snr.to_numpy()
    drop = {j if snr[i] >= snr[j] else i for i, j in pairs}
    return df.drop(index=df.index[list(drop)]), len(drop)


def run(ctx: Ctx):
    cfg = ctx.cfg["catalogue"]
    out = ctx.stage_dir("8")
    parts = ["sources", "pass1", "clusters", "baseline", "gaia", "segregation", "labels", "routing", "noise", "pass2"]
    parts = [p for p in parts if p != "routing" or ctx.path("3b", "star_routing.csv").exists()]
    df = load_objects(ctx, parts)
    df["psf_pass"] = df.psf_pass.fillna(1).astype(int)

    # ---- calibrated predictions for every galaxy row ----
    gal = (df.cluster_semantic == "Galaxy") & (df.anomaly_flag == 0)
    gidx = df.index[gal]
    if len(gidx):
        r = predict_galaxies(ctx, df.loc[gidx])          # fusion XGBoost + calibration, all galaxies at once
        P, M = r["morph"], r["merger"]
        for k, c in enumerate(PCOLS):
            df.loc[gal, c] = P[:, k]
        df.loc[gal, "pred_class"] = np.array(MORPH_CLASSES)[P.argmax(1)]
        df.loc[gal, "p_merger"] = M                  # probability only: a 0/1 pred_merger would repeat it
        df.loc[gal, "pred_entropy"] = predictive_entropy(P)

    # ---- identity + provenance ----
    pts = load_json(ctx.require("1a", "pointings.json"))["pointings"]
    man = load_json(ctx.require("1a", "manifest.json"))
    det_file = {p: Path(v["bands"][v["detection_band"]]).name for p, v in pts.items()}
    df["tile_id"] = df.pointing.map(det_file)
    df["product_type"] = df.tile_id.map(lambda f: man.get(f, {}).get("product_type"))
    df["ndrizim"] = df.tile_id.map(lambda f: man.get(f, {}).get("ndrizim"))
    from .. import __version__
    df["pipeline_version"] = f"p2p-{__version__}"
    # empty when HDBSCAN failed (Stage 2e could not apply the rule): keep it empty, do not turn it into False
    df["is_transitional"] = df.is_transitional.map(
        lambda v: None if pd.isna(v) or str(v).strip() == "" else str(v).strip().lower() in ("true", "1", "1.0"))
    df["classification"] = df["cluster_semantic"]    # final Star / Galaxy / Noise after clustering + gating

    # ---- PSF / optical calibration columns for the star track ----
    psfm = load_json(ctx.require("1d", "psf_match.json"))
    scale = df.pointing.map({p: v.get("pixscale_arcsec", np.nan) for p, v in psfm.items()}).astype(float)
    # each star's FWHM with the SAME 2-D Gaussian fit Stage 5a uses for the ePSF, in the detection band of the
    # un-matched cutout, so star_fwhm_arcsec and pointing_epsf_fwhm_arcsec can be compared directly
    from .stage1d_psf_match import crop_center
    info = load_json(ctx.require("1a", "pointings.json"))
    raw = np.load(ctx.require("1c", "cutouts_raw.npy"), mmap_mode="r")
    det_ch = {p: info["bands"].index(v["detection_band"]) for p, v in info["pointings"].items()}
    star = df.cluster_semantic == "Star"
    df["star_fwhm_arcsec"] = np.nan
    df.loc[star, "star_fwhm_arcsec"] = [
        fwhm_fit(crop_center(np.asarray(raw[int(i), det_ch[p]], np.float32), ctx.cfg["psf_match"]["stamp"]))
        for i, p in zip(df.loc[star, "cutout_index"], df.loc[star, "pointing"])]
    df["star_fwhm_arcsec"] = (df["star_fwhm_arcsec"] * scale).round(4)
    match_ids = {int(i) for v in psfm.values() for i in v.get("star_ids", [])}
    df["used_in_psf_match"] = df.object_id.isin(match_ids)
    # the MEASURED matched FWHM (Stage 1d), empty when the target band used the Gaussian fallback
    df["pointing_matched_fwhm_arcsec"] = df.pointing.map(
        {p: (v.get("matched_fwhm_arcsec") if v.get("target_psf_measured") else np.nan) for p, v in psfm.items()}
    ).astype(float).round(4)
    rep = ctx.path("5a", "epsf_report.json")
    if rep.exists():
        ep = {p: v.get("fwhm_px", np.nan) for p, v in load_json(rep).items()}
        df["pointing_epsf_fwhm_arcsec"] = (df.pointing.map(ep).astype(float) * scale).round(4)
    for c in ("used_in_epsf", "epsf_isolated"):
        if c in df:
            df[c] = as_bool(df[c])
    df["is_pseudo_label"] = np.where(df.cluster_semantic == "Galaxy", ~df.gz_label.notna(), None)
    df, n_dup = deduplicate(df, cfg["dedup_radius_arcsec"])
    df["designation"] = unique_designations(iau_designation(df.ra, df.dec, cfg["designation_prefix"]), df.ra)

    # exact Gaia DR3 ids (19 digits do not fit a float64; a merge with missing values would turn them into floats)
    gm = pd.read_csv(ctx.require("2c", "gaia_match.csv"), dtype={"gaia_source_id": "string"})
    ids = gm.set_index("object_id")["gaia_source_id"]
    df["gaia_source_id"] = df.object_id.map(ids).where(lambda v: v.notna() & (v != "-1"), None)
    df["gaia_match"] = as_bool(df.gaia_hit)                       # every row: misclassified Gaia stars stay visible
    df["cr_candidate"] = df.cr_candidate.map(
        lambda v: None if pd.isna(v) or str(v).strip() == "" else str(v).strip().lower() in ("true", "1", "1.0")) \
        if "cr_candidate" in df else None

    # CANDELS (Kartaltepe 2015) match for every galaxy (Stage 7a evaluates only the Phase-7 region)
    for c in ("k15_class", "k15_merger_frac", "k15_sep_arcsec"):
        df[c] = None
    k15 = load_candels(ctx, list(load_json(ctx.require("1a", "manifest.json")).values()))
    galaxy_rows = (df.cluster_semantic == "Galaxy").to_numpy()
    if k15 is not None and len(k15) and galaxy_rows.any():
        from .stage7a_taxonomy import K15
        g = df[galaxy_rows]
        kidx, ksep, khit, koff = match_with_offset(g.ra, g.dec, k15.ra, k15.dec, 0.5)
        F = k15[list(K15)].to_numpy(float)
        main = np.array(K15)[np.nanargmax(np.nan_to_num(F, nan=-1), axis=1)]
        j = np.clip(kidx, 0, None)
        df.loc[galaxy_rows, "k15_class"] = np.where(khit, main[j], None)
        df.loc[galaxy_rows, "k15_merger_frac"] = np.where(khit, k15.merger.to_numpy(float)[j], np.nan)
        df.loc[galaxy_rows, "k15_sep_arcsec"] = np.where(khit, np.round(ksep, 4), np.nan)
        ctx.log.info(f"[Stage 8] CANDELS: {int(khit.sum())} of {int(galaxy_rows.sum())} galaxies matched "
                     f"(offset removed dRA {koff[0]:+.3f}\", dDec {koff[1]:+.3f}\")")
    else:
        ctx.log.info("[Stage 8] no CANDELS table covering the field - k15_* columns empty")
    df = df.rename(columns=RENAME)
    for c in MORPH_COLS:                                          # statmorph -99 = could not measure -> empty
        if c in df:
            v = pd.to_numeric(df[c], errors="coerce")
            df[c] = v.where(~(v <= -98.9), np.nan)

    # a required column that an earlier stage did not produce is written empty, so the C1-C75 layout never shifts
    bands = load_json(ctx.require("1a", "pointings.json"))["bands"]
    cols = catalogue_columns(bands)
    missing = [n for n, *_ in cols if n not in df]
    if missing:
        ctx.log.info(f"[Stage 8] WARNING: no values from earlier stages for {missing}; written as empty columns")
        for n in missing:
            df[n] = None

    # ---- sparse schema, column by column ----
    for name, _, filled_for, _ in cols:
        if name not in df:
            continue
        ok = allowed(filled_for)
        df[name] = df[name].astype(object)
        if ok is not None:
            df.loc[~df.classification.isin(ok), name] = None
        if filled_for.startswith("Star (Gaia-matched)"):
            df.loc[~as_bool(df.gaia_hit), name] = None
        if "anomaly_flag 0" in filled_for:
            df.loc[df.anomaly_flag != 0, name] = None
    ordered = [n for n, *_ in cols]
    cat = df[ordered].sort_values("object_id")
    sparse_ok = all(cat.loc[~cat.classification.isin(allowed(f)), n].isna().all()
                    for n, _, f, _ in cols if allowed(f) is not None)

    # ---- delivery checks (Objective 5) ----
    g = cat[(cat.classification == "Galaxy") & (cat.anomaly_flag == 0)]
    psum = g[PCOLS].astype(float).sum(axis=1)
    checks = {"every_galaxy_has_calibrated_probs": bool(np.allclose(psum, 1.0, atol=1e-4)),
              "every_galaxy_has_merger_prob": bool(g.p_merger.notna().all()),
              "every_galaxy_has_heritage_flag": bool(g.is_pseudo_label.notna().all()),
              "every_measured_galaxy_at_pass_2": bool((g[g.snr >= ctx.cfg["morphology"]["min_snr"]].psf_pass == 2).all()),
              "class_columns_empty_for_other_classifications": bool(sparse_ok),
              "designations_unique": bool(cat.designation.is_unique),
              "column_order_as_specified": list(cat.columns) == ordered and list(cat.columns[:75]) == ORDER}
    pm = g.p_merger.astype(float)
    notes = {"columns": len(cat.columns), "first_extra_column": cat.columns[75] if len(cat.columns) > 75 else None,
             "p_merger_range": [round(float(pm.min()), 4), round(float(pm.max()), 4)] if len(pm) else None}
    if len(pm) and float(pm.max() - pm.min()) < 0.1:
        notes["merger_model"] = (f"p_merger spans only {pm.min():.3f}-{pm.max():.3f}: the merger head is effectively "
                                 "constant - report it as not trainable with this number of GZH mergers")
        ctx.log.info(f"[Stage 8] NOTE: {notes['merger_model']}")
    save_json({**checks, "duplicates_merged": n_dup, "notes": notes}, out / "delivery_checks.json")
    assert all(checks.values()), f"Delivery check failed: {checks}"
    cat.to_csv(out / "catalog.csv", index=False)
    fill = {"c1": f"{bands[0]} - {bands[1]}" if len(bands) > 1 else "(needs 2 bands)",
            "c2": f"{bands[1]} - {bands[2]}" if len(bands) > 2 else "(needs 3 bands)"}
    pd.DataFrame([{"column": n, "group": g_, "filled_for": f, "rows_filled": int(cat[n].notna().sum()),
                   "description": d.format(**fill)}
                  for n, g_, f, d in cols]).to_csv(out / "catalog_columns.csv", index=False)
    heritage = g.is_pseudo_label.astype(str).value_counts().to_dict()
    ctx.log.info(f"[Stage 8] catalog.csv: {len(cat)} rows x {cat.shape[1]} columns ({n_dup} duplicates merged) | "
                 f"{cat.classification.value_counts().to_dict()} | galaxy labels from GZH (False) / model (True): "
                 f"{heritage}")
    ctx.log.info(f"[Stage 8] delivery checks: {checks}")
    return {"rows": len(cat), "checks": checks}

In [ ]:
%%writefile src/p2p/stages/stage9_galaxy_atlas.py
"""Stage 9 - Galaxy image atlas: a colour PNG of every galaxy in the catalogue.

Purpose : let a person look at every classified galaxy, and hand the images on
          together with catalog.csv.
Method  : for each catalogue row with classification == Galaxy, the registered,
          cosmic-ray-cleaned cutout of Stage 1c (native resolution, all bands)
          is cleaned of the cosmic rays left outside the target's footprint (display
          only - see clean_cosmic_rays), background-subtracted (median of the cutout border), put on one
          flux scale with the AB zero-points of its pointing, and combined into an
          RGB image (reddest band = R, bluest = B; with two bands G is their mean,
          with one band the image is grey) with a colour-preserving asinh stretch
          (Lupton et al. 2004): the softening is set from the sky noise of the
          border and the top from the bright end of the galaxy. Images are saved
          as PNG with a 256-colour palette (compressed), in one folder per
          predicted class, and written to galaxy_images.zip together with
          galaxy_images_index.csv, which links every image to its catalogue row
          (object_id, designation, class, probabilities, GZH label).
          gallery_<class>.png shows the most confident galaxies of each class;
          galaxy_atlas_overview.png shows a few of every class. A catalogue galaxy
          whose central light is mostly removed by the cleaner is a cosmic ray that
          reached the galaxy track: it is flagged cr_suspect in the index, left out
          of the galleries and shown (uncleaned) in gallery_cr_suspects.png.
Output  : images/<class>/<designation>.png, galaxy_images_index.csv,
          galaxy_images.zip, gallery_<class>.png, galaxy_atlas_overview.png,
          gallery_cr_suspects.png, atlas_summary.json
"""
from __future__ import annotations

import re
import shutil
import zipfile

import numpy as np
import pandas as pd
from PIL import Image

from ..context import Ctx, StageSkipped, load_json, save_json

CLASSES = ["Smooth", "FaceOnSpiral", "EdgeOn", "Irregular", "Ambiguous"]
PCOLS = {"Smooth": "p_smooth", "FaceOnSpiral": "p_spiral", "EdgeOn": "p_edgeon",
         "Irregular": "p_irregular", "Ambiguous": "p_ambiguous"}


def _wavelength(band: str) -> float:
    """F475W -> 475, F110W (WFC3/IR) -> 1100: HST filter names give the pivot in nm (IR in 10 nm)."""
    m = re.search(r"(\d+)", band)
    if not m:
        return 0.0
    v = float(m.group(1))
    return v * 10 if v < 200 else v          # F105W..F160W are IR filters: 1050..1600 nm


def _border(a: np.ndarray, w: int = 8) -> np.ndarray:
    """Border pixels of a stamp that have data (uncovered pixels are NaN or exactly 0 in the cutouts)."""
    b = np.concatenate([a[:w].ravel(), a[-w:].ravel(), a[w:-w, :w].ravel(), a[w:-w, -w:].ravel()])
    b = b[np.isfinite(b) & (b != 0)]
    return b if b.size >= 20 else np.array([0.0, 1.0])


def _covered(a: np.ndarray) -> np.ndarray:
    return np.isfinite(a) & (a != 0)


def clean_cosmic_rays(cut: np.ndarray, order: list[int], cfg: dict) -> np.ndarray:
    """Display-only cosmic-ray removal (the measurements of Stages 1c-8 are not touched).
    Single exposures leave many cosmic rays OUTSIDE the target footprint (Stage 1c cleans the target's
    footprint only). A pixel of band b is replaced when it is (1) sharp in b: > nsigma sky-sigma above the
    median of its 8 neighbours and > 2x that median, and (2) a single-band event: its S/N in b is more than
    `cr_ratio` times its S/N in every other band. Real sources are smooth at the PSF scale (FWHM ~2.6 px),
    so they fail (1), and they appear in every band, so they fail (2). The mask is grown by one pixel and
    replaced with the 5x5 median of the band; two passes catch the inside of wide tracks."""
    from scipy import ndimage
    ring = np.ones((3, 3), bool); ring[1, 1] = False
    nsig, ratio = float(cfg.get("cr_nsigma", 5.0)), float(cfg.get("cr_ratio", 3.0))
    cov = np.stack([_covered(np.asarray(cut[b], np.float32)) for b in range(cut.shape[0])])
    a = np.nan_to_num(np.asarray(cut, np.float32), nan=0.0).copy()
    nb = len(order)
    for _ in range(int(cfg.get("cr_passes", 2))):
        sig = np.array([1.4826 * np.median(np.abs(_border(a[b]) - np.median(_border(a[b])))) or 1e-6 for b in range(a.shape[0])])
        snr = np.stack([(a[b] - np.median(_border(a[b]))) / sig[b] for b in range(a.shape[0])])
        for b in order:
            ringmed = ndimage.median_filter(a[b], footprint=ring, mode="nearest")
            sharp = (a[b] - ringmed > nsig * sig[b]) & (a[b] > 2 * np.maximum(ringmed, 0) )
            oth = [c for c in order if c != b]
            if oth:
                # only bands that COVER the pixel can say 'nothing is there'; where no other band covers it,
                # the pixel cannot be tested and is left alone (chip gaps, frame edges)
                o_snr = np.stack([np.where(cov[c], np.clip(snr[c], 1.0, None), np.nan) for c in oth])
                any_cov = np.isfinite(o_snr).any(axis=0)
                others = np.nanmax(np.where(np.isfinite(o_snr), o_snr, -np.inf), axis=0)
                single = any_cov & (snr[b] > ratio * others)
            else:
                single = np.ones_like(sharp)
            sharp &= cov[b]
            seed = sharp & single
            # hysteresis: grow each sharp seed through connected single-band pixels (S/N > 3), so the inside of
            # long, wide tracks (not sharp against their own neighbours) goes too; a red galaxy has no seed
            lab, n = ndimage.label(single & (snr[b] > 3.0))
            if n:
                hit_lab = np.unique(lab[seed & (lab > 0)])
                seed = seed | np.isin(lab, hit_lab[hit_lab > 0])
            m = ndimage.binary_dilation(seed, iterations=1)
            if m.any():
                med5 = ndimage.median_filter(np.where(m, np.nan, a[b]), size=5, mode="nearest")
                fill = np.where(np.isfinite(med5), med5, np.median(_border(a[b])))
                a[b] = np.where(m, fill, a[b])
    return a


def removed_fraction(before: np.ndarray, after: np.ndarray, half: int = 6) -> float:
    """Share of the significant light (> 2 sky sigma) in the central (2*half)^2 pixels, summed over bands,
    that the cleaner removed."""
    h, w = before.shape[-2:]
    tot = rem = 0.0
    for i in range(before.shape[0]):
        b0 = np.nan_to_num(np.asarray(before[i], np.float32))
        a0 = np.nan_to_num(np.asarray(after[i], np.float32))
        sky = _border(b0)
        bg = np.median(sky)
        sig = 1.4826 * np.median(np.abs(sky - bg)) or 1e-6
        c0 = b0[h // 2 - half:h // 2 + half, w // 2 - half:w // 2 + half] - bg
        c1 = a0[h // 2 - half:h // 2 + half, w // 2 - half:w // 2 + half] - bg
        sig_px = c0 > 2 * sig
        tot += c0[sig_px].sum()
        rem += np.clip(c0 - c1, 0, None)[sig_px].sum()
    return float(np.clip(rem / tot, 0, 1)) if tot > 0 else 0.0


def colour_image(cut: np.ndarray, order: list[int], scale: np.ndarray, cfg: dict) -> np.ndarray:
    """cut: (bands, H, W) in e-/s; order: channel indices from bluest to reddest; scale: AB flux factor per band.
    Returns uint8 RGB (H, W, 3)."""
    chans = []
    for b in order:
        raw_b = np.asarray(cut[b], np.float32)
        a = np.nan_to_num(raw_b, nan=0.0)
        a = np.where(_covered(raw_b), (a - np.median(_border(a))) * scale[b], 0.0)
        chans.append(a)
    if len(chans) >= 3:
        bl, gr, rd = chans[0], chans[len(chans) // 2], chans[-1]
    elif len(chans) == 2:
        bl, rd = chans
        gr = 0.5 * (bl + rd)
    else:
        bl = gr = rd = chans[0]
    inten = (bl + gr + rd) / 3.0
    sky = _border(inten)
    sigma = 1.4826 * np.median(np.abs(sky - np.median(sky))) or float(np.std(sky)) or 1e-6
    h, w = inten.shape
    core = inten[h // 4: 3 * h // 4, w // 4: 3 * w // 4]
    top = max(float(np.percentile(core, cfg.get("top_percentile", 99.7))), cfg.get("min_top_sigma", 15) * sigma)
    soft = cfg.get("soft_sigma", 2.0) * sigma
    f = np.arcsinh(np.clip(inten, 0, None) / soft) / np.arcsinh(top / soft)
    with np.errstate(divide="ignore", invalid="ignore"):
        k = np.where(inten > 0, f / inten, 0.0)
    rgb = np.stack([rd * k, gr * k, bl * k], axis=-1)
    mx = rgb.max(axis=-1, keepdims=True)                    # keep the hue when a channel saturates
    rgb = np.where(mx > 1, rgb / np.maximum(mx, 1e-9), rgb)
    return (np.clip(rgb, 0, 1) * 255 + 0.5).astype(np.uint8)


def save_png(rgb: np.ndarray, path, cfg: dict):
    im = Image.fromarray(rgb, "RGB")
    up = int(cfg.get("upscale", 1))
    if up > 1:
        im = im.resize((im.width * up, im.height * up), Image.NEAREST)
    colours = int(cfg.get("palette_colours", 256))
    if 2 <= colours <= 256:
        im = im.quantize(colors=colours, method=Image.Quantize.MEDIANCUT, dither=Image.Dither.NONE)
    im.save(path, format="PNG", optimize=True)


def safe_name(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9+\-.]+", "_", str(s)).strip("_")


def gallery(rows: pd.DataFrame, images: dict, title: str, path, ncol: int):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    n = len(rows)
    nrow = max(1, int(np.ceil(n / ncol)))
    fig, axs = plt.subplots(nrow, ncol, figsize=(1.9 * ncol, 2.15 * nrow + 0.5), squeeze=False)
    for ax in axs.flat:
        ax.axis("off")
    for ax, (_, r) in zip(axs.flat, rows.iterrows()):
        ax.imshow(images[r.object_id], interpolation="nearest")
        ax.set_title(f"{str(r.designation).replace('P2P ', '')}\n{r.pred_class} p={r.p_max:.2f}", fontsize=6.5)
    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    fig.savefig(path, dpi=110, facecolor="white")
    plt.close(fig)


def run(ctx: Ctx):
    cfg = ctx.cfg.get("atlas", {})
    if not cfg.get("enabled", True):
        raise StageSkipped("atlas.enabled is false in config.yaml")
    out = ctx.stage_dir("9")
    cat = pd.read_csv(ctx.require("8", "catalog.csv"), low_memory=False)
    gal = cat[cat.classification == "Galaxy"].copy()
    if cfg.get("min_snr"):
        gal = gal[gal.snr >= float(cfg["min_snr"])]
    if gal.empty:
        raise StageSkipped("no galaxies in catalog.csv")
    pc = [PCOLS[c] for c in CLASSES if PCOLS[c] in gal]
    gal["p_max"] = gal[pc].max(axis=1)
    gal["pred_class"] = gal.pred_class.fillna("Unclassified")
    if cfg.get("max_galaxies"):
        gal = gal.sort_values("p_max", ascending=False).head(int(cfg["max_galaxies"]))

    src = pd.read_csv(ctx.require("1c", "sources.csv"), usecols=["object_id", "cutout_index", "pointing"])
    gal = gal.merge(src.rename(columns={"pointing": "_pt"}), on="object_id", how="left")
    raw = np.load(ctx.require("1c", "cutouts_raw.npy"), mmap_mode="r")
    info = load_json(ctx.require("1a", "pointings.json"))
    bands = info["bands"]
    order = sorted(range(len(bands)), key=lambda i: _wavelength(bands[i]))

    img_dir = out / "images"
    if img_dir.exists():
        shutil.rmtree(img_dir)
    images, paths, cr_frac = {}, [], []
    for _, r in gal.iterrows():
        pt = info["pointings"].get(r._pt, {})
        zp = pt.get("zp_ab", {})
        ref = zp.get(pt.get("detection_band"), 25.0) if zp else 25.0
        scale = np.array([10 ** (-0.4 * (zp.get(b, ref) - ref)) if b in zp else 1.0 for b in bands], np.float32)
        have = [i for i in order if b_ok(bands[i], pt)]
        stamp = np.asarray(raw[int(r.cutout_index)], np.float32)
        removed = 0.0
        if cfg.get("clean_cosmic_rays", True):
            clean = clean_cosmic_rays(stamp, have or order, cfg)
            removed = removed_fraction(stamp, clean)
            stamp = clean
        cr_frac.append(removed)
        rgb = colour_image(stamp, have or order, scale, cfg)
        cls_dir = img_dir / safe_name(r.pred_class)
        cls_dir.mkdir(parents=True, exist_ok=True)
        p = cls_dir / f"{safe_name(r.designation)}.png"
        save_png(rgb, p, cfg)
        images[r.object_id] = rgb
        paths.append(str(p.relative_to(out)))
    gal["image"] = paths
    gal["cr_removed_frac"] = np.round(cr_frac, 3)
    # most of the target's own central light was a single-band, sharp event: a cosmic ray, not a galaxy
    gal["cr_suspect"] = gal.cr_removed_frac > float(cfg.get("cr_suspect_fraction", 0.5))

    keep = ["object_id", "designation", "ra", "dec", "pointing", "pred_class", "p_max"] + pc + \
           [c for c in ("pred_entropy", "p_merger", "gz_label", "is_pseudo_label", "mag", "snr", "split_tag") if c in gal] + \
           ["cr_removed_frac", "cr_suspect", "image"]
    idx = gal[keep].sort_values(["pred_class", "p_max"], ascending=[True, False])
    idx.to_csv(out / "galaxy_images_index.csv", index=False)

    zpath = out / "galaxy_images.zip"
    with zipfile.ZipFile(zpath, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as z:
        z.write(out / "galaxy_images_index.csv", "galaxy_images_index.csv")
        for p in paths:
            z.write(out / p, p)
    if not cfg.get("keep_png_folder", True):
        shutil.rmtree(img_dir)

    n_gal = int(cfg.get("gallery_n", 36))
    ncol = int(cfg.get("gallery_columns", 6))
    per_class = {}
    good = idx[~idx.cr_suspect]
    for c in CLASSES + ["Unclassified"]:
        sub = good[good.pred_class == c].head(n_gal)
        per_class[c] = int((idx.pred_class == c).sum())
        if len(sub):
            gallery(sub, images, f"Stage 9: {c} — the {len(sub)} most confident of {per_class[c]} galaxies",
                    out / f"gallery_{c}.png", ncol)
    per_row = int(cfg.get("overview_per_class", 6))
    ov = pd.concat([good[good.pred_class == c].head(per_row) for c in CLASSES if (good.pred_class == c).any()])
    if len(ov):
        gallery(ov, images, "Stage 9: galaxy atlas overview (most confident per class)",
                out / "galaxy_atlas_overview.png", per_row)
    sus = idx[idx.cr_suspect].sort_values("cr_removed_frac", ascending=False).head(n_gal)
    if len(sus):
        raw_imgs = {}
        for _, r in sus.iterrows():
            k = int(gal.loc[gal.object_id == r.object_id, "cutout_index"].iloc[0])
            pt = info["pointings"].get(gal.loc[gal.object_id == r.object_id, "_pt"].iloc[0], {})
            zp = pt.get("zp_ab", {}); ref = zp.get(pt.get("detection_band"), 25.0) if zp else 25.0
            scale = np.array([10 ** (-0.4 * (zp.get(b, ref) - ref)) if b in zp else 1.0 for b in bands], np.float32)
            raw_imgs[r.object_id] = colour_image(np.asarray(raw[k], np.float32), order, scale, cfg)
        gallery(sus, raw_imgs, f"Stage 9: catalogue 'galaxies' that look like cosmic rays "
                f"({int(idx.cr_suspect.sum())}; uncleaned view, check before use)", out / "gallery_cr_suspects.png", ncol)

    size_mb = sum((out / p).stat().st_size for p in paths) / 1e6 if cfg.get("keep_png_folder", True) else None
    summary = {"galaxies_imaged": len(paths), "per_class": per_class, "bands_rgb_order_blue_to_red": [bands[i] for i in order],
               "png_total_mb": round(size_mb, 2) if size_mb is not None else None,
               "zip_mb": round(zpath.stat().st_size / 1e6, 2), "palette_colours": int(cfg.get("palette_colours", 256)),
               "cr_suspects": int(gal.cr_suspect.sum()),
               "cr_suspect_rule": f"> {cfg.get('cr_suspect_fraction', 0.5)} of the central light removed as single-band, sharp pixels"}
    save_json(summary, out / "atlas_summary.json")
    ctx.log.info(f"[Stage 9] {len(paths)} galaxy images ({summary['png_total_mb']} MB PNG, zip {summary['zip_mb']} MB) "
                 f"| per class {per_class} | RGB = {summary['bands_rgb_order_blue_to_red'][::-1]} "
                 f"| look like cosmic rays: {summary['cr_suspects']} (galaxy_images_index.csv: cr_suspect)")
    return {"galaxies_imaged": len(paths), "zip_mb": summary["zip_mb"]}


def b_ok(band: str, pointing: dict) -> bool:
    """A band the pointing actually has (missing bands are left out of the colour image)."""
    return band not in pointing.get("missing_bands", [])

### 2.5 Tools — `tools/`
- `check_catalog_columns.py`
- `download_catalogs.py`
- `download_mast.py`
- `inspect_catalog.py`
- `inspect_fits.py`
- `show_stage.py`

In [ ]:
%%writefile tools/check_catalog_columns.py
#!/usr/bin/env python
"""Check that the catalogue files in data/catalogs have the columns that
config.yaml maps (catalogs.<kind>.columns). Run this BEFORE the pipeline when
you add a new GZH / Gaia / CANDELS file.

    python tools/check_catalog_columns.py [--config config.yaml]
"""
from __future__ import annotations

import argparse
import difflib
import sys
from pathlib import Path

import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from p2p.catalogs import K15_CLASS_COLUMNS, _candidates, _text_format, find_catalog  # noqa: E402
from p2p.context import Ctx  # noqa: E402


def header(path: Path):
    n = path.name.lower()
    if n.endswith((".fits", ".fit", ".fits.gz")):
        from astropy.table import Table
        return Table.read(path).colnames
    sep, comment = _text_format(path)
    return [c.strip() for c in pd.read_csv(path, nrows=0, sep=sep, comment=comment).columns]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="config.yaml")
    ctx = Ctx(ROOT, ap.parse_args().config)
    for kind in ("gzh", "gaia", "candels"):
        p = find_catalog(ctx, kind)
        print(f"\n=== {kind}: {ctx.rel(p) if p else 'NOT FOUND in ' + str(ctx.catalogs)} ===")
        if p is None:
            continue
        cols = header(p)
        low = {c.strip().lower(): c for c in cols}
        required = {"ra", "dec"} | (set(K15_CLASS_COLUMNS) if kind == "candels" else set())
        for std, want in ctx.cfg["catalogs"][kind]["columns"].items():
            found = next((low[c.lower()] for c in _candidates(want) if c.lower() in low), None)
            if found is not None:
                print(f"  OK       {std:15s} <- {found}")
            else:
                close = difflib.get_close_matches(_candidates(want)[0], cols, n=3, cutoff=0.4)
                tag = "MISSING!" if std in required else "missing "
                print(f"  {tag} {std:15s} <- {want}   closest: {close}"
                      + ("   (REQUIRED - the pipeline stops)" if std in required else "   (optional)"))
        print(f"  ({len(cols)} columns in the file)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile tools/download_catalogs.py
#!/usr/bin/env python
"""Download the reference catalogues into data/catalogs (needs internet).

    python tools/download_catalogs.py gzh                                   # Galaxy Zoo: Hubble Table 4
    python tools/download_catalogs.py gaia --ra 53.12 --dec -27.80          # Gaia DR3 cone around a pointing
    python tools/download_catalogs.py candels                               # Kartaltepe+2015 (VizieR J/ApJS/221/11/table3)
    python tools/download_catalogs.py all --ra 53.12 --dec -27.80

Gaia can also be left to the pipeline: set catalogs.gaia.query_online: true in config.yaml.
After downloading, run:  python tools/check_catalog_columns.py
"""
from __future__ import annotations

import argparse
import shutil
import urllib.parse
import urllib.request
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
CATS = ROOT / "data" / "catalogs"
GZH_URL = "https://zooniverse-data.s3.amazonaws.com/galaxy-zoo-hubble/gz_hubble_main.csv.gz"


def gzh():
    out = CATS / "gz_hubble_main.csv.gz"
    if out.exists():
        print(f"[gzh] already there: {out.relative_to(ROOT)}")
        return
    print(f"[gzh] downloading {GZH_URL} ...")
    tmp = out.with_suffix(".part")
    with urllib.request.urlopen(GZH_URL) as r, open(tmp, "wb") as f:
        shutil.copyfileobj(r, f, length=1 << 20)
    tmp.rename(out)
    print(f"[gzh] saved {out.relative_to(ROOT)} ({out.stat().st_size / 1e6:.0f} MB)")


def gaia(ra, dec, radius):
    from astroquery.gaia import Gaia
    q = ("SELECT source_id, ra, dec, parallax, pmra, pmdec, phot_g_mean_mag, teff_gspphot "
         "FROM gaiadr3.gaia_source "
         f"WHERE 1 = CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', {ra}, {dec}, {radius}))")
    print(f"[gaia] cone ra={ra} dec={dec} r={radius} deg ...")
    t = Gaia.launch_job(q).get_results()
    out = CATS / f"gaia_dr3_{ra:.3f}_{dec:+.3f}.csv"
    t.to_pandas().to_csv(out, index=False)
    print(f"[gaia] {len(t)} sources -> {out.relative_to(ROOT)}")
    print("       (one pointing per file is fine only if you have ONE pointing; for several pointings use "
          "catalogs.gaia.query_online: true, or one larger radius that covers all of them)")


def candels():
    """Kartaltepe et al. (2015) FRACTIONAL table = VizieR J/ApJS/221/11/table3 (GOODS-S, H < 24.5),
    saved as data/catalogs/candels_k15.csv with the VizieR column labels (f(Sph), f(Dk), ...)."""
    out = CATS / "candels_k15.csv"
    if out.exists():
        print(f"[candels] already there: {out.relative_to(ROOT)}")
        return
    url = ("https://tapvizier.cds.unistra.fr/TAPVizieR/tap/sync?REQUEST=doQuery&LANG=ADQL&FORMAT=csv&QUERY="
           + urllib.parse.quote('SELECT * FROM "J/ApJS/221/11/table3"'))
    print("[candels] downloading VizieR J/ApJS/221/11/table3 (fractional classifications) ...")
    tmp = out.with_suffix(".part")
    with urllib.request.urlopen(url) as r, open(tmp, "wb") as f:
        shutil.copyfileobj(r, f, length=1 << 20)
    tmp.rename(out)
    import pandas as pd
    t = pd.read_csv(out, nrows=5)
    n = sum(1 for _ in open(out)) - 1
    print(f"[candels] {n} rows -> {out.relative_to(ROOT)}\n  columns: {', '.join(t.columns[:20])} ...")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("what", nargs="?", default="all", choices=["gzh", "gaia", "candels", "all"],
                    help="which catalogue (default: all)")
    ap.add_argument("--ra", type=float, help="pointing centre RA [deg] (for gaia)")
    ap.add_argument("--dec", type=float, help="pointing centre Dec [deg] (for gaia)")
    ap.add_argument("--radius", type=float, default=0.06, help="cone radius [deg] (for gaia)")
    a = ap.parse_args()
    CATS.mkdir(parents=True, exist_ok=True)
    if a.what in ("gzh", "all"):
        gzh()
    if a.what == "all" and (a.ra is None or a.dec is None):
        print("[gaia] skipped: no --ra/--dec given. Either rerun with them, or set "
              "catalogs.gaia.query_online: true in config.yaml and the pipeline fetches Gaia itself.")
    elif a.what in ("gaia", "all"):
        if a.ra is None or a.dec is None:
            ap.error("gaia needs --ra and --dec (the centre of your pointing; tools/inspect_fits.py prints it)")
        gaia(a.ra, a.dec, a.radius)
    if a.what in ("candels", "all"):
        candels()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile tools/download_mast.py
#!/usr/bin/env python
"""Download the best Level-3 HST _drc products for a sky position from MAST into data/raw (needs internet).

    python tools/download_mast.py --ra 53.12 --dec -27.80 --dry-run          # show what would be downloaded
    python tools/download_mast.py --ra 53.12 --dec -27.80                    # one file per colour (auto filters)
    python tools/download_mast.py --ra 53.12 --dec -27.80 --filters F435W F606W F814W

Per filter, the file must CONTAIN the position; combined products (several exposures, cosmic rays rejected)
are preferred over single exposures, then the longest exposure. The other bands prefer files that overlap the
red (detection) band's file, so all bands form one pointing. See src/p2p/archive.py.
To search, download AND run the pipeline in one go, use tools/run_field.py or the app (streamlit run app.py).
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--ra", type=float, required=True)
    ap.add_argument("--dec", type=float, required=True)
    ap.add_argument("--radius", type=float, default=0.03, help="search radius in degrees")
    ap.add_argument("--instrument", default="ACS/WFC", help="ACS/WFC | WFC3/UVIS | WFC3/IR")
    ap.add_argument("--filters", nargs="+", default=["auto"], help="'auto' = one per colour, or a list e.g. F435W F606W F814W")
    ap.add_argument("--single-ok", action="store_true", default=True, help=argparse.SUPPRESS)
    ap.add_argument("--combined-only", action="store_true", help="never take single-exposure files")
    ap.add_argument("--out", default=str(ROOT / "data" / "raw"))
    ap.add_argument("--dry-run", action="store_true", help="list what would be downloaded")
    a = ap.parse_args()
    from p2p import archive
    filters = "auto" if a.filters == ["auto"] else [f.upper() for f in a.filters]
    ch = archive.search(a.ra, a.dec, filters, a.instrument, a.radius, allow_single=not a.combined_only)
    if not ch:
        print("nothing found that covers this position")
        return
    if not a.dry_run:
        archive.download(ch, a.out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile tools/export_model.py
#!/usr/bin/env python
"""Export the trained models of a run as a MODEL BUNDLE, so fields without Galaxy Zoo labels can be classified.

    python tools/export_model.py --name goods_s_2026-10-24                  # from outputs/
    python tools/export_model.py --name goods_s --outputs runs/2026-10-25_101500/outputs

Writes models/<name>/ with the Stage 4b/4c files (XGBoost morphology and merger models, ensemble weights,
temperature and Platt calibration) and model_card.json (bands, training field, label counts, dev/test scores). Use it with
supervised.mode: apply and supervised.model_bundle: models/<name> (tools/run_field.py and the app do this).
"""
from __future__ import annotations

import argparse
import json
import shutil
import sys
import time
from pathlib import Path

import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--name", required=True)
    ap.add_argument("--outputs", default="outputs", help="the run's outputs folder")
    ap.add_argument("--dest", default="models")
    ap.add_argument("--config", default="config.yaml")
    a = ap.parse_args()
    from p2p.context import Ctx, load_json
    from p2p.modes import BUNDLE_DIRS, BUNDLE_FILES
    out = Path(a.outputs) if Path(a.outputs).is_absolute() else ROOT / a.outputs
    run_log = load_json(out / "run_log.json") if (out / "run_log.json").exists() else {}
    for k in ("4b", "4c"):
        st = run_log.get(k, {}).get("summary", {})
        if isinstance(st, dict) and st.get("mode") == "apply":
            sys.exit(f"{out} is itself an apply-mode run (stage {k} copied a bundle) - export from a TRAINING run.")
    dest = (ROOT / a.dest / a.name)
    if dest.exists():
        sys.exit(f"{dest} exists - choose another --name")
    for k, d in BUNDLE_DIRS.items():
        (dest / d).mkdir(parents=True, exist_ok=True)
        for f in BUNDLE_FILES[k]:
            src = out / d / f
            if not src.exists() and f not in ("dev_metrics.json", "xgb_merger.joblib"):
                shutil.rmtree(dest)
                sys.exit(f"missing {src} - run Stages 4a-4c first")
            if src.exists():
                shutil.copy2(src, dest / d / f)
    pts = load_json(out / "stage1a_acquisition" / "pointings.json")
    lab = load_json(out / "stage3a_gzh_labels" / "label_summary.json")
    train_counts = {c: int(v.get("train", 0)) for c, v in lab.get("per_split_class", {}).items()}
    # training-set feature medians, for the covariate-shift table of Stage 7c in apply mode
    ctx = Ctx(ROOT, a.config)
    ctx.out = out
    from p2p.features import supervised_splits
    tr = supervised_splits(ctx)["train"]
    med = {c: float(pd.to_numeric(tr[c], errors="coerce").median())
           for c in ("snr", "mag", "r_half", "gini", "asymmetry", "concentration") if c in tr}
    test = load_json(out / "stage6a_test_eval" / "test_metrics.json") if (out / "stage6a_test_eval" / "test_metrics.json").exists() else {}
    card = {"name": a.name, "created": time.strftime("%Y-%m-%d %H:%M:%S"),
            "trained_on": {p: {"centre": v.get("centre"), "files": {b: Path(f).name for b, f in v.get("bands", {}).items()}}
                           for p, v in pts["pointings"].items()},
            "bands": pts["bands"], "train_class_counts": train_counts, "n_train_labels": int(sum(train_counts.values())),
            "train_feature_medians": med,
            "dev_metrics": {k: v for k, v in load_json(out / "stage4b_xgboost_fusion" / "dev_metrics.json").items()
                            if k in ("accuracy", "macro_f1", "macro_f1_95ci", "balanced_acc", "nll", "n")},
            "calibration": load_json(out / "stage4c_calibration" / "temperature.json"),
            "test_metrics": {k: test.get(k) for k in ("macro_f1", "macro_f1_ci95", "balanced_acc", "ece", "n")},
            "source_outputs": out.name if not str(out).startswith(str(ROOT)) else str(out.relative_to(ROOT))}
    (dest / "model_card.json").write_text(json.dumps(card, indent=1))
    size = sum(f.stat().st_size for f in dest.rglob("*") if f.is_file()) / 1e6
    print(f"model bundle -> {dest.relative_to(ROOT)} ({size:.0f} MB); bands {card['bands']}, "
          f"{card['n_train_labels']} training labels")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile tools/get_fits.py
#!/usr/bin/env python
"""Download HST _drc/_drz images (one per filter) for one or more sky positions - DOWNLOAD ONLY, no pipeline run.

Each target gets its own folder data/fields/<name>/, so fields never mix with data/raw.

    # look first: prints what was found per filter and which file would be taken (downloads nothing)
    python tools/get_fits.py --target 189.23,62.24,GOODS-N --list

    # 3 fields x 3 filters = 9 files
    python tools/get_fits.py --target 189.23,62.24,GOODS-N --target 53.16,-27.78,GOODS-S-2 --target 214.83,52.83,EGS

    # other instrument / own filters
    python tools/get_fits.py --target 189.23,62.24,GOODS-N --instrument WFC3/IR
    python tools/get_fits.py --target 189.23,62.24,GOODS-N --filters F435W F606W F850LP

Selection per filter (only the filter name is matched, so ACS entries such as "F814W;CLEAR2L" count):
  the image must cover the position; images made from several exposures (cosmic rays removed) come first.
  The filters are chosen TOGETHER: per filter the 15 images covering most of a 3.4' box on the target are
  kept; every combination is scored by the sky area covered by ALL filters (target inside it); among
  combinations within --area-tolerance (10 %) of the largest such area, those without an image centred
  more than --max-offset (2.5') away win, then the deepest. The printout gives that all-filter area. Skycell mosaics (several GB) are skipped. Use --allow-single to accept
  single-exposure images when a filter has nothing else.
To run the pipeline on a downloaded field afterwards:  python tools/run_field.py ... (or copy the folder's
files into data/raw/ after moving the old ones away).
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

DEFAULT_FILTERS = {  # first available of each colour group is taken
    "ACS/WFC": [["F435W", "F475W", "F555W"], ["F606W", "F625W"], ["F814W", "F775W", "F850LP"]],
    "WFC3/UVIS": [["F438W", "F475W", "F390W"], ["F606W", "F555W"], ["F814W", "F775W"]],
    "WFC3/IR": [["F105W", "F110W"], ["F125W", "F140W"], ["F160W"]],
}


def wavelength(band: str) -> float:
    import re
    m = re.search(r"(\d+)", band)
    return float(m.group(1)) if m else 0.0


def parse_target(s):
    parts = [p.strip() for p in s.split(",")]
    if len(parts) < 2:
        raise argparse.ArgumentTypeError("use RA,Dec[,name] in degrees, e.g. 189.23,62.24,GOODS-N")
    from p2p.field import parse_coord
    ra, dec = parse_coord(parts[0], parts[1])
    name = parts[2] if len(parts) > 2 and parts[2] else f"ra{ra:.3f}_dec{dec:+.3f}"
    return ra, dec, name.replace(" ", "_").replace("/", "-")


def filter_set(v) -> set:
    return {x.strip().upper() for x in str(v).replace(",", ";").split(";") if x.strip()}


def candidates(obs, ra, dec, band, allow_single):
    """Observations of one filter that cover the position, best first."""
    import numpy as np
    from p2p.archive import contains, is_combined, product_kind
    df = obs[[band in filter_set(v) for v in obs["filters"]]].copy()
    n_all = len(df)
    if df.empty:
        return df, f"{band}: 0 images"
    df["kind"] = df.obs_id.map(product_kind)
    df = df[df.kind != "mosaic"]
    if df.empty:
        return df, f"{band}: {n_all} images, all of them multi-GB skycell mosaics (skipped)"
    df["combined"] = df.obs_id.map(is_combined).astype(bool)
    cosd = np.cos(np.radians(dec))
    df["offset_arcmin"] = 60 * np.hypot((df.s_ra - ra) * cosd, df.s_dec - dec)
    df["covers"] = np.array([contains(s, ra, dec) for s in df.s_region], dtype=bool)
    n_cov = int(df.covers.sum())
    # a footprint that cannot be parsed is accepted if its centre is within 1.5' (ACS/WFC is 3.4' wide)
    df = df[df.covers | (df.offset_arcmin < 1.5)]
    n_comb = int(df.combined.sum())
    if not allow_single:
        df = df[df.combined]
    df = df.sort_values(["combined", "covers", "t_exptime", "offset_arcmin"], ascending=[False, False, False, True])
    note = (f"{band}: {n_all} images, {n_cov} cover the position, {n_comb} of those combined"
            + ("" if allow_single else " (single exposures skipped)"))
    return df, note


def pick_product(Observations, row, ext):
    prods = Observations.get_product_list(str(row.obsid)).to_pandas()
    fn = prods.productFilename.astype(str)
    for e in ext:
        hit = prods[fn.str.lower().str.startswith(str(row.obs_id).lower()) & fn.str.lower().str.endswith(e)]
        if len(hit):
            return hit.iloc[0]
    return None


def coverage_masks(df, ra, dec, half_arcmin=3.0, step_arcmin=0.05):
    """Boolean coverage of each candidate footprint on a common grid around the target (local tangent plane)."""
    import numpy as np
    from matplotlib.path import Path as MPath
    from p2p.archive import _polygons
    g = np.arange(-half_arcmin, half_arcmin + 1e-9, step_arcmin)
    xx, yy = np.meshgrid(g, g)
    pts = np.column_stack([xx.ravel(), yy.ravel()])
    cosd = np.cos(np.radians(dec))
    masks = []
    for s_region in df.s_region:
        m = np.zeros(len(pts), bool)
        for poly in _polygons(s_region):
            q = poly.copy()
            q[:, 0] = ((q[:, 0] - ra + 180) % 360 - 180) * cosd * 60
            q[:, 1] = (q[:, 1] - dec) * 60
            m |= MPath(q).contains_points(pts)
        masks.append(m)
    centre = int(np.argmin((pts ** 2).sum(1)))
    return masks, centre, step_arcmin ** 2


def central_box(ra, dec, half_arcmin=3.0, step_arcmin=0.05, box_half=1.7):
    import numpy as np
    g = np.arange(-half_arcmin, half_arcmin + 1e-9, step_arcmin)
    xx, yy = np.meshgrid(g, g)
    return ((np.abs(xx) <= box_half) & (np.abs(yy) <= box_half)).ravel()


def choose_jointly(obs, ra, dec, groups, a, Observations, ext, per_filter=15):
    import itertools
    import numpy as np
    cands = []
    for group in groups:
        rows = []
        for rank, band in enumerate(group):
            df, note = candidates(obs, ra, dec, band, a.allow_single)
            print("    " + note)
            if not df.empty:
                rows.append(df.assign(band=band, band_rank=rank))
        c = pd_concat(rows)
        if c is not None and len(c) > per_filter:
            # keep the images that cover the most of a 3.4' box centred on the target (an ACS field placed on
            # the target): a shallow image aligned with the others beats a deep one 2' away
            m, _, _ = coverage_masks(c, ra, dec)
            box = central_box(ra, dec)
            c = c.assign(central=[round(float((mi & box).sum() / box.sum()), 2) for mi in m])
            c = c.sort_values(["combined", "central", "t_exptime"], ascending=[False, False, False]).head(per_filter)
        cands.append(c)
    if any(c is None or c.empty for c in cands):
        missing = [g for g, c in zip(groups, cands) if c is None or c.empty]
        for g in missing:
            print(f"    -> nothing usable for {'/'.join(g)}" + ("" if a.allow_single else "  (try --allow-single)"))
        cands = [c for c in cands if c is not None and not c.empty]
        if not cands:
            return {}
    masks = [coverage_masks(c, ra, dec) for c in cands]
    blocked = [set() for _ in cands]
    while True:
        best = None
        for combo in itertools.product(*[range(len(c)) for c in cands]):
            if any(i in blocked[k] for k, i in enumerate(combo)):
                continue
            common = np.logical_and.reduce([masks[k][0][i] for k, i in enumerate(combo)])
            if not common[masks[0][1]]:
                continue                                        # the target itself must have every band
            area = float(common.sum() * masks[0][2])
            depth = float(sum(np.log10(max(cands[k].iloc[i].t_exptime, 1)) for k, i in enumerate(combo)))
            pref = -sum(int(cands[k].iloc[i].band_rank) for k, i in enumerate(combo))
            single = sum(not bool(cands[k].iloc[i].combined) for k, i in enumerate(combo))
            # images centred far from the target are larger multi-pointing products: most of their area has one
            # band only (single-band detections = cosmic-ray candidates that fill the source cap)
            far = sum(float(cands[k].iloc[i].offset_arcmin) > a.max_offset for k, i in enumerate(combo))
            best = best or []
            best.append((combo, area, depth, pref, single, common, far))
        if not best:
            print("    -> no combination of images covers the target in every filter")
            return {}
        top = max(b[1] for b in best)
        ok = [b for b in best if b[1] >= (1 - a.area_tolerance) * top]
        combo, area, depth, _, _, common, _ = max(ok, key=lambda b: (-b[4], -b[6], b[3], b[2], b[1]))
        picked, failed = {}, False
        for k, i in enumerate(combo):
            row = cands[k].iloc[i]
            p = pick_product(Observations, row, ext)
            if p is None:
                blocked[k].add(i)
                failed = True
                break
            frac = float((masks[k][0][i] & common).sum() / max(masks[k][0][i].sum(), 1))
            row = row.copy()
            row["ov"] = frac
            picked[k] = (row.band, row, p)
        if failed:
            continue
        print(f"    area covered by ALL {len(combo)} filters: {area:.1f} arcmin^2 "
              f"({area / 11.56:.0%} of one ACS/WFC field); best possible here {top:.1f} arcmin^2")
        return picked


def pd_concat(rows):
    import pandas as pd
    return pd.concat(rows) if rows else None


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--target", action="append", type=parse_target, required=True, help="RA,Dec[,name]")
    ap.add_argument("--instrument", default="ACS/WFC", choices=list(DEFAULT_FILTERS))
    ap.add_argument("--filters", nargs="+", help="exact filters, e.g. F435W F606W F814W (default: one per colour)")
    ap.add_argument("--radius-arcmin", type=float, default=2.0, help="search radius around the position")
    ap.add_argument("--allow-single", action="store_true", help="accept single-exposure images (cosmic rays!)")
    ap.add_argument("--out", default=str(ROOT / "data" / "fields"), help="one sub-folder per target is made here")
    ap.add_argument("--list", action="store_true", help="only show what would be downloaded")
    ap.add_argument("--gaia", action="store_true", help="also save the Gaia DR3 stars around each target")
    ap.add_argument("--max-offset", type=float, default=2.5,
                    help="images centred further than this (arcmin) are avoided when an alternative exists")
    ap.add_argument("--area-tolerance", type=float, default=0.10,
                    help="combinations within this fraction of the largest all-band area compete on depth")
    a = ap.parse_args()

    import numpy as np
    from astroquery.mast import Observations
    groups = [[f.upper()] for f in a.filters] if a.filters else DEFAULT_FILTERS[a.instrument]
    ext = ["_drz.fits", "_drc.fits"] if a.instrument == "WFC3/IR" else ["_drc.fits", "_drz.fits"]
    total = 0
    for ra, dec, name in a.target:
        print(f"\n=== {name}: RA {ra:.5f}  Dec {dec:+.5f}  ({a.instrument}, radius {a.radius_arcmin}')")
        obs = Observations.query_criteria(coordinates=f"{ra} {dec}", radius=f"{a.radius_arcmin / 60} deg",
                                          obs_collection="HST", instrument_name=a.instrument,
                                          dataproduct_type="image", calib_level=3)
        obs = obs.to_pandas()
        print(f"    {len(obs)} level-3 {a.instrument} images found nearby; filters present: "
              f"{', '.join(sorted(set().union(*[filter_set(v) for v in obs['filters']]) - {'CLEAR1L', 'CLEAR2L', 'CLEAR1S', 'CLEAR2S'})) or '-'}")
        out_dir = Path(a.out) / name
        # The images of the different filters are chosen TOGETHER: every combination of the best candidates
        # (one per filter) is scored by the sky area that ALL filters cover (the target must be inside it).
        # Filters taken by different HST programmes are often offset by 1-2' (an ACS field is 3.4' wide):
        # picking each filter by exposure alone can leave much of the field with one or two bands, where a
        # source has no colours and a one-band source counts as a cosmic-ray candidate. Among combinations
        # within --area-tolerance of the largest common area, the deepest (sum of log exposure) wins.
        picked = choose_jointly(obs, ra, dec, groups, a, Observations, ext)
        for gi in sorted(picked):
            band, row, p = picked[gi]
            dest = out_dir / str(p.productFilename)
            kind = "combined" if row.combined else "SINGLE exposure"
            print(f"    -> {band}: {p.productFilename}  [{kind}, {row.t_exptime:.0f} s, "
                  f"{row.offset_arcmin:.1f}' from centre, {row.ov:.0%} of this image has all filters, "
                  f"{float(p['size']) / 1e6:.0f} MB]")
            if a.list:
                continue
            out_dir.mkdir(parents=True, exist_ok=True)
            if dest.exists():
                print("       already downloaded")
            else:
                status, msg, _ = Observations.download_file(str(p.dataURI), local_path=str(dest))
                print(f"       download: {status} {msg or ''}")
                if status != "COMPLETE":
                    continue
            total += 1
        if a.gaia and not a.list:
            from p2p.archive import gaia_cone
            gaia_cone(ra, dec, 0.06, out_dir)
    if not a.list:
        print(f"\n{total} file(s) in {a.out}/<field name>/")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile tools/inspect_catalog.py
#!/usr/bin/env python
"""Summarise the final catalogue.

    python tools/inspect_catalog.py                                   # outputs/stage8_catalogue/catalog.csv
    python tools/inspect_catalog.py path/to/catalog.csv
"""
from __future__ import annotations

import sys

import numpy as np
import pandas as pd


def main(path="outputs/stage8_catalogue/catalog.csv"):
    cat = pd.read_csv(path)
    print(f"{path}: {len(cat):,} rows x {cat.shape[1]} columns\n")
    print("=== rows per population ===")
    print(cat.classification.value_counts().to_string(), "\n")
    print("=== rows per split (buffer = inside a collar) ===")
    print(cat.split_tag.fillna("buffer").value_counts().to_string(), "\n")
    if "cr_candidate" in cat:
        print("=== cosmic-ray checks (Stage 1c) ===")
        print(f"  cosmic-ray candidates  : {int((cat.cr_candidate.astype(str) == 'True').sum())} "
              f"(no other band covers: {int(cat.cr_candidate.isna().sum())})")
        print(f"  footprints with masked cosmic-ray pixels: {int((cat.cr_pixels_masked > 0).sum())}")
    if "classification_override" in cat:
        print("=== classification differs from the cluster name (Stage 2e) ===")
        print(cat.classification_override.fillna("(cluster name kept)").value_counts().to_string(), "\n")
    gal = cat[(cat.classification == "Galaxy") & (cat.anomaly_flag == 0)]
    pc = ["p_smooth", "p_spiral", "p_edgeon", "p_irregular", "p_ambiguous"]
    print("=== galaxies ===")
    print(f"  probabilities sum to 1 : {np.allclose(gal[pc].sum(axis=1), 1, atol=1e-4)}")
    print(f"  label heritage         : {gal.is_pseudo_label.astype(str).value_counts().to_dict()}  "
          "(False = GZH label, True = model inference only)")
    print("  predicted classes      :", gal.pred_class.value_counts().to_dict())
    pm = gal.p_merger.astype(float)
    print(f"  merger probability     : median {pm.median():.3f}, range {pm.min():.3f} - {pm.max():.3f}")
    lab = gal[gal.gz_label.notna()]
    if len(lab):
        print(f"  prediction = GZH label on {(lab.gz_label == lab.pred_class).mean():.1%} of {len(lab)} labelled "
              "galaxies (includes training rows: NOT a test score - see stage6a_test_eval/test_metrics.json)")
    print("\n=== sample rows ===")
    show = ["designation", "classification", "split_tag", "gz_label", "pred_class", "p_merger", "pred_entropy",
            "mag", "snr", "gini", "m20", "sersic_n"]
    print(gal[[c for c in show if c in gal]].head(10).to_string(index=False))


if __name__ == "__main__":
    main(*sys.argv[1:])

In [ ]:
%%writefile tools/inspect_fits.py
#!/usr/bin/env python
"""Look inside an HST drizzled product: extensions, key header values, WCS,
pixel statistics, and a picture of SCI and WHT.

    python tools/inspect_fits.py data/raw/<file>_drc.fits
"""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from p2p.fitsio import load_image, read_meta, valid_mask  # noqa: E402


def main(path):
    import matplotlib.pyplot as plt
    from astropy.io import fits
    from astropy.visualization import AsinhStretch, ImageNormalize, PercentileInterval

    path = Path(path)
    with fits.open(path) as h:
        print("=== Extensions ===")
        h.info()
    m = read_meta(path)
    print("\n=== What the pipeline reads (Stage 1a) ===")
    for k in ("instrument", "filter", "exptime", "ndrizim", "date_obs", "targname", "bunit", "photflam", "photplam",
              "zp_ab", "pixscale_arcsec", "naxis1", "naxis2", "ra_centre", "dec_centre", "product_type"):
        print(f"  {k:16s} = {m[k]}")
    sci, wht, w = load_image(path)
    val = valid_mask(sci, wht)
    d = sci[val]
    med = np.median(d)
    mad = 1.4826 * np.median(np.abs(d - med))
    print("\n=== Pixels ===")
    print(f"  valid (WHT > 0)  = {val.mean():.1%} of the frame")
    print(f"  sky median       = {med:.5f}   robust sigma = {mad:.5f}  ({m['bunit']})")
    print(f"  min / max        = {d.min():.4g} / {d.max():.4g}")
    norm = ImageNormalize(np.where(val, sci, np.nan), interval=PercentileInterval(99.5), stretch=AsinhStretch(0.05))
    ny, nx = sci.shape
    c = np.s_[ny // 2 - 256:ny // 2 + 256, nx // 2 - 256:nx // 2 + 256]
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(np.where(val, sci, np.nan), origin="lower", cmap="gray", norm=norm)
    ax[0].set_title(f"SCI {m['filter']} (asinh)")
    ax[1].imshow(sci[c], origin="lower", cmap="gray", norm=norm)
    ax[1].set_title("central 512 px")
    if wht is not None:
        ax[2].imshow(wht, origin="lower", cmap="viridis")
        ax[2].set_title("WHT (0 = no data)")
    for a in ax:
        a.set_xticks([])
        a.set_yticks([])
    fig.suptitle(path.name)
    fig.tight_layout()
    out = path.parent / (path.name.split(".fits")[0] + "_preview.png")
    fig.savefig(out, dpi=90)
    print(f"\nPreview saved: {out}")
    plt.show()


if __name__ == "__main__":
    if len(sys.argv) < 2:
        sys.exit(__doc__)
    main(sys.argv[1])

In [ ]:
%%writefile tools/show_stage.py
#!/usr/bin/env python
"""Explain one stage: its purpose (from the stage file), what it wrote, and the
headline numbers from its JSON outputs.

    python tools/show_stage.py 1c
    python tools/show_stage.py 6a --config my_other_config.yaml
"""
from __future__ import annotations

import argparse
import importlib
import json
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from p2p.context import STAGES, Ctx  # noqa: E402


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("stage")
    ap.add_argument("--config", default="config.yaml")
    a = ap.parse_args()
    info = {k: (m, d, t) for k, m, d, t in STAGES}
    if a.stage not in info:
        sys.exit(f"unknown stage {a.stage}; choose from {list(info)}")
    mod, folder, title = info[a.stage]
    ctx = Ctx(ROOT, a.config)
    doc = importlib.import_module(f"p2p.stages.{mod}").__doc__
    print(f"Stage {a.stage} - {title}\nsrc/p2p/stages/{mod}.py\n")
    print(doc.strip(), "\n")
    out = ctx.out / folder
    if not out.exists():
        print(f"(no outputs yet in {ctx.rel(out)} - run: python run_pipeline.py --only {a.stage})")
        return
    print(f"Files in {ctx.rel(out)}:")
    for f in sorted(out.iterdir()):
        print(f"  {f.name:45s} {f.stat().st_size / 1e6:8.2f} MB")
    for f in sorted(out.glob("*.json")):
        try:
            d = json.loads(f.read_text())
        except Exception:
            continue
        text = json.dumps(d, indent=1, default=str)
        print(f"\n--- {f.name} ---\n" + (text if len(text) < 2500 else text[:2500] + "\n ... (truncated)"))
    log = ctx.out / "run_log.json"
    if log.exists():
        r = json.loads(log.read_text()).get(a.stage)
        if r:
            print(f"\nlast run: {r['status']} in {r['seconds']} s at {r['finished']}")


if __name__ == "__main__":
    main()

In [ ]:
!{sys.executable} run_pipeline.py --list

**Model summaries** — `model.summary()` of every network of the pipeline (QC autoencoder, representation autoencoder, and the frozen ResNet-18 and EfficientNet-B0 feature extractors), built from the config; written to `outputs/model_summaries.txt`.

In [ ]:
!{sys.executable} run_pipeline.py --summary

## 3. Data

**Images** go in `data/raw/`: HST ACS/WFC `_drc` (or `_drz`) FITS files, three filters per pointing. Several pointings at different RA/Dec are fine: Stage 1a groups files that overlap by ≥ 50 % into one pointing, and Stage 0 splits the sky of every pointing into train / dev / test with a buffer collar.

**Catalogues** go in `data/catalogs/` (the file name must contain the keyword):
- Galaxy Zoo: Hubble (`gz_hubble…`) — required, morphology labels
- Gaia DR3 (`gaia…`) — star ground truth; with several pointings it is easiest to set `catalogs.gaia.query_online: true` in `config.yaml`
- CANDELS Kartaltepe et al. 2015 (`candels…` / `k15…`) — cross-catalogue test

On Colab you can upload files to the Drive folder, or download them with the tools below (needs internet).

In [ ]:
# Option A: upload from your computer (Colab only)
if IN_COLAB:
    from google.colab import files
    # up = files.upload()          # then move FITS to data/raw and tables to data/catalogs
    pass
print(sorted(os.listdir("data/raw")), sorted(os.listdir("data/catalogs")))

In [ ]:
# Option B: download from MAST / Gaia / VizieR (needs internet).
# get_fits.py downloads ONLY (no pipeline run), one folder per position: data/fields/<name>/
# --list shows what would be taken (nothing is downloaded); add --instrument WFC3/IR or WFC3/UVIS for other cameras
# !{sys.executable} tools/get_fits.py --list --target 53.12,-27.80,GOODS-S
# !{sys.executable} tools/get_fits.py --target 53.12,-27.80,GOODS-S --gaia
# Then copy that folder's _drc.fits files into data/raw/ (move old files away first).
# !{sys.executable} tools/download_catalogs.py gzh
# !{sys.executable} tools/download_catalogs.py candels

In [ ]:
# Check the inputs before running
!{sys.executable} tools/check_catalog_columns.py
import glob
for f in sorted(glob.glob("data/raw/*.fits")):
    print(f)
# !{sys.executable} tools/inspect_fits.py data/raw/<file>_drc.fits

## 4. Run the pipeline

### 4.1 Helpers
`show_stage(key)` prints the stage's purpose and headline numbers and displays every figure it wrote.

In [ ]:
import json, glob
from IPython.display import Image, display, Markdown

def show_stage(key, max_images=20, width=900):
    """Explain one stage and show its outputs (text summary + all PNG figures)."""
    import subprocess
    r = subprocess.run([sys.executable, "tools/show_stage.py", key], capture_output=True, text=True)
    print(r.stdout + r.stderr)
    sys.path.insert(0, "src")
    from p2p.context import STAGE_DIR
    folder = os.path.join("outputs", STAGE_DIR[key])
    pngs = sorted(glob.glob(os.path.join(folder, "*.png")))
    for p in pngs[:max_images]:
        display(Markdown(f"**{os.path.basename(p)}**"))
        display(Image(filename=p, width=width))
    if len(pngs) > max_images:
        print(f"... {len(pngs) - max_images} more figures in {folder}")

`!{sys.executable} …` runs the scripts with the same Python as this notebook's kernel (a plain `!python` can pick up a different Python in local Jupyter).

### 4.2 Run everything at once
Uncomment to run all 28 stages in order (the order is fixed in `src/p2p/context.py`: Stage 5 runs before Stage 4; Stages 6a-7c are skipped while the test is sealed). `--from <stage>` resumes after an interruption.

In [ ]:
# !{sys.executable} run_pipeline.py
# !{sys.executable} run_pipeline.py --from 2a
# !{sys.executable} run_pipeline.py --from 1a --to 1e

### 4.3 Or run stage by stage
Each stage reads the saved outputs of the earlier ones, so you can stop and continue at any point.

#### Phase 0 — Data governance

**Stage 0 — Sky-region partitioning**

fix the train / dev / test / external split BY SKY REGION before any pixel is analysed (`partition.pointing_splits`). Check the GZH table per pointing and split in the log and `split_regions.png`.

In [ ]:
!{sys.executable} run_pipeline.py --only 0
show_stage("0")

#### Phase 1 — Ingestion and instrument-aware preprocessing

**Stage 1a — Archive acquisition / local scan**

check every FITS file (SCI, WHT, WCS), read zero-points, group the files into pointings; `pointings.json` lists each pointing's name, splits, bands and detection band.

In [ ]:
!{sys.executable} run_pipeline.py --only 1a
show_stage("1a")

**Stage 1b — Frame quality control (CAE)**

flag detector artefacts with a convolutional autoencoder trained on train / dev sky tiles. Check `qc_training.png` (train vs validation loss, best epoch; the log shows `no gain k/patience` for early stopping) and the flag maps.

In [ ]:
!{sys.executable} run_pipeline.py --only 1b
show_stage("1b")

**Stage 1c — Detection, deblending, aligned cutouts**

locate every source and register all filters into one cutout.

In [ ]:
!{sys.executable} run_pipeline.py --only 1c
show_stage("1c")

**Stage 1d — Cross-filter PSF homogenisation**

make every band of a cutout share one resolution, so chromatic PSF differences are not mistaken for galaxy structure.

In [ ]:
!{sys.executable} run_pipeline.py --only 1d
show_stage("1d")

**Stage 1e — Pass-1 photometry and morphology**

a provisional table of physical parameters for every source.

In [ ]:
!{sys.executable} run_pipeline.py --only 1e
show_stage("1e")

#### Phase 2 — Unsupervised object segregation

**Stage 2a — Representation learning + fusion**

one feature space that captures both pixel structure and physics. Check `cae_training.png` (train / validation loss, best epoch) and `pca_scree.png` (red line = parallel analysis; right panel: the kept components should NOT track the band background levels).

In [ ]:
!{sys.executable} run_pipeline.py --only 2a
show_stage("2a")

**Stage 2b — Unsupervised clustering**

GMM (k by minimum BIC) and HDBSCAN on the train split. Check `gmm_k_selection.png` (elbow / BIC / ICL, held-out log-likelihood, diagonal vs full) and the `GMM check` / `HDBSCAN check` lines of the log (stability and density structure). The external check against Gaia / GZH / noise evidence is printed by Stage 2d.

In [ ]:
!{sys.executable} run_pipeline.py --only 2b
show_stage("2b")

**Stage 2c — Baseline segregation + Gaia match**

a simple, transparent Star / Galaxy / Noise separator - the yardstick the learned segregation (Stages 2a-2d) must beat (Objective 2).

In [ ]:
!{sys.executable} run_pipeline.py --only 2c
show_stage("2c")

**Stage 2d — Semantic audit**

turn abstract GMM cluster IDs into Star / Galaxy / Noise, and measure how good the learned segregation is against Gaia.

In [ ]:
!{sys.executable} run_pipeline.py --only 2d
show_stage("2d")

**Stage 2e — Continuum handling + outlier gating**

do not throw away real galaxies that sit between populations, and do not silently drop true anomalies either.

In [ ]:
!{sys.executable} run_pipeline.py --only 2e
show_stage("2e")

#### Phase 3 — Label anchoring and three-way routing

**Stage 3a — GZH label anchoring**

attach crowd-sourced morphology labels to the galaxy track.

In [ ]:
!{sys.executable} run_pipeline.py --only 3a
show_stage("3a")

**Stage 3b — Star calibration routing**

point sources skip morphology classification and become PSF tracers.

In [ ]:
!{sys.executable} run_pipeline.py --only 3b
show_stage("3b")

**Stage 3c — Noise false-positive model**

characterise spurious detections - the Noise population has no external catalogue, so this model is also its validation.

In [ ]:
!{sys.executable} run_pipeline.py --only 3c
show_stage("3c")

#### Phase 5 — Optical calibration feedback (runs before Phase 4)

**Stage 5a — Empirical PSF construction**

a localised, high-quality PSF built from the pipeline's own stars.

In [ ]:
!{sys.executable} run_pipeline.py --only 5a
show_stage("5a")

**Stage 5b — Pass-2 morphology refinement**

physics-calibrated morphology for the galaxy track.

In [ ]:
!{sys.executable} run_pipeline.py --only 5b
show_stage("5b")

#### Phase 4 — Supervised multi-task learning and ensemble

**Stage 4a — CNN features (ResNet-18 + EfficientNet-B0, ImageNet, frozen)**

every galaxy cutout -> R, G, B = F775W, F606W, F435W (one asinh stretch, colours kept) -> ImageNet mean / std -> 224 x 224 -> ResNet-18 (512 features) and EfficientNet-B0 (1,280 features). No CNN is trained: `model_summary.txt` lists both networks layer by layer.

In [ ]:
!{sys.executable} run_pipeline.py --only 4a
show_stage("4a")

In [ ]:
# model.summary() of the two frozen feature extractors
print(open("outputs/stage4a_cnn_features/model_summary.txt").read())

**Stage 4b — ENSEMBLE (feature fusion) + XGBoost**

1,792 CNN features + 14 physical features = 1,806 per galaxy; ensemble weights w1 (ResNet-18), w2 (EfficientNet-B0), w3 (physical) chosen by 5-fold CV (`weight_search.png`); XGBoost (5 classes) with the number of trees chosen by the out-of-fold log-loss (`training_history_xgboost.png`); a second XGBoost for mergers.

In [ ]:
!{sys.executable} run_pipeline.py --only 4b
show_stage("4b")

In [ ]:
# Training history of the Stage 4b XGBoost (fusion) - the Keras way
import matplotlib.pyplot as plt
history = json.load(open("outputs/stage4b_xgboost_fusion/training_history_xgboost.json"))   # mean over the 5 CV folds
plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(history['loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title('XGBoost (fusion) Loss')
plt.xlabel('Boosting rounds')
plt.ylabel('Loss')
plt.legend()

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(history['accuracy'], label='Train Acc')
plt.plot(history['val_accuracy'], label='Val Acc')
plt.title('XGBoost (fusion) Accuracy')
plt.xlabel('Boosting rounds')
plt.ylabel('Accuracy')
plt.legend()

plt.show()
print(open("outputs/stage4b_xgboost_fusion/model_summary.txt").read())

**Stage 4c — Probability calibration**

temperature scaling (morphology) and Platt scaling (merger) on the out-of-fold predictions of the fusion model; accuracy, log-loss and ECE before / after.

In [ ]:
!{sys.executable} run_pipeline.py --only 4c
show_stage("4c")

**Stage 4d — Controlled ablation**

the same XGBoost on each feature source alone (ResNet-18, EfficientNet-B0, physical, CNN only), the full fusion, the fusion with 128 px input, and the previous three-model ensemble, all on the same 5 folds (`ablation.png`).

In [ ]:
!{sys.executable} run_pipeline.py --only 4d
show_stage("4d")

#### Phase 6 — In-domain evaluation and explainability

**Stage 6a — In-domain test (run once)**

the final, honest in-domain score.

> Stages 6a-7c are **sealed** while `evaluation.test_sealed: true` (they are skipped). When the model is final, set `test_sealed: false` in the config cell and run `run_pipeline.py --from 6a` **once**.

In [ ]:
!{sys.executable} run_pipeline.py --only 6a
show_stage("6a")

**Stage 6b — Uncertainty audit**

does the model hesitate where humans disagree?

In [ ]:
!{sys.executable} run_pipeline.py --only 6b
show_stage("6b")

**Stage 6c — Explainable AI (SHAP)**

exact TreeSHAP of the fusion XGBoost: the share of each class decision that rests on ResNet-18, EfficientNet-B0 and the physical features (`shap_blocks.csv`, `shap_summary.png`), and the most confident correct test galaxy per class (`examples.png`).

In [ ]:
!{sys.executable} run_pipeline.py --only 6c
show_stage("6c")

#### Phase 7 — Cross-catalogue robustness

**Stage 7a — Taxonomy alignment + ceiling**

separate label-scheme differences from model failure.

In [ ]:
!{sys.executable} run_pipeline.py --only 7a
show_stage("7a")

**Stage 7b — Cross-catalogue evaluation**

how well does the frozen model generalise to an independent expert scheme?

In [ ]:
!{sys.executable} run_pipeline.py --only 7b
show_stage("7b")

**Stage 7c — Generalisation disentanglement**

report true degradation, not a raw score drop.

In [ ]:
!{sys.executable} run_pipeline.py --only 7c
show_stage("7c")

#### Phase 8 — Unified catalogue

**Stage 8 — Unified catalogue (catalog.csv)**

one deliverable that a scientist can filter with a single column.

In [ ]:
!{sys.executable} run_pipeline.py --only 8
show_stage("8")

#### Phase 9 — Galaxy image atlas

**Stage 9 — Galaxy image atlas (PNG + zip)**

a colour image of every catalogue galaxy; galaxies whose light is mostly single-band and sharp are flagged `cr_suspect` in the index.

In [ ]:
!{sys.executable} run_pipeline.py --only 9
show_stage("9")

## 5. Results — `catalog.csv`

In [ ]:
!{sys.executable} tools/inspect_catalog.py

In [ ]:
import pandas as pd
cat = pd.read_csv("outputs/stage8_catalogue/catalog.csv")
cols = pd.read_csv("outputs/stage8_catalogue/catalog_columns.csv")
print(cat.shape)
display(cat.head(10))
display(cols)
print(json.dumps(json.load(open("outputs/stage8_catalogue/delivery_checks.json")), indent=2))

In [ ]:
# Run log: status and duration of every stage
log = json.load(open("outputs/run_log.json"))
pd.DataFrame.from_dict(log, orient="index")

## Notes
- **Test seal**: while developing, keep `evaluation.test_sealed: true` - Stages 6a-7c are skipped and every choice is made by cross-validation on train + dev. For the final, one-time evaluation set it to `false` and run `!{sys.executable} run_pipeline.py --from 6a` once.
- **Re-running on new data**: put the new FITS files in `data/raw/` (remove the old ones), set `partition.pointing_splits: []` (automatic 70/15/15 stripes) or list the new pointings, then run from Stage 0. Outputs are overwritten in `outputs/`; copy the folder first if you want to keep a run.
- **Several pointings**: Gaia per field (`tools/download_catalogs.py gaia --ra … --dec …` for each) or `catalogs.gaia.query_online: true`. Wholly external pointings are listed by file-name substring in `partition.external_patterns`.
- **Colab sessions end** after inactivity; because the project folder is on Drive, re-run Section 1 and continue with `--from <stage>`.